# PlantVision — houseplant disease pipeline

The run's version is `MODEL_VERSION` in §1; what each release changed and
measured is in `docs/MODEL_HISTORY.md`.

End-to-end deterministic training notebook that produces every artefact
`server/` reads at runtime:

```
models/
├── classifier.pth              ← frozen-DINOv2 ViT-S/14 + MLP head (full state_dict)
├── classifier.onnx             ← FP32 export (backbone+head, single graph)
├── classifier_int8.onnx        ← dynamic INT8 (production)   [absent if INT8 drift > 0.05]
├── classifier_fp32.onnx        ← FP32 ONNX (only present if INT8 drift gate failed)
├── classes.json                ← {architecture, class_names, training.img_size, inference.tta, ood, ...}
├── temperature.json            ← {temperature: T, tta}
├── conformal.json              ← RAPS calibration {q_hat, alpha, ...}
├── detector.{pt,onnx}          ← Stage 1 (re-trained in §3 from PlantDoc bboxes by default)
├── training_metrics.json
└── model_card.json
```

## Class set — 9 trainable classes (v2.x)

```
gray_mold_botrytis, healthy, leaf_scorch_underwatering,
leaf_spot, not_a_plant, nutrient_deficiency,
pests, powdery_mildew, viral_disease
```

`pests` is a single combined head for mealybugs + scale (the two most
visually-separable houseplant pests). It is a first-class trainable class
— §2 fetches its iNat supply unconditionally.

`leaf_scorch_underwatering` and `nutrient_deficiency` are sourced from
free-text iNat queries (`leaf scorch`, `chlorosis`, …). They land at a
60-row per-class floor (`THRESH_TRAIN_PER_CLASS`); logit adjustment +
label smoothing absorbs the noisier free-text labels. Flip
`INCLUDE_PHYSIOLOGICAL_CLASSES = False` in §1 to roll back to the
7-class head if those sources go dark.

The two remaining v2.0 design classes — `edema`, `sunburn_or_scorch` —
stay deferred. Their bilingual content cards live in
`server/diseases_data.py` already, so flipping them on is a §1 +
data-fetch change, not a content port.

## Why these decisions

* **Frozen DINOv2 ViT-S/14 + trained head — not distillation.** The §3
  frozen DINOv2-S linear probe already reaches ~0.91 top-1 on
  `val_realworld`, proving the features separate the 8 classes. With no
  model-size constraint and ~10× single-core CPU latency headroom, the
  highest-ceiling, lowest-risk move is to deploy the frozen ViT-S/14
  (21M params, 384-d CLS embedding) plus a trained MLP head, rather than
  distilling into a weaker, INT8-hostile MobileNetV3 student (vanilla KD
  degrades exactly when the teacher is much stronger — DIST/DKD
  territory). The EfficientNetV2-S teacher and the KD student were
  removed (§4/§5).
* **Loss-side balancing only.** Logit Adjustment (Menon ICLR 2021,
  τ=0.5); the `WeightedRandomSampler` is **off** by default to avoid
  double-correction on rare classes. A source-weighted sampler
  down-weights field-crop PlantDoc rows so the empirical training prior
  shifts toward indoor houseplants.
* **Three independent val slices, one ship gate.** `val_realworld` =
  primary disease macro-F1 gate (≥ 0.70 — must not regress below the
  live model, no class with
  recall=0, healthy-bias < 1.5). `val_indomain` drives early stopping.
  `val_studio` (PlantVillage) is a transfer/background-bias monitor —
  never blocking.
* **Two dedicated calibration splits**, both held out from the train
  set: `temperature_cal` (~5 %) for temperature scaling, `conformal_cal`
  (~10 %) for RAPS quantile fitting. Both are restricted to the
  deployment distribution (`plantdoc + inat`), never PlantVillage, so
  conformal exchangeability holds at deploy.
* **Dynamic INT8 ships only if `val_realworld` disease F1 drops by
  < 0.05** vs FP32. Transformers/ViTs quantise best with *dynamic* INT8
  (ORT official guidance) — activation ranges are computed at run time,
  no calibration set, and the weight-heavy MatMuls get quantised for
  ~2–3× CPU speedup at ~0 accuracy loss. On drift > 0.05 the build does
  NOT rename FP32 as INT8 — it writes `classifier_fp32.onnx` and leaves
  `classifier_int8.onnx` absent.
* **Trustworthy comparison.** The int8-vs-fp32 ordering is reported with
  a **paired bootstrap** CI (report Stage 0) in `model_card.json`; treat
  the ordering as real only when `significant=true`.
* **Test-time augmentation (off by default).** `USE_TTA_HFLIP` enables
  2× horizontal-flip TTA. When on, the ship-gate F1, temperature,
  conformal and MaxLogit are all fit through the same flip-averaging and
  `inference.tta='hflip'` is written so the server reproduces it.
* **Fail-loud everywhere** and **ship-gate-then-write.** All canonical
  declarative artefacts in `models/` are written AFTER the ship gate
  passes; a failed run leaves the prior build untouched.

## Training modes (gamma / combined / frontier)

One run, one mode, selected by `TRAIN_MODE` in §1. Every mode shares this
exact recipe (data, losses, calibration, export, ship gate); only the
backbone and whether §5 distills the student change:

| `TRAIN_MODE` | Backbone | Produces | `model_family` in classes.json |
|---|---|---|---|
| `gamma` | DINOv2 ViT-S/14 | server bundle only (cheapest run) | `gamma` |
| **`combined`** (default) | **DINOv2 ViT-S/14** | **gamma server bundle + aqua on-device TFLite** | `gamma` (+ `aqua`) |
| `frontier` | DINOv2 ViT-B/14 (~86M) | high-capacity server bundle | `frontier` |

`frontier` is a plain mode — nothing gates it. Judge it like any other run:
the §6 ship gate, then a paired bootstrap against the incumbent.

The notebook maximises **gamma**: the backbone is unchanged (the
frozen-probe ~0.91 ceiling is validated) and LoRA capacity is rank 16 on
the last 8 blocks (the §3 probe showed the *features*, not the head, cap
weak-class recall). Evaluation reports **per-class precision CIs** beside
the recall CIs because the weak physiological classes are precision-bound. `model_family` is written into
`classes.json` + `model_card.json` and surfaced by the server's
`/api/capabilities`.

Total training time on a single A100-40GB is ≈ 20–35 min including data
fetch (the frozen backbone trains only the head + LoRA adapters).


## §1 — Setup: deps, schema, taxonomy, eval helpers

This cell is idempotent — re-running it does not redo anything besides
imports and folder creation. It also:

* installs Colab-compatible deps and binds the exact resolved environment,
* sets every seed,
* defines the `CropAnnotation` sidecar schema,
* declares the canonical `CLASSES` list (the alphabetical order is
  what `ImageFolder` will produce at training time and what every
  downstream artefact expects).

**Colab A100 quickstart.** Pick a GPU (A100) runtime, add a Colab secret
`KAGGLE_API_TOKEN` with *Notebook access* ON (so §2 pulls the in-domain
supplement and the real-app-photo DoctorP sources), then *Run all*. Licences
are recorded per row as provenance and never gated on (study-project policy,
2026-09-21).
`TRAIN_MODE='combined'` (§1) trains the gamma teacher, distills the aqua
student and exports both bundles in one ~30–40 min run; the §6 ship gate blocks
the write unless `val_realworld` disease macro-F1 ≥ 0.70. Combined mode is
fail-closed: an Aqua/TFLite error aborts the bundle; start a fresh runtime with
`TRAIN_MODE='gamma'` if a gamma-only run is intended. Any Colab Python works:
the Aqua converter (TensorFlow 2.19 + onnx2tf) runs in its own CPython 3.12
env that the end of §1 provisions with `uv` and tests on a throwaway student,
so a toolchain fault stops the run in minutes, before any training.


In [ ]:
# §1 — Setup
import os
# Set BEFORE huggingface_hub is imported or it has no effect. The Xet
# 'reconstructing file / downloading bytes' bars are ~40% of the lines in a
# captured run log and carry no information once the download succeeds --
# the per-source '+= N' counts and the source-floor gate are what actually
# say whether the data arrived. Errors and warnings still print.
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
# Required by torch.use_deterministic_algorithms on CUDA BLAS paths. Set it
# before torch is imported so seed-replicate comparisons are reproducible.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
# Kept to RESTORE it later: Ultralytics pops it when detector.train() returns
# (see assert_determinism below).
_CUBLAS_WORKSPACE_CONFIG = os.environ['CUBLAS_WORKSPACE_CONFIG']
import os, sys, subprocess, random, json, hashlib, pathlib, re, shutil, time, zipfile, math
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, asdict
from pathlib import Path, PurePosixPath
from typing import List, Optional

# DELIBERATE COLAB RUNTIME-RESOLUTION EXCEPTION. CUDA-enabled torch and
# torchvision must remain the ABI-matched pair supplied by the active Colab
# image; forcing a generic PyPI wheel can silently remove/mismatch CUDA. The
# remaining packages use compatibility floors for the same fresh-image
# constraint. This is not claimed to be a byte-reproducible package lock:
# §6 seals EVERY resolved distribution version plus Python/CUDA/cuDNN into a
# canonical recipe_environment_sha256, and the checksummed run manifest is a
# bound bundle artifact. Any runtime drift therefore creates a different
# bundle identity instead of masquerading as the same recipe. A production
# retrain container should replace this exception with a tested wheel lock.
PACKAGE_RESOLUTION_POLICY = 'colab_cuda_abi_runtime_resolved_manifest_bound_v1'
PIP_PKGS = [
    # torch / torchvision are left as low floors on purpose: Colab ships a
    # CUDA-matched build, and a higher floor would force a slow, possibly
    # CUDA-mismatched reinstall. The rest are bumped to current stable
    # floors (Colab installs the latest satisfying them).
    'torch>=2.2', 'torchvision',
    'ultralytics>=8.3', 'onnx>=1.18', 'onnxruntime>=1.22',
    # torch>=2.10 imports onnxscript at torch.onnx.export module load.
    'onnxscript>=0.2',
    'onnxslim>=0.1.82',  # YOLO ONNX export uses it; pre-install avoids a mid-run AutoUpdate
    'imagehash', 'Pillow', 'scikit-learn>=1.6', 'pandas',
    'peft>=0.14',  # LoRA fine-tune of the DINOv2 backbone (§4)
    'tqdm', 'matplotlib',
    'requests>=2.34', 'datasets>=3.2', 'PyYAML',
    'sympy',  # onnxruntime.quantization.shape_inference (quant_pre_process) needs it
    'kagglehub',  # default production source floors require the §2 supplement
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PIP_PKGS])

# torchao compatibility shim. Recent peft hard-rejects an OLD torchao via
# is_torchao_available(), which raises ImportError inside get_peft_model()
# (§4) -- Colab preinstalls torchao 0.10.0 while peft wants > 0.16.0. Standard
# LoRA never touches torchao and we quantise via onnxruntime, not torchao, so
# drop the incompatible build and let peft's optional-integration probe return
# False cleanly. Best-effort; checked via find_spec so we don't import (and
# thus cache) torchao before removing it.
import importlib.util as _ilu
if _ilu.find_spec('torchao') is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'],
                   check=False)
    print('Removed incompatible torchao (unused; unblocks peft LoRA in §4).')

import numpy as np, requests, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from tqdm import tqdm

# ── Reproducibility ─────────────────────────────────────────────
SEED = 0
# Single source of truth for the version tag we write into
# ``classes.json`` and ``model_card.json``. Production servers pin
# behaviour to this string via ``classes.json -> model_version``.
MODEL_VERSION = '2.7.5'
# Immutable torch.hub source. A floating default branch is a model lever: the
# same notebook and data could silently load different backbone code. This is
# the official DINOv2 commit audited for this recipe (2026-06-03).
DINOV2_REPO_REF = 'facebookresearch/dinov2:7764ea0f912e53c92e82eb78a2a1631e92725fc8'


# ── SUPERSEDED 2.7.3 ARCHIVE NARRATIVE (DO NOT USE AS EVIDENCE) ──
# The 2026-08-23 pipeline audit invalidated the old causal story that used
# archive-derived counts such as 3/31 -> 4/31, 5/62 and a 'never-trained OOD'
# result to declare particular representation/sampling levers settled. The
# archive mixed denominators, inference stages and raw-vs-served gate semantics;
# supplement rows also included model-era overlap. Its private 182-image set is
# 151 healthy + 31 studio hibiscus-scorch rows, not deployment OOD, and robust
# geometry found rotated/noised pseudoreplicates that row-level metrics counted
# as independent samples. Those historical point estimates, p-values and
# 'falsified/settled' conclusions are therefore untrusted and must not drive the
# active recipe. See docs/ROADMAP.md for the corrected status.
#
# The active 2.7.5 recipe instead makes this set diagnostic-only, evaluates
# served semantics, resamples capture families, and requires model-era-clean v9
# provenance. One mathematical statement survives independently of the archive:
# argmax(z / T) == argmax(z) for every scalar T > 0, so temperature calibration
# can change confidence/set size but cannot repair argmax recall.

# ── Per-class decision bias (Z1) ────────────────────────────────
# A constant added to one class's LOGIT. Empty dict = off, which is the
# default and what every bundle to date shipped.
#
# WHY IT IS HERE AND NOT IN pipeline.py. A bias applied at serving time would
# silently invalidate the calibration: temperature and the RAPS conformal
# q-hat are fit on the logits the model produces, and shifting one afterwards
# moves every probability they were fit against. Declared here it is FOLDED
# INTO THE HEAD'S FINAL BIAS VECTOR before the ONNX export (adding a constant
# to a class logit is exactly adding it to that row of the last Linear's bias),
# so §6 fits temperature, INT8 and conformal on the biased logits and the
# server needs no change at all. classes.json records the value for provenance,
# NOT as an instruction to re-apply it.
#
# WHEN TO USE IT. The motivating measurement, on the deployed 2.6.8 over the
# 585 supplement rows it never trained on: leaf_scorch recall 0.739 -> 0.841 at
# +1.0, healthy recall UNCHANGED at 0.934, other-disease accuracy 0.810 ->
# 0.795, overall 0.807 -> 0.821. Most of that class's errors sit within ~1
# logit of the boundary.
#
# DO NOT set this from the §6 sweep of the same run — that is selection on the
# split you then quote. Pick it on temperature_cal, set it here, re-run, and
# read the result on the holdout.
CLASS_LOGIT_BIAS = {}          # e.g. {'leaf_scorch_underwatering': 1.0}
# ── Training modes (gamma / combined / frontier) ────────────────
# ONE knob picks what a run PRODUCES. Every mode shares the same recipe --
# same data, losses, calibration, ONNX export and ship gate -- and differs
# only in backbone size and whether §5 distills the on-device student.
#   gamma    — the production server model alone: frozen DINOv2 ViT-S/14 +
#              LoRA + MLP head. The cheapest run; skips §5's TensorFlow
#              toolchain entirely, which also leaves room for the
#              production-faithful detector-crop eval in §6.
#   combined — gamma AND aqua from ONE run (the DEFAULT, and what ships
#              today): trains the ViT-S/14 teacher, distills the
#              MobileNetV3-Small student in §5, and exports BOTH the gamma
#              server bundle (models/) and the aqua on-device TFLite
#              (models/aqua/).
#   frontier — the high-capacity server model: frozen DINOv2 ViT-B/14 (~86M)
#              + LoRA + MLP head, GPU inference. A PLAIN TRAINING MODE: it is
#              not gated behind a probe or a go/no-go: pick it, run it, and
#              judge the result the same way every other run is judged -- the
#              §6 ship gate plus a paired bootstrap against the incumbent.
#
# `export_family` is what lands in `classes.json -> model_family`, i.e. the
# SERVER contract (aqua | gamma | frontier surfaced by /api/capabilities). It
# is deliberately NOT the mode name: a `combined` run ships a `gamma` server
# bundle alongside the `aqua` TFLite, so the mode describes the RUN and the
# family describes the ARTEFACT.
TRAIN_MODE = 'combined'
TRAIN_MODES = {
    'gamma':    {'hub_id': 'dinov2_vits14', 'img_size': 224, 'on_device': False,
                 'distill': False, 'export': 'onnx_int8', 'export_family': 'gamma',
                 'desc': 'production server model alone (ViT-S/14)'},
    # The student's teacher backbone IS gamma (ViT-S/14 @224) so it distills
    # from the strongest small teacher; the student has its own input size
    # (STUDENT_IMG, set in §5).
    'combined': {'hub_id': 'dinov2_vits14', 'img_size': 224, 'on_device': True,
                 'distill': True, 'student_arch': 'mobilenetv3_small',
                 'export': 'tflite_int8', 'export_family': 'gamma',
                 'desc': 'gamma server bundle + distilled aqua on-device TFLite'},
    'frontier': {'hub_id': 'dinov2_vitb14', 'img_size': 224, 'on_device': False,
                 'distill': False, 'export': 'onnx_int8', 'export_family': 'frontier',
                 'desc': 'high-capacity server model (ViT-B/14, ~86M params)'},
}
if TRAIN_MODE == 'aqua':      # pre-2.7 name for the gamma+aqua combo run
    print("TRAIN_MODE='aqua' is the pre-2.7 name for the combo run — using 'combined'.")
    TRAIN_MODE = 'combined'
if TRAIN_MODE not in TRAIN_MODES:
    raise ValueError(f'TRAIN_MODE={TRAIN_MODE!r} not in {list(TRAIN_MODES)}')
FAMILY_CONFIG = TRAIN_MODES[TRAIN_MODE]
# The fold happens in §6, AFTER §5 has distilled the aqua student from the
# unbiased teacher. Validate only after TRAIN_MODE/TRAIN_MODES exist: the old
# placement crashed with NameError whenever CLASS_LOGIT_BIAS was non-empty.
if CLASS_LOGIT_BIAS and FAMILY_CONFIG.get('distill'):
    raise SystemExit(
        f"CLASS_LOGIT_BIAS={CLASS_LOGIT_BIAS} with TRAIN_MODE='{TRAIN_MODE}': the bias is "
        "folded in §6, after §5 distils the aqua student, so gamma and aqua would ship "
        "different decision boundaries. Use TRAIN_MODE='gamma', or clear the bias.")
# Server-contract family for this run's PRIMARY bundle. Downstream code that
# labels an artefact reads this; code that decides what to RUN reads TRAIN_MODE.
MODEL_FAMILY = FAMILY_CONFIG['export_family']
MODEL_FAMILIES = TRAIN_MODES          # pre-2.7 alias
print(f'train mode = {TRAIN_MODE}  ({FAMILY_CONFIG["desc"]})  '
      f'backbone={FAMILY_CONFIG["hub_id"]}  img_size={FAMILY_CONFIG["img_size"]}  '
      f'-> classes.json model_family={MODEL_FAMILY}')

# ── Run boundary / hardware preflight ───────────────────────────
# This notebook is a training recipe, not a CPU smoke test. Accidentally
# pressing Run-all on a laptop used to start multi-day CPU training. Refuse
# before any model/data work; inference belongs in the server/eval tools.
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required for training; CPU training is disabled. '
                       'Use paired_eval.py for inference-only evaluation.')
# Combined mode's Aqua converter (TF 2.19 + onnx2tf) has no CPython 3.13+
# wheel, so it does not run in this kernel: the end of §1 provisions it in its
# own CPython 3.12 env and converts a throwaway student BEFORE any training. A
# toolchain fault still stops the run here, not after the teacher.
RUN_STARTED_AT = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
RUN_ID = (f'{MODEL_VERSION}-{TRAIN_MODE}-s{SEED}-'
          f'{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}-'
          f'{os.urandom(4).hex()}')
# ``models`` is the current-run staging root throughout this self-contained
# notebook. Remove only the exact files this recipe owns; never glob or delete
# arbitrary downloads. The manifest-only bundler later ignores every sibling.
_MODEL_STAGE = Path('models'); _MODEL_STAGE.mkdir(parents=True, exist_ok=True)
_OWNED_STAGE_FILES = {
    'classifier.pth', 'classifier.onnx', 'classifier_int8.onnx',
    'classifier_fp32.onnx', 'classifier_raw.onnx', 'detector.pt',
    'detector.onnx', 'classes.json', 'temperature.json', 'conformal.json',
    'ood_embedding.npz', 'training_metrics.json', 'model_card.json',
    'dinov2_probe.json', 'run_log.jsonl', 'run_manifest.json',
    'data_manifest.json', 'bundle_manifest.json',
}
for _owned_name in sorted(_OWNED_STAGE_FILES):
    (_MODEL_STAGE / _owned_name).unlink(missing_ok=True)
_aqua_stage = _MODEL_STAGE / 'aqua'
if _aqua_stage.exists():
    shutil.rmtree(_aqua_stage)
AQUA_EXPORT_SUCCEEDED = False
print(f'fresh run_id={RUN_ID}; cleared only notebook-owned staging outputs')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Determinism ─────────────────────────────────────────────────
# One helper, called here and again at the top of §4, §5 and §6. Ultralytics'
# trainer calls ``unset_deterministic()`` when ``detector.train()`` returns:
# use_deterministic_algorithms(False), cudnn.deterministic = False, and it pops
# CUBLAS_WORKSPACE_CONFIG. Without the re-assert, §4/§5 trained
# non-deterministically after §3, which is how 2.7.5 was trained
# (`cudnn_deterministic: false` in its card).
#
# ``warn_only=True``: an op with no deterministic CUDA kernel — e.g. the
# AdaptiveAvgPool2d backward in the Aqua MobileNetV3 student — warns instead of
# aborting a Colab run hours in. Every such op is recorded in
# DETERMINISM_WARNINGS (op -> phases) and sealed into run_manifest.json, so a
# replicate comparison knows exactly which kernels it cannot repeat bitwise.
import warnings
DETERMINISM_WARNINGS: dict = {}
_DETERMINISM_PHASE = 's1_setup'
_NONDETERMINISTIC_MARKER = 'does not have a deterministic implementation'
_showwarning_before_determinism = warnings.showwarning


def _record_nondeterministic_op(message, category, filename, lineno, file=None, line=None):
    text = str(message)
    if _NONDETERMINISTIC_MARKER in text:
        op = text.split(_NONDETERMINISTIC_MARKER, 1)[0].strip()[:200]
        phases = DETERMINISM_WARNINGS.setdefault(op, [])
        if _DETERMINISM_PHASE not in phases:
            phases.append(_DETERMINISM_PHASE)
    return _showwarning_before_determinism(message, category, filename, lineno, file, line)


warnings.showwarning = _record_nondeterministic_op


def determinism_state() -> dict:
    return {
        'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
        'warn_only': torch.is_deterministic_algorithms_warn_only_enabled(),
        'cudnn_deterministic': bool(torch.backends.cudnn.deterministic),
        'cudnn_benchmark': bool(torch.backends.cudnn.benchmark),
        'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
    }


def assert_determinism(phase: str) -> dict:
    """(Re-)assert deterministic torch for ``phase`` and return the live state."""
    global _DETERMINISM_PHASE
    _DETERMINISM_PHASE = phase
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = _CUBLAS_WORKSPACE_CONFIG
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)
    # 'default' prints each call site once. Re-inserted at the FRONT of the
    # filter list on every call, so an 'ignore' a library installed since the
    # last call cannot hide a non-deterministic op from the recorder above.
    warnings.filterwarnings('default', message=f'.*{_NONDETERMINISTIC_MARKER}.*')
    state = determinism_state()
    if not (state['deterministic_algorithms'] and state['cudnn_deterministic']):
        raise RuntimeError(f'determinism re-assert failed for {phase}: {state}')
    print(f'determinism asserted for {phase}: {state}')
    if 'log_run' in globals():
        log_run({'phase': f'determinism:{phase}', **state})
    return state


assert_determinism('s1_setup')
# TF32 on Ampere+ (A100): accuracy-neutral speedup for any fp32 matmul/conv
# OUTSIDE the bf16 autocast (the training forward/backward is already bf16, so
# this only touches residual fp32 paths). It does not change the exported /
# shipped numerics (ONNX export + parity run on CPU), so it is a free win.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

device = 'cuda'
print(f'device={device}  torch={torch.__version__}')

# ── Sidecar metadata schema ─────────────────────────────────────
@dataclass
class CropAnnotation:
    plant_id: str
    species: Optional[str]
    diagnosis: str
    severity: str               # low | medium | high | unknown
    leaf_or_whole_plant: str    # leaf | whole_plant
    capture_session: str
    source: str                 # inat | coco | plantvillage | plantdoc
    label_confidence: float     # in [0, 1]
    bbox: List[int]             # [x, y, w, h]; [0,0,0,0] for whole-image
    # Deployment-domain bucket the row was judged into UPSTREAM (v2.6.12+):
    # 'houseplant' | 'garden_ornamental' | 'garden_edible' | ... — free-form on
    # purpose. The notebook RECORDS and REPORTS this, it never GATES on it: the
    # domain policy lives in the offline labeling pipeline, so broadening the
    # domain again (houseplants -> garden ornamentals -> kitchen-garden edibles
    # like cucumber/tomato) is a DATASET decision that needs no notebook change
    # and no new enum here. None for the sources that carry no per-row domain.
    subject_domain: Optional[str] = None
    # Per-image rights/provenance. Public corpora are not source-wide grants:
    # every accepted row must carry the image's own license and landing page.
    license_id: Optional[str] = None
    license_url: Optional[str] = None
    creator: Optional[str] = None
    source_url: Optional[str] = None
    source_item_id: Optional[str] = None
    attribution: Optional[str] = None
    immutable_source_id: Optional[str] = None
    capture_family_id: Optional[str] = None
    first_seen_model_version: Optional[str] = None

def write_sidecar(p: Path, ann: CropAnnotation):
    p.with_suffix('.json').write_text(json.dumps(asdict(ann), ensure_ascii=False, indent=2))
def read_sidecar(p: Path):
    s = p.with_suffix('.json'); return json.loads(s.read_text()) if s.exists() else None

# ── Folder layout ───────────────────────────────────────────────
ROOT          = pathlib.Path('data_v2')
CLS_ROOT      = ROOT / 'classification'
TRAIN_ROOT    = CLS_ROOT / 'train'
VAL_ROOT      = CLS_ROOT / 'val'
VAL_INDOMAIN  = VAL_ROOT / 'indomain'
VAL_PRIMARY   = VAL_ROOT / 'realworld'              # ship gate
VAL_STUDIO    = VAL_ROOT / 'plantvillage_studio'    # transfer monitor
TEMP_CAL      = VAL_ROOT / 'temperature_cal'        # INT8 + temperature
CONFORMAL_CAL = VAL_ROOT / 'conformal_cal'          # RAPS
HOLDOUT       = VAL_ROOT / 'holdout'                 # PERMANENT, never trained

INCLUDE_PHYSIOLOGICAL_CLASSES = True
# 'pests' is a SINGLE combined head for the two most visually-separable
# houseplant pests (mealybug cottony masses + scale waxy bumps). One
# well-supplied class trains far more reliably than two thin ones and the
# user-facing care advice is nearly identical, so they share one label. It
# is a FIRST-CLASS trainable class — no opt-in flag; §2 fetches its supply
# unconditionally. Spider mites stay excluded — their signal is a damage
# texture that overlaps early leaf scorch / nutrient stipple.
BASE_CLASSES = [
    'gray_mold_botrytis', 'healthy', 'leaf_spot',
    'not_a_plant', 'pests', 'powdery_mildew', 'viral_disease',
]
PHYSIOLOGICAL_CLASSES = ['leaf_scorch_underwatering', 'nutrient_deficiency']
CLASSES = sorted(
    BASE_CLASSES
    + (PHYSIOLOGICAL_CLASSES if INCLUDE_PHYSIOLOGICAL_CLASSES else [])
)
# `_drop_low_green_content` floor for gray_mold_botrytis — necrotic
# Botrytis foliage is mostly brown, so the gate is set low to keep
# gray-on-brown leaf shots while still rejecting agar plates / fruit.
BOTRYTIS_GREEN_MIN_RATIO = 0.04
SPLIT_ROOTS = (VAL_INDOMAIN, VAL_PRIMARY, VAL_STUDIO, TEMP_CAL, CONFORMAL_CAL,
               HOLDOUT)

def ensure_layout():
    for c in CLASSES:
        (TRAIN_ROOT / c).mkdir(parents=True, exist_ok=True)
    for d in SPLIT_ROOTS:
        d.mkdir(parents=True, exist_ok=True)
ensure_layout()

# ── Eval helpers (used by every later phase) ────────────────────
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# Eval/calibration resize ratio — SINGLE SOURCE OF TRUTH. The eval/cal
# transform is Resize(int(img_size * EVAL_RESIZE_RATIO)) -> CenterCrop(img_size)
# (a mild centre zoom). temperature.json + the conformal q_hat are fit on THIS
# distribution and the server reproduces it from classes.json -> training.
# eval_resize_ratio, so every train-side use must read this one constant
# (its value is also written into classes.json in §6). Hardcoding 1.05 in more
# than one place is exactly how train/serve crop parity silently drifts.
EVAL_RESIZE_RATIO = 1.05
# Scale to half the box's vCPUs, capped at 12 — heavy CPU augs
# (JpegRecompress, WhiteBalanceShift, DefocusOrMotionBlur) bottleneck the
# A100 if we cap too low; more workers than vCPU/2 adds context-switch
# overhead without further throughput.
EVAL_WORKERS = min(12, max(2, (os.cpu_count() or 2) // 2))
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def iter_images(root: Path):
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def count_images(root):
    return sum(1 for _ in iter_images(root)) if Path(root).exists() else 0

def _seed_worker(worker_id):
    s = (SEED + worker_id) % (2**32)
    np.random.seed(s); random.seed(s)

def _loader_kwargs(workers, **extra):
    """DataLoader kwargs. Caller `extra` overrides defaults so e.g.
    `persistent_workers=False` from a one-off cell takes effect — important
    after a cell interrupt leaves stale workers attached to a closed dataset.
    """
    kw = dict(num_workers=workers,
              pin_memory=(device == 'cuda'),
              worker_init_fn=_seed_worker)
    if workers > 0:
        kw['persistent_workers'] = True
        kw['prefetch_factor'] = 2
    kw.update(extra)
    return kw

import inspect as _inspect
_IMAGEFOLDER_ALLOW_EMPTY = (
    'allow_empty' in _inspect.signature(ImageFolder.__init__).parameters
)

def make_eval_loader(root, img_size, batch=128, workers=EVAL_WORKERS, class_to_idx=None):
    """Build a deterministic eval loader whose label space matches training.

    `ImageFolder` indexes classes by sorted directory listing — an eval
    slice missing one trained class would silently shift the label IDs.
    `class_to_idx` is the train-time map; we remap `target` so y_true is
    comparable to model `argmax` on every slice.
    """
    tx = T.Compose([T.Resize(int(img_size * EVAL_RESIZE_RATIO)),
                    T.CenterCrop(img_size), T.ToTensor(), NORM])
    if_kwargs = {'allow_empty': True} if _IMAGEFOLDER_ALLOW_EMPTY else {}
    target_transform = None
    if class_to_idx is not None:
        tmp = ImageFolder(str(root), **if_kwargs)
        unknown = [c for c in tmp.classes if c not in class_to_idx]
        if unknown:
            raise RuntimeError(
                f'Eval folder {root} has classes outside training taxonomy: {unknown}.'
            )
        slice_to_train = {tmp.class_to_idx[c]: class_to_idx[c] for c in tmp.classes}
        target_transform = slice_to_train.__getitem__
    ds = ImageFolder(str(root), transform=tx, target_transform=target_transform,
                     **if_kwargs)
    if len(ds) == 0:
        raise RuntimeError(f'Eval folder is empty: {root}.  Run §2 first.')
    dl = DataLoader(ds, batch_size=batch, shuffle=False, **_loader_kwargs(workers))
    return ds, dl

@torch.inference_mode()
def evaluate(model, dl, class_names, device, name='', logit_sink=None):
    """Disease macro-F1 + per-class recall + confusion matrix + healthy-bias.

    ``logit_sink``: an optional list that receives the per-batch logits this
    function already computes. §6's Z1 bias sweep needs exactly these numbers
    and used to re-run the whole forward pass to get them — ~2k images through
    the INT8 session with hflip TTA, i.e. two full passes over val_realworld for
    one set of logits. Handing them out costs a few MB and removes the duplicate.
    """
    model.eval()
    y_true, y_pred = [], []
    for x, y in dl:
        logits = model(x.to(device, non_blocking=True))
        if logit_sink is not None:
            logit_sink.append(logits.detach().cpu().numpy())
        y_pred.extend(logits.argmax(1).detach().cpu().tolist())
        y_true.extend(y.tolist())
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names) if n not in ('healthy', 'not_a_plant')]
    f1 = float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))
    # Disease macro-F1 over only the disease classes PRESENT in this slice
    # (support > 0). PlantVillage val_studio supplies just a few of the 9
    # classes, so the plain macro-F1 above averages in hard zeros for the
    # absent classes and badly understates real transfer (0.43 vs ~0.99).
    # This is the honest studio number; equals disease_macro_f1 on
    # fully-populated slices (realworld / indomain / test).
    supported_idx = [i for i in disease_idx if int((y_true == i).sum()) > 0]
    supported_f1 = float(f1_score(y_true, y_pred, labels=supported_idx,
                                  average='macro', zero_division=0)) if supported_idx else 0.0
    recall = recall_score(y_true, y_pred, labels=list(range(len(class_names))),
                          average=None, zero_division=0)
    # Per-class PRECISION too: the physiological classes are precision-bound
    # (they over-trigger), so recall alone hides their real failure mode. Used
    # by the §4 precision-aware checkpoint selection and shipped (with CIs) in
    # the model card. zero_division=0 => a never-predicted class scores 0.
    precision = precision_score(y_true, y_pred, labels=list(range(len(class_names))),
                               average=None, zero_division=0)
    h = class_names.index('healthy') if 'healthy' in class_names else None
    bias = float((y_pred == h).mean() / max((y_true == h).mean(), 1e-9)) if h is not None else 0.0
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))).tolist()
    # Canonical split id (strip the 'final '/'ep N ' print prefix + the test
    # suffix) so model_card.json / training_metrics.json carry a clean,
    # comparable split name per metric block.
    split = (name.replace('final ', '').replace(' (stable held-out)', '')
             .strip() or name)
    print(f'[{name}] disease_macro_f1={f1:.3f}  '
          f'supported_disease_macro_f1={supported_f1:.3f} '
          f'(n_classes={len(supported_idx)})  '
          f'healthy_bias={bias:.2f}  n={len(y_true)}')
    if 'val_realworld' in name or 'final' in name:
        for c, r in zip(class_names, recall):
            print(f'    recall[{c:25s}] = {r:.3f}')
    return {'name': name, 'split': split, 'n': int(len(y_true)),
            'disease_macro_f1': f1,
            'supported_disease_macro_f1': supported_f1,
            'n_supported_disease_classes': len(supported_idx),
            'healthy_bias': bias,
            'per_class_recall': {c: float(r) for c, r in zip(class_names, recall)},
            'per_class_precision': {c: float(p) for c, p in zip(class_names, precision)},
            'confusion_matrix': cm}

# Persisted run log — every epoch appends a row, useful for post-mortem.
RUN_LOG = Path('models/run_log.jsonl')
RUN_LOG.parent.mkdir(parents=True, exist_ok=True)
RUN_LOG.write_text('', encoding='utf-8')  # fresh run; never append stale epochs
def log_run(row: dict):
    row = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **row}
    with RUN_LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

def _kaggle_load_colab_config():
    """Bridge Kaggle + HuggingFace config values from Colab userdata into os.environ.

    Colab userdata entries are NOT auto-exported as environment variables -- they must
    be read explicitly with google.colab.userdata -- which is why saving a
    KAGGLE_API_TOKEN entry alone left the downloader seeing no config.
    This copies the value into the env so both our checks and kagglehub find
    it. No-op outside Colab, when a var is already set, or when the entry is
    unset / lacks notebook access."""
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return
    # HF_TOKEN rides along: §2 pulls PlantDoc, PlantVillage and the COCO mirror
    # from the HF Hub, where anonymous requests are rate-limited. huggingface_hub
    # reads HF_TOKEN from the environment on its own once it is exported here.
    for name in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY', 'HF_TOKEN'):
        if os.environ.get(name):
            continue
        try:
            val = userdata.get(name)
        except Exception:
            val = None  # entry missing / access not granted
        if val:
            os.environ[name] = str(val).strip()


def kaggle_install_api_config():
    """Install a Kaggle API config file (uploaded via Colab) where kagglehub
    looks. Accepts the new-format API key (a one-line value -> ~/.kaggle/
    access_token, or JSON carrying an access_token/token field) and, for
    convenience, a legacy kaggle.json {username, key}. Sets the matching env
    vars for this kernel and restricts the file to owner-only access (the
    kaggle client requires that). Returns True on success; prints guidance and
    returns False outside Colab.

    Call this in a cell BEFORE running §2 (or just save a Colab userdata entry named
    KAGGLE_API_TOKEN, which needs no upload at all)."""
    import stat
    cfg_dir = Path.home() / '.kaggle'
    cfg_dir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files  # type: ignore
    except Exception:
        print('Not running in Colab. Provide Kaggle API config another way:\n'
              '  - Colab userdata / env KAGGLE_API_TOKEN=<value>  (new format), or\n'
              '  - write the value to ~/.kaggle/access_token, or\n'
              '  - legacy ~/.kaggle/kaggle.json {"username","key"}.')
        return False
    print('Select your Kaggle API config file — the new-format API key '
          '(saved as access_token) or a legacy kaggle.json:')
    up = files.upload()
    if not up:
        print('No file uploaded.')
        return False
    name, body = next(iter(up.items()))
    text = body.decode('utf-8', 'ignore').strip()
    try:
        data = json.loads(text)
    except Exception:
        data = None
    dest = kind = None
    if isinstance(data, dict) and data.get('username') and data.get('key'):
        dest = cfg_dir / 'kaggle.json'
        dest.write_text(json.dumps({'username': data['username'], 'key': data['key']}))
        os.environ['KAGGLE_USERNAME'] = str(data['username'])
        os.environ['KAGGLE_KEY'] = str(data['key'])
        kind = 'legacy kaggle.json'
    else:
        token = None
        if isinstance(data, dict):
            token = data.get('access_token') or data.get('token') or data.get('key')
        elif text and '\n' not in text and 1 < len(text) < 8192:
            token = text  # a bare one-line value (the new format)
        if token:
            dest = cfg_dir / 'access_token'
            dest.write_text(str(token).strip())
            os.environ['KAGGLE_API_TOKEN'] = str(token).strip()
            kind = 'new-format access_token'
    if dest is None:
        print(f'Unrecognised Kaggle config file {name!r}. Expected a new-format '
              'one-line API key or a legacy kaggle.json {"username","key"}.')
        return False
    try:
        dest.chmod(stat.S_IRUSR | stat.S_IWUSR)  # owner read/write only
    except Exception:
        pass
    print(f'Installed {kind} -> {dest}')
    return True


# ── Aqua export toolchain (combined mode only) ──────────────────
# The audited converter is onnx2tf on TensorFlow 2.19, and TF 2.19 has no
# CPython 3.13+ wheel -- newer Colab images run 3.13+ and cannot switch back.
# So the converter gets its OWN interpreter: uv provisions a CPython 3.12 env
# with the pinned toolchain below, and §5 runs onnx2tf there in a subprocess
# on the ONNX this kernel exports. TensorFlow is never installed into this
# kernel, so the export cannot move torch / numpy / protobuf under a live run
# (the protobuf trap §5 used to document). Verification stays HERE, through
# ai-edge-litert (wheels for CPython 3.10-3.14): the runtime that checks the
# graph is not the one that wrote it.
# `tensorflow`, not `tensorflow-cpu`: tf-keras depends on the `tensorflow`
# distribution, so pinning tensorflow-cpu installed both. onnx2tf 1.28.8 is the
# newest release that resolves against TF 2.19 (2.x pins numpy==2.2.6 and
# TF 2.21); it imports tf_keras, onnx_graphsurgeon and ai_edge_litert at load
# time, uses onnxruntime / sne4onnx on some paths (the old kernel install got
# those from the kernel; a fresh env has to list them) and calls the onnxsim CLI.
AQUA_EXPORT_PYTHON = '3.12'
AQUA_EXPORT_TOOLCHAIN = (
    'tensorflow==2.19.0', 'tf-keras==2.19.0', 'onnx2tf==1.28.8', 'onnx==1.22.0',
    'onnx-graphsurgeon==0.6.1', 'onnxsim==0.7.3', 'sng4onnx==2.0.1',
    'sne4onnx==2.0.1', 'onnxruntime==1.30.0', 'ai-edge-litert==2.2.0',
    'numpy==2.1.3', 'protobuf==5.29.6', 'psutil==7.2.2',
)
AQUA_KERNEL_LITERT = 'ai-edge-litert==2.2.0'   # the interpreter §5 and §6 verify with
AQUA_EXPORT_ENV = ((Path('/content') if Path('/content').is_dir() else Path.cwd())
                   / '.aqua_export_env')
AQUA_EXPORT_ENVIRONMENT = None   # filled by provision_aqua_export_env; §6 seals it


def provision_aqua_export_env():
    """Create (or reuse) the CPython 3.12 export env, and install the TFLite
    interpreter into this kernel. Idempotent; returns the env's python."""
    global AQUA_EXPORT_ENVIRONMENT
    import importlib.metadata as _md
    _py = AQUA_EXPORT_ENV / 'bin' / 'python'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    _uv = [sys.executable, '-m', 'uv']
    if not _py.is_file():
        # A folder without its interpreter is an interrupted earlier attempt.
        shutil.rmtree(AQUA_EXPORT_ENV, ignore_errors=True)
        # uv fetches a standalone CPython 3.12 when the image has none.
        subprocess.check_call([*_uv, 'venv', '-q', '--python', AQUA_EXPORT_PYTHON,
                               str(AQUA_EXPORT_ENV)])
    subprocess.check_call([*_uv, 'pip', 'install', '-q', '--python', str(_py),
                           *AQUA_EXPORT_TOOLCHAIN])
    # This kernel gets the interpreter only, with every core package it may
    # already have imported held at its installed version: pip can add, never move.
    _held = []
    for _name in ('numpy', 'torch', 'torchvision', 'protobuf', 'onnx', 'onnxruntime'):
        try:
            _held.append(f'{_name}=={_md.version(_name)}')
        except _md.PackageNotFoundError:
            pass
    _constraints = AQUA_EXPORT_ENV / 'kernel_constraints.txt'
    _constraints.write_text('\n'.join(_held) + '\n', encoding='utf-8')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           AQUA_KERNEL_LITERT, '-c', str(_constraints)])
    _freeze = subprocess.run([*_uv, 'pip', 'freeze', '--python', str(_py)],
                             check=True, capture_output=True, text=True).stdout
    _env_python = subprocess.run([str(_py), '-c', 'import sys; print(sys.version)'],
                                 check=True, capture_output=True, text=True).stdout.strip()
    AQUA_EXPORT_ENVIRONMENT = {
        'python': _env_python,
        'toolchain': list(AQUA_EXPORT_TOOLCHAIN),
        'distributions': sorted(_l for _l in _freeze.splitlines() if _l.strip()),
        'kernel_interpreter': AQUA_KERNEL_LITERT,
    }
    return _py


def aqua_onnx_to_tflite(onnx_path, out_dir):
    """ONNX -> the ONE FLOAT32 .tflite onnx2tf emits, converted in the export env.

    onnx2tf may emit INT8/FLOAT16 analysis siblings, but only the
    *_float32.tflite graph satisfies the Android serving contract."""
    _py = AQUA_EXPORT_ENV / 'bin' / 'python'
    if not _py.is_file():
        raise RuntimeError('Aqua export env missing; run provision_aqua_export_env() (§1)')
    # A clean interpreter: none of this kernel's import paths, the env's bin/ first
    # (onnx2tf shells out to `onnxsim`), and no GPU -- the teacher may still hold it.
    _env = {k: v for k, v in os.environ.items()
            if k not in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP')}
    _env.update(PATH=f"{_py.parent}{os.pathsep}{os.environ.get('PATH', '')}",
                PYTHONNOUSERSITE='1', CUDA_VISIBLE_DEVICES='', TF_CPP_MIN_LOG_LEVEL='2')
    _code = ('import sys, onnx2tf\n'
             'onnx2tf.convert(input_onnx_file_path=sys.argv[1], output_folder_path=sys.argv[2],\n'
             '                output_dynamic_range_quantized_tflite=True,\n'
             '                copy_onnx_input_output_names_to_tflite=True, non_verbose=True)\n')
    _proc = subprocess.run([str(_py), '-c', _code, str(onnx_path), str(out_dir)],
                           env=_env, capture_output=True, text=True, timeout=1800)
    if _proc.returncode != 0:
        _tail = '\n'.join((_proc.stdout + _proc.stderr).strip().splitlines()[-25:])
        raise RuntimeError(
            f'onnx2tf failed in the export env (exit {_proc.returncode}):\n{_tail}')
    # Require the ONE exact FLOAT32 graph; artifact substitution is forbidden.
    _emitted = sorted(Path(out_dir).glob('*.tflite'))
    _float32_emitted = [p for p in _emitted if 'float32' in p.name.lower()]
    if len(_float32_emitted) != 1:
        _listing = [p.name for p in Path(out_dir).iterdir()] if Path(out_dir).exists() else []
        raise RuntimeError(
            f'onnx2tf must produce exactly one FLOAT32 TFLite; found '
            f'{len(_float32_emitted)} under {out_dir} (contents: {_listing})')
    _chosen = _float32_emitted[0]
    return _chosen


def aqua_verify_tflite(tflite_path, module, x_nchw, n_classes):
    """The shipped graph loads, runs and tracks the torch student: FLOAT32 I/O,
    the app's [1,H,W,3] -> [1,C] shapes (AquaClassifier.kt refuses anything
    else), and max |logit diff| < 1e-2 on x_nchw. Returns that difference."""
    try:
        from ai_edge_litert.interpreter import Interpreter as _TFLInterp
    except Exception:
        from tensorflow.lite import Interpreter as _TFLInterp  # type: ignore
    _interp = _TFLInterp(model_path=str(tflite_path)); _interp.allocate_tensors()
    _ind, _outd = _interp.get_input_details()[0], _interp.get_output_details()[0]
    if np.dtype(_ind['dtype']) != np.dtype(np.float32) or np.dtype(_outd['dtype']) != np.dtype(np.float32):
        raise RuntimeError(
            f'Aqua serving contract requires FLOAT32 I/O; got '
            f'input={_ind["dtype"]}, output={_outd["dtype"]}')
    _h, _w = int(x_nchw.shape[2]), int(x_nchw.shape[3])
    if list(_ind['shape']) != [1, _h, _w, 3] or list(_outd['shape']) != [1, n_classes]:
        raise RuntimeError(
            f'Aqua app contract is input [1,{_h},{_w},3] -> output [1,{n_classes}]; '
            f'got {list(_ind["shape"])} -> {list(_outd["shape"])}')
    _x = np.transpose(x_nchw.detach().cpu().numpy(), (0, 2, 3, 1)).astype(np.float32)
    _interp.set_tensor(_ind['index'], _x)
    _interp.invoke()
    _tfl_logits = _interp.get_tensor(_outd['index']).reshape(-1)
    with torch.no_grad():
        _torch_logits = module(x_nchw).detach().float().cpu().numpy().reshape(-1)
    _argmax_ok = int(_tfl_logits.argmax()) == int(_torch_logits.argmax())
    _md = float(np.max(np.abs(_tfl_logits - _torch_logits)))
    if not _md < 1e-2:
        raise RuntimeError(f'TFLite/torch logit drift too large: {_md:.3e}')
    # The logit scale beside the diff: a diff is only evidence when the logits
    # it compares are not themselves ~0.
    print(f'Aqua FLOAT32 TFLite parity OK  max_abs_logit_diff={_md:.3e}  '
          f'max_abs_logit={float(np.max(np.abs(_torch_logits))):.3e}  '
          f'argmax_match={_argmax_ok}')
    return _md


# Preflight: the whole §5 export path -- ONNX export here, onnx2tf in the env,
# the app-contract and parity checks here -- on §5's starting point (ImageNet
# MobileNetV3-Small + a fresh head of the run's class count) at §5's STUDENT_IMG
# (224). Not random init: that gives ~1e-8 logits, which pass any parity bound
# (the first 2.7.5 preflight read 1.2e-15). A few minutes now
# instead of a teacher-length run lost at §5. Its RNG draws are forked so they
# do not shift anything the run seeds.
if FAMILY_CONFIG.get('distill'):
    import tempfile
    import torchvision as _tv
    _pf_dir = Path(tempfile.mkdtemp(prefix='aqua_export_preflight_'))
    try:
        provision_aqua_export_env()
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(SEED)
            _pf_model = _tv.models.mobilenet_v3_small(weights='IMAGENET1K_V1')
            _pf_model.classifier[-1] = nn.Linear(
                _pf_model.classifier[-1].in_features, len(CLASSES))
            _pf_model = _pf_model.eval()
            _pf_x = torch.randn(1, 3, 224, 224)
        torch.onnx.export(_pf_model, _pf_x, str(_pf_dir / 'preflight.onnx'),
                          opset_version=18, input_names=['input'], output_names=['logits'],
                          dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}})
        aqua_verify_tflite(aqua_onnx_to_tflite(_pf_dir / 'preflight.onnx', _pf_dir / 'tflite'),
                           _pf_model, _pf_x, len(CLASSES))
    except Exception as _pf_err:
        raise RuntimeError(
            'combined mode: the Aqua export toolchain failed its preflight, before any '
            "training -- nothing was trained or written. Fix the cause above, or set "
            "TRAIN_MODE='gamma' for the server bundle alone.") from _pf_err
    finally:
        shutil.rmtree(_pf_dir, ignore_errors=True)
    print(f"Aqua export preflight OK: CPython {AQUA_EXPORT_ENVIRONMENT['python'].split()[0]} "
          f'env at {AQUA_EXPORT_ENV} converts an ImageNet MobileNetV3-Small to the app contract')
    log_run({'phase': 'aqua_export_preflight', **AQUA_EXPORT_ENVIRONMENT})


### Kaggle credentials (required by the default production recipe)

Add a Colab secret named `KAGGLE_API_TOKEN` (new format), or set `KAGGLE_UPLOAD_CREDS = True` in the next cell and upload a token file (or legacy `kaggle.json`). Individual Kaggle fetchers fail soft so a deliberately non-production diagnostic can inspect other sources, but the default `REQUIRE_SUPPLEMENT=True` plus strict source-floor gate fails closed before training. A production Run-all therefore requires working Kaggle credentials and the supplement; missing access is never silently substituted. Licences are recorded, not gated.

In [ ]:
# Kaggle + HuggingFace credentials for §2.
#   HF_TOKEN (optional, read scope): PlantDoc / PlantVillage / the COCO mirror
#   come from the HF Hub, where anonymous requests are rate-limited and slower.
#   Add it as a Colab secret the same way as KAGGLE_API_TOKEN; the sync below
#   exports it and huggingface_hub picks it up.
#   Easiest: add a Colab secret named KAGGLE_API_TOKEN (toggle 'Notebook
#   access' ON for it). Colab secrets are NOT auto-exported as env vars, so
#   the sync below pulls it in for you -- no upload needed.
#   Alternative: set KAGGLE_UPLOAD_CREDS = True to upload a token file / kaggle.json.
KAGGLE_UPLOAD_CREDS = False
_kaggle_load_colab_config()
_have_kaggle = bool(os.environ.get('KAGGLE_API_TOKEN')
                    or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')))
if not _have_kaggle and KAGGLE_UPLOAD_CREDS:
    _have_kaggle = bool(kaggle_install_api_config())
print('HF_TOKEN detected — HF Hub downloads will be authenticated.'
      if os.environ.get('HF_TOKEN') else
      'HF_TOKEN not set — HF Hub downloads stay anonymous (rate-limited but '
      'functional; §2 will print the same notice).')
if _have_kaggle:
    print('Kaggle credentials detected — the §2 supplement will run (incl. the\n          eval-eligible DoctorP real-photo slice that seeds the weak-class ship gate).')
else:
    print('Kaggle: no credentials yet. Add a Colab secret KAGGLE_API_TOKEN '
          '(Notebook access ON) and re-run THIS cell, or set '
          'KAGGLE_UPLOAD_CREDS = True to upload a token file. The §2 '
          'supplement is optional and will be skipped without it.')


## §2 — Data acquisition + group-aware splits

Public sources only. No manual upload. Every saved image gets a
sidecar JSON so the splitter can be group-aware (`(source, plant_id)`
goes together, never split across train and a held-out set).

| Source | Role |
|---|---|
| **PlantDoc** (HF) | real-world train + `val_realworld` seed |
| **PlantVillage** (HF) | capped train + `val_studio`. PlantVillage **never** enters `val_realworld`, `temp_cal` or `conformal_cal` (would break exchangeability) |
| **iNaturalist** | healthy houseplant leaves (Ficus / Monstera / …), Botrytis supplement (**`Green Leaves` annotation** off for the fungus class), Animalia for NAP, and free-text physiological queries (`leaf scorch`, `chlorosis`, …) for the v2.2 deferred classes |
| **COCO val2017** | `not_a_plant` images, plant-like categories filtered out |
| **Open Images V7** | `not_a_plant` indoor-scene supplement (Couch / Lamp / Vase / Bookcase / …) — closes the indoor-clutter gap behind production NAP failures |
| **Kaggle ornamental** (`pappu54/indoor-plant-disease-dataset-23-classes`) | v2.4: rose / hibiscus / chrysanthemum / **money-plant (Epipremnum, a true houseplant)** leaves with REAL disease morphology on NATURAL backgrounds — the closest public supply to the deployment domain for the weak classes (`powdery_mildew`, `viral_disease`, `leaf_spot`, `nutrient_deficiency`, `leaf_scorch`). **Train-only**, moderately down-weighted (partly augmented). Drops rust / toxicity / wilt / downy / aphid / blight / necrosis. |
| **DoctorP** (`alexanderuzhinskiy/the-doctorp-project-dataset`) | v2.4: ~4k REAL plant-doctor **app photos** (gardener's framing, no augmentation). The only public source that supplies the starved classes in their deployment framing — **Grey mold + Botrytis cinerea → gray_mold_botrytis**, **Mealybug + Scale → pests** (cottony masses / waxy bumps on the plant, not iNat entomology macros) — plus real powdery_mildew / viral / leaf_spot / nutrient_deficiency / scorch / healthy. **EVAL-ELIGIBLE** (the one Kaggle source allowed into the held-out splits): full sampler weight, seeds the real-photo eval slice. |

**v2.5.1 data-quality fixes.** (a) The Kaggle disease sets are ~85% baked-in augmentations (e.g. abdulahad0296's `Aloe_Anthracnose` is 54 originals vs 712 synthetic rotations); `_stratified_folder_order` now draws ORIGINAL captures before augmented copies, so the per-class cap buys distinct real photos (~4.6× more unique images at the same count) instead of rotations the §4 augmentation already produces. (b) The kagglehub download cache (~4 GB) is purged once every source has copied into `TRAIN_ROOT`, so the run fits a ~60 GB disk.

**v2.4 data-quality fixes** (from a hands-on audit of every source):
the `Cactus_Dactylopius` -> `pests` mapping was removed (outdoor
prickly-pear field shots, not indoor mealybug/scale); the Kaggle
houseplant set's **black-background mask-segmented Aloe** images (a
background shortcut + heavy augmentation) are now rejected at fetch by a
near-black-pixel test; the noisy `'yellowing leaves plant'` iNat query
(~9.7k generic-yellowing obs) was dropped from `nutrient_deficiency`;
and `kaggle_houseplant` is further down-weighted. v2.4 also adds **DoctorP** (real plant-doctor app photos) as the first
EVAL-ELIGIBLE supplement: it gives `gray_mold_botrytis` and `pests` a
real (if small, ~76-173/class) in-domain supply in their deployment
framing, and — because `source='doctorp'` is in every `make_*_split`
`allowed_sources` — it **seeds `val_realworld` / `val_indomain` /
`temp_cal` / `conformal_cal` with real gardener-view photos across the
whole taxonomy**, so the ship gate no longer measures only PlantDoc
field crops. Residual limitation: `gray_mold_botrytis` is still the
thinnest class (~96 real + crop proxies), and DoctorP spans
houseplants + garden/orchard plants (real phone photos, but not
exclusively indoor).

Set `REBUILD_GENERATED_SPLITS = False` if you only want to top-up an
existing run.


In [ ]:
# §2 — Data acquisition + group-aware splits
import functools, threading
from io import BytesIO
from PIL import Image as PILImage

UA = {'User-Agent': 'PlantVision/2.6 (research; contact: local-notebook)'}
REBUILD_GENERATED_SPLITS = True

if REBUILD_GENERATED_SPLITS:
    for d in (TRAIN_ROOT, VAL_ROOT):
        if d.exists():
            shutil.rmtree(d)
    ensure_layout()

# Module-level Session: TCP keep-alive across the per-image fetches in
# this cell. Per-call ``headers=`` still wins over the session default.
_SESSION = requests.Session()
_SESSION.headers.update(UA)

_RETRY_STATUS = (429, 500, 502, 503, 504)

# Cross-call dedup of remote fetches (iNat photos can appear under
# multiple taxa). Guarded by a lock because we update it from worker
# threads in the ThreadPoolExecutor pools below.
_FETCH_SEEN_SHA: set = set()
_FETCH_SEEN_LOCK = threading.Lock()

# ── HTTP helpers (exponential backoff on 429/5xx) ────────────────
def http_get_json(url, params=None, timeout=30, retries=4):
    """GET + .json(). Logs every backoff so a starved class makes the
    upstream failure visible in the run log."""
    last, delay = None, 0.8
    for attempt in range(retries):
        try:
            r = _SESSION.get(url, params=params, timeout=timeout)
            if r.status_code in _RETRY_STATUS:
                last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                print(f'  http_get_json {url} attempt {attempt+1}/{retries} '
                      f'-> {r.status_code}; sleeping {delay:.1f}s')
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.json()
        except Exception as e:
            last = e
            print(f'  http_get_json {url} attempt {attempt+1}/{retries} '
                  f'-> {type(e).__name__}: {e}; sleeping {delay:.1f}s')
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_get_json exhausted retries: {url}')

def http_download(url, dest, timeout=120, retries=4):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.tmp')
    last, delay = None, 1.0
    for _ in range(retries):
        try:
            with _SESSION.get(url, stream=True, timeout=timeout) as r:
                if r.status_code in _RETRY_STATUS:
                    last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                    time.sleep(delay); delay *= 2; continue
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk: f.write(chunk)
            tmp.replace(dest); return dest
        except Exception as e:
            last = e
            if tmp.exists():
                try: tmp.unlink()
                except Exception: pass
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_download exhausted retries: {url}')

def http_get_bytes(url, timeout=20, retries=4):
    """Returns body bytes or None — callers treat None as a soft skip
    so a single bad image doesn't abort the surrounding pool."""
    delay = 0.5
    for _ in range(retries):
        try:
            r = _SESSION.get(url, timeout=timeout)
            if r.status_code in _RETRY_STATUS:
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.content
        except Exception:
            time.sleep(delay); delay *= 2
    return None

def _stable_frac(text: str) -> float:
    return int(hashlib.sha1(text.encode('utf-8')).hexdigest()[:12], 16) / float(16**12 - 1)

def _save_rgb(img, out: Path, quality=92):
    out.parent.mkdir(parents=True, exist_ok=True)
    img.convert('RGB').save(out, quality=quality)

def _valid_image_bytes(body, min_side=160):
    try:
        im = PILImage.open(BytesIO(body)).convert('RGB')
        if min(im.size) < min_side: return None
        return im
    except Exception:
        return None

def _claim_sha(body: bytes) -> bool:
    """Try to claim a SHA-256. Returns True if this is the first time we've
    seen these bytes (and the caller may proceed), False on duplicate."""
    sha = hashlib.sha256(body).hexdigest()
    with _FETCH_SEEN_LOCK:
        if sha in _FETCH_SEEN_SHA:
            return False
        _FETCH_SEEN_SHA.add(sha)
    return True

def _save_remote_body(body, fp, ann_factory):
    """Claim/validate/save one already-fetched body in caller order."""
    if fp.exists(): return 0
    if not body: return 0
    if not _claim_sha(body): return 0
    img = _valid_image_bytes(body)
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, ann_factory())
    return 1

def _fetch_save_remote(url, fp, ann_factory, *, timeout=20):
    """Sequential convenience wrapper around ``_save_remote_body``."""
    return _save_remote_body(http_get_bytes(url, timeout=timeout), fp, ann_factory)

# Network-bound fetch pools wait on HTTP, not CPU, so the worker count is
# sized to concurrent connections rather than vCPUs.
NET_WORKERS = 16
_NET_CHUNK = 64

# ── Connection-pool sizing (must match the widest thread pool) ───
# requests' default HTTPAdapter keeps pool_maxsize=10 connections PER HOST, but
# the fetchers below run NET_WORKERS=16 threads against ONE host at a time
# (iNat's S3 bucket, then Open Images' S3, then Zenodo). The 6 surplus
# connections had nowhere to go: urllib3 logged 'Connection pool is full,
# discarding connection: <host>' and closed the socket, so the next fetch paid a
# fresh TCP + TLS handshake. On a cell that makes tens of thousands of requests
# that is pure latency, and the log noise was the symptom, not the problem.
# Sizing the pool to the thread pool removes the cause, so the warnings stop
# without anything being suppressed.
#   pool_connections — how many DISTINCT-HOST pools stay cached (iNat static +
#     iNat S3 + OpenImages S3 + Zenodo + HF + …). Cheap: an idle pool is a dict
#     entry, not a socket.
#   max_retries=0 — retry policy stays in the explicit backoff loops above.
#     Letting the adapter retry too would silently square the attempt count and
#     hide upstream failures the run log is supposed to surface.
_POOL_MAXSIZE = max(NET_WORKERS, EVAL_WORKERS) + 4
_HTTP_ADAPTER = requests.adapters.HTTPAdapter(
    pool_connections=32, pool_maxsize=_POOL_MAXSIZE, max_retries=0)
_SESSION.mount('https://', _HTTP_ADAPTER)
_SESSION.mount('http://', _HTTP_ADAPTER)
print(f'HTTP session: pool_maxsize={_POOL_MAXSIZE} '
      f'(NET_WORKERS={NET_WORKERS}, EVAL_WORKERS={EVAL_WORKERS}), pool_connections=32')

# ── HuggingFace auth ────────────────────────────────────────────
# PlantDoc + PlantVillage + the COCO mirror all come from the HF Hub. Anonymous
# requests are rate-limited and slower, which is what
# 'You are sending unauthenticated requests to the HF Hub' is warning about —
# and on a bad day it is a 429 mid-fetch, not just a slow one. The loaders below
# used to pass token=False, which forces anonymous EVEN IF a token is present,
# so the warning could not be resolved by configuring anything.
# Now: use a token when one exists (Colab secret / env HF_TOKEN, bridged in §1),
# and fall back to explicit-anonymous when it does not — same behaviour as
# before, but now it is a consequence of having no token rather than of the code
# refusing to use one. Read once here so every call site agrees.
_HF_TOKEN = (os.environ.get('HF_TOKEN')
             or os.environ.get('HUGGING_FACE_HUB_TOKEN') or '').strip() or None
_HF_AUTH = _HF_TOKEN or False      # False = explicitly anonymous (huggingface_hub API)
_HF_RESOLVED_REVISIONS = {}

def _hf_resolve_dataset_revision(repo_id: str) -> str:
    """Resolve Hub ``main`` once, then use and record that exact commit.

    Dataset repos are mutable. Passing the returned SHA to every download
    prevents a long fetch from mixing two revisions, while the source/data
    manifests make revision drift between fresh runs explicit.
    """
    cached = _HF_RESOLVED_REVISIONS.get(repo_id)
    if cached is not None:
        return cached['resolved_commit']
    from huggingface_hub import HfApi
    info = HfApi(token=_HF_TOKEN).dataset_info(repo_id=repo_id, revision='main')
    revision = str(getattr(info, 'sha', '') or '').lower()
    if re.fullmatch(r'[0-9a-f]{40}', revision) is None:
        raise RuntimeError(f'{repo_id}: Hub returned invalid dataset commit {revision!r}')
    _HF_RESOLVED_REVISIONS[repo_id] = {
        'repo_id': repo_id, 'requested_revision': 'main',
        'resolved_commit': revision, 'loads': {},
    }
    return revision

def _record_hf_dataset_load(repo_id, config_name, split, dataset):
    """Bind the exact datasets-library fingerprints to the Hub commit."""
    entry = _HF_RESOLVED_REVISIONS[repo_id]
    key = f'{config_name or "default"}:{split or "all"}'
    if hasattr(dataset, 'keys'):
        fingerprints = {str(k): str(getattr(v, '_fingerprint', ''))
                        for k, v in dataset.items()}
        versions = {str(k): str(getattr(getattr(v, 'info', None), 'version', ''))
                    for k, v in dataset.items()}
    else:
        fingerprints = {str(split or 'all'): str(getattr(dataset, '_fingerprint', ''))}
        versions = {str(split or 'all'):
                    str(getattr(getattr(dataset, 'info', None), 'version', ''))}
    if any(not value for value in fingerprints.values()):
        raise RuntimeError(f'{repo_id}: datasets fingerprint missing for {key}')
    entry['loads'][key] = {
        'fingerprints': dict(sorted(fingerprints.items())),
        'dataset_versions': dict(sorted(versions.items())),
    }
print('HuggingFace Hub: ' + ('authenticated (HF_TOKEN found)' if _HF_TOKEN else
      'ANONYMOUS — rate-limited and slower. Add a Colab secret HF_TOKEN '
      '(Notebook access ON) with read scope to remove the warning below.'))

def _run_fetch_jobs(jobs, target_n, workers=NET_WORKERS):
    """Fetch ``jobs`` concurrently, but consume results in JOB order.

    Draining ``as_completed`` and breaking at ``target_n`` (the behaviour
    this replaces) let HTTP timing decide which images entered the corpus:
    the fastest responses won the race, so two runs of identical code at
    SEED=0 produced different training sets, and `_claim_sha` dedup
    ownership shifted with them. Jobs are submitted a chunk at a time and
    awaited in submission order — the fetches still overlap, the surplus
    past ``target_n`` is never submitted, and the selected prefix is
    reproducible. Callers over-collect ``jobs`` to absorb fetch failures.
    """
    n = 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for i in range(0, len(jobs), _NET_CHUNK):
            if n >= target_n:
                break
            chunk = jobs[i:i + _NET_CHUNK]
            # Fetches overlap, but SHA ownership + writes happen here in job
            # order. Previously workers called _claim_sha concurrently, so the
            # comment promised determinism while HTTP timing still chose which
            # duplicate source/label survived.
            futs = [pool.submit(http_get_bytes, u) for u, _f, _a in chunk]
            for (_u, fp, ann_factory), fut in zip(chunk, futs, strict=True):
                if n >= target_n:
                    break
                n += _save_remote_body(fut.result(), fp, ann_factory)
    return n

# ── iNaturalist (research-grade + commercial-compatible photos) ──
INAT_BASE = 'https://api.inaturalist.org/v1'
INAT_LEAF_TERM_ID, INAT_LEAF_TERM_VALUE_ID = 36, 38   # iNat 'Leaves' annotation, value='Green Leaves'
# Research-grade is an identification-quality grade, not a reuse grant, and
# iNat licenses each PHOTO separately. STUDY-PROJECT POLICY (2026-09-21): the
# licence is RECORDED per row as provenance and never gated on. Measured on
# 2026-09-21 against the live API: 83.0M research-grade Plantae observations
# with photos, 10.8M of them under cc0/cc-by/cc-by-sa — the 2026-08-25 licence
# filter would have discarded ~87% of the supply. Photo quality does not depend
# on the licence (it is the uploader's account default), so nothing about the
# eligibility of a photo changes; only what the sidecar records does.
INAT_KNOWN_PHOTO_LICENSES = {
    'cc0': 'https://creativecommons.org/publicdomain/zero/1.0/',
    'cc-by': 'https://creativecommons.org/licenses/by/4.0/',
    'cc-by-sa': 'https://creativecommons.org/licenses/by-sa/4.0/',
    'cc-by-nc': 'https://creativecommons.org/licenses/by-nc/4.0/',
    'cc-by-nd': 'https://creativecommons.org/licenses/by-nd/4.0/',
    'cc-by-nc-sa': 'https://creativecommons.org/licenses/by-nc-sa/4.0/',
    'cc-by-nc-nd': 'https://creativecommons.org/licenses/by-nc-nd/4.0/',
    'pd': 'https://creativecommons.org/publicdomain/mark/1.0/',
    'gfdl': 'https://www.gnu.org/licenses/fdl-1.3.html',
}
_INAT_LICENSE_TALLY = {}

@functools.lru_cache(maxsize=64)
def inat_resolve_taxon(name, rank='genus'):
    res = http_get_json(f'{INAT_BASE}/taxa', {'q': name, 'rank': rank}).get('results') or []
    return res[0]['id'] if res else None

def inat_observations(taxon_id, max_n, leaf_only=True):
    rows, page, per = [], 1, 100
    while len(rows) < max_n:
        params = {'taxon_id': taxon_id, 'quality_grade': 'research', 'photos': 'true',
                  'per_page': per, 'page': page, 'order': 'desc', 'order_by': 'created_at'}
        if leaf_only:
            params['term_id'] = INAT_LEAF_TERM_ID
            params['term_value_id'] = INAT_LEAF_TERM_VALUE_ID
        batch = (http_get_json(f'{INAT_BASE}/observations', params).get('results') or [])
        if not batch: break
        rows.extend(batch)
        if len(batch) < per: break
        page += 1; time.sleep(0.25)
    return rows[:max_n]

def _photo_url(photo, size='medium'):
    raw = photo.get('url') or ''
    return re.sub(r'/square\.', f'/{size}.', raw) if raw.startswith('http') else None

def _inat_photo_provenance(obs, photo):
    """Return sidecar rights/provenance metadata for one iNat photo.

    Record-only: every photo is accepted. The licence code ('unlicensed' when
    iNat reports none, i.e. all rights reserved) is tallied and written into
    the sidecar so a bundle can report what it was trained on."""
    code = str(photo.get('license_code') or '').strip().lower() or 'unlicensed'
    _INAT_LICENSE_TALLY[code] = _INAT_LICENSE_TALLY.get(code, 0) + 1
    user = obs.get('user') or {}
    creator = (user.get('name') or user.get('login') or '').strip() or None
    obs_id, photo_id = obs.get('id'), photo.get('id')
    return {
        'license_id': code,
        'license_url': INAT_KNOWN_PHOTO_LICENSES.get(code),
        'creator': creator,
        'source_url': (obs.get('uri') or
                       (f'https://www.inaturalist.org/observations/{obs_id}'
                        if obs_id is not None else None)),
        'source_item_id': f'observation:{obs_id}/photo:{photo_id}',
        'attribution': (photo.get('attribution') or '').strip() or creator,
    }

def _save_one_inat(item, dest, klass, tx, label_confidence):
    obs, ph, provenance = item
    url = _photo_url(ph, 'medium')
    if not url: return 0
    fp = dest / f'inat_{tx}_{obs["id"]}_{ph["id"]}.jpg'
    if fp.exists(): return 0
    body = http_get_bytes(url)
    if not body: return 0
    if not _claim_sha(body): return 0
    img = _valid_image_bytes(body)
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, CropAnnotation(
        plant_id=str(obs.get('id')),
        species=(obs.get('taxon') or {}).get('name'),
        diagnosis=klass, severity='unknown', leaf_or_whole_plant='leaf',
        capture_session=f'inat_taxon_{tx}', source='inat',
        label_confidence=label_confidence, bbox=[0, 0, 0, 0],
        **provenance))
    return 1

def download_inat_class(taxon_ids, dest, klass, max_per_taxon=160, leaf_only=True,
                        workers=8, label_confidence=0.7, photos_per_obs=1):
    dest.mkdir(parents=True, exist_ok=True)
    # Dedup resolved taxon ids. iNat treats some genera as synonyms of one
    # another (e.g. Botryotinia -> Botrytis, both taxon 324264), so two names
    # in a *_TAXA list can resolve to the SAME id. Without this, the second id
    # re-fetches the identical observations + photos that SHA-dedup then drops
    # — wasted API calls/bandwidth, and on the thinnest classes (gray mold)
    # that is the slow tail of §2. dict.fromkeys preserves first-seen order.
    taxon_ids = list(dict.fromkeys(taxon_ids))
    n = 0
    for tx in taxon_ids:
        try:
            obs_list = inat_observations(tx, max_per_taxon, leaf_only=leaf_only)
        except Exception as e:
            print(f'  iNat taxon {tx} failed: {e}'); continue
        items = [(o, p, prov) for o in obs_list
                 for p in (o.get('photos') or [])[:photos_per_obs]
                 if (prov := _inat_photo_provenance(o, p)) is not None]
        if not items: continue
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futs = [pool.submit(_save_one_inat, it, dest, klass, tx, label_confidence) for it in items]
            for f in futs:          # submission order: keeps _claim_sha dedup reproducible
                n += f.result()
    return n

INAT_HEALTHY_TAXA = [
    ('Ficus','genus'),('Monstera','genus'),('Epipremnum','genus'),('Calathea','genus'),
    ('Spathiphyllum','genus'),('Dracaena','genus'),('Sansevieria','genus'),('Philodendron','genus'),
    # Variegated / vividly patterned ornamentals as HEALTHY negatives.
    # Their natural mottling + bold colour is the dominant viral_disease
    # false positive (the classifier reads it as mosaic — see MODEL_CARD).
    # iNat research-grade 'Calathea' skews to plain-green specimens, so the
    # strongly-patterned cultivars were under-represented; these add them.
    # (Goeppertia is the current genus for many 'Calathea'; both kept for
    # yield. No green-content filter runs on the healthy class, so vivid
    # red/pink leaves are retained rather than dropped.)
    ('Goeppertia','genus'),('Maranta','genus'),('Stromanthe','genus'),('Ctenanthe','genus'),
    ('Caladium','genus'),('Aglaonema','genus'),('Codiaeum','genus'),('Coleus','genus'),
    ('Fittonia','genus'),('Hypoestes','genus'),('Begonia','genus'),('Tradescantia','genus'),
]
# A genus taxon_id query on iNat already returns every descendant
# species, so the genus rows alone cover all Botrytis spp. The
# species-level rows we used to list added ~0 deduped yield (measured
# ~138 foliage images either way) while re-downloading photos that SHA
# dedup then dropped, so they were cut to save the redundant fetches.
INAT_BOTRYTIS_TAXA = [
    # 'Botryotinia' is an iNat SYNONYM of 'Botrytis' (both resolve to taxon
    # 324264, verified live 2026-06), so it contributes 0 distinct observations.
    # download_inat_class() dedups the resolved ids, so the synonym never
    # double-fetches; it is kept only as a resolver fallback in case iNat ever
    # re-splits the genera.
    ('Botrytis','genus'),('Botryotinia','genus'),
]

def _green_content_ratio(img) -> float:
    """Fraction of foliage-green pixels in HSV. Cheap heuristic for filtering
    fungal observations that aren't actually on a leaf (agar plates, dead
    tissue, fruit). Indoor leaf shots score 0.30–0.70; non-foliage close-ups
    score < 0.05."""
    # Pillow HSV (H,S,V each 0-255), no OpenCV dependency. The green-hue
    # band [50,120]/255 ≈ 70-170° reproduces the old OpenCV [35,85]/179.
    arr = np.asarray(img.convert('HSV'))
    h, s, v = arr[..., 0], arr[..., 1], arr[..., 2]
    mask = (h >= 50) & (h <= 120) & (s >= 40) & (v >= 50)
    return float(mask.mean())

def _green_ratio_for_path(p: Path):
    try:
        return p, _green_content_ratio(PILImage.open(p))
    except Exception:
        return p, None

def _drop_low_green_content(class_dir: Path, min_ratio: float = 0.10) -> int:
    """Remove images whose green-content ratio falls below the floor.
    Sidecars are removed alongside. PIL+cv2 work is GIL-released so we
    parallelise across EVAL_WORKERS threads."""
    paths = list(iter_images(class_dir))
    if not paths:
        return 0
    dropped = 0
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        for p, ratio in pool.map(_green_ratio_for_path, paths):
            if ratio is None or ratio >= min_ratio:
                continue
            sc = p.with_suffix('.json')
            try:
                p.unlink()
                if sc.exists(): sc.unlink()
                dropped += 1
            except OSError:
                pass
    return dropped

def fetch_inat_supplement(healthy_per_taxon=180, botrytis_per_taxon=2400):
    healthy_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_HEALTHY_TAXA) if t]
    n = download_inat_class(healthy_ids, TRAIN_ROOT / 'healthy', 'healthy',
                            max_per_taxon=healthy_per_taxon, leaf_only=True,
                            label_confidence=0.75)
    print(f'iNat healthy += {n}')
    # Botrytis is a fungus, not a plant — iNat's `Green Leaves` annotation
    # returns ~0 hits, so we fetch leaf_only=False and accept some non-leaf
    # hosts. The green-content gate filters obvious non-foliage. v2.2 raises
    # botrytis_per_taxon to compensate for Openverse being dropped.
    botrytis_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_BOTRYTIS_TAXA) if t]
    # v2.4: pull up to 3 photos per RESEARCH-GRADE observation (the API
    # returns ~2.5 photos/obs and the code previously kept only the first),
    # ~1.6x-ing the high-confidence botrytis foliage supply WITHOUT dropping to
    # the noisier needs_id grade. The green-content gate below still filters
    # non-foliage (agar plates / herbarium / fruit macros).
    n = download_inat_class(botrytis_ids, TRAIN_ROOT / 'gray_mold_botrytis', 'gray_mold_botrytis',
                            max_per_taxon=botrytis_per_taxon, leaf_only=False,
                            label_confidence=0.55, photos_per_obs=3)
    ft = _fetch_inat_freetext_class('gray_mold_botrytis', GRAY_MOLD_FREETEXT_QUERIES,
                                    source='inat', per_query=140, label_confidence=0.5)
    print(f'iNat gray_mold_botrytis free-text += {ft}')
    dropped = _drop_low_green_content(TRAIN_ROOT / 'gray_mold_botrytis', min_ratio=BOTRYTIS_GREEN_MIN_RATIO)
    print(f'iNat gray_mold_botrytis supplement += {n}, dropped {dropped} non-foliage')

# ── COCO val2017 → not_a_plant ──────────────────────────────────
COCO_CACHE = ROOT / 'cache_coco_val2017'
# Mirror used when images.cocodataset.org serves a CloudFront certificate
# whose SAN doesn't list the host (intermittent, currently broken).
# ``pcuenq/coco-2017-mirror`` exposes the canonical val2017.zip (816 MB)
# and annotations_trainval2017.zip (253 MB) as direct downloadable files
# under their canonical filenames — same SHA, same internal layout.
_COCO_HF_MIRROR = 'pcuenq/coco-2017-mirror'
# COCO's instances JSON licenses every image separately (ids 1-3 are the NC
# variants, 4-8 BY / BY-SA / BY-ND / public-domain / USG). Record-only: the id
# and licence name go into the sidecar; nothing is filtered on it.
_COCO_LICENSE_TALLY = {}

def _coco_tally(im):
    _k = str(im.get('license'))
    _COCO_LICENSE_TALLY[_k] = _COCO_LICENSE_TALLY.get(_k, 0) + 1

def _coco_download(canonical_url: str, hf_filename: str, dest: Path) -> Path:
    """Fetch a COCO asset, HF-MIRROR-FIRST. images.cocodataset.org has been
    serving a CloudFront cert whose SAN omits the host, so the canonical-first
    order burned ~15 s of SSL retries + an alarming log line every run before
    falling back. Prefer pcuenq/coco-2017-mirror (identical SHA + layout) and
    only hit the canonical host if the mirror is unreachable."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    try:
        from huggingface_hub import hf_hub_download
        _revision = _hf_resolve_dataset_revision(_COCO_HF_MIRROR)
        src = hf_hub_download(_COCO_HF_MIRROR, hf_filename, repo_type='dataset',
                              revision=_revision, token=_HF_AUTH)
        shutil.copy(src, dest)
        return dest
    except Exception as e:  # noqa: BLE001 — mirror down: try the canonical host
        print(f'  COCO HF mirror {_COCO_HF_MIRROR} failed ({type(e).__name__}: {e}); '
              f'trying canonical {canonical_url}')
        return http_download(canonical_url, dest)

def fetch_coco_for_nap(target_n=1500):
    COCO_CACHE.mkdir(parents=True, exist_ok=True)
    img_zip, ann_zip = COCO_CACHE/'val2017.zip', COCO_CACHE/'annotations.zip'
    if not (COCO_CACHE/'val2017').exists():
        print('Downloading COCO val2017 (~1 GB)…')
        _coco_download('https://images.cocodataset.org/zips/val2017.zip',
                       'val2017.zip', img_zip)
        with zipfile.ZipFile(img_zip) as z: z.extractall(COCO_CACHE)
    if not (COCO_CACHE/'annotations').exists():
        _coco_download(
            'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
            'annotations_trainval2017.zip', ann_zip,
        )
        with zipfile.ZipFile(ann_zip) as z: z.extractall(COCO_CACHE)
    ann = json.loads((COCO_CACHE/'annotations'/'instances_val2017.json').read_text())
    license_by_id = {int(x['id']): x for x in ann.get('licenses', [])}
    # COCO is not exhaustively labelled, so dropping food / dining categories
    # cuts the worst plant-leak offenders without losing the indoor-clutter
    # frames we want for NAP.
    plant_like_names = {
        'potted plant', 'vase',
        'broccoli', 'carrot', 'banana', 'apple', 'orange',
        'sandwich', 'cake', 'donut', 'pizza', 'hot dog',
        'dining table', 'bowl',
    }
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in plant_like_names}
    plant_imgs = {a['image_id'] for a in ann['annotations'] if a['category_id'] in plant_cat}
    eligible = [im for im in ann['images'] if im['id'] not in plant_imgs]
    if not eligible:
        raise RuntimeError('COCO produced no eligible non-plant images')
    rng = random.Random(42); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for im in eligible[:target_n]:
        src, dst = COCO_CACHE/'val2017'/im['file_name'], dest/f'coco_{im["id"]}.jpg'
        if not src.exists() or dst.exists(): continue
        shutil.copy(src, dst)
        lic = license_by_id.get(im.get('license')) or {}
        _coco_tally(im)
        write_sidecar(dst, CropAnnotation(plant_id=f'coco_{im["id"]}', species=None,
            diagnosis='not_a_plant', severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_val2017', source='coco', label_confidence=0.99, bbox=[0,0,0,0],
            license_id=f'coco:{im.get("license")}', license_url=lic.get('url'),
            source_url=im.get('flickr_url') or im.get('coco_url'),
            source_item_id=str(im['id']), attribution=lic.get('name')))
        n += 1
    print(f'COCO: not_a_plant += {n}')

# ── iNat Animalia → not_a_plant (indoor-clutter approximation) ─
# COCO is too studio-clean to teach the OOD gate about cats / dogs / birds
# in a leaf photo. iNat research-grade Animalia gets tagged
# source='inat_animalia' so the source-weighted sampler can distinguish
# pet-in-frame NAP from real-leaf iNat rows.
INAT_ANIMALIA_KINGDOM = 'Animalia'

def fetch_inat_animalia_for_nap(target_n=500):
    kingdom_id = inat_resolve_taxon(INAT_ANIMALIA_KINGDOM, rank='kingdom')
    if not kingdom_id:
        print('iNat Animalia kingdom not resolvable; skipping NAP supplement')
        return
    obs = inat_observations(kingdom_id, target_n, leaf_only=False)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    jobs = []
    for ob in obs:
        for ph in (ob.get('photos') or []):
            if len(jobs) >= target_n * 2: break  # oversample for skip slack
            provenance = _inat_photo_provenance(ob, ph)
            if provenance is None: continue
            url = _photo_url(ph, 'medium')
            if not url: continue
            fp = dest / f'inat_animalia_{ob["id"]}_{ph["id"]}.jpg'
            obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
            ann = lambda obs_id=obs_id, taxon_name=taxon_name, provenance=provenance: CropAnnotation(
                plant_id=f'inat_animalia_{obs_id}', species=taxon_name,
                diagnosis='not_a_plant', severity='low',
                leaf_or_whole_plant='whole_plant',
                capture_session=f'inat_animalia_{kingdom_id}',
                source='inat_animalia', label_confidence=0.99,
                bbox=[0, 0, 0, 0], **provenance)
            jobs.append((url, fp, ann))
        if len(jobs) >= target_n * 2: break
    n = _run_fetch_jobs(jobs, target_n)
    print(f'iNat Animalia: not_a_plant += {n}')

# ── Open Images V7 → not_a_plant (indoor-clutter NAP) ─────────────
# v5 bbox CSV is the canonical validation-bbox file referenced by V6/V7
# (validation annotations didn't change between revisions).
OI_CACHE = ROOT / 'cache_openimages_v7'
OI_CDN_BASE = 'https://s3.amazonaws.com/open-images-dataset/validation/'
OI_VALIDATION_BBOX_URL = 'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv'
OI_CLASS_CSV_URL = 'https://storage.googleapis.com/openimages/2018_04/class-descriptions-boxable.csv'
OI_VALIDATION_METADATA_URL = ('https://storage.googleapis.com/openimages/2018_04/'
                              'validation/validation-images-with-rotation.csv')
# Open Images photos are CC BY 2.0 by construction of the dataset; the licence
# key is recorded per row as provenance, never gated on.
_OI_LICENSE_TALLY = {}

# MIDs resolved at runtime from class-descriptions-boxable.csv.
OI_INDOOR_LABELS = {
    'Couch', 'Vase', 'Lamp', 'Bookcase', 'Bed', 'Refrigerator',
    'Coffee table', 'Television', 'Pillow', 'Curtain', 'Toilet',
    'Sink', 'Furniture', 'Microwave oven', 'Oven', 'Sofa bed',
    'Chair', 'Desk', 'Cabinetry', 'Wardrobe', 'Nightstand',
}
OI_PLANT_LABELS = {
    'Houseplant', 'Plant', 'Flower', 'Tree', 'Leaf', 'Fruit',
    'Flowerpot',
}

def _oi_load_label_map():
    """LabelName MID → human-readable class name."""
    csv_local = OI_CACHE / 'class-descriptions-boxable.csv'
    if not csv_local.exists():
        OI_CACHE.mkdir(parents=True, exist_ok=True)
        http_download(OI_CLASS_CSV_URL, csv_local)
    mid_to_name = {}
    with csv_local.open() as fh:
        for line in fh:
            parts = line.rstrip('\n').split(',', 1)
            if len(parts) == 2:
                mid_to_name[parts[0]] = parts[1]
    return mid_to_name

def _oi_license_key(url):
    return re.sub(r'^https?://', '', str(url or '').strip().lower()).rstrip('/')

def _oi_load_image_metadata():
    """Validation ImageID -> rights/attribution metadata (record-only)."""
    import csv
    local = OI_CACHE / 'validation-images-with-rotation.csv'
    if not local.exists():
        http_download(OI_VALIDATION_METADATA_URL, local)
    out = {}
    with local.open(newline='', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        required = {'ImageID', 'License', 'OriginalLandingURL', 'Author'}
        if not reader.fieldnames or required - set(reader.fieldnames):
            raise RuntimeError(
                f'Open Images metadata missing columns {sorted(required - set(reader.fieldnames or []))}')
        for row in reader:
            key = _oi_license_key(row.get('License')) or 'missing'
            _OI_LICENSE_TALLY[key] = _OI_LICENSE_TALLY.get(key, 0) + 1
            image_id = (row.get('ImageID') or '').strip()
            if not image_id or image_id in out:
                raise RuntimeError(f'Open Images metadata has empty/duplicate ImageID {image_id!r}')
            out[image_id] = row
    if not out:
        raise RuntimeError('Open Images metadata is empty')
    return out

def fetch_openimages_indoor(target_n=600):
    """NAP supplement: indoor-clutter images from Open Images V7 validation."""
    OI_CACHE.mkdir(parents=True, exist_ok=True)
    bbox_csv = OI_CACHE / 'validation-annotations-bbox.csv'
    if not bbox_csv.exists():
        print('Downloading Open Images V5 validation bbox CSV (~25 MB)…')
        http_download(OI_VALIDATION_BBOX_URL, bbox_csv)
    mid_to_name = _oi_load_label_map()
    indoor_mids = {m for m, n in mid_to_name.items() if n in OI_INDOOR_LABELS}
    plant_mids  = {m for m, n in mid_to_name.items() if n in OI_PLANT_LABELS}
    if not indoor_mids:
        print('Open Images: no indoor labels resolved; skipping NAP supplement')
        return
    per_image = {}  # image_id → set(label_mid)
    with bbox_csv.open() as fh:
        fh.readline()  # header
        for line in fh:
            parts = line.split(',', 3)
            if len(parts) < 3: continue
            per_image.setdefault(parts[0], set()).add(parts[2])
    eligible = [img for img, labs in per_image.items()
                if labs & indoor_mids and not (labs & plant_mids)]
    image_meta = _oi_load_image_metadata()
    _missing_meta = sum(1 for img in eligible if img not in image_meta)
    if _missing_meta:
        print(f'  Open Images: {_missing_meta} eligible rows have no metadata row '
              '(kept; licence recorded as unknown)')
    rng = random.Random(SEED + 11); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    # Oversample 2× so the pool has slack for misses (404s, decode failures,
    # files already present) and still hits target_n.
    jobs = []
    for img_id in eligible[:target_n * 2]:
        fp = dest / f'oi_{img_id}.jpg'
        meta = image_meta.get(img_id) or {}
        ann = lambda img_id=img_id, meta=meta: CropAnnotation(
            plant_id=f'oi_{img_id}', species=None,
            diagnosis='not_a_plant', severity='low',
            leaf_or_whole_plant='whole_plant',
            capture_session='openimages_v7_validation',
            source='openimages_indoor', label_confidence=0.99,
            bbox=[0, 0, 0, 0],
            license_id=('cc-by-2.0' if _oi_license_key(meta.get('License')) ==
                        'creativecommons.org/licenses/by/2.0'
                        else (_oi_license_key(meta.get('License')) or 'unknown')),
            license_url=meta.get('License'), creator=(meta.get('Author') or None),
            source_url=(meta.get('OriginalLandingURL') or None),
            source_item_id=img_id,
            attribution=((meta.get('Author') or '').strip() + ' / CC BY 2.0').strip(' /'))
        jobs.append((OI_CDN_BASE + img_id + '.jpg', fp, ann))
    n = _run_fetch_jobs(jobs, target_n)
    print(f'Open Images indoor: not_a_plant += {n}')

# ── iNat → physiological classes ───────────────────────────────
# Free-text iNat queries for damage-related observations. Noisy (no curated
# annotation for "chlorosis" — iNat's Leaf annotations top out at
# `Coloured Leaves`). per_class is set high (400) because free-text recall
# is low and the round-robin claim below splits the deduped pool evenly
# across the two physiological classes. label_confidence is set low so
# label-smoothing dilutes the residual query noise.
# The queries themselves are constrained to iconic_taxa='Plantae' and an
# operator review blocklist (below) at fetch time, so non-plant matches
# and hand-rejected observations never enter the pool in the first place.
# Stage 0 (report): broaden the in-domain free-text supply for the weak
# classes so BOTH their train and their (ratio-fixed) eval slices grow —
# the eval CIs were +/-0.2 at n~=19-28, too wide to certify a change. More
# DISTINCT query terms is the lever that adds distinct observations (raising
# per_class alone did not: the v2.3.x runs were supply-, not cap-, bound).
INAT_PHYSIOLOGICAL_QUERIES = {
    'leaf_scorch_underwatering': [
        # Pruned 6 free-text queries that returned 0 research-grade+photo
        # observations on two live checks (2026-06): 'underwatered houseplant',
        # 'leaf margin necrosis drought', 'brown leaf tips houseplant',
        # 'wilting houseplant', 'crispy leaf edges indoor', 'leaf scorch indoor
        # plant'. They only spent API calls; removing them loses no images.
        #
        # v2.5.2 — TIGHTENED FOR PRECISION. The shipped model over-triggers
        # leaf_scorch (val_realworld precision 0.523, the worst class): it had
        # become a catch-all for any brown/stressed leaf. The cause was this
        # list mixing morphologies that are NOT under-watering scorch, taught
        # at the noisiest label_confidence (0.5) and the highest sampler weight
        # (inat_physiological=1.2). Removed:
        #   'sunburn leaf'        -> bleached/tan sun-exposed patches; a DISTINCT
        #                            morphology, and the job of the deferred
        #                            `sunburn_or_scorch` class, not this one.
        #   'wilting leaves drought', 'leaf curl drought'
        #                         -> turgor loss / curl: droopy-but-green or
        #                            rolled leaves, visually unlike brown crispy
        #                            scorch (and 'wilt'/'turgor' are already
        #                            DROPPED on the DoctorP side for the same
        #                            reason). Wilting overlaps root-rot/over-water.
        #   'leaf edge browning', 'leaf browning underwatering'
        #                         -> generic 'browning' that also fits nutrient
        #                            deficiency (K/marginal), salt burn and normal
        #                            senescence — the dominant cross-class noise.
        # Kept are the tight desiccation/scorch terms (crispy brown margins/tips,
        # heat scorch = same morphology). 11 terms remain (>= the supply-breadth
        # floor). The in-domain supply that ACTUALLY anchors this class —
        # DoctorP Burn/Marginal-necrosis/Leaves-scorch + Spider-plant leaf-tip
        # necrosis + Hibiscus scorch — is unchanged.
        'leaf scorch',
        'leaf tip burn', 'crispy brown leaf edges',
        'dehydrated plant leaves', 'water stress leaf',
        'scorched leaves heat', 'drought stressed plant',
        'desiccated leaves',
        'heat stress foliage',
        'parched plant leaves',
        'shrivelled leaves drought',
    ],
    'nutrient_deficiency': [
        # v2.4: 'yellowing leaves plant' removed — it returned ~9.7k obs that
        # are a generic-yellowing catch-all (senescence / overwatering / light),
        # the dominant label-noise source for nutrient_deficiency.
        # v2.5 pruned 10 free-text queries that returned 0 research-grade+photo
        # observations on two live checks (2026-06) -- iNat free-text doesn't
        # carry those phrasings -- plus 'marginal leaf yellowing', which a live
        # check measured at ~91% of this yield and is generic-yellowing noise.
        # The deficiency signal comes from MPLD + DoctorP; the residual iNat
        # free-text is TRAIN-ONLY (inat_physiological is in no eval/cal split),
        # so DoctorP's higher-confidence rows anchor the held-out slices.
        'chlorosis leaf', 'iron chlorosis',
        'nitrogen deficiency leaf', 'magnesium deficiency leaf',
        'interveinal chlorosis',
        'leaf yellowing nitrogen',
        'pale young leaves nutrient',
        'sulphur deficiency leaf',
        'purple leaves phosphorus',
        'general leaf yellowing deficiency',
    ],
}

def _inat_freetext_observations(query, per_query, iconic_taxa=None):
    """Wrap the existing iNat layer for q= free-text search.

    ``iconic_taxa`` (e.g. ``'Plantae'``) restricts results to a top-level
    iNat group. It is passed for the physiological queries: a free-text
    search for ``'leaf scorch'`` / ``'chlorosis'`` otherwise returns
    insects, fungi and birds whose common names share those words, which
    is the dominant label-noise source for those classes. It is NOT passed
    for the gray-mold queries, whose observations are legitimately filed
    under the fungus taxon (iconic_taxon=Fungi)."""
    base = INAT_BASE + '/observations'
    params = {
        'q': query, 'quality_grade': 'research',
        'per_page': 100, 'photos': 'true', 'order': 'desc',
        'order_by': 'created_at',
    }
    if iconic_taxa:
        params['iconic_taxa'] = iconic_taxa
    page, out = 1, []
    while len(out) < per_query and page <= 5:
        params['page'] = page
        try:
            data = http_get_json(base, params=params)
        except Exception as e:
            print(f'  inat_freetext({query!r}) failed: {e}')
            break
        results = (data.get('results') or [])
        if not results: break
        out.extend(results)
        page += 1
    return out[:per_query]

# Free-text gray-mold supplement. Live checks (2026-06) show this path is
# nearly empty — only 3 of the 12 original queries return anything and the
# whole list yields ~4 observations — so 9 zero-yield queries were pruned.
# gray_mold_botrytis is carried by DoctorP (~96) + the iNat genus taxon
# (~138 foliage), NOT this free-text path; the 3 survivors are kept only as
# a marginal top-up.
GRAY_MOLD_FREETEXT_QUERIES = [
    'gray mold botrytis', 'botrytis leaf rot', 'botrytis grey mould',
]
def _fetch_inat_freetext_class(cls, queries, source, *, per_query=120,
                               label_confidence=0.5, severity='unknown',
                               leaf_or_whole='leaf'):
    """Free-text iNat search -> save photos under ``cls`` with the given
    ``source`` tag. Tops up supply-starved classes (e.g. gray mold, which
    has no PlantDoc / PlantVillage label and is iNat-taxon-capped) beyond
    what the taxon-id query alone returns."""
    dest = TRAIN_ROOT / cls; dest.mkdir(parents=True, exist_ok=True)
    saved = 0
    for q in queries:
        obs = _inat_freetext_observations(q, per_query)
        jobs = []
        for ob in obs:
            for ph in (ob.get('photos') or [])[:1]:
                provenance = _inat_photo_provenance(ob, ph)
                if provenance is None:
                    continue
                url = _photo_url(ph, 'medium')
                if not url:
                    continue
                fp = dest / f'inat_ft_{cls}_{ob["id"]}_{ph["id"]}.jpg'
                obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
                ann = lambda obs_id=obs_id, taxon_name=taxon_name, q=q, provenance=provenance: CropAnnotation(
                    plant_id=f'inat_ft_{obs_id}', species=taxon_name,
                    diagnosis=cls, severity=severity, leaf_or_whole_plant=leaf_or_whole,
                    capture_session=f'inat_ft_{q.replace(" ", "_")}',
                    source=source, label_confidence=label_confidence, bbox=[0, 0, 0, 0],
                    **provenance)
                jobs.append((url, fp, ann))
        saved += _run_fetch_jobs(jobs, len(jobs))
    return saved

def _save_inat_phys_observation(cls, q, ob, seen_obs) -> bool:
    """Try to save one iNat free-text observation under ``cls``. Returns
    True if a photo landed (and the obs is now claimed in ``seen_obs``),
    False otherwise. Failed photo fetches do NOT add to ``seen_obs`` so
    the sibling class can still try the same obs."""
    obs_id = ob['id']
    for ph in (ob.get('photos') or []):
        provenance = _inat_photo_provenance(ob, ph)
        if provenance is None:
            continue
        url = _photo_url(ph, 'medium')
        if not url:
            continue
        fp = TRAIN_ROOT / cls / f'inat_phys_{cls}_{obs_id}_{ph["id"]}.jpg'
        if fp.exists():
            seen_obs.add(obs_id)
            return True
        body = http_get_bytes(url)
        if not body or not _claim_sha(body):
            continue
        img = _valid_image_bytes(body)
        if img is None:
            continue
        _save_rgb(img, fp)
        write_sidecar(fp, CropAnnotation(
            plant_id=f'inat_phys_{obs_id}',
            species=(ob.get('taxon') or {}).get('name'),
            diagnosis=cls, severity='medium',
            leaf_or_whole_plant='leaf',
            capture_session=f'inat_phys_{q.replace(" ", "_")}',
            source='inat_physiological',
            label_confidence=0.55, bbox=[0, 0, 0, 0],
            **provenance))
        seen_obs.add(obs_id)
        return True
    return False

def fetch_inat_physiological(per_class=400):
    """Free-text iNat queries → physiological classes.

    Cross-class observation dedup is fairness-critical: a
    chlorosis-and-scorch photo can't land in both classes (the label
    signal would be contradictory). The two classes share several visual
    modes (yellowing leaves, generalised wilting), so first-come-first-
    served on the dict-insertion order silently privileges
    ``leaf_scorch_underwatering`` and starves ``nutrient_deficiency``
    of ~20% of its supply.

    Fix: fetch the candidate observation pools for every class up front,
    then claim observations in **round-robin** order. Each cycle every
    still-hungry class gets exactly one new observation (skipping any
    already claimed by a sibling), so both classes converge toward
    their target at the same rate. The green-content gate is
    intentionally NOT applied here — its HSV mask rejects pale-yellow
    chlorosis and brown leaf scorch as readily as it rejects agar
    plates.
    """
    targets: dict[str, int] = {}
    for cls in INAT_PHYSIOLOGICAL_QUERIES:
        dest = TRAIN_ROOT / cls
        dest.mkdir(parents=True, exist_ok=True)
        n_have = sum(1 for _ in iter_images(dest))
        if n_have >= per_class:
            print(f'iNat physiological ({cls}): already have {n_have}, skipping')
            continue
        targets[cls] = per_class - n_have
    if not targets:
        return

    pools: dict[str, list] = {}
    for cls, target in targets.items():
        queries = INAT_PHYSIOLOGICAL_QUERIES[cls]
        per_query = max(1, target // len(queries))
        pool = []
        for q in queries:
            pool.extend(
                (q, ob)
                for ob in _inat_freetext_observations(q, per_query, iconic_taxa='Plantae')
            )
        pools[cls] = pool

    seen_obs: set = set()
    cursors = {cls: 0 for cls in targets}
    saved = {cls: 0 for cls in targets}
    done: set = set()
    while len(done) < len(targets):
        for cls in list(targets):
            if cls in done:
                continue
            if saved[cls] >= targets[cls]:
                done.add(cls); continue
            advanced = False
            while cursors[cls] < len(pools[cls]):
                q, ob = pools[cls][cursors[cls]]
                cursors[cls] += 1
                if ob['id'] in seen_obs:
                    continue
                if _save_inat_phys_observation(cls, q, ob, seen_obs):
                    saved[cls] += 1
                    advanced = True
                    break
            if not advanced:
                done.add(cls)

    for cls, n in saved.items():
        print(f'iNat physiological ({cls}): += {n}')

# ── Curated abiotic-stress supplement: MPLD (report Stage 2) ─────
# The weak physiological classes are otherwise fed by noisy iNat free-text
# alone. A curated crop-domain nutrient-deficiency set teaches the deficiency
# SIGNAL (interveinal chlorosis, marginal necrosis) at a HIGH label_confidence
# the iNat free-text supply can't. Domain scoping (so the eval distribution is
# never corrupted): it is a CROP, not a houseplant, so its images land in
# TRAIN_ROOT only, are excluded from every eval/cal split (no make_*_split
# allowed_sources lists the crop source), and §4 down-weights it in the
# sampler — exactly how PlantVillage studio imagery is scoped.
# (CoLeaf-DB was retired: Mendeley/Cloudflare 403s datacenter IPs incl. Colab,
# so it never downloaded from Colab; MPLD is the active crop-nutrient set.)
INCLUDE_ABIOTIC_SUPPLEMENTS = True
ABIOTIC_CACHE = ROOT / '_abiotic_cache'   # under data_v2/ (gitignored);
# survives REBUILD_GENERATED_SPLITS (it only wipes train/ + val/), so the
# zips are not re-downloaded on every split rebuild.

def _save_archive_member(target_cls, source, name_prefix, name_in_zip, body, *,
                         label_confidence, species, severity='medium', min_side=160,
                         subject_domain=None, license_id=None, license_url=None,
                         creator=None, source_url=None, attribution=None,
                         immutable_source_id=None, capture_family_id=None,
                         first_seen_model_version=None,
                         output_stem=None, exif_transpose=False):
    """Validate one in-archive image's bytes -> save under target_cls with
    a sidecar, sharing the cross-source SHA dedup. ``name_prefix`` makes the
    on-disk stem unique across archives (two zips can hold a '1.jpg');
    ``source`` is the sampler/weight tag. ``subject_domain`` is recorded
    verbatim (no allowlist) so the split reports below can measure the
    broadened deployment domain. Returns 1 on save else 0."""
    stem = re.sub(r'[^A-Za-z0-9_.-]', '_',
                  output_stem or f'{name_prefix}_{Path(name_in_zip).stem}')
    fp = TRAIN_ROOT / target_cls / f'{stem}.jpg'
    if fp.exists():
        return 0
    if not _claim_sha(body):
        return 0
    if exif_transpose:
        # A row that ships detector boxes was labelled in a browser tool, and
        # browsers show the EXIF-ROTATED frame: store that frame so the boxes
        # fit it. It is also the frame the server classifies (main.py applies
        # exif_transpose to every upload).
        try:
            from PIL import ImageOps as _ImageOps
            img = _ImageOps.exif_transpose(PILImage.open(BytesIO(body))).convert('RGB')
            img = img if min(img.size) >= min_side else None
        except Exception:
            img = None
    else:
        img = _valid_image_bytes(body, min_side=min_side)
    if img is None:
        return 0
    _save_rgb(img, fp)
    write_sidecar(fp, CropAnnotation(
        plant_id=capture_family_id or stem, species=species,
        diagnosis=target_cls, severity=severity, leaf_or_whole_plant='leaf',
        capture_session=(f'{source}:{capture_family_id}' if capture_family_id else source),
        source=source,
        label_confidence=label_confidence, bbox=[0, 0, 0, 0],
        subject_domain=subject_domain, license_id=license_id, license_url=license_url,
        creator=creator, source_url=source_url,
        source_item_id=stem, attribution=attribution,
        immutable_source_id=(immutable_source_id or
                             f'sha256:{hashlib.sha256(body).hexdigest()}'),
        capture_family_id=capture_family_id,
        first_seen_model_version=first_seen_model_version))
    return 1

# ── Curated abiotic-stress supplement #2: MPLD (report Stage 2) ──
# A SECOND, species-diverse nutrient-deficiency source so the deficiency
# signal is not tied to coffee morphology alone. MPLD (Maize Phosphorus
# Leaf Deficiency Database, 224×224; Zenodo 10041514; CC BY 4.0) is an
# expert-controlled maize phosphorus set shot with five cameras under
# three regimes. Only the deficiency folders are imported:
#   '-P'   = full phosphorus starvation        -> nutrient_deficiency
#   '-P50' = half-dose (partial) deficiency     -> nutrient_deficiency
#   '_C'   = control (fully fed = HEALTHY)       -> EXCLUDED
# Dropping '_C' keeps single-nutrient zips only (no 'healthy'/'mixed'
# members), so a fully-fed leaf is never mislabelled deficient. Same
# domain scoping: maize is a CROP, so rows are TRAIN-only (no
# make_*_split allowed_sources lists 'mpld') and §4 down-weights the
# 'mpld' source (0.5) so the crop domain can't pull the LA prior off the
# houseplant target. P-deficiency adds purpling + interveinal chlorosis,
# a complementary nutrient-deficiency signature (chlorosis/necrosis).
_MPLD_URL = 'https://zenodo.org/api/records/10041514/files/MPLD_databse.zip/content'
MPLD_DEFICIENCY_DIRS = ('-P', '-P50')   # '_C' (control / healthy) is excluded

def fetch_mpld_phosphorus_deficiency(cap=700, label_confidence=0.90):
    """MPLD maize phosphorus-deficiency zip -> nutrient_deficiency. Streams
    members straight from the archive (no full extract), importing ONLY the
    '-P' / '-P50' deficiency folders and skipping the '_C' control so a
    fully-fed leaf is never labelled deficient. Shuffles across the five
    cameras + two deficiency levels and caps the total for a balanced,
    bandwidth-bounded subset; SHA-dedups against every other source. Fails
    soft (via _try_fetch) if Zenodo is unreachable -- iNat physiological
    still meets the per-class floor, so an outage neither blocks the
    run nor corrupts the class."""
    ABIOTIC_CACHE.mkdir(parents=True, exist_ok=True)
    dest = TRAIN_ROOT / 'nutrient_deficiency'
    dest.mkdir(parents=True, exist_ok=True)
    archive = ABIOTIC_CACHE / 'mpld.zip'
    try:
        http_download(_MPLD_URL, archive)
    except Exception as e:
        print(f'  MPLD: download failed ({type(e).__name__}: {e}); skipping')
        return 0
    saved = 0
    try:
        with zipfile.ZipFile(archive) as z:
            members = []
            for m in z.namelist():
                if m.endswith('/') or Path(m).suffix.lower() not in IMG_EXTS:
                    continue
                # path layout: MPLD_databse/<camera>/<level>/<file>
                parts = m.split('/')
                if len(parts) >= 4 and parts[-2] in MPLD_DEFICIENCY_DIRS:
                    members.append(m)
            random.Random(SEED).shuffle(members)
            for m in members:
                if saved >= cap:
                    break
                try:
                    body = z.read(m)
                except Exception:
                    continue
                parts = m.split('/')
                camera, level = parts[-3], parts[-2]
                saved += _save_archive_member(
                    'nutrient_deficiency', 'mpld', f'mpld_{camera}_{level}', m, body,
                    label_confidence=label_confidence,
                    species=f'Zea mays (MPLD {level} phosphorus deficiency)')
    except zipfile.BadZipFile:
        print('  MPLD: corrupt archive; clearing cache + skipping')
        archive.unlink(missing_ok=True)
        return 0
    print(f'MPLD nutrient_deficiency total: += {saved}')
    return saved

# ── Kaggle houseplant-disease supplement (operator opt-in) ───────
# Real INDOOR houseplant/succulent disease photos -- the in-domain supply
# the keyless HF/iNat sources under-provide. Requires Kaggle credentials in
# the environment (KAGGLE_USERNAME + KAGGLE_KEY, or ~/.kaggle/kaggle.json);
# FAIL-SOFT when absent so a keyless run is unaffected. TRAIN-ONLY: the
# 'kaggle_houseplant' source is in NO make_*_split allowed_sources, so it
# never enters an eval/calibration slice. It is moderately down-weighted in
# the sampler (the set may be partly studio-framed; unverified until a run
# prints its structure) and its LA prior is decoupled lower, exactly like
# the MPLD crop supplement.
INCLUDE_KAGGLE_SUPPLEMENTS = True
# Free the kagglehub download cache (~4 GB extracted) once every Kaggle source
# has copied its images into TRAIN_ROOT. Purging buys disk headroom but costs a
# full ~5 GB re-download on the next run, so it is conditional rather than
# unconditional: 'auto' purges only when free space is actually tight. Force
# with True/False.
PURGE_KAGGLE_CACHE = 'auto'
PURGE_KAGGLE_CACHE_MIN_FREE_GB = 25
KAGGLE_HOUSEPLANT_SLUG = 'abdulahad0296/indoor-plant-disease-detection-dataset'
# Best-effort folder-name -> our-class map (keys are _pv_normalise'd). UNMAPPED
# folders are DROPPED (never guessed into a class); every discovered folder +
# count + resolved mapping is printed so the operator can refine this map
# against the first run before trusting the import.
# Folder names here are <Species>_<Condition> (e.g. 'Aloe_LeafSpot',
# 'Cactus_Dactylopius_Opuntia'), so we map on the CONDITION keyword inside the
# normalised folder name, not the whole string. A DROP list is checked FIRST so
# a condition outside our taxonomy is never force-mapped by a looser keyword:
#   * rust           -> different morphology (we drop rust everywhere, cf. PlantDoc)
#   * *toxicity*     -> a nutrient TOXICITY is the OPPOSITE signal to a deficiency
#   * bacterial wilt -> a vascular/systemic disease, not a leaf spot
# Anything still unmatched is dropped (never guessed); every folder + count +
# resolved class is printed so the mapping stays auditable per run.
KAGGLE_DROP_KEYWORDS = ('rust', 'toxicity', 'bacterial_wilt', 'wilt_disease',
                        'root_rot', 'edema', 'oedema')
# (keyword, our_class), checked in order; first substring hit wins. sunburn /
# heat scorch / leaf-tip necrosis map to leaf_scorch_underwatering (consistent
# with how that class is already fed by iNat 'sunburn leaf' / 'scorched leaves
# heat' / 'leaf tip burn'). 'withering' is deliberately NOT mapped: on snake
# plant etc. it is usually over-watering / root rot, not under-watering scorch,
# and the scorch class is the weakest -- purity beats the extra (redundant)
# volume. anthracnose is a fungal leaf-spot morphology (cf. the PlantDoc map);
# Dactylopius (cochineal) is a scale-like pest.
KAGGLE_CONDITION_KEYWORDS = [
    ('healthy', 'healthy'),
    ('anthracnose', 'leaf_spot'), ('leaf_spot', 'leaf_spot'), ('leafspot', 'leaf_spot'),
    ('bacterial_spot', 'leaf_spot'), ('fungal', 'leaf_spot'),
    # v2.4: 'dactylopius'/'cochineal' removed — the only folder they matched
    # (Cactus_Dactylopius_Opuntia) is OUTDOOR prickly-pear field shots, not the
    # indoor mealybug/scale close-up the 'pests' class targets.
    ('mealybug', 'pests'), ('mealy', 'pests'), ('scale', 'pests'),
    # v2.4: 'sunburn' removed — the only match (Aloe_Sunburn) is black-bg
    # segmented + augmented and ~indistinguishable from Aloe_Healthy.
    ('scorch', 'leaf_scorch_underwatering'),
    ('tip_necrosis', 'leaf_scorch_underwatering'),
    ('underwater', 'leaf_scorch_underwatering'), ('under_watering', 'leaf_scorch_underwatering'),
    ('chlorosis', 'nutrient_deficiency'), ('deficien', 'nutrient_deficiency'),
    ('powdery', 'powdery_mildew'),
    ('botrytis', 'gray_mold_botrytis'), ('gray_mold', 'gray_mold_botrytis'),
    ('grey_mold', 'gray_mold_botrytis'), ('gray_mould', 'gray_mold_botrytis'),
    ('mosaic', 'viral_disease'), ('virus', 'viral_disease'),
]

def _kaggle_map_folder(folder_norm):
    """Resolve a normalised <species>_<condition> folder name to one of CLASSES,
    or None. DROP keywords win first; then the first matching condition keyword;
    anything unmatched (or mapped to a class not in the taxonomy) returns None."""
    if any(k in folder_norm for k in KAGGLE_DROP_KEYWORDS):
        return None
    for kw, cls in KAGGLE_CONDITION_KEYWORDS:
        if kw in folder_norm and cls in CLASSES:
            return cls
    return None

# Kaggle selection policy: download the latest revision visible to the supplied
# credentials. This is deliberately operator-selected mutable-latest behavior.
# The resolved numeric revision is still captured from kagglehub's cache path and
# bound into the run/data manifests, so a completed run is auditable even though
# its revision was not predetermined. Strict source schema, rights, provenance,
# source floors and per-image corpus hashing still fail closed after download.
KAGGLE_REVISION_SELECTION_POLICY = 'latest_at_download_time'
_KAGGLE_RESOLVED_REVISIONS = {}

def _kaggle_download(slug):
    """Download a Kaggle dataset via kagglehub; return the local root Path or
    None. Fail-soft: missing package / credentials / network each return None
    with a printed reason so a keyless run is wholly unaffected.

    NOT fail-soft about contamination: a benchmark-poisoning slug raises here,
    at the single choke point every Kaggle source passes through. It was
    previously checked only inside `fetch_kaggle_ornamental_disease`, which left
    five other call sites (houseplant, species, MPLD, DoctorP, the supplement)
    able to pull the benchmark in with no check at all — the precise mistake the
    deny-list exists to make impossible rather than merely unlikely."""
    import os
    _assert_source_allowed(slug)
    if not INCLUDE_KAGGLE_SUPPLEMENTS:
        return None
    try:
        import kagglehub
    except Exception:
        try:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'],
                           check=True)
            import kagglehub
        except Exception as e:
            print(f'  Kaggle: kagglehub unavailable ({type(e).__name__}); skipping')
            return None
    # Pull a Colab secret (KAGGLE_API_TOKEN) into the env first -- Colab
    # secrets are not auto-exported, so without this a set secret looks
    # like 'no credentials'. Accept BOTH the new-format API token (env
    # KAGGLE_API_TOKEN or ~/.kaggle/access_token) and legacy username+key.
    _kaggle_load_colab_config()
    cfg = Path.home() / '.kaggle'
    has_creds = bool(
        os.environ.get('KAGGLE_API_TOKEN')                                     # new (env)
        or (cfg / 'access_token').exists()                                     # new (file)
        or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))  # legacy env
        or (cfg / 'kaggle.json').exists()                                      # legacy file
    )
    if not has_creds:
        print('  Kaggle: no credentials. Add a Colab secret KAGGLE_API_TOKEN '
              '(with Notebook access ON) or env KAGGLE_API_TOKEN, or run '
              'kaggle_install_api_config(); legacy KAGGLE_USERNAME+KAGGLE_KEY / '
              'kaggle.json also work. Skipping the Kaggle supplement')
        return None
    try:
        _root = Path(kagglehub.dataset_download(slug))
        # kagglehub cache roots end in .../versions/<integer>. Preserve that
        # stable dataset revision without leaking an ephemeral absolute path.
        _parts = list(_root.parts); _resolved_version = None
        if 'versions' in _parts:
            _vi = len(_parts) - 1 - _parts[::-1].index('versions')
            if _vi + 1 < len(_parts):
                _resolved_version = _parts[_vi + 1]
        _KAGGLE_RESOLVED_REVISIONS[slug] = {
            'dataset_slug': slug, 'requested_handle': slug,
            'selection_policy': KAGGLE_REVISION_SELECTION_POLICY,
            'kaggle_version': _resolved_version}
        return _root
    except Exception as e:
        print(f'  Kaggle latest-version download failed for {slug} '
              f'({type(e).__name__}: {e}); skipping')
        return None

def _is_augmented_filename(path) -> bool:
    """True for a baked-in augmentation copy (synthetic rotation/flip/jitter),
    as opposed to an original capture. The big Kaggle disease sets ship these
    with an ``Augmented_*`` / ``*_aug*`` filename next to a handful of base
    photos — e.g. abdulahad0296's ``Aloe_Anthracnose`` is 54 originals vs 712
    augmented (measured 2026-06). 'augment' as a substring is specific enough
    that no real plant-photo filename trips it."""
    return 'augment' in Path(path).name.lower()

def _stratified_folder_order(items, seed=SEED):
    """Round-robin index order across source folders so one big folder can't
    dominate a thin class. ``items`` is a list of ``(path, folder_norm)``; we
    bucket indices by ``folder_norm``, ORDER ORIGINAL CAPTURES BEFORE BAKED-IN
    AUGMENTATIONS within each bucket (deterministic shuffle inside each group),
    then interleave one index per folder per cycle. The per-class save loops
    stop at their cap, so this makes the documented 'diverse cross-folder
    sample' actually hold — plain random.sample drew folders PROPORTIONALLY,
    letting e.g. a 3k-image Rose_Black_Spot folder swamp a 500-image sibling
    in the same class pool (or Aloe swamp the other species in healthy).

    Originals-first matters as much as cross-folder: the Kaggle disease sets are
    ~85% synthetic augmentations of a few dozen base images, so a proportional
    draw spent the per-class cap on near-duplicate rotations the §4 training
    augmentation ALREADY produces — burning unique-capture diversity for nothing.
    Serving originals first (augmented only as fallback once originals run out)
    maximises distinct real photos at the SAME cap. No-op for un-augmented
    sources (DoctorP, rice nutrient): every file lands in the 'original' group.
    Single-folder pools still get a (now originals-first) deterministic order."""
    from collections import defaultdict, deque
    buckets = defaultdict(list)
    for i, (_p, norm) in enumerate(items):
        buckets[norm].append(i)
    rng = random.Random(seed)
    queues = []
    for norm in sorted(buckets):            # deterministic folder order
        idxs = buckets[norm]
        orig = [i for i in idxs if not _is_augmented_filename(items[i][0])]
        augd = [i for i in idxs if _is_augmented_filename(items[i][0])]
        rng.shuffle(orig); rng.shuffle(augd)
        queues.append(deque(orig + augd))   # originals drained before augmented
    order = []
    while queues:
        nxt = []
        for q in queues:
            order.append(q.popleft())
            if q:
                nxt.append(q)
        queues = nxt
    return order


def _images_by_relative_parent(root):
    """Group images by their complete source-relative parent path.

    Leaf basenames are not identities: separate branches routinely contain
    folders named ``healthy``, ``images`` or ``train``. Merging those buckets
    can transfer one branch's mapping/deny verdict to another and defeats
    folder-stratified sampling. POSIX keys are portable and deterministic."""
    out = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rel_parent = p.parent.relative_to(root).as_posix()
            out.setdefault(rel_parent, []).append(p)
    return out


def _is_black_bg_segmented(body, thr=18, frac=0.5):
    """True if > ``frac`` of pixels are near-black (max channel <= ``thr``) — the
    signature of a leaf MASK-SEGMENTED onto a black background. Such images teach
    a background shortcut and are usually augmented duplicates, so we reject them
    from the Kaggle houseplant set. Measured: real-photo folders score ~0.00-0.03
    black-frac, segmented Aloe ~0.78-0.83, so 0.5 separates them cleanly."""
    try:
        im = PILImage.open(BytesIO(body)).convert('RGB')
        im.thumbnail((96, 96))
        a = np.asarray(im).astype(np.int16)
        return float((a.max(axis=2) <= thr).mean()) > frac
    except Exception:
        return False

def fetch_kaggle_houseplant_disease(slug=KAGGLE_HOUSEPLANT_SLUG, per_class_cap=250,
                                    label_confidence=0.70):
    """Real indoor houseplant/succulent photos from Kaggle -> TRAIN. Maps each
    <Species>_<Condition> folder to our taxonomy by CONDITION keyword (dropping
    unmapped/ambiguous ones), then for each class draws a DIVERSE cross-folder
    sample up to per_class_cap (so one Aloe-heavy folder can't dominate a thin
    class), SHA-dedups against every other source, and prints the per-folder
    mapping + the per-class totals. label_confidence is moderate because the set
    is partly augmented. TRAIN-ONLY (source 'kaggle_houseplant'); fail-soft."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    pools = {}   # our_class -> list[(path, folder_norm)]
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    by_folder = _images_by_relative_parent(root)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    for rel_folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(rel_folder)
        mapped = _kaggle_map_folder(norm)
        print(f'    {rel_folder!r:42s} n={len(files):5d} -> {mapped or "DROP"}')
        if mapped is not None:
            pools.setdefault(mapped, []).extend((f, norm) for f in files)
    saved_per_class = {}
    for cls, items in pools.items():
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        order = _stratified_folder_order(items)
        got = 0
        for idx in order:
            if got >= per_class_cap:
                break
            path, norm = items[idx]
            try:
                body = path.read_bytes()
            except Exception:
                continue
            if _is_black_bg_segmented(body):   # v2.4: drop mask-segmented
                continue                        # (black-bg) leaves
            got += _save_archive_member(
                cls, 'kaggle_houseplant', f'kaggle_{norm}', path.name, body,
                label_confidence=label_confidence,
                species=f'houseplant (Kaggle {slug.split("/")[-1]})')
        saved_per_class[cls] = got
    total = sum(saved_per_class.values())
    print(f'  Kaggle houseplant per-class: {saved_per_class}')
    print(f'  Kaggle houseplant total: += {total}')
    return total

# ── Kaggle real-houseplant HEALTHY supplement (curated, not scraped) ──────
# A large, manually-curated set of REAL indoor houseplant photos across ~47
# species (kacpergregorowicz/house-plant-species). All healthy specimens, so
# every folder maps to `healthy`. Value is DOMAIN + SPECIES DIVERSITY (real
# houseplants incl. variegated species) which reduces the healthy<->disease
# false positives, not volume -- so it's capped low and down-weighted. A
# diverse cross-species sample, TRAIN-ONLY (the labels are species, not
# disease-verified, so they don't enter the eval/cal slices).
#
# RESTORED 2026-08: the 2.6.13 re-sourcing commit deleted this constant but
# kept the function below, whose signature reads it as a DEFAULT ARGUMENT --
# and defaults evaluate at `def` time, so the whole of §2 died with a
# NameError before a single image was fetched. The INCLUDE_KAGGLE_HEALTHY_
# SPECIES=False guard at the call site could not save it: the flag gates the
# CALL, not the DEF. Keep the constant next to the function it feeds.
KAGGLE_SPECIES_SLUG = 'kacpergregorowicz/house-plant-species'

def fetch_kaggle_houseplant_species(slug=KAGGLE_SPECIES_SLUG, cap=400,
                                    label_confidence=0.70):
    """Curated real-houseplant species photos -> healthy (TRAIN-ONLY). Pools
    images across all species folders, draws a DIVERSE cap-sized sample so no
    one species dominates, SHA-dedups, prints the species folders it saw.
    Fail-soft via _kaggle_download."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = _images_by_relative_parent(root)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    print(f'  Kaggle {slug}: {len(by_folder)} species folders -> healthy')
    pool = [(f, _pv_normalise(rel_folder))
            for rel_folder, files in by_folder.items() for f in files]
    (TRAIN_ROOT / 'healthy').mkdir(parents=True, exist_ok=True)
    got = 0
    for idx in _stratified_folder_order(pool):
        if got >= cap:
            break
        path, norm = pool[idx]
        try:
            body = path.read_bytes()
        except Exception:
            continue
        got += _save_archive_member(
            'healthy', 'kaggle_species', f'kaggle_sp_{norm}', path.name, body,
            label_confidence=label_confidence, species=f'houseplant ({norm})')
    print(f'  Kaggle healthy (species): += {got}')
    return got

# ── Kaggle ornamental disease supplement (v2.4; re-sourced 2026-07-24) ──────
# The garden/ornamental leg of the v2.6.12 general-plant pivot: rose, aloe vera
# and friends with REAL disease morphology on NATURAL backgrounds — the closest
# public supply to the broadened deployment domain (houseplants + outdoor/garden
# ornamentals, no field crops).
#
# 2026-07-24 — THE ORIGINAL SLUG IS GONE. `pappu54/indoor-plant-disease-dataset-
# 23-classes` (2,544 images: 600 each of healthy/leaf_spot/viral_disease/
# powdery_mildew + 77 nutrient + 67 scorch) started returning HTTP 403
# `Permission 'datasets.get' was denied` and `datasets/list?user=pappu54` now
# returns an EMPTY list — the owner deleted or unlisted it, so it is not coming
# back. It vanished silently mid-run on 2026-07-20 (fail-soft `_kaggle_download`
# + one 'skipping' line in a 900-line log), taking ~2.26k TRAIN images with it;
# see the REQUIRED_SOURCE_FLOORS gate at the end of this cell, which now makes
# exactly this failure fatal instead of silent.
#
# Replacements below are live, permissively licensed, and were folder-inventoried
# via the Kaggle API before being wired in. NOTE the honest gap: neither supplies
# `viral_disease`, which was the single biggest loss (600 images) — closing that
# needs a new ornamental mosaic-virus source or collected scans, NOT this list.
#
# TRAIN-ONLY (source 'kaggle_ornamental'): these slugs are in NO make_*_split
# allowed_sources, so they never enter an eval/cal slice. Fail-soft per slug.
# DROP list (checked first): rust (morphology we drop everywhere), *toxicity*
# (opposite of a deficiency), *wilt* (vascular/systemic), downy (downy !=
# powdery mildew), aphid (!= mealybug/scale 'pests'), blight/necrosis/blotch
# (ambiguous -> unmapped).
# ── Benchmark-contaminated sources: NEVER TRAIN ON THESE ─────────
# `server/tools/build_ood_benchmark.py` builds the only never-trained eval set
# this project has out of `shuvokumarbasak4004/flower-leaf-diseases...`. Any
# source that re-publishes those photographs is therefore eval-poison: training
# on it does not merely inflate the benchmark, it destroys the instrument, and
# nothing downstream would notice — the numbers would simply get better.
#
# `aditisalvi04/indoor-plant-disease-dataset` is the live example and the reason
# this list exists in code rather than in a doc. It looks like exactly the
# supply this corpus is missing: 49,878 files, an explicit `Hibiscus_Scorch`
# folder of 1,200, on a project whose weakest class is leaf_scorch. Sampled 60
# of those files evenly and pHashed them against the benchmark: **60/60 match at
# Hamming <= 6, median Hamming 0** — the same photographs. `Hibiscus_Healthy` is
# 20/20. Its OTHER genera are clean (Aloe_Sunburn 0/30, Money_Plant 0/20,
# Rose_Black_Spot 0/6, Turmeric_Blotch 0/15) and are genuine candidate supply —
# hence a per-folder deny-list, not a per-slug one.
#
# THE WHOLE `hibiscus` BRANCH IS DENIED, not just the two folders that matched.
# `Hibiscus_Necrosis` reads 0/30 against our 213 benchmark clusters and was
# recorded as clean supply on exactly that evidence. It is not. Its upstream is
# `flower_data/*/hibiscus/death_leaf` in the deny-listed
# `shuvokumarbasak4004/...` — 12/12 of an independent fresh sample match it at
# Hamming **0**. It read clean only because the benchmark was built from that
# source's `healthy` + `scorch` folders and left `death_leaf` out. The four
# candidate folders are one-to-one with the four upstream subclasses.
#
# The lesson generalises past this slug: **a pHash gate against the benchmark
# only detects the frames we happened to sample into it.** A re-upload of the
# benchmark's SOURCE passes that gate while carrying the same leaves, the same
# session and the same background — everything that makes the eval set
# independent. So gate new folders against the benchmark's upstream datasets
# too, not only against the 213 clusters, and deny at the branch a leaked
# folder belongs to rather than at the individual folder.
#
# ADDING A SOURCE: pHash a sample of it against the benchmark FIRST. A dataset
# that is "never trained" is not automatically usable; it also has to not be the
# thing we grade with, nor a re-upload of what we grade with.
KAGGLE_DENY_FOLDERS = {
    # substring-matched, so this covers Hibiscus_{Scorch,Healthy,Necrosis,Blight}
    'aditisalvi04/indoor-plant-disease-dataset': ('hibiscus',),
}
KAGGLE_DENY_SLUGS = {
    # This IS the benchmark. Its own official split is 100% pHash-leaked too
    # (2,800 scorch files -> 62 clusters, 61/61 test clusters also in train).
    'shuvokumarbasak4004/flower-leaf-diseases-dataset-new-and-update',
}

def _assert_source_allowed(slug, folder_norm=''):
    """Raise if (slug, folder) is benchmark-contaminated. Fail LOUD, not skip:
    a silently-dropped folder looks identical to a source that simply had no
    matching images, and this is exactly the mistake worth being unable to make
    by accident."""
    s = (slug or '').strip().lower()
    if s in KAGGLE_DENY_SLUGS:
        raise RuntimeError(
            f'BENCHMARK CONTAMINATION: {slug!r} is the source of the never-trained OOD '
            f'benchmark (server/tools/build_ood_benchmark.py). Training on it destroys '
            f'even its limited diagnostic value. The private v2 artifact is not a clean '
            f'OOD/release gate; see docs/DATASET_LABELING_PLAN.md.')
    bad = KAGGLE_DENY_FOLDERS.get(s, ())
    if folder_norm and any(b in folder_norm for b in bad):
        raise RuntimeError(
            f'BENCHMARK CONTAMINATION: {slug!r} folder {folder_norm!r} re-publishes the '
            f'benchmark or the dataset the benchmark was built from (matched at pHash '
            f'Hamming 0). Other folders of this slug are clean — exclude this one, do '
            f'not skip the source.')
    return True

KAGGLE_ORNAMENTAL_SLUGS = (
    # rose + aloe vera, <species>_<condition> folders, CC0. Usable after DROP:
    # healthy 389, leaf_spot 287 (black_spot + leafspot), powdery_mildew 133.
    'itselif/ornamental-plant-diseases-master-dataset',
    # FlowerNet rose leaves (Rajbongshi et al. 2022, Mendeley), CC BY 4.0.
    # 'Original Dataset/' only — all real captures, no augmented copies.
    # Usable after DROP: healthy 404 ('Fresh Leaf'), leaf_spot 313 ('Black
    # Spot'); 'Downy Mildew' (200) is dropped by the `downy` keyword.
    'warcoder/rose-leaves-disease-detection',
)
KAGGLE_ORN_DROP = ('rust', 'toxicity', 'wilt', 'downy', 'aphid', 'blight',
                   'necrosis', 'blotch')
KAGGLE_ORN_KEYWORDS = [
    ('healthy', 'healthy'),
    # FlowerNet names its healthy class 'Fresh Leaf'.
    ('fresh', 'healthy'),
    ('powdery', 'powdery_mildew'),
    ('mosaic', 'viral_disease'), ('virus', 'viral_disease'),
    ('black_spot', 'leaf_spot'), ('leaf_spot', 'leaf_spot'),
    ('leafspot', 'leaf_spot'), ('septoria', 'leaf_spot'),
    ('bacterial_leaf_spot', 'leaf_spot'), ('anthracnose', 'leaf_spot'),
    ('chlorosis', 'nutrient_deficiency'), ('deficien', 'nutrient_deficiency'),
    ('scorch', 'leaf_scorch_underwatering'),
]
def _orn_map_folder(folder_norm, slug='', rel_norm=''):
    """Resolve a normalised <plant>_<condition> folder to one of CLASSES or None.
    Contamination check first, then DROP keywords, then the first matching
    condition keyword.

    A contaminated FOLDER excludes that folder and nothing else — which is what
    the deny-list's own message tells you to do ('Other folders of this slug are
    clean — exclude this one, do not skip the source'). Raising here would make
    a 49,878-file dataset whose Aloe / Rose / Money-Plant / Turmeric folders are
    all clean unusable because two Hibiscus folders are not. Slug-level hits
    still raise, in `_kaggle_download`."""
    try:
        _assert_source_allowed(slug, rel_norm or folder_norm)
    except RuntimeError as e:
        print(f'    DENIED (benchmark contamination): {folder_norm!r} — {e}')
        return None
    if any(k in folder_norm for k in KAGGLE_ORN_DROP):
        return None
    for kw, cls in KAGGLE_ORN_KEYWORDS:
        if kw in folder_norm and cls in CLASSES:
            return cls
    return None

def fetch_kaggle_ornamental_disease(slugs=KAGGLE_ORNAMENTAL_SLUGS, per_class_cap=300,
                                    label_confidence=0.75):
    """Ornamental disease sets -> TRAIN. Maps each <plant>_<condition> folder to
    our taxonomy by CONDITION keyword (dropping the ambiguous / out-of-taxonomy
    ones), POOLS ACROSS SLUGS, then for each class draws a DIVERSE cross-folder
    sample to per_class_cap, SHA-dedups against every other source, and prints
    the per-folder mapping + per-class totals.

    Pooling before the cap (rather than one capped fetch per slug) is what keeps
    `per_class_cap` meaning what it says and lets `_stratified_folder_order`
    interleave ACROSS datasets — the bucket key is `<slug-tag>/<folder>`, so a
    single large rose folder can no longer swamp a thin aloe one. A slug that
    fails to download is skipped and the rest still load (fail-soft), which is
    safe here only because REQUIRED_SOURCE_FLOORS re-checks the total below.
    TRAIN-ONLY (source 'kaggle_ornamental')."""
    if isinstance(slugs, str):
        slugs = (slugs,)
    pools, seen_any = {}, False
    for slug in slugs:
        root = _kaggle_download(slug)     # slug-level deny-list runs in there
        if root is None:
            continue
        tag = slug.split('/')[-1][:24]
        # Full relative-parent keys keep same-named leaves in different
        # branches separate. Otherwise files[0] alone supplied the deny/mapping
        # verdict for a merged bucket and a denied Hibiscus branch could ride
        # behind an allowed branch named e.g. ``healthy``.
        by_folder = _images_by_relative_parent(root)
        if not by_folder:
            print(f'  Kaggle {slug}: no images found under {root}')
            continue
        seen_any = True
        print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
        for rel_folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
            norm = _pv_normalise(rel_folder)
            mapped = _orn_map_folder(norm, slug, rel_norm=norm)
            print(f'    {rel_folder!r:46s} n={len(files):5d} -> {mapped or "DROP"}')
            if mapped is not None:
                # Bucket on slug+folder so stratification interleaves datasets.
                pools.setdefault(mapped, []).extend((f, f'{tag}/{norm}') for f in files)
    if not seen_any:
        print('  Kaggle ornamental: no slug downloaded; skipping')
        return 0
    saved_per_class = {}
    for cls, items in pools.items():
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        order = _stratified_folder_order(items)
        got = 0
        for idx in order:
            if got >= per_class_cap:
                break
            path, norm = items[idx]
            try:
                body = path.read_bytes()
            except Exception:
                continue
            got += _save_archive_member(
                cls, 'kaggle_ornamental', f'orn_{norm}', path.name, body,
                label_confidence=label_confidence,
                species=f'garden ornamental (Kaggle {norm.split("/")[0]})')
        saved_per_class[cls] = got
    total = sum(saved_per_class.values())
    print(f'  Kaggle ornamental per-class: {saved_per_class}')
    print(f'  Kaggle ornamental total: += {total}')
    return total

# ── DoctorP plant-doctor app dataset (v2.4, REAL deployment-domain) ──────────
# alexanderuzhinskiy/the-doctorp-project-dataset: ~4k REAL user phone photos
# from a plant-doctor app, labelled by condition (no augmentation, gardener's
# framing). It is the ONLY public source that supplies the chronically-starved
# classes in their DEPLOYMENT framing: Grey mold + Botrytis cinerea ->
# gray_mold_botrytis, Mealybug + Scale -> pests (cottony masses / waxy bumps ON
# a plant, not iNat entomology macros), plus real powdery_mildew / viral /
# leaf_spot / nutrient_deficiency / scorch / healthy. Small per class (76-351)
# and low-res (~128 px), but real.
#
# UNLIKE the other Kaggle sources it is EVAL-ELIGIBLE: source 'doctorp' is in
# every make_*_split allowed_sources, so a held-out fraction seeds val_realworld
# / val_indomain / temp_cal / conformal_cal. This is the v2.4 fix for the
# field-crop-seeded ship gate — the gate now measures real gardener-view photos
# across the whole taxonomy, not just PlantDoc crop leaves. Full sampler weight
# (it IS the deployment distribution). Fail-soft via _kaggle_download.
DOCTORP_SLUG = 'alexanderuzhinskiy/the-doctorp-project-dataset'
# v2.6: the pepper sibling from the SAME DoctorP project — 291 real app photos,
# median ~90 KB (higher-res than the main set's ~128 px thumbnails), none tiny.
# Maps through the SAME DOCTORP keyword/drop rules and adds REAL eval-eligible
# rows where the gate is thinnest: +31 nutrient_deficiency (the n=25 eval slice),
# + smaller viral/powdery/leaf_spot/scorch. Pepper is a container plant in the
# same gardener-photo framing the main DoctorP already spans (houseplants +
# garden/orchard), so it joins the eval/cal splits as the 'doctorp_pepper'
# source. No gray-mold here (that class stays iNat/DoctorP-bound).
DOCTORP_PEPPER_SLUG = 'alexanderuzhinskiy/pepper-disease-classification-dataset-doctorp'
# Both official Kaggle data cards are CC BY-NC-SA 4.0. STUDY-PROJECT POLICY
# (2026-09-21): the licence is recorded on every DoctorP row and the source is
# part of the default recipe again. It is the only public source of real
# plant-doctor APP photos, it is eval-eligible, and it supplies the starved
# classes (nutrient, gray mold): the 2026-08-25 opt-out removed 1,558 rows
# (doctorp 1,403 + doctorp_pepper 155 in the 2.7.2 inventory) from the mix.
DOCTORP_LICENSE_ID = 'cc-by-nc-sa-4.0'
DOCTORP_LICENSE_URL = 'https://creativecommons.org/licenses/by-nc-sa/4.0/'
DOCTORP_DROP = ('spider_mite', 'mite', 'rust', 'scab', 'esca', 'blight', 'rot',
                'shot_hole', 'hole', 'shute', 'gall', 'aphid', 'lichen', 'thrip',
                'edema', 'oedema', 'whitefly', 'miner', 'mechanical', 'turgor',
                'deformation', 'downy', 'wilt', 'foliage', 'sooty', 'frost',
                'canker', 'beetle', 'ants', 'slug', 'caterpillar', 'wireworm',
                'monilia', 'eyespot', 'chaff', 'polypore', 'anthocyanosis',
                'gryllotalpa', 'coccomyces', 'cyclamen', 'yellow', 'black_spot',
                # 'tubercular' (DoctorP 'Tubercular necrosis', ~40 imgs): a
                # pathogen lesion, NOT abiotic under-watering scorch — the bare
                # 'necrosis' keyword below would otherwise mislabel it into
                # leaf_scorch_underwatering. 'Marginal leaf necrosis' / 'Leaves
                # scorch' / 'Burn' still map to leaf_scorch correctly.
                'blossom', 'pear', 'tubercular')
DOCTORP_KEYWORDS = [
    ('grey_mold', 'gray_mold_botrytis'), ('gray_mold', 'gray_mold_botrytis'),
    ('botrytis', 'gray_mold_botrytis'),
    ('mealybug', 'pests'), ('mealy', 'pests'), ('scale', 'pests'),
    ('powdery', 'powdery_mildew'),
    ('nutrient', 'nutrient_deficiency'), ('chlorosis', 'nutrient_deficiency'),
    ('deficien', 'nutrient_deficiency'),
    ('mosaic', 'viral_disease'), ('virus', 'viral_disease'),
    ('bacterial_spot', 'leaf_spot'), ('leaf_spot', 'leaf_spot'),
    ('leafspot', 'leaf_spot'), ('septoria', 'leaf_spot'), ('anthracnose', 'leaf_spot'),
    ('burn', 'leaf_scorch_underwatering'), ('scorch', 'leaf_scorch_underwatering'),
    ('necrosis', 'leaf_scorch_underwatering'),
    ('healthy', 'healthy'),
]
def _doctorp_map_folder(folder_norm):
    if any(k in folder_norm for k in DOCTORP_DROP):
        return None
    for kw, cls in DOCTORP_KEYWORDS:
        if kw in folder_norm and cls in CLASSES:
            return cls
    return None

def fetch_doctorp_disease(slug=DOCTORP_SLUG, per_class_cap=400, label_confidence=0.8,
                          source='doctorp', label='DoctorP'):
    """DoctorP app photos -> TRAIN + held-out eval (EVAL-ELIGIBLE real-photo
    source). ``source`` tags the rows for the sampler/splitter: the main set is
    'doctorp'; the pepper sibling (same DoctorP project, same real app-photo
    framing, higher-res) is 'doctorp_pepper'. Maps each condition folder by the
    shared DOCTORP keyword/drop rules (dropping out-of-taxonomy / deferred
    conditions), draws a diverse cross-folder sample per class to per_class_cap,
    SHA-dedups against every other source, prints the folder mapping + per-class
    totals. Fail-soft."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = _images_by_relative_parent(root)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    pools = {}
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    for rel_folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(rel_folder)
        mapped = _doctorp_map_folder(norm)
        if mapped is not None:
            print(f'    {rel_folder!r:34s} n={len(files):5d} -> {mapped}')
            pools.setdefault(mapped, []).extend((f, norm) for f in files)
    saved_per_class = {}
    for cls, items in pools.items():
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        order = _stratified_folder_order(items)
        got = 0
        for idx in order:
            if got >= per_class_cap:
                break
            path, norm = items[idx]
            try:
                body = path.read_bytes()
            except Exception:
                continue
            got += _save_archive_member(
                cls, source, f'{source}_{norm}', path.name, body,
                label_confidence=label_confidence,
                species=f'plant ({label} {norm})',
                license_id=DOCTORP_LICENSE_ID, license_url=DOCTORP_LICENSE_URL,
                source_url=f'https://www.kaggle.com/datasets/{slug}',
                attribution='Alexander Uzhinskiy / CC BY-NC-SA 4.0',
                # DoctorP app photos are uniformly ~128px; the default 160px
                # floor rejected EVERY one (the source contributed 0). It's the
                # eval-eligible real-photo source, so accept its native size.
                # (The pepper sibling is higher-res and clears 120 trivially.)
                min_side=120)
        saved_per_class[cls] = got
    total = sum(saved_per_class.values())
    print(f'  {label} per-class: {saved_per_class}')
    print(f'  {label} total: += {total}')
    return total


# ── PlantDoc (HF) → real-world train + val_realworld seed ───────
_PD_FUNGAL_KEYWORDS = (
    # Indoor-relevant leaf-spot morphologies. Rust, scab, and whole-plant
    # systemic patterns are intentionally OUT — pooling them with cercospora
    # / anthracnose / alternaria collapses recall on the actual indoor pattern.
    'leaf spot', 'mold', 'anthracnose', 'septoria', 'target spot',
    'early blight', 'late blight', 'leaf blight',
)
_PD_DROP_KEYWORDS = (
    'spider mite', 'mites',
    'rust',                     # very different morphology — drop
    'scab',                     # apple scab is more rust-like
    'black rot', 'fruit rot',   # fruit-stage disease, not leaf
)

def map_plantdoc_label(raw):
    s = re.sub(r'\s+', ' ', str(raw).strip().lower())
    if any(k in s for k in _PD_DROP_KEYWORDS):
        return None
    if 'bacterial spot' in s or 'bacterial blight' in s:
        return 'leaf_spot'
    if 'powdery mildew' in s:
        return 'powdery_mildew'
    if 'yellow virus' in s or 'mosaic virus' in s or 'curl virus' in s:
        return 'viral_disease'
    if any(k in s for k in _PD_FUNGAL_KEYWORDS):
        return 'leaf_spot'
    if 'leaf' in s:
        return 'healthy'
    return None

def _clip_bbox_xywh(bbox, w, h, pad=0.06):
    """Clip an XYWH (COCO-style) bbox to image bounds with padding.

    Strictly assumes XYWH (the HF card for `agyaatcoder/PlantDoc` confirms
    XYWH). Reject bboxes whose bottom-right corner falls outside the image:
    in valid XYWH `x + bw ≤ w` always holds, so an XYXY tuple sneaking
    through past the image centre will overflow this check.
    """
    x, y, bw, bh = [float(v) for v in bbox]
    if bw <= 1 or bh <= 1 or w <= 1 or h <= 1:
        return None
    # 5 % slack absorbs off-by-one annotations; anything past that is
    # either a real out-of-bounds bbox or XYXY misread as XYWH.
    if (x + bw) > w * 1.05 or (y + bh) > h * 1.05:
        return None
    if bw > w or bh > h:
        return None
    px, py = bw * pad, bh * pad
    x0, y0 = max(0, int(round(x - px))), max(0, int(round(y - py)))
    x1, y1 = min(w, int(round(x + bw + px))), min(h, int(round(y + bh + py)))
    if x1 - x0 < 32 or y1 - y0 < 32:
        return None
    return x0, y0, x1, y1

def _plantdoc_image_label(row, ds_split):
    """Resolve a PlantDoc image-level class label as a string, used as a
    fallback when the HF revision ships image-level labels instead of
    bbox-level ``objects``."""
    if 'label' in row:
        feat = ds_split.features.get('label') if hasattr(ds_split, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

def fetch_plantdoc(max_train_per_class=500, max_val_per_class=120):
    try:
        ds = _hf_load_with_retry('agyaatcoder/PlantDoc')
    except Exception as e:
        print(f'PlantDoc HF download failed after retries: {e}')
        print('  HuggingFace 500/503 on /api/datasets/.../tree/... resolves in '
              'a few minutes — re-run this cell shortly.')
        return
    counts, dropped, no_obj_fallback = {}, 0, 0
    for split in [s for s in ('train', 'test') if s in ds]:
        out_base = VAL_PRIMARY if split == 'test' else TRAIN_ROOT
        cap = max_val_per_class if split == 'test' else max_train_per_class
        # HF returns rows clustered by class — shuffle so the cap samples
        # a wider mix.
        order = list(range(len(ds[split])))
        random.Random(SEED + (10 if split == 'test' else 11)).shuffle(order)
        # Boxes, categories and ids are read WITHOUT decoding the photo; it is
        # decoded (by the split's own HF decoder — same pixels as before) only
        # when a crop or the full frame is actually written. Once a class hits
        # its cap, and on every re-run, a row is a dict lookup, not a decode.
        _rows, _decode = _hf_lazy_images(ds[split])
        for i in tqdm(order, desc=f'PlantDoc/{split}'):
            row = _rows[i]
            img = None
            objects = row.get('objects') or {}
            bboxes = objects.get('bbox') or []
            cats = objects.get('category') or []
            # Object-level path (preferred — HF revisions ≥ 1.x).
            wrote_any = False
            for j, (bbox, cat) in enumerate(zip(bboxes, cats)):
                target = map_plantdoc_label(cat)
                if target is None or target not in CLASSES:
                    dropped += 1; continue
                key = (split, target)
                if counts.get(key, 0) >= cap:
                    continue
                image_id = row.get('image_id', i)
                stem = f'plantdoc_{split}_{image_id}_{j}'
                out = out_base / target / f'{stem}.jpg'
                # Same outcome as checking after the clip: an existing crop was
                # written from these same bytes, so its box was valid.
                if out.exists(): wrote_any = True; continue
                if img is None:
                    img = _decode(row['image']).convert('RGB')
                box = _clip_bbox_xywh(bbox, *img.size)
                if box is None: dropped += 1; continue
                crop = img.crop(box)
                _save_rgb(crop, out)
                write_sidecar(out, CropAnnotation(
                    plant_id=f'plantdoc_{split}_{image_id}',
                    species=str(cat).split(' leaf')[0],
                    diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                    capture_session=f'plantdoc_{split}', source='plantdoc',
                    label_confidence=0.85, bbox=[int(v) for v in bbox]))
                counts[key] = counts.get(key, 0) + 1
                wrote_any = True
            if wrote_any:
                continue
            # Image-level fallback. Triggered when the HF revision drops
            # object-detection annotations and ships one category per image.
            raw = _plantdoc_image_label(row, ds[split])
            target = map_plantdoc_label(raw) if raw is not None else None
            if target is None or target not in CLASSES:
                dropped += 1; continue
            key = (split, target)
            if counts.get(key, 0) >= cap:
                continue
            image_id = row.get('image_id', i)
            stem = f'plantdoc_{split}_{image_id}_full'
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            if img is None:
                img = _decode(row['image']).convert('RGB')
            _save_rgb(img, out)
            write_sidecar(out, CropAnnotation(
                plant_id=f'plantdoc_{split}_{image_id}',
                species=str(raw).split(' leaf')[0] if raw else None,
                diagnosis=target, severity='unknown', leaf_or_whole_plant='whole_plant',
                capture_session=f'plantdoc_{split}', source='plantdoc',
                label_confidence=0.75, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
            no_obj_fallback += 1
    print(f'PlantDoc counts: {counts}\nPlantDoc dropped: {dropped}'
          f'  image-level fallbacks: {no_obj_fallback}')

# ── PlantVillage (HF) → capped train + val_studio ───────────────
def _pv_normalise(s):
    return re.sub(r'[^a-z0-9_]+', '_', str(s).strip().lower()).strip('_')

PV_LABEL_MAP_RAW = {
    'Pepper,_bell___Bacterial_spot': 'leaf_spot',
    'Pepper__bell___Bacterial_spot': 'leaf_spot',
    'Tomato___Bacterial_spot':       'leaf_spot',
    'Peach___Bacterial_spot':        'leaf_spot',
    'Apple___healthy': 'healthy', 'Blueberry___healthy': 'healthy',
    'Cherry_(including_sour)___healthy': 'healthy', 'Corn_(maize)___healthy': 'healthy',
    'Grape___healthy': 'healthy',
    'Pepper,_bell___healthy': 'healthy', 'Pepper__bell___healthy': 'healthy',
    'Potato___healthy': 'healthy', 'Raspberry___healthy': 'healthy',
    'Soybean___healthy': 'healthy', 'Strawberry___healthy': 'healthy',
    'Tomato___healthy': 'healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'leaf_spot',
    'Tomato___Septoria_leaf_spot': 'leaf_spot',
    'Tomato___Target_Spot':        'leaf_spot',
    'Tomato___Leaf_Mold':          'leaf_spot',
    'Tomato___Early_blight':       'leaf_spot',
    'Tomato___Late_blight':        'leaf_spot',
    'Potato___Early_blight':       'leaf_spot',
    'Potato___Late_blight':        'leaf_spot',
    # Apple scab is rust-family morphology; keep OUT of leaf_spot.
    'Apple___Apple_scab':          None,
    # Apple black rot is a fruit disease — many images are whole-fruit.
    'Apple___Black_rot':           None,
    # Rust morphology is dropped wholesale.
    'Apple___Cedar_apple_rust':    None,
    'Corn_(maize)___Common_rust_': None,
    'Corn_(maize)___Northern_Leaf_Blight': 'leaf_spot',
    'Grape___Black_rot':           None,                  # fruit disease
    'Grape___Esca_(Black_Measles)':'leaf_spot',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 'leaf_spot',
    'Orange___Haunglongbing_(Citrus_greening)':   None,   # HLB ≠ leaf-spot
    'Cherry_(including_sour)___Powdery_mildew': 'powdery_mildew',
    'Squash___Powdery_mildew':                  'powdery_mildew',
    'Tomato___Tomato_mosaic_virus':           'viral_disease',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'viral_disease',
    'Tomato___Spider_mites Two-spotted_spider_mite': None,
    # Strawberry leaf scorch is physiological / abiotic, not fungal.
    'Strawberry___Leaf_scorch':  None,
    'Background_without_leaves': 'not_a_plant',
}
PV_LABEL_MAP = {_pv_normalise(k): v for k, v in PV_LABEL_MAP_RAW.items()}
# Each candidate is (hf_repo, hf_config_name_or_None, splits_tuple). The
# config name is required for mohanty/PlantVillage (ships three configs;
# load_dataset() refuses to pick one). BrandonFors has a single default
# config, so explicit name is None there.
PV_DATASET_CANDIDATES = [
    ('BrandonFors/Plant-Diseases-PlantVillage-Dataset', None,    ('train', 'test')),
    ('mohanty/PlantVillage',                            'color', ('train', 'test')),
]

def _hf_lazy_images(split, column='image'):
    """-> (rows, decode) for an HF split with an Image column.

    ``rows[i]`` returns row i with ``column`` left as its raw {'bytes', 'path'}
    payload — boxes, labels and ids cost no JPEG decode — and ``decode(raw)``
    runs the split's OWN Image feature decoder, i.e. exactly the EXIF/mode
    handling ``split[i]`` applies, only for the rows that need pixels. A
    column that is not a decoding HF Image feature is returned unchanged."""
    feat = (getattr(split, 'features', None) or {}).get(column)
    if feat is None or not hasattr(feat, 'decode_example') or not getattr(feat, 'decode', True):
        return split, (lambda v: v)
    from datasets import Image as _HFImage
    return split.cast_column(column, _HFImage(decode=False)), feat.decode_example


def _hf_load_with_retry(name, config_name=None, *, split=None, retries=3, base_delay=5.0):
    """`datasets.load_dataset` with exponential-backoff retry on transient
    HuggingFace 5xx errors."""
    from datasets import load_dataset
    last = None
    delay = base_delay
    for attempt in range(retries):
        try:
            revision = _hf_resolve_dataset_revision(name)
            dataset = load_dataset(name, config_name, split=split,
                                   revision=revision, token=_HF_AUTH)
            _record_hf_dataset_load(name, config_name, split, dataset)
            return dataset
        except Exception as e:
            last = e
            msg = str(e)
            transient = (
                '500' in msg or '502' in msg or '503' in msg or '504' in msg
                or 'temporarily unavailable' in msg.lower()
                or 'connection' in msg.lower() or 'timeout' in msg.lower()
            )
            if attempt + 1 == retries or not transient:
                raise
            print(f'  HF {name} attempt {attempt + 1}/{retries} failed ({type(e).__name__}); '
                  f'retrying in {delay:.0f}s…')
            time.sleep(delay)
            delay *= 3
    if last is not None:
        raise last
    raise RuntimeError(f'_hf_load_with_retry exhausted retries for {name}')

def _load_first_available_hf_dataset(candidates):
    errors = []
    for entry in candidates:
        if len(entry) == 2:
            name, splits = entry; config_name = None
        else:
            name, config_name, splits = entry
        try:
            out = {}
            for split in [s for s in splits if s]:
                out[split] = _hf_load_with_retry(name, config_name, split=split)
            label = name + (f' [{config_name}]' if config_name else '')
            print(f'PlantVillage source: {label}')
            return name, out
        except Exception as e:
            errors.append(f'{name}: {e}')
    raise RuntimeError('All PlantVillage HF downloads failed:\n' + '\n'.join(errors))

def _row_label(row, ds):
    if 'label' in row:
        feat = ds.features.get('label') if hasattr(ds, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

# PV `healthy` is uniform-bg / agar-plate; iNat `healthy` is real foliage.
# Cap PV-only contribution low so iNat dominates healthy.
PV_PER_CLASS_TRAIN_OVERRIDE = {'healthy': 250}

def fetch_plantvillage_into_train_and_studio(max_train_per_class=900, max_studio_per_class=300, studio_hash_frac=0.15):
    try:
        name, dsets = _load_first_available_hf_dataset(PV_DATASET_CANDIDATES)
    except Exception as e:
        print(f'PlantVillage HF download failed: {e}'); return
    # When the candidate ships a real `test` split, use IT as the canonical
    # val_studio source so train rows always go to TRAIN_ROOT.
    test_present = 'test' in dsets
    # Skip image DECODE for the ~90% of rows we drop or cap: read labels off
    # the undecoded column and decode only the few we actually save. Falls back
    # to normal decoding if the cast isn't applicable.
    try:
        from datasets import Image as _HFImage
        for _k in list(dsets):
            if 'image' in dsets[_k].features and isinstance(dsets[_k].features['image'], _HFImage):
                dsets[_k] = dsets[_k].cast_column('image', _HFImage(decode=False))
    except Exception as _e:  # noqa: BLE001 — degrade to full decode
        print(f'  PV: decode-skip unavailable ({type(_e).__name__}); decoding all rows')
    counts, dropped = {}, 0
    for split, ds in dsets.items():
        order = list(range(len(ds)))
        random.Random(SEED + (0 if split == 'train' else 1)).shuffle(order)
        for i in tqdm(order, desc=f'PlantVillage/{split}'):
            row = ds[i]
            raw = _row_label(row, ds)
            norm = _pv_normalise(raw)
            target = PV_LABEL_MAP.get(norm)
            if target is None or target not in CLASSES:
                dropped += 1; continue
            _imgcol = row['image']
            filename = (row.get('image/filename') or row.get('filename')
                        or (_imgcol.get('path') if isinstance(_imgcol, dict) else None)
                        or f'{split}_{i}')
            stem = f'pv_{_pv_normalise(filename)}_{i}'
            if test_present:
                use_studio = (split == 'test')
            else:
                use_studio = (_stable_frac(stem) < studio_hash_frac)
            out_base = VAL_STUDIO if use_studio else TRAIN_ROOT
            if use_studio:
                cap = max_studio_per_class
            else:
                cap = PV_PER_CLASS_TRAIN_OVERRIDE.get(target, max_train_per_class)
            key = ('studio' if use_studio else 'train', target)
            if counts.get(key, 0) >= cap:
                continue
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            img = _imgcol
            if isinstance(img, dict):          # decode=False -> {'bytes','path'}
                img = (PILImage.open(BytesIO(img['bytes'])) if img.get('bytes')
                       else PILImage.open(img['path']))
            elif not hasattr(img, 'convert'):
                img = PILImage.open(img)
            _save_rgb(img.convert('RGB'), out)
            write_sidecar(out, CropAnnotation(
                plant_id=stem, species=str(raw).split('___')[0].replace('_', ' '),
                diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                capture_session=f'plantvillage_{split}', source='plantvillage',
                label_confidence=0.9, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
    print(f'PV counts: {counts}\nPV dropped/ignored: {dropped}')

# ── Group-aware splits ──────────────────────────────────────────
def make_group_split(dest_root, ratio, seed, label, allowed_sources=None):
    """Move a fraction of TRAIN_ROOT into dest_root.

    Every `(source, plant_id)` group lands together IN ONE SPLIT — including
    when its images span MULTIPLE classes. PlantDoc is a detection set whose
    crops from a single photo can map to different diseases (one box
    leaf_spot, another healthy); they share a plant_id, so deciding per-class
    independently would leak the same photo across train and a held-out split
    (exactly what the §2 leakage guard caught). We therefore collect groups
    GLOBALLY across classes, choose per-class by `ratio`, and move EVERY file
    of a chosen group regardless of which class dir it sits in.

    `allowed_sources` constrains which sources can populate the split — used
    to keep PlantVillage out of the real-world / calibration splits.
    """
    rng = random.Random(seed)
    # Pass 1 — collect every group's files across ALL classes.
    group_files = {}                 # (source, pid) -> [(cls, img_path), ...]
    per_class_ids = {}               # cls -> [ (source, pid), ... ] (unique)
    for cls in CLASSES:
        cls_dir = TRAIN_ROOT / cls
        if not cls_dir.exists():
            continue
        seen_in_cls = set()
        for img in iter_images(cls_dir):
            sc = read_sidecar(img) or {}
            source = sc.get('source', 'unknown')
            if allowed_sources is not None and source not in allowed_sources:
                continue
            g = (source, sc.get('plant_id', img.stem))
            group_files.setdefault(g, []).append((cls, img))
            if g not in seen_in_cls:
                per_class_ids.setdefault(cls, []).append(g)
                seen_in_cls.add(g)
    # Pass 2 — choose groups per class by `ratio`. A group already chosen via
    # another class counts toward this class's quota (it's one shared photo),
    # so cross-class groups never double-draw and always co-locate.
    chosen = set()
    for cls in CLASSES:
        ids = list(per_class_ids.get(cls, []))
        if not ids:
            continue
        rng.shuffle(ids)
        k = int(len(ids) * ratio)
        if ratio > 0:
            k = max(1, k)
        if k <= 0:
            continue
        taken = [g for g in ids if g in chosen]          # already destined here
        for g in ids:
            if len(taken) >= k:
                break
            if g not in chosen:
                taken.append(g)
        chosen.update(taken)
    # Pass 3 — move every file of every chosen group (across all its classes).
    moved = 0
    for g in chosen:
        for cls, img in group_files[g]:
            out = dest_root / cls
            out.mkdir(parents=True, exist_ok=True)
            dst = out / img.name
            if dst.exists():
                raise RuntimeError(f'Destination already exists: {dst}')
            shutil.move(str(img), str(dst))
            sc_path = img.with_suffix('.json')
            if sc_path.exists():
                shutil.move(str(sc_path), str(dst.with_suffix('.json')))
            moved += 1
    print(f'{label} populated  (moved={moved})')

# ── Content-hash memo ────────────────────────────────────────────
# §3 runs FOUR tree-wide pHash passes and TWO SHA passes over largely the same
# files: _dedup_train_in_place, dedup_each_split, assert_no_exact_split_leakage,
# resolve_phash_split_leakage and audit_cross_class_phash_leakage twice (the
# second is the holdout-vs-train pass). TRAIN alone is ~13.6k images, and every
# pHash decodes and resizes a JPEG — so most of §3's wall-clock is the same
# work repeated. Keyed on (path, size, mtime) rather than path alone so a
# rebuilt or replaced file is never served a stale hash; entries for files that
# were moved or dropped simply go unused (a few MB at this corpus size).
_HASH_MEMO: dict = {}

def _memo_key(p: Path):
    try:
        st = Path(p).stat()
        return (str(p), st.st_size, int(st.st_mtime))
    except OSError:
        return None

def _sha256_file(p: Path) -> str:
    k = _memo_key(p)
    if k is not None and ('sha', k) in _HASH_MEMO:
        return _HASH_MEMO[('sha', k)]
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    v = h.hexdigest()
    if k is not None:
        _HASH_MEMO[('sha', k)] = v
    return v

def _phash_file(p: Path):
    import imagehash
    k = _memo_key(p)
    if k is not None and ('ph', k) in _HASH_MEMO:
        return _HASH_MEMO[('ph', k)]
    try:
        v = imagehash.phash(PILImage.open(p).convert('RGB'))
    except Exception:
        v = None
    if k is not None:
        _HASH_MEMO[('ph', k)] = v
    return v

def _dedup_train_in_place(root: Path, phash_threshold: int = 6) -> int:
    """Per-class SHA-256 + pHash dedup on TRAIN_ROOT before the splitters
    run. PIL.open + hashing both release the GIL, so we parallelise the
    per-file work and keep the per-class hash-set membership check serial."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists():
            continue
        paths = sorted(iter_images(cls_dir))
        if not paths:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            shas   = list(pool.map(_sha256_file, paths))
            phashes = list(pool.map(_phash_file, paths))
        seen_sha, seen_ph = set(), []
        for p, sha, ph in zip(paths, shas, phashes):
            if sha in seen_sha:
                drop.append(p); continue
            seen_sha.add(sha)
            if ph is None:
                drop.append(p); continue
            if any(ph - existing <= phash_threshold for existing in seen_ph):
                drop.append(p); continue
            seen_ph.append(ph)
    for p in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except OSError:
            pass
    print(f'pre-split dedup({root}): dropped={len(drop)}')
    return len(drop)

# ── The PERMANENT never-trained holdout (R0a) ────────────────────
# WHY THIS EXISTS. Measured on the deployed 2.6.8 bundle over all 1,907
# eval-eligible supplement rows: 0.964 on the 1,322 it trained on, 0.807 on the
# 585 it never saw. And on those 1,322 shared rows, 2.6.8 vs 2.7.1 is 35 gained
# / 32 lost, p=0.807 — dead flat. As currently routed the supplement is a
# training-set-membership detector, not an accuracy instrument: a model looks
# good on it exactly to the degree it memorised it. Every 2.7.x verdict to date
# was read partly off that.
#
# WHY A HASH BUCKET AND NOT A RATIO DRAW. `make_group_split` shuffles the group
# list with a seeded RNG, which is reproducible only while the corpus is
# identical. The corpus grows every release, so a ratio draw silently
# reshuffles membership and the "holdout" is a different set of images each run
# — useless for comparing runs. Bucketing a stable hash of the GROUP KEY
# instead makes membership a property of the image, not of the draw: rows keep
# their verdict as the corpus grows, and a row that was ever held out is never
# re-admitted. No state file, nothing to lose.
#
# Group-aware for the same reason every other split is: the unit is
# `(source, plant_id)`, so all crops of one photo land on the same side.
#
# SCOPE. Supplement `eval_ok` rows only. Those are the in-situ, human/agent-
# judged photographs that look like what the product actually sees; the other
# sources are either studio (PlantVillage), detection crops (PlantDoc), or
# not_a_plant negatives, and pulling those into a holdout would weaken the ship
# gate without making the instrument more like production.
#
# HONEST LIMIT: this is clean for 2.7.2 forward and NOT comparable to 2.6.8 or
# 2.7.x, which trained on rows now inside it. It starts measuring on the first
# 2.7.2 run and gets more valuable every release. Do not backfill it.
HOLDOUT_PERMILLE = 300          # ~30% of eval-eligible supplement -> ~550 rows
HOLDOUT_SOURCES = {'supplement'}

def _holdout_bucket(source, plant_id) -> int:
    """Stable 0-999 bucket for a group key. sha1 (not hash()) because Python's
    string hash is salted per process — `hash()` here would silently reshuffle
    the holdout on every kernel restart."""
    key = f'{source}|{plant_id}'.encode()
    return int.from_bytes(hashlib.sha1(key).digest()[:4], 'big') % 1000

def make_holdout(permille=None, label='holdout'):
    """Move every group whose stable bucket is below `permille` into HOLDOUT.

    Runs BEFORE the four ratio splits so it cannot be drained by them: whatever
    it takes is gone before val_realworld / val_indomain / temperature_cal /
    conformal_cal draw. That ordering is the guarantee — a holdout that is
    populated last is a holdout of whatever the other splits did not want.
    """
    permille = HOLDOUT_PERMILLE if permille is None else permille
    if permille <= 0:
        print(f'{label}: DISABLED (HOLDOUT_PERMILLE={permille})')
        return
    group_files, meta = {}, {}
    for cls in CLASSES:
        cls_dir = TRAIN_ROOT / cls
        if not cls_dir.exists():
            continue
        for img in iter_images(cls_dir):
            sc = read_sidecar(img) or {}
            source = sc.get('source', 'unknown')
            if source not in HOLDOUT_SOURCES:
                continue
            g = (source, sc.get('plant_id', img.stem))
            group_files.setdefault(g, []).append((cls, img))
            meta.setdefault(g, sc)
    chosen = [g for g in group_files if _holdout_bucket(*g) < permille]
    moved, by_class, by_domain = 0, {}, {}
    for g in chosen:
        for cls, img in group_files[g]:
            out = HOLDOUT / cls
            out.mkdir(parents=True, exist_ok=True)
            dst = out / img.name
            if dst.exists():
                raise RuntimeError(f'Destination already exists: {dst}')
            shutil.move(str(img), str(dst))
            sc_path = img.with_suffix('.json')
            if sc_path.exists():
                shutil.move(str(sc_path), str(dst.with_suffix('.json')))
            moved += 1
            by_class[cls] = by_class.get(cls, 0) + 1
            dom = (meta.get(g) or {}).get('subject_domain') or 'unspecified'
            by_domain[dom] = by_domain.get(dom, 0) + 1
    print(f'{label} populated  (moved={moved} from {len(chosen)}/{len(group_files)} groups, '
          f'bucket<{permille}/1000)')
    print(f'  {label} per class:  {dict(sorted(by_class.items()))}')
    print(f'  {label} per domain: {dict(sorted(by_domain.items(), key=lambda kv: -kv[1]))}')
    # A holdout too thin per class cannot carry a per-class verdict, and a
    # holdout that swallowed a class cannot leave one for the ship gate.
    # Include zero-support classes: iterating only by_class hid a completely
    # absent class (viral_disease=0 in the reviewed 2.7.3 holdout).
    _thin = {c: by_class.get(c, 0) for c in CLASSES if by_class.get(c, 0) < 20}
    if _thin:
        print(f'  WARN {label}: classes under 20 rows — read these per-class numbers as '
              f'directional only: {_thin}')
    if not group_files:
        print(f'  WARN {label}: no {sorted(HOLDOUT_SOURCES)} groups in TRAIN_ROOT to draw '
              f'from — the supplement did not load, or it already ran this session. '
              f'The improvement metric will be MISSING from §6.')
    elif moved == 0:
        # Distinguish "already drawn" from "the bucket is broken". After a
        # successful draw every row LEFT in TRAIN_ROOT hashes above the bucket by
        # construction, so a second call moving nothing is correct, not alarming.
        already = sum(count_images(HOLDOUT / c) for c in CLASSES)
        if already:
            print(f'  {label}: already populated ({already} images); the {len(group_files)} '
                  f'groups still in train all hash >= {permille}/1000, as they should.')
        else:
            print(f'  WARN {label}: {len(group_files)} candidate groups and NOT ONE hashed '
                  f'below {permille}/1000 into an EMPTY holdout. Suspect the bucket or the '
                  f'group keys, not luck.')

def make_val_realworld(ratio=0.15, seed=2):
    # Physiological-class sources join the realworld pool so the ship-gate
    # per-class recall check sees real rows. Every NAP source is included
    # so NAP recall is measured against indoor clutter, not just COCO.
    make_group_split(VAL_PRIMARY, ratio, seed, 'val_realworld',
                     allowed_sources={'inat', 'coco', 'plantdoc',
                                      'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp', 'doctorp_pepper', 'supplement'})
def make_val_indomain(ratio=0.10, seed=0):
    make_group_split(VAL_INDOMAIN, ratio, seed, 'val_indomain',
                     allowed_sources={'inat', 'coco', 'plantdoc',
                                      'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp', 'doctorp_pepper', 'supplement'})
def make_temperature_cal(ratio=0.12, seed=3):
    # All trained classes need calibration samples. Include every source
    # that feeds a trained class (NAP + physiological).
    make_group_split(TEMP_CAL, ratio, seed, 'temperature_cal',
                     allowed_sources={'plantdoc', 'inat',
                                      'coco', 'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp', 'doctorp_pepper', 'supplement'})
def make_conformal_cal(ratio=0.15, seed=1):
    make_group_split(CONFORMAL_CAL, ratio, seed, 'conformal_cal',
                     allowed_sources={'plantdoc', 'inat',
                                      'coco', 'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp', 'doctorp_pepper', 'supplement'})

# ── NAP hard negatives (leaf-like-but-not-leaf / partial plants /
#    blurry green textures) ─────────────────────────────────────────
# The easy COCO / Animalia / Open-Images negatives sit far from the plant
# MaxLogit distribution, so plant and NAP MaxLogit overlap and the §6 OOD
# gate auto-disables (negative separation). These two fetchers add the HARD
# negatives that actually overlap real leaf close-ups — green, textured,
# leaf-adjacent frames — which is what flips separation positive and lets
# the MaxLogit floor switch on.
def fetch_coco_hard_negatives(target_n=600, max_plant_area_frac=0.25):
    """COCO indoor frames that DO contain a potted plant / vase but only as
    a small / partial element (summed plant bbox area < max_plant_area_frac
    of the image). These 'partial plant in clutter' shots are exactly what
    fetch_coco_for_nap deliberately excludes — and exactly what the OOD gate
    confuses with a real leaf. Reuses the COCO cache fetch_coco_for_nap
    already downloaded."""
    ann_path = COCO_CACHE / 'annotations' / 'instances_val2017.json'
    img_dir = COCO_CACHE / 'val2017'
    if not ann_path.exists() or not img_dir.exists():
        print('  COCO cache missing (run fetch_coco_for_nap first) — skipping hard negatives')
        return
    ann = json.loads(ann_path.read_text())
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in {'potted plant', 'vase'}}
    license_by_id = {int(x['id']): x for x in ann.get('licenses', [])}
    image_by_id = {im['id']: im for im in ann['images']}
    meta = {im['id']: (im['width'], im['height'], im['file_name']) for im in ann['images']}
    frac = {}
    for a in ann['annotations']:
        if a['category_id'] not in plant_cat:
            continue
        iw, ih, _ = meta.get(a['image_id'], (0, 0, None))
        if not iw or not ih:
            continue
        bw, bh = a['bbox'][2], a['bbox'][3]
        frac[a['image_id']] = frac.get(a['image_id'], 0.0) + (bw * bh) / float(iw * ih)
    eligible = [iid for iid, f in frac.items() if 0.0 < f < max_plant_area_frac]
    rng = random.Random(SEED + 23); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for iid in eligible:
        if n >= target_n:
            break
        _iw, _ih, fname = meta[iid]
        src, dst = img_dir / fname, dest / f'coco_hardneg_{iid}.jpg'
        if not src.exists() or dst.exists():
            continue
        shutil.copy(src, dst)
        im = image_by_id[iid]
        lic = license_by_id.get(im.get('license')) or {}
        _coco_tally(im)
        write_sidecar(dst, CropAnnotation(
            plant_id=f'coco_hardneg_{iid}', species=None, diagnosis='not_a_plant',
            severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_hard_partial_plant', source='coco_hard_partial_plant',
            label_confidence=0.95, bbox=[0, 0, 0, 0],
            license_id=f'coco:{im.get("license")}', license_url=lic.get('url'),
            source_url=im.get('flickr_url') or im.get('coco_url'),
            source_item_id=str(iid), attribution=lic.get('name')))
        n += 1
    print(f'COCO hard negatives (partial plant): not_a_plant += {n}')

# Green, leaf-like textures that are NOT a houseplant leaf and are
# taxonomically disjoint from every disease / healthy genus: grasses,
# mosses, liverworts, lichens, green algae. These overlap the plant
# MaxLogit distribution far more than animals / furniture do.
INAT_HARD_NEGATIVE_TAXA = [
    ('Poaceae', 'family'),          # grasses / lawn blades
    ('Bryophyta', 'phylum'),        # mosses
    ('Marchantiophyta', 'phylum'),  # liverworts
    ('Lecanoromycetes', 'class'),   # lichens
    ('Chlorophyta', 'phylum'),      # green algae
]
def fetch_inat_hard_negatives(per_taxon=200, target_n=900):
    """iNat research-grade green-texture hard negatives -> not_a_plant."""
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    saved = 0
    for name, rank in INAT_HARD_NEGATIVE_TAXA:
        if saved >= target_n:
            break
        tid = inat_resolve_taxon(name, rank)
        if not tid:
            print(f'  iNat hard-neg taxon {name!r} not resolvable; skipping')
            continue
        obs = inat_observations(tid, per_taxon, leaf_only=False)
        jobs = []
        for ob in obs:
            for ph in (ob.get('photos') or [])[:1]:
                provenance = _inat_photo_provenance(ob, ph)
                if provenance is None:
                    continue
                url = _photo_url(ph, 'medium')
                if not url:
                    continue
                fp = dest / f'inat_hardneg_{ob["id"]}_{ph["id"]}.jpg'
                obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
                ann = lambda obs_id=obs_id, taxon_name=taxon_name, name=name, provenance=provenance: CropAnnotation(
                    plant_id=f'inat_hardneg_{obs_id}', species=taxon_name,
                    diagnosis='not_a_plant', severity='low',
                    leaf_or_whole_plant='whole_plant',
                    capture_session=f'inat_hardneg_{name}',
                    source='inat_hard_greentexture', label_confidence=0.97,
                    bbox=[0, 0, 0, 0], **provenance)
                jobs.append((url, fp, ann))
        n = _run_fetch_jobs(jobs, target_n - saved)
        saved += n
        print(f'  iNat hard-neg {name}: += {n}')
    print(f'iNat hard negatives (green texture): not_a_plant += {saved}')

# ── iNat → pest classes (mealybug / scale) ───────────────
# The two most visually-separable houseplant pests: mealybugs
# (Pseudococcidae — cottony white masses) and scale (Diaspididae /
# Coccidae — waxy / armoured bumps). Research-grade observations are the
# insect ON a plant, so leaf_only=False and the green-content gate is NOT
# applied (the pest mass, not foliage green, is the signal). Tagged
# source=\'inat_pest\' (in-domain weight) so the sampler treats them as real
# houseplant rows, not crop supplements (full source weight in §4).
INAT_PEST_TAXA = {
    # Single combined 'pests' class — pull from every common houseplant
    # mealybug + scale family into one label: mealybugs = Pseudococcidae;
    # armoured scale = Diaspididae; soft scale = Coccidae; felt scale =
    # Eriococcidae. Broad family coverage makes 'pests' a strong,
    # well-supplied class rather than a thin one.
    'pests': [('Pseudococcidae', 'family'), ('Diaspididae', 'family'),
              ('Coccidae', 'family'), ('Eriococcidae', 'family')],
}
def fetch_inat_pests(per_taxon=800, target_per_class=900):
    for cls, taxa in INAT_PEST_TAXA.items():
        dest = TRAIN_ROOT / cls; dest.mkdir(parents=True, exist_ok=True)
        saved = 0
        for name, rank in taxa:
            if saved >= target_per_class:
                break
            tid = inat_resolve_taxon(name, rank)
            if not tid:
                print(f'  iNat pest taxon {name!r} not resolvable; skipping'); continue
            obs = inat_observations(tid, per_taxon, leaf_only=False)
            jobs = []
            for ob in obs:
                for ph in (ob.get('photos') or [])[:1]:
                    provenance = _inat_photo_provenance(ob, ph)
                    if provenance is None:
                        continue
                    url = _photo_url(ph, 'medium')
                    if not url:
                        continue
                    fp = dest / f'inat_pest_{cls}_{ob["id"]}_{ph["id"]}.jpg'
                    obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
                    ann = lambda obs_id=obs_id, taxon_name=taxon_name, name=name, cls=cls, provenance=provenance: CropAnnotation(
                        plant_id=f'inat_pest_{obs_id}', species=taxon_name,
                        diagnosis=cls, severity='medium', leaf_or_whole_plant='leaf',
                        capture_session=f'inat_pest_{name}', source='inat_pest',
                        label_confidence=0.7, bbox=[0, 0, 0, 0], **provenance)
                    jobs.append((url, fp, ann))
            saved += _run_fetch_jobs(jobs, len(jobs))
        print(f'iNat pest ({cls}): += {saved}')


# ── Our Kaggle eval-supplement (agent-collected, in-domain houseplant) ──────
# maxayyt5/houseplant-disease-eval-supplement: OUR offline-collected, vision-
# agent-judged (verified_by=agent_vision) houseplant photos for the weak
# classes. This fetcher reads WHATEVER canonical-class folders the dataset
# ships (gray_mold_botrytis / leaf_scorch_underwatering / nutrient_deficiency
# today; healthy/ + not_a_plant/ hard-negative folders as the collection plan
# adds them — docs/DATASET_LABELING_PLAN.md), so the dataset grows without a
# code change. Unlike the field-crop proxies it is in-domain and therefore
# EVAL-ELIGIBLE: source 'supplement' is in every make_*_split allowed_sources,
# so a held-out fraction thickens the thin real-world eval slices (esp.
# nutrient_deficiency, eval n=25). Each image is its own (source, plant_id)
# group -> a clean per-image holdout, no within-supplement leakage.
# CAVEAT: rows are AGENT-judged, not human-verified, so a ship gate measured on
# them is a PROXY until verified production volume exists. As of supplement
# v3 a HARD domain gate is applied upstream, so the weak classes -- incl.
# gray_mold -- carry no commercial agriculture. The IN-domain side of that
# gate keeps widening (v8b: + outdoor/garden ornamentals; next: + domestic
# kitchen-garden edibles), which is why nothing here reads the value: the
# manifest's subject_domain is recorded + reported, never gated on, so a new
# bucket costs zero notebook changes; role/hn_kind columns in the
# manifest tag the healthy/ + not_a_plant/ hard negatives (each folder still
# maps to its canonical class); hn_kind now also drives train-only routing of
# the boundary negatives (see fetch_kaggle_supplement). Fail-soft via
# _kaggle_download.
KAGGLE_SUPPLEMENT_SLUG = 'maxayyt5/houseplant-disease-eval-supplement'
# The offline pipeline stamps every supplement row with its OWN eval/train
# verdict in `split_hint` ('eval_ok' | 'train_only'): rows below its label-
# confidence floor, plus anything else it does not want grading a model, are
# 'train_only'. Until v2.6.13 the notebook ignored the column, so those rows
# still drained into val_realworld / val_indomain / temperature_cal /
# conformal_cal at the shipped ratios (~52% of them) and the ship gate was
# measured — and the temperature + conformal calibration FIT — on rows the
# dataset build had already judged too noisy to grade with. Set False only for
# a deliberate run against an older supplement version that predates the column.
#
# FALLBACK (2026-08): the published Kaggle build may predate `split_hint` while
# still carrying the `confidence` column the hint was DERIVED from. Rather than
# refuse the run or silently revert to grading on everything, the routing falls
# back to `confidence >= SUPPLEMENT_EVAL_CONF_FLOOR` — the same >=0.80 floor the
# offline pipeline applies (docs/DATASET_LABELING_PLAN.md v8 item 3). The strict
# flag now only fires when NEITHER column exists, i.e. when there is no per-row
# basis for the eval/train call at all. A row whose confidence cannot be parsed
# routes train-only, like any other unreadable verdict.
STRICT_SUPPLEMENT_SPLIT_HINT = True
# ── Per-row provenance: RECORDED, NOT REQUIRED (study-project policy, 2026-09-21)
# The 2026-08-25 draft made every row carry a licence, an immutable source id, a
# capture family and a first-seen model version, and raised on the first row
# without them. Dataset v8 (3,389 rows) carries none of those columns, so that
# contract could not load the corpus every shipped bundle was trained on. The
# three STRICT_* knobs below now default to False: when a column is present it is
# validated and used exactly as before; when it is absent the row gets the
# documented fallback and the run prints ONE warning with the count. Set a knob
# to True only for a supplement revision that promises the column.
#   filter_verdict absent       -> every row is treated as 'keep' (v8 was QA'd
#                                  offline: docs/DATASET_LABELING_PLAN.md "Corpus
#                                  verification"); present -> non-keep rows are
#                                  skipped and bench_leak / phash_mismatch RAISE
#   license_id absent           -> recorded as 'unknown'; never a reason to skip
#   immutable_source_id absent  -> sha256 of the image bytes
#   capture_family_id absent    -> none; the row is its own group (v8 behaviour)
#   first_seen_model_version absent -> routing falls back to split_hint; the
#                                  permanent HOLDOUT carve (sha1 bucket, drawn
#                                  first) is what keeps a never-trained slice
# LIGHT COLLECTION (2.7.5). Nothing below stops a run any more; each case is
# resolved the safe way and counted in ONE summary line:
#   image with no manifest row  -> loads TRAIN-ONLY, class from its folder
#                                  (SUPPLEMENT_LOAD_UNLISTED_IMAGES). Dropping new
#                                  photos into <class>/ is a complete submission.
#                                  It can never become eval/holdout data by accident.
#   capture family disagreements -> the FAMILY routes as one unit: any train-side
#                                  or already-seen member makes every member
#                                  train-only; a family of eval rows stays eval
#                                  together (one split group, plant_id = family).
#                                  Families may span classes.
#   reused immutable_source_id  -> the later row is the same photograph again
#                                  and is skipped.
#   <name>.txt beside <name>.jpg -> YOLO boxes for the Stage 1 detector (below).
# The v8 holdout is untouched by all of this: v8 rows carry no family, no source
# id and no box file, so they route exactly as before.
STRICT_SUPPLEMENT_FILTER_VERDICT = False
SUPPLEMENT_REQUIRED_INDEPENDENCE_METHOD = 'capture_family_id+phash_v1'
STRICT_SUPPLEMENT_LICENSE_PROVENANCE = False
STRICT_SUPPLEMENT_MODEL_PROVENANCE = False
_SUPPLEMENT_LICENSE_TALLY = {}
_SUPPLEMENT_PROVENANCE_FALLBACKS = {}
_SUPPLEMENT_PROVENANCE_RESOLVED = {}
SUPPLEMENT_LOAD_UNLISTED_IMAGES = True
# ── Detector boxes from the supplement (2.7.5) ──────────────────────────
# Optional, per image: a YOLO label file beside the photo — <class>/<name>.txt
# for <class>/<name>.jpg, one `cls cx cy w h` row per box, normalised 0-1. That
# is the plain export of makesense.ai, CVAT, Label Studio and Roboflow, so no
# CSV editing is involved. Class ids are ignored (Stage 1 is single-class
# plant_leaf: box every leaf or damaged area you would want Stage 2 to look
# at); YOLO-seg polygon rows are reduced to their bounding box; a trailing
# confidence column is tolerated. Reading them costs one small text read per
# BOXED image, from the directory listing §2 already makes — no image is
# decoded. §2 writes them to ONE index file, and §3 hard-links the boxed
# train-split photos into the detector set (build_yolo_detection_dataset_*).
SUPPLEMENT_BOXES_INDEX = ROOT / '_supplement_boxes.json'
SUPPLEMENT_BOX_MIN_SIDE = 0.01      # normalised; smaller boxes are annotation slips

def _parse_yolo_boxes(text):
    """YOLO label text -> [(cx, cy, w, h), ...] normalised and clipped to the
    frame, plus the number of rows that were dropped as malformed. Rows with
    pixel coordinates (any value well outside 0-1) are dropped, not rescaled:
    without the image size they cannot be read correctly."""
    boxes, bad = [], 0
    for line in text.splitlines():
        parts = line.split()
        if not parts:
            continue
        try:
            vals = [float(v) for v in parts[1:]]
        except ValueError:
            bad += 1
            continue
        if len(vals) in (4, 5):                         # cx cy w h [conf]
            cx, cy, w, h = vals[:4]
            x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
            coords = vals[:4]
        elif len(vals) >= 6 and len(vals) % 2 == 0:     # polygon x1 y1 x2 y2 ...
            xs, ys = vals[0::2], vals[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
            coords = vals
        else:
            bad += 1
            continue
        if not all(math.isfinite(v) and -0.02 <= v <= 1.02 for v in coords):
            bad += 1
            continue
        x1, y1, x2, y2 = max(0.0, x1), max(0.0, y1), min(1.0, x2), min(1.0, y2)
        if x2 - x1 < SUPPLEMENT_BOX_MIN_SIDE or y2 - y1 < SUPPLEMENT_BOX_MIN_SIDE:
            bad += 1
            continue
        boxes.append((round((x1 + x2) / 2, 6), round((y1 + y2) / 2, 6),
                      round(x2 - x1, 6), round(y2 - y1, 6)))
    return boxes, bad
# Eval eligibility is relative to the newest model being judged. A row first
# seen by that model or any earlier release is train-only even when an old,
# permanent split_hint still says eval_ok. Use literal 'never' for genuinely
# unseen rows; a row without the column is routed by split_hint alone.
SUPPLEMENT_EVAL_THROUGH_MODEL_VERSION = MODEL_VERSION
def _supplement_version_key(value):
    m = re.fullmatch(r'v?(\d+)\.(\d+)\.(\d+)', str(value or '').strip())
    if not m:
        raise ValueError(f'expected semantic model version, got {value!r}')
    return tuple(map(int, m.groups()))
_SUPPLEMENT_EVAL_VERSION_KEY = _supplement_version_key(
    SUPPLEMENT_EVAL_THROUGH_MODEL_VERSION)
# Known licence ids -> URL, for the sidecar. A lookup table, not an allowlist.
SUPPLEMENT_KNOWN_LICENSES = {
    'cc0': 'https://creativecommons.org/publicdomain/zero/1.0/',
    'cc-by-2.0': 'https://creativecommons.org/licenses/by/2.0/',
    'cc-by-4.0': 'https://creativecommons.org/licenses/by/4.0/',
    'cc-by-sa-2.0': 'https://creativecommons.org/licenses/by-sa/2.0/',
    'cc-by-sa-4.0': 'https://creativecommons.org/licenses/by-sa/4.0/',
    'cc-by-nc-4.0': 'https://creativecommons.org/licenses/by-nc/4.0/',
    'cc-by-nc-sa-4.0': 'https://creativecommons.org/licenses/by-nc-sa/4.0/',
    'public-domain': None,
    'project-owned': None,
}
SUPPLEMENT_EVAL_CONF_FLOOR = 0.80
# Stale-upload guard. The offline rounds land in the Kaggle dataset only when
# `kaggle datasets version` is actually run, and a stale build is invisible: it
# loads fine, passes every gate, and quietly trains + grades on a previous era's
# corpus. On 2026-08 the published build was 2,539 rows (a v7-era snapshot with
# NO leaf_spot and no garden ornamentals) while the round notes described 3,125.
# Set to 0 to train deliberately on an older build.
SUPPLEMENT_MIN_ROWS = 3000
def fetch_kaggle_supplement(slug=KAGGLE_SUPPLEMENT_SLUG, per_class_cap=None,
                            label_confidence=0.85):
    """OUR eval-supplement -> TRAIN + held-out eval (EVAL-ELIGIBLE). Reads every
    present <canonical-class>/ folder (new classes/hard-negative folders are
    picked up automatically), SHA-dedups against every other source, prints
    per-class totals. Fail-soft. No per-class cap (per_class_cap=None): the supplement is fully curated + agent-judged + pHash-deduped, so a cap only silently discarded curated boundary hard-negatives (the precision lever)."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    # Route curated *boundary* hard-negatives to a TRAIN-ONLY source. Boundary
    # negs are the precision lever (near-miss healthy/not-a-plant that pins the
    # decision boundary); when they share source='supplement' with everything
    # else, make_*_split MOVES ~2/3 of them into the eval/val splits and only
    # the remainder trains -- so most of the lever never reaches the model.
    # 'supplement_boundary' is listed in NO make_*_split allowed_sources, so
    # every boundary neg stays in TRAIN. Fail-loud: a missing manifest / hn_kind
    # column / zero boundary rows means the tag we depend on is gone -- raise
    # rather than silently reverting to the eval-eligible behaviour.
    import csv as _csv
    _man = root / 'manifest.csv'
    if not _man.exists():
        raise RuntimeError(
            f'eval-supplement manifest.csv missing at {_man}: cannot route '
            'boundary hard-negatives train-only (tag depended on by v3 precision).')
    with open(_man, newline='') as _f:
        _rows = list(_csv.DictReader(_f))
    if not _rows or 'hn_kind' not in _rows[0]:
        raise RuntimeError(
            'eval-supplement manifest.csv has no hn_kind column: cannot route '
            'boundary hard-negatives train-only.')
    # Log what actually arrived BEFORE anything is routed: a stale upload is
    # otherwise indistinguishable from a current one in a 900-line log.
    _cols = list(_rows[0].keys())
    _cls_counts = {}
    for r in _rows:
        _c = (r.get('class') or '?').strip()
        _cls_counts[_c] = _cls_counts.get(_c, 0) + 1
    print(f'  Kaggle supplement manifest: {len(_rows)} rows, columns={_cols}')
    print(f'  Kaggle supplement manifest per-class: '
          f'{dict(sorted(_cls_counts.items(), key=lambda kv: -kv[1]))}')
    if SUPPLEMENT_MIN_ROWS and len(_rows) < SUPPLEMENT_MIN_ROWS:
        raise RuntimeError(
            f'STALE SUPPLEMENT UPLOAD: manifest.csv has {len(_rows)} rows, below the '
            f'{SUPPLEMENT_MIN_ROWS} floor. The Kaggle dataset is almost certainly an '
            'older build -- an offline collection round only reaches training once '
            '`kaggle datasets version -p ... -r zip` has actually run. Training on a '
            'previous era\'s corpus silently reverts the data fixes AND breaks every '
            'version-over-version A/B (two models on two corpora are not comparable). '
            'FIX: publish the current build, or lower/zero SUPPLEMENT_MIN_ROWS as a '
            'deliberate, reviewed edit.')
    # Join manifest rows to bytes by their COMPLETE POSIX-relative path. A
    # basename dict silently overwrote duplicate names in different classes or
    # subfolders, transferring confidence/split/boundary metadata across rows.
    _meta, _disk_stem_owner, _source_id_owner, _family_members = {}, {}, {}, {}
    for _line, r in enumerate(_rows, start=2):
        _raw = (r.get('filename') or '').strip().replace('\\', '/')
        _rel = PurePosixPath(_raw)
        _key = _rel.as_posix().lstrip('./')
        _cls = (r.get('class') or '').strip()
        if (not _key or _rel.is_absolute() or '..' in _rel.parts or
                not _cls or not _rel.parts or _rel.parts[0] != _cls):
            raise RuntimeError(
                f'eval-supplement manifest line {_line}: unsafe filename/class '
                f'identity: filename={_raw!r}, class={_cls!r}')
        if _key in _meta:
            raise RuntimeError(
                f'eval-supplement manifest has duplicate path key {_key!r}')
        _license_id = (r.get('license_id') or '').strip().lower()
        _source_url = (r.get('source_url') or '').strip()
        _creator = (r.get('creator') or '').strip() or None
        _source_id = (r.get('immutable_source_id') or '').strip()
        _family_id = (r.get('capture_family_id') or '').strip()
        _first_seen = (r.get('first_seen_model_version') or '').strip().lower()
        for _col, _val in (('immutable_source_id', _source_id),
                           ('capture_family_id', _family_id),
                           ('first_seen_model_version', _first_seen)):
            if not _val:
                _SUPPLEMENT_PROVENANCE_FALLBACKS[_col] = (
                    _SUPPLEMENT_PROVENANCE_FALLBACKS.get(_col, 0) + 1)
        if STRICT_SUPPLEMENT_MODEL_PROVENANCE and (
                not _source_id or not _family_id or not _first_seen):
            raise RuntimeError(
                f'eval-supplement manifest line {_line}: STRICT_SUPPLEMENT_MODEL_PROVENANCE '
                'requires immutable_source_id, capture_family_id and an explicit '
                "first_seen_model_version ('never' means no model has trained on it)")
        r['_duplicate_of'] = None
        if _source_id:
            if _source_id in _source_id_owner:
                # The same source photograph listed twice (a re-export, a
                # resize): keep the first row, skip this one.
                r['_duplicate_of'] = _source_id_owner[_source_id]
                _SUPPLEMENT_PROVENANCE_RESOLVED['duplicate immutable_source_id skipped'] = (
                    _SUPPLEMENT_PROVENANCE_RESOLVED.get(
                        'duplicate immutable_source_id skipped', 0) + 1)
            else:
                _source_id_owner[_source_id] = _key
        if _family_id:
            _family_members.setdefault(_family_id, []).append(_key)
        if not _first_seen or _first_seen == 'never':
            _seen_by_eval_model = False
        else:
            try:
                _seen_by_eval_model = (
                    _supplement_version_key(_first_seen) <= _SUPPLEMENT_EVAL_VERSION_KEY)
            except ValueError as e:
                raise RuntimeError(
                    f'eval-supplement manifest line {_line}: invalid '
                    f'first_seen_model_version={_first_seen!r}') from e
        _SUPPLEMENT_LICENSE_TALLY[_license_id or 'unknown'] = (
            _SUPPLEMENT_LICENSE_TALLY.get(_license_id or 'unknown', 0) + 1)
        if STRICT_SUPPLEMENT_LICENSE_PROVENANCE:
            if _license_id not in SUPPLEMENT_KNOWN_LICENSES or not _source_url:
                raise RuntimeError(
                    f'eval-supplement manifest line {_line}: STRICT_SUPPLEMENT_LICENSE_PROVENANCE '
                    f'requires a known license_id and a source_url (got '
                    f'license_id={_license_id!r}, source_url={_source_url!r})')
            if _license_id.startswith('cc-by') and not _creator:
                raise RuntimeError(
                    f'eval-supplement manifest line {_line}: {_license_id} requires creator')
        r['_license_id'] = _license_id or 'unknown'
        r['_license_url'] = ((r.get('license_url') or '').strip() or
                             SUPPLEMENT_KNOWN_LICENSES.get(_license_id))
        r['_source_url'] = _source_url or None
        r['_creator'] = _creator
        r['_immutable_source_id'] = _source_id or None
        r['_capture_family_id'] = _family_id or None
        r['_first_seen_model_version'] = _first_seen or None
        r['_seen_by_eval_model'] = _seen_by_eval_model
        _disk_stem = re.sub(
            r'[^A-Za-z0-9_.-]', '_',
            'supp_' + PurePosixPath(_key).with_suffix('').as_posix())
        if _disk_stem in _disk_stem_owner:
            raise RuntimeError(
                f'eval-supplement paths {_disk_stem_owner[_disk_stem]!r} and '
                f'{_key!r} collapse to the same on-disk stem {_disk_stem!r}')
        _disk_stem_owner[_disk_stem] = _key
        r['_disk_stem'] = _disk_stem
        _meta[_key] = r
    # One member of a capture family in a past training set means the family is
    # not unseen: the whole family inherits the earliest first-seen.
    for _fam, _keys in _family_members.items():
        _seen = [_meta[k]['_seen_by_eval_model'] for k in _keys]
        if any(_seen) and not all(_seen):
            for k in _keys:
                _meta[k]['_seen_by_eval_model'] = True
            _SUPPLEMENT_PROVENANCE_RESOLVED['family rows marked seen with their family'] = (
                _SUPPLEMENT_PROVENANCE_RESOLVED.get(
                    'family rows marked seen with their family', 0) + _seen.count(False))
    _has_filter = 'filter_verdict' in _rows[0]
    if not _has_filter:
        _msg = ('eval-supplement manifest.csv has no filter_verdict column: every row '
                'is treated as keep; the offline QA (tools/filter_supplement.py) is not '
                'provable from this manifest.')
        if STRICT_SUPPLEMENT_FILTER_VERDICT:
            raise RuntimeError(_msg + ' STRICT_SUPPLEMENT_FILTER_VERDICT=True refuses it.')
        print('  WARN: ' + _msg)
    _methods = {(r.get('filter_independence_method') or '').strip() for r in _rows}
    if _methods != {SUPPLEMENT_REQUIRED_INDEPENDENCE_METHOD}:
        _msg = (f'eval-supplement independence methods {sorted(_methods)!r}; '
                f'{SUPPLEMENT_REQUIRED_INDEPENDENCE_METHOD!r} is the one that catches '
                'rotated/noised capture families, so pHash-only rows may hide siblings.')
        if STRICT_SUPPLEMENT_FILTER_VERDICT:
            raise RuntimeError(_msg)
        print('  WARN: ' + _msg)
    if _SUPPLEMENT_PROVENANCE_FALLBACKS:
        print(f'  provenance defaults applied (blank/absent columns, normal for v8 and '
              f'for light rows): {_SUPPLEMENT_PROVENANCE_FALLBACKS}')
    print(f'  Kaggle supplement licence tally: {dict(sorted(_SUPPLEMENT_LICENSE_TALLY.items()))}')
    _filter_counts = {}
    if _has_filter:
        for r in _rows:
            _v = (r.get('filter_verdict') or '').strip().lower() or 'missing'
            _filter_counts[_v] = _filter_counts.get(_v, 0) + 1
        if _filter_counts.get('bench_leak', 0):
            raise RuntimeError(
                f'eval-supplement contains {_filter_counts["bench_leak"]} '
                'benchmark-leak row(s); refusing to contaminate train/eval.')
        # The vocabulary server/tools/filter_supplement.py writes into this
        # column, plus the older labeller spellings. The tool's manifests used
        # to fail here on 'dup_intra' / 'tiny' / 'corrupt' — a contract break
        # between the QA tool and the notebook that consumes its output.
        _KNOWN_FILTER_VERDICTS = {
            'keep', 'blurry', 'bench_leak',
            'too_small', 'tiny', 'duplicate', 'dup_intra', 'dup_cross',
            'missing', 'corrupt', 'phash_mismatch',
        }
        if _filter_counts.get('phash_mismatch', 0):
            raise RuntimeError(
                f'eval-supplement has {_filter_counts["phash_mismatch"]} row(s) whose '
                'bytes are not the photograph the manifest describes; re-publish it.')
        _unknown_filter = set(_filter_counts) - _KNOWN_FILTER_VERDICTS
        if _unknown_filter:
            raise RuntimeError(
                f'eval-supplement has unknown filter_verdict values: '
                f'{sorted(_unknown_filter)}')
        print(f'  Kaggle supplement filter verdicts: {_filter_counts}')
    _boundary = {n for n, r in _meta.items()
                 if (not _has_filter or (r.get('filter_verdict') or '').strip().lower() == 'keep')
                 if (r.get('hn_kind') or '').strip() == 'boundary'}
    if not _boundary:
        raise RuntimeError(
            'eval-supplement manifest.csv lists 0 boundary rows: cannot route '
            'boundary hard-negatives train-only.')
    _has_hint = 'split_hint' in _rows[0]
    _has_conf = 'confidence' in _rows[0]
    _basis = ('split_hint' if _has_hint else
              f'confidence>={SUPPLEMENT_EVAL_CONF_FLOOR}' if _has_conf else 'none')
    if STRICT_SUPPLEMENT_SPLIT_HINT and _basis == 'none':
        raise RuntimeError(
            'eval-supplement manifest.csv has NEITHER a split_hint NOR a confidence '
            "column: there is no per-row basis for the eval/train call, so every "
            'sub-floor positive would drain into the held-out splits and grade + '
            'CALIBRATE the model that ships. FIX: publish a supplement build that '
            'writes one of the two columns, or set STRICT_SUPPLEMENT_SPLIT_HINT='
            'False to accept the old (every-non-boundary-row-eval-eligible) '
            'behaviour deliberately.')
    print(f'  Kaggle supplement: eval/train routing basis = {_basis}')
    if _basis == 'none':
        print('  WARN: no routing basis — every non-boundary row stays '
              'eval-eligible (pre-2.6.13 behaviour)')

    def _route(name):
        """-> (source_tag, subject_domain, species, label_confidence) for one file.

        Routing precedence: boundary hard-negatives first (the precision lever,
        always TRAIN-ONLY at its own low sampler weight), then any row already
        seen by the candidate/reference-model era, then the manifest's
        own split_hint. Anything that is not explicitly 'eval_ok' — 'train_only',
        blank, or a hint this notebook has never seen — routes TRAIN-ONLY: an
        unrecognised verdict must never PROMOTE a row into a held-out slice.
        Nothing is dropped; train-only rows keep the full 'supplement' sampler
        weight, which is also what protects aqua's data-hungry distillation
        volume (docs/DATASET_LABELING_PLAN.md v8 item 3)."""
        if name not in _meta:
            raise RuntimeError(
                f'eval-supplement image {name!r} has no manifest row')
        r = _meta[name]
        try:
            conf = min(max(float((r.get('confidence') or '').strip()), 0.0), 1.0)
            conf_ok = True
        except ValueError:
            conf, conf_ok = label_confidence, False
        hint = (r.get('split_hint') or '').strip().lower()
        if name in _boundary:
            src = 'supplement_boundary'
        elif r.get('_seen_by_eval_model'):
            src = 'supplement_train_only'
        elif _basis == 'none':
            src = 'supplement'
        elif _has_hint:
            src = 'supplement' if hint == 'eval_ok' else 'supplement_train_only'
        else:
            # Derived basis: an unparseable confidence is an unreadable verdict,
            # so it routes train-only like an unrecognised split_hint.
            src = ('supplement' if conf_ok and conf >= SUPPLEMENT_EVAL_CONF_FLOOR
                   else 'supplement_train_only')
        # Recorded verbatim — no allowlist, no mapping. A new upstream bucket
        # (e.g. 'garden_edible' for the kitchen-garden broadening) shows up in
        # the split reports the run after it is collected, with no code change.
        dom = (r.get('subject_domain') or '').strip() or None
        plant = (r.get('subject_plant') or '').strip()
        species = f'{plant or dom or "plant"} (eval-supplement)'
        return src, dom, species, conf

    print(f'  Kaggle supplement: routing {len(_boundary)} boundary '
          'hard-negatives to TRAIN-ONLY (source=supplement_boundary)')
    # Route every listed row up front so a capture family moves as ONE unit.
    # A family with any train-side member (boundary, train_only, already seen)
    # trains as a whole — an eval row with a training sibling would grade the
    # model on a photograph it learned. Otherwise its eval rows stay eval
    # together: they share plant_id, so every split keeps them in one group.
    _planned = {}
    for _k, _r in _meta.items():
        if _r.get('_duplicate_of'):
            continue
        if _has_filter and (_r.get('filter_verdict') or '').strip().lower() != 'keep':
            continue
        _planned[_k] = _route(_k)
    _fam_srcs = {}
    for _k, _plan in _planned.items():
        _f = _meta[_k].get('_capture_family_id')
        if _f:
            _fam_srcs.setdefault(_f, set()).add(_plan[0])
    for _k, _plan in list(_planned.items()):
        _f = _meta[_k].get('_capture_family_id')
        if _f and _plan[0] == 'supplement' and len(_fam_srcs[_f]) > 1:
            _planned[_k] = ('supplement_train_only',) + tuple(_plan[1:])
            _SUPPLEMENT_PROVENANCE_RESOLVED['eval rows trained with their family'] = (
                _SUPPLEMENT_PROVENANCE_RESOLVED.get('eval rows trained with their family', 0) + 1)
    _multi_eval_families = sum(
        1 for _f, _ks in _family_members.items()
        if _fam_srcs.get(_f) == {'supplement'} and len(_ks) > 1)

    def _unlisted_row(key):
        stem = re.sub(r'[^A-Za-z0-9_.-]', '_',
                      'supp_' + PurePosixPath(key).with_suffix('').as_posix())
        if stem in _disk_stem_owner:
            return None
        _disk_stem_owner[stem] = key
        return {'_disk_stem': stem, '_license_id': 'unknown', '_license_url': None,
                '_source_url': None, '_creator': None, '_immutable_source_id': None,
                '_capture_family_id': None, '_first_seen_model_version': None,
                '_seen_by_eval_model': False, '_duplicate_of': None, '_unlisted': True}

    saved_per_class, routed, domains = {}, {}, {}
    _family_sources = {}   # capture_family_id -> {source tags it was routed to}
    _box_index, _box_stats = {}, {'label_files': 0, 'boxed_images': 0, 'boxes': 0,
                                  'rows_dropped': 0, 'files_without_a_box': 0}
    _listed_on_disk = set()
    for cls in CLASSES:
        cls_dir = root / cls
        if not cls_dir.is_dir():
            continue
        # ONE listing per class: images, plus any YOLO label files beside them.
        _listing = sorted(p for p in cls_dir.rglob('*') if p.is_file())
        files = [p for p in _listing if p.suffix.lower() in IMG_EXTS]
        _label_files = {p.with_suffix(''): p for p in _listing if p.suffix.lower() == '.txt'}
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        got = 0
        for path in files:
            if per_class_cap is not None and got >= per_class_cap:
                break
            _key = path.relative_to(root).as_posix()
            _row = _meta.get(_key)
            if _row is None:
                if not SUPPLEMENT_LOAD_UNLISTED_IMAGES:
                    raise RuntimeError(
                        f'eval-supplement image {_key!r} has no manifest row')
                _row = _unlisted_row(_key)
                if _row is None:
                    _SUPPLEMENT_PROVENANCE_RESOLVED['unlisted images skipped (name clash)'] = (
                        _SUPPLEMENT_PROVENANCE_RESOLVED.get(
                            'unlisted images skipped (name clash)', 0) + 1)
                    continue
                _meta[_key] = _row
                # Never eval: an unlisted photo has no eval verdict, whatever the basis.
                _planned[_key] = ('supplement_train_only', None,
                                  'plant (eval-supplement)', label_confidence)
                _SUPPLEMENT_PROVENANCE_RESOLVED['unlisted images loaded train-only'] = (
                    _SUPPLEMENT_PROVENANCE_RESOLVED.get('unlisted images loaded train-only', 0) + 1)
            else:
                _listed_on_disk.add(_key)
            if _key not in _planned:        # non-keep verdict, or a duplicate source id
                continue
            try:
                body = path.read_bytes()
            except Exception:
                continue
            _src, _dom, _species, _conf = _planned[_key]
            _family_id = _row.get('_capture_family_id')
            if _family_id:
                _family_sources.setdefault(_family_id, set()).add(_src)
            _boxes = []
            _lbl = _label_files.get(path.with_suffix(''))
            if _lbl is not None:
                _box_stats['label_files'] += 1
                try:
                    _boxes, _bad = _parse_yolo_boxes(
                        _lbl.read_text(encoding='utf-8', errors='replace'))
                except OSError:
                    _boxes, _bad = [], 1
                _box_stats['rows_dropped'] += _bad
                if not _boxes:
                    _box_stats['files_without_a_box'] += 1
            _saved = _save_archive_member(
                cls, _src, f'supp_{cls}', path.name, body,
                label_confidence=_conf, species=_species, subject_domain=_dom,
                license_id=_row.get('_license_id'),
                license_url=_row.get('_license_url'),
                creator=_row.get('_creator'), source_url=_row.get('_source_url'),
                attribution=((_row.get('attribution') or '').strip() or
                             _row.get('_creator')),
                immutable_source_id=_row.get('_immutable_source_id'),
                capture_family_id=_family_id,
                first_seen_model_version=_row.get('_first_seen_model_version'),
                output_stem=_row['_disk_stem'], exif_transpose=bool(_boxes))
            if _boxes and (TRAIN_ROOT / cls / f"{_row['_disk_stem']}.jpg").is_file():
                _box_index[f"{cls}/{_row['_disk_stem']}.jpg"] = {
                    'group': _family_id or _row['_disk_stem'], 'boxes': _boxes}
                _box_stats['boxed_images'] += 1
                _box_stats['boxes'] += len(_boxes)
            if _saved:
                routed[_src] = routed.get(_src, 0) + 1
                domains[_dom or 'unspecified'] = domains.get(_dom or 'unspecified', 0) + 1
            got += _saved
        saved_per_class[cls] = got
    # Invariant of the family routing above, checked on what was actually
    # saved: an eval_ok row and a train-side sibling carry DIFFERENT source
    # tags, so the (source, plant_id) split guard would not keep them apart.
    _straddling = sorted(f for f, s in _family_sources.items()
                         if 'supplement' in s and len(s) > 1)
    if _straddling:
        raise RuntimeError(
            f'eval-supplement family routing broke: {len(_straddling)} capture '
            f'family(ies) hold an eval row AND a train-side sibling (e.g. {_straddling[:5]})')
    SUPPLEMENT_BOXES_INDEX.parent.mkdir(parents=True, exist_ok=True)
    SUPPLEMENT_BOXES_INDEX.write_text(json.dumps(_box_index, sort_keys=True), encoding='utf-8')
    print(f'  Kaggle supplement detector boxes: {_box_stats} -> {SUPPLEMENT_BOXES_INDEX}')
    if _multi_eval_families:
        print(f'  Kaggle supplement: {_multi_eval_families} capture family(ies) hold several '
              'eval rows; each stays in one split group')
    if _SUPPLEMENT_PROVENANCE_RESOLVED:
        print(f'  Kaggle supplement provenance, resolved without stopping: '
              f'{dict(sorted(_SUPPLEMENT_PROVENANCE_RESOLVED.items()))}')
    _listed_missing = [k for k in _planned
                       if not _meta[k].get('_unlisted') and k not in _listed_on_disk]
    if _listed_missing:
        print(f'  WARN: {len(_listed_missing)} kept manifest row(s) have no image under a '
              f'class folder (e.g. {_listed_missing[:3]}) — a stale manifest, or files '
              'outside CLASSES; they are not loaded')
    # Handed to the end of §2, which writes the re-stamped next manifest.
    globals()['_SUPPLEMENT_NEXT'] = {'columns': _cols, 'rows': list(_meta.items()),
                                     'has_hint': _has_hint, 'has_conf': _has_conf}
    total = sum(saved_per_class.values())
    print(f'  Kaggle supplement per-class: {saved_per_class}')
    print(f'  Kaggle supplement routing: {dict(sorted(routed.items(), key=lambda kv: -kv[1]))}')
    print(f'  Kaggle supplement subject_domain: '
          f'{dict(sorted(domains.items(), key=lambda kv: -kv[1]))}')
    print(f'  Kaggle supplement total: += {total}')
    if total and not routed.get('supplement'):
        raise RuntimeError(
            f'EVAL-ELIGIBLE SUPPLEMENT EMPTY: {total} supplement rows loaded but '
            "NONE is eval-eligible — no row carries split_hint='eval_ok', so the "
            'weak-class ship-gate slices AND the temperature/conformal calibration '
            'sets collapse to their pre-supplement state. FIX: check the offline '
            "pipeline's eval/train verdict, or set STRICT_SUPPLEMENT_SPLIT_HINT="
            'False for a deliberate run against an older supplement version.')
    return total


# ── Run the pipeline ────────────────────────────────────────────
def _try_fetch(tag, fn, *args, **kwargs):
    """Run a fetcher, log + continue on any exception. Upstream availability
    is the dominant failure mode for §2; a single transient failure after
    the slow fetches wastes ten minutes of work.

    NOTE: the first param is ``tag`` (not ``label``) on purpose — a fetcher
    (``fetch_doctorp_disease``) takes its own ``label=`` kwarg, and naming this
    one ``label`` shadowed it (``multiple values for argument 'label'``)."""
    try:
        return fn(*args, **kwargs)
    except KeyboardInterrupt:
        raise
    except Exception as exc:  # noqa: BLE001 — by design; see above
        print(f'{tag} skipped: {type(exc).__name__}: {exc}')
        log_run({'phase': 'fetch_skipped', 'label': tag,
                 'error_type': type(exc).__name__, 'error': str(exc)})
        return None

_try_fetch('inat_supplement', fetch_inat_supplement,
           healthy_per_taxon=180, botrytis_per_taxon=2400)
# not_a_plant is over-served (P 0.99/R 0.97); halved 1500->750 in the
# 2026-06-19 data audit to rebalance the sampler budget toward weak classes
# (the OOD bank still draws from coco + the green-texture/partial-plant hard-negs).
_try_fetch('coco_nap', fetch_coco_for_nap, target_n=750)
_try_fetch('inat_animalia_nap', fetch_inat_animalia_for_nap, target_n=500)
_try_fetch('openimages_indoor', fetch_openimages_indoor, target_n=600)
_try_fetch('coco_hard_negatives', fetch_coco_hard_negatives, target_n=600)
_try_fetch('inat_hard_negatives', fetch_inat_hard_negatives, per_taxon=200, target_n=900)
_try_fetch('plantdoc', fetch_plantdoc,
           max_train_per_class=500, max_val_per_class=120)
_try_fetch('plantvillage', fetch_plantvillage_into_train_and_studio,
           max_train_per_class=900, max_studio_per_class=300)
if INCLUDE_PHYSIOLOGICAL_CLASSES:
    # Stage 0: higher target so the broadened query set is fully drawn.
    _try_fetch('inat_physiological', fetch_inat_physiological, per_class=1500)
    if INCLUDE_ABIOTIC_SUPPLEMENTS:
        # nutrient_deficiency's crop signal comes from MPLD + the Kaggle
        # nutrient/ornamental sets + DoctorP. (CoLeaf-DB was removed: it 403'd
        # from Colab/datacenter IPs and never downloaded.)
        # MPLD (maize P-deficiency, Zenodo) downloads fine. Capped LOW on
        # purpose: a crop-domain representation aid for the deficiency *signal*,
        # not in-domain samples — kept comparable to the other thin classes so
        # it doesn't inflate nutrient_deficiency's logit-adjustment prior.
        _try_fetch('mpld_phosphorus_deficiency', fetch_mpld_phosphorus_deficiency,
                   cap=150)

if INCLUDE_KAGGLE_SUPPLEMENTS:
    # Operator opt-in; fail-soft without Kaggle creds. Runs BEFORE the
    # pre-split dedup so its rows are deduped + train-scoped like the rest.
    _try_fetch('kaggle_houseplant', fetch_kaggle_houseplant_disease,
               per_class_cap=250)
    # NOTE: do not re-add a rice/field-crop nutrient source here. The
    # 2026-06-19 data-quality audit dropped one because field-crop imagery fed
    # nutrient_deficiency -- the class the supplement is being de-cropped to
    # fix, and still the weakest on precision.
    # Curated real-houseplant species photos -> healthy (variegation diversity).
    # SKIPPED by default: it's a 4.85 GB Kaggle download — the single biggest
    # time sink in this cell (~5 min, bandwidth-bound) — for only ~400 images of
    # the ALREADY-abundant `healthy` class. iNat (Ficus/Calathea/Sansevieria/…)
    # + PlantVillage + ornamental already supply healthy variegation. Flip to
    # True to re-enable.
    INCLUDE_KAGGLE_HEALTHY_SPECIES = False
    if INCLUDE_KAGGLE_HEALTHY_SPECIES:
        _try_fetch('kaggle_houseplant_species', fetch_kaggle_houseplant_species,
                   cap=400)
    # v2.4: ornamental/houseplant disease set (rose/hibiscus/money-plant) —
    # real disease morphology on natural backgrounds for the weak classes.
    _try_fetch('kaggle_ornamental', fetch_kaggle_ornamental_disease,
               per_class_cap=600)   # pre-v5: top up weak classes (scorch/viral/powdery/nutrient) from existing headroom
    # v2.4: DoctorP — REAL plant-doctor app photos. Supplies the starved
    # classes (gray mold / mealybug / scale / nutrient deficiency) in their
    # deployment framing AND is eval-eligible (seeds the real-photo eval slice).
    _try_fetch('doctorp', fetch_doctorp_disease, per_class_cap=400)
    # v2.6: pepper sibling (same DoctorP project) — REAL, higher-res, EVAL-
    # ELIGIBLE photos that thicken the thinnest eval slice (nutrient_deficiency)
    # plus viral/powdery/leaf_spot/scorch. Tagged 'doctorp_pepper' so it joins
    # the held-out splits like 'doctorp'. Runs before the pre-split dedup.
    _try_fetch('doctorp_pepper', fetch_doctorp_disease,
               slug=DOCTORP_PEPPER_SLUG, source='doctorp_pepper',
               label='DoctorP-pepper', per_class_cap=400)

    # Disk hygiene. Every Kaggle source has now copied what it needs into
    # TRAIN_ROOT, so the kagglehub cache (~/.cache/kagglehub: indoor ~2.3 GB +
    # ornamental ~4 GB extracted) is dead weight for the rest of the run.
    # Nothing downstream reads it, but purging costs a full re-download next
    # run -- hence conditional on disk actually being tight. Best-effort.
    try:
        _kcache = Path.home() / '.cache' / 'kagglehub'
        _free_gb = shutil.disk_usage(Path.home()).free / 1e9
        _tight = _free_gb < PURGE_KAGGLE_CACHE_MIN_FREE_GB
        _purge = PURGE_KAGGLE_CACHE is True or (PURGE_KAGGLE_CACHE == 'auto' and _tight)
        if _purge and _kcache.exists():
            _freed = sum(f.stat().st_size for f in _kcache.rglob('*') if f.is_file())
            shutil.rmtree(_kcache, ignore_errors=True)
            print(f'  purged kagglehub cache (~{_freed/1e9:.1f} GB freed; '
                  f'{_free_gb:.0f} GB was free)')
        elif _kcache.exists():
            print(f'  kagglehub cache kept ({_free_gb:.0f} GB free) — a re-run '
                  'skips the ~5 GB Kaggle re-download')
    except Exception as _e:
        print(f'  kagglehub cache purge skipped: {type(_e).__name__}: {_e}')

# OUR eval-supplement (in-domain, agent-collected) — always run: it is the
# in-domain supply for the weak classes and the EVAL-ELIGIBLE source that
# thickens the thin held-out slices. Train-scoped + deduped like the rest.
#
# CALLED DIRECTLY, NOT VIA _try_fetch. _try_fetch swallows every exception so a
# transient upstream outage doesn't waste ten minutes of fetching — correct for
# an optional source, WRONG here. This fetcher's raises are data-integrity
# assertions (missing manifest, missing hn_kind tag, no routing basis, stale
# upload) and REQUIRE_SUPPLEMENT makes a supplement-less corpus fatal anyway. Run
# through _try_fetch they printed one 'skipped' line and the run continued to die
# ~200 lines later on `supplement_boundary == 0` -- a message that misdiagnoses
# the cause as a missing hn_kind tag whatever the real failure was. Let them out.
fetch_kaggle_supplement(per_class_cap=None)

# 'pests' is a first-class trainable class — always fetch its supply.
_try_fetch('inat_pests', fetch_inat_pests, per_taxon=800, target_per_class=900)

# Dedup BEFORE splitting. The previous order — split first, dedup TRAIN_ROOT
# in §3 — left near-duplicates straddling train and held-out splits because
# make_group_split moves bytes by (source, plant_id) and only a SHA-256
# audit catches identical bytes (not pHash near-dupes).
_dedup_train_in_place(TRAIN_ROOT, phash_threshold=6)

# The holdout draws FIRST — see make_holdout's docstring. Order is the guarantee.
make_holdout()
make_val_realworld(); make_val_indomain(); make_temperature_cal(); make_conformal_cal()

def report_class_counts(root):
    print(f'\n{root}:')
    for c in CLASSES:
        print(f'  {c:25s} {count_images(root/c):5d}')
for r in (TRAIN_ROOT, VAL_PRIMARY, VAL_INDOMAIN, TEMP_CAL, CONFORMAL_CAL, VAL_STUDIO,
          HOLDOUT):
    report_class_counts(r)

# Per-class minimum training count. Most classes share the 150 default —
# the smallest count where KD distillation has produced trainable signal
# in practice for this taxonomy. ``gray_mold_botrytis`` is supplied by
# iNat alone (PlantDoc has no Botrytis label, PV's standard release has
# no Botrytis label) and ships chronically thin. Measured May 2026: the
# full Botrytis taxa set yields ~138 foliage images after the
# green-content gate, of which ~81 land in train after the four-way
# split drainage — that is the genuine iNat-only ceiling, not a
# starvation symptom. The 70 floor sits ~11 rows below it so a normal
# run passes while a real iNat outage (ConnectTimeout truncation,
# rate-limit wall) still fails loud. The §6 ship gate — disease
# macro-F1 ≥ 0.70 with no zero-recall class — is the real quality
# backstop for an under-supplied class. Openverse used to top this
# class up; removed in v2.2 because its anonymous tier was rate-capped
# below the notebook's burst rate.
THRESH_TRAIN_DEFAULT = 150
THRESH_TRAIN_PER_CLASS = {
    # v2.4: gray_mold is no longer iNat-only. It is now fed by (a) iNat
    # research-grade botrytis at 3 photos/obs (~1.6x the old ~138 ceiling),
    # (b) DoctorP real app photos (Grey mold + Botrytis cinerea, ~96).
    # Floor kept low
    # so a partial-source outage still trains, but the realistic train
    # count is several hundred; the ship-gate per-class recall CI on
    # val_realworld (now seeded with REAL DoctorP gray-mold rows) is the
    # honest quality backstop for this structurally-hardest class.
    'gray_mold_botrytis': 70,
    # 60 is the floor at which source-weighted sampling produces trainable
    # signal for thin iNat-free-text classes; nutrient_deficiency is now
    # additionally supplemented by the curated MPLD set (train-only).
    'leaf_scorch_underwatering': 90,
    'nutrient_deficiency': 90,
    # 'pests' is a strong, broadly-supplied class — it uses the
    # THRESH_TRAIN_DEFAULT (150) floor like the other real classes.
}
THRESH_VAL = 5

def _train_floor_for(cls):
    return THRESH_TRAIN_PER_CLASS.get(cls, THRESH_TRAIN_DEFAULT)

under_train = [c for c in CLASSES if count_images(TRAIN_ROOT/c) < _train_floor_for(c)]
under_val   = [c for c in CLASSES if count_images(VAL_PRIMARY/c) < THRESH_VAL]
if under_train or under_val:
    msg = []
    if under_train:
        floors = {c: _train_floor_for(c) for c in under_train}
        msg.append(f'TRAIN_ROOT under per-class floor: {floors}')
    if under_val:   msg.append(f'VAL_PRIMARY < {THRESH_VAL}: {under_val}')
    raise RuntimeError('\n'.join(msg) +
        '\nTRAIN_ROOT < threshold means an upstream source for that class is '
        'starved. For gray_mold_botrytis the supply chain is iNat alone '
        '(PlantDoc has no Botrytis label); other classes pull from '
        'PlantDoc + PV + iNat. Check the per-source log lines above for '
        'download failures or rate-limit backoffs, then re-run §2 or '
        'drop additional real-world photos into TRAIN_ROOT/<class>/.')

# ── Leakage guard + per-source coverage report (v2.5) ───────────
# Group-aware splits MOVE each (source, plant_id) group wholesale out of a
# shrinking TRAIN_ROOT, so by construction no group can span two splits. This
# is the runtime assertion that PROVES it for this run: a future refactor that
# accidentally copies instead of moves, or splits before dedup, trips here
# instead of silently inflating eval scores.
_SPLIT_ROOTS = {'train': TRAIN_ROOT, 'val_realworld': VAL_PRIMARY, 'holdout': HOLDOUT,
                'val_indomain': VAL_INDOMAIN, 'temperature_cal': TEMP_CAL,
                'conformal_cal': CONFORMAL_CAL}
_group_to_splits, _per_source, _per_domain = {}, {}, {}
_per_license, _missing_rights = {}, {}
_RIGHTS_REQUIRED_SOURCES = {
    'inat', 'inat_animalia', 'inat_physiological', 'inat_hard_greentexture',
    'inat_pest', 'coco', 'coco_hard_partial_plant', 'openimages_indoor',
    'supplement', 'supplement_train_only', 'supplement_boundary',
}
for _split, _root in _SPLIT_ROOTS.items():
    for _cls in CLASSES:
        _d = _root / _cls
        if not _d.exists():
            continue
        for _img in iter_images(_d):
            _sc = read_sidecar(_img) or {}
            _src = _sc.get('source', 'unknown')
            _gid = (_src, _sc.get('plant_id', _img.stem))
            _group_to_splits.setdefault(_gid, set()).add(_split)
            _per_source[_src] = _per_source.get(_src, 0) + 1
            _lic = (_sc.get('license_id') or '').strip().lower()
            if _lic:
                _per_license[_lic] = _per_license.get(_lic, 0) + 1
            elif _src in _RIGHTS_REQUIRED_SOURCES:
                _missing_rights[_src] = _missing_rights.get(_src, 0) + 1
            _dom = _sc.get('subject_domain') or 'unspecified'
            _per_domain.setdefault(_split, {})
            _per_domain[_split][_dom] = _per_domain[_split].get(_dom, 0) + 1
if _missing_rights:
    print(f'  provenance: license_id absent on {_missing_rights} rows (per source) — '
          'recorded, not gated; cached legacy rows carry no licence column')
_leaked = {g: sorted(v) for g, v in _group_to_splits.items() if len(v) > 1}
if _leaked:
    _ex = list(_leaked.items())[:5]
    raise RuntimeError(
        f'DATA LEAKAGE: {len(_leaked)} (source, plant_id) group(s) span multiple '
        f'splits - e.g. {_ex}. Splits must be disjoint by group; verify dedup ran '
        'BEFORE make_*_split and that splits MOVE (not copy) files.')
print('leakage guard OK - every (source, plant_id) group lands in exactly one split')
print('per-source image counts:', dict(sorted(_per_source.items(), key=lambda kv: -kv[1])))

# ── Per-split subject_domain report (v2.6.12+ domain broadening) ─
# The deployment domain is widening — houseplants -> outdoor/garden ornamentals
# -> kitchen-garden edibles (cucumber & co.) — but until now NOTHING in a run
# said how much of each split actually IS the new domain. That is the same
# blind spot that let a 2.26k-image train-only source vanish unnoticed: what the
# gate cannot see, the gate cannot defend. 'unspecified' is the honest label for
# every source that carries no per-row domain (all the public corpora); read the
# val_realworld line to know whether the ship gate can measure the pivot at all.
print('per-split subject_domain:')
for _split in _SPLIT_ROOTS:
    _d = dict(sorted(_per_domain.get(_split, {}).items(), key=lambda kv: -kv[1]))
    print(f'  {_split:16s} {_d}')

# ── Eval-supplement presence gate (pre-A100 safety) ─────────────
# OUR eval-supplement (maxayyt5/houseplant-disease-eval-supplement, source
# 'supplement') is the in-domain houseplant source that BOTH seeds the
# eval-eligible weak-class ship-gate slices (gray_mold / leaf_scorch /
# nutrient_deficiency) AND carries the v3 precision hard-negatives (boundary
# `healthy` + `not_a_plant` textures). It is fetched FAIL-SOFT, so a Kaggle
# 403 prints a single 'skipping' line among hundreds and the run proceeds on a
# supplement-less corpus: the weak-class eval collapses to the v2.5.0-era
# thinness and the leaf_scorch precision fix is absent, yet the ship gate still
# likely passes -- so the regression below v2.6.0 ships SILENTLY. On an
# expensive A100 run that is far too easy to miss, so fail fast here, before
# training. Override only for a deliberate supplement-less experiment.
REQUIRE_SUPPLEMENT = True
_supp_n = _per_source.get('supplement', 0)
_supp_boundary_n = _per_source.get('supplement_boundary', 0)
_supp_train_only_n = _per_source.get('supplement_train_only', 0)
if REQUIRE_SUPPLEMENT and _supp_boundary_n == 0:
    raise RuntimeError(
        'BOUNDARY LEVER MISSING: source "supplement_boundary" contributed 0 '
        'images. The eval-supplement loaded but the train-only boundary routing '
        'produced nothing -- the manifest hn_kind=boundary tag is likely gone. '
        'The v3 precision hard-negatives depend on it. FIX: confirm the private '
        'dataset manifest.csv still tags boundary rows, re-run \u00a72.')
if REQUIRE_SUPPLEMENT and _supp_n == 0:
    raise RuntimeError(
        'EVAL-SUPPLEMENT MISSING: source "supplement" contributed 0 images. '
        'maxayyt5/houseplant-disease-eval-supplement did not load -- almost '
        'always a Kaggle auth/permission 403 on the PRIVATE dataset (note the '
        'other, public, Kaggle sources above DID load, so the KAGGLE_API_TOKEN '
        'is valid but its account lacks access to THIS dataset). The weak-class '
        'eval slices and the v3 precision hard-negatives depend on it; training '
        'without it regresses below v2.6.0. FIX: set the Colab secret '
        'KAGGLE_API_TOKEN to the account that owns/can read the private dataset, '
        're-run §2, and confirm this prints a non-zero count. Set '
        'REQUIRE_SUPPLEMENT=False only for a deliberate supplement-less run.')
print(f'eval-supplement OK: {_supp_n} eval-eligible images from source '
      f'"supplement" (+{_supp_train_only_n} split_hint=train_only, '
      f'+{_supp_boundary_n} boundary — both TRAIN-ONLY)')

# ── Required-source floors (2026-07-24) ─────────────────────────
# GENERALISES the supplement gate above to every load-bearing source.
#
# WHY: on 2026-07-20 the ornamental slug `pappu54/indoor-plant-disease-dataset-
# 23-classes` began returning HTTP 403 (owner deleted it). `_kaggle_download` is
# fail-soft, so the run printed ONE 'skipping' line and trained on a corpus
# missing 2,264 TRAIN images — 600 each of leaf_spot / viral_disease /
# powdery_mildew / healthy. Because 'kaggle_ornamental' is TRAIN-ONLY, the
# val_realworld ship gate could not see the loss and PASSED; the damage showed
# up only in val_indomain (disease macro-F1 0.865 -> 0.809, leaf_spot recall
# 0.852 -> 0.695, viral_disease 0.952 -> 0.792), and the run was written up as a
# clean win for an unrelated fix. A silent corpus change also invalidates every
# version-over-version A/B, because the two models no longer share a dataset.
#
# The floors are deliberately WELL BELOW each source's healthy yield: this gate
# exists to catch a source that DISAPPEARED, not to police normal drift. Set a
# floor to 0 (or drop the key) to retire a source on purpose — that is a
# deliberate, reviewable edit, which is exactly the property the fail-soft path
# lacked. STRICT_SOURCE_FLOORS=False downgrades it to a warning for a
# deliberately degraded experiment.
STRICT_SOURCE_FLOORS = True
REQUIRED_SOURCE_FLOORS = {
    # Core public corpora — the backbone of every class.
    'plantdoc':                1500,
    'plantvillage':            2000,
    'inat':                    1500,
    'inat_pest':               1000,
    # Real gardener-photo sources; eval-eligible, so a loss also thins the gate.
    'doctorp':                 1000,
    'doctorp_pepper':            80,
    # NAP / hard negatives — the not_a_plant head and the OOD bank depend on them.
    'coco':                     500,
    'inat_animalia':            300,
    'openimages_indoor':        400,
    'coco_hard_partial_plant':  120,
    'inat_hard_greentexture':   600,
    # Domain supplements.
    'kaggle_houseplant':        400,
    # v2.6.12 pivot: the garden-ornamental leg. THIS is the source that vanished.
    'kaggle_ornamental':        400,
    'inat_physiological':       300,
    'mpld':                     100,
}
_missing, _under = [], []
for _src, _floor in sorted(REQUIRED_SOURCE_FLOORS.items()):
    if _floor <= 0:
        continue
    _got = _per_source.get(_src, 0)
    if _got == 0:
        _missing.append(_src)
    elif _got < _floor:
        _under.append(f'{_src}: {_got} < {_floor}')
if _missing or _under:
    _msg = ['SOURCE INVENTORY BELOW FLOOR — refusing to train on a silently '
            'truncated corpus.']
    if _missing:
        _msg.append(f'  MISSING ENTIRELY (0 images): {_missing}')
    if _under:
        _msg.append(f'  UNDER FLOOR: {_under}')
    _msg.append(
        '  Scroll up for the matching fetcher line: a Kaggle 403 (dataset '
        'deleted/unlisted, or the token lost access), an HF/Zenodo outage, or a '
        'rate-limit backoff. Training past this point produces a model that is '
        'NOT comparable to any other version, and a train-only source loss is '
        'INVISIBLE to the val_realworld ship gate. FIX the source and re-run '
        '§2, or -- if the source is gone for good -- re-point it and lower/zero '
        'its floor in REQUIRED_SOURCE_FLOORS as a deliberate, reviewed edit. '
        'Set STRICT_SOURCE_FLOORS=False only for a knowingly degraded run.')
    if STRICT_SOURCE_FLOORS:
        raise RuntimeError('\n'.join(_msg))
    print('WARN: ' + '\n'.join(_msg))
else:
    print(f'source floors OK: {len(REQUIRED_SOURCE_FLOORS)} required sources '
          f'all present and above floor')

# Persist the inventory so a later A/B can PROVE two runs shared a corpus.
# §6 folds this into training_metrics.json + model_card.json.
_noncommercial_used = {s: _per_source.get(s, 0)
                       for s in ('doctorp', 'doctorp_pepper')
                       if _per_source.get(s, 0)}
# STUDY-PROJECT POLICY: licences are provenance, not a gate. The tallies say
# what the bundle was trained on (they ship in model_card.json ->
# source_inventory.license_policy); nothing here can stop a run.
SOURCE_LICENSE_POLICY = {
    'policy': 'study_project_record_only',
    'gated_on_license': False,
    'per_image_rights_recorded_sources': sorted(_RIGHTS_REQUIRED_SOURCES),
    'rows_without_license_id': dict(_missing_rights),
    'noncommercial_sources_used': _noncommercial_used,
    'license_tally': {
        'inat_photos': dict(sorted(_INAT_LICENSE_TALLY.items())),
        'coco_license_ids': dict(sorted(_COCO_LICENSE_TALLY.items())),
        'openimages': dict(sorted(_OI_LICENSE_TALLY.items())),
        'supplement': dict(sorted(_SUPPLEMENT_LICENSE_TALLY.items())),
    },
}
SOURCE_INVENTORY = {
    'per_source': dict(sorted(_per_source.items(), key=lambda kv: -kv[1])),
    'total_images': int(sum(_per_source.values())),
    'required_floors': dict(REQUIRED_SOURCE_FLOORS),
    'strict': bool(STRICT_SOURCE_FLOORS),
    'per_license': dict(sorted(_per_license.items())),
    'license_policy': SOURCE_LICENSE_POLICY,
    'huggingface_revisions': {
        k: _HF_RESOLVED_REVISIONS[k] for k in sorted(_HF_RESOLVED_REVISIONS)},
    'kaggle_revisions': dict(sorted(_KAGGLE_RESOLVED_REVISIONS.items())),
    # Domain composition per split — ships in model_card.json so a later A/B can
    # ask 'did the gate contain the broadened domain?' the same way it asks
    # 'did the two runs share a corpus?'.
    'per_split_subject_domain': {
        _s: dict(sorted(_v.items(), key=lambda kv: -kv[1]))
        for _s, _v in _per_domain.items()},
}
(ROOT / 'source_inventory.json').write_text(json.dumps(SOURCE_INVENTORY, indent=2))
log_run({'phase': 'source_inventory', **SOURCE_INVENTORY})

# ── The next supplement manifest, re-stamped by this run (2.7.5) ──────────
# The bookkeeping half of first_seen_model_version, done by the run instead of
# by hand. Every supplement photo still in TRAIN_ROOT after the splits trains
# this MODEL_VERSION; models/supplement_manifest_next.csv is the supplement's
# manifest.csv with that column stamped on exactly those rows (a row that
# already names a version keeps it), plus a train_only row for every unlisted
# photo that loaded. Held-out rows are untouched, so the permanent holdout keeps
# its membership. Publish it as the NEXT revision's manifest.csv (with the next
# MODEL_VERSION) and that run's eval slices are exactly the rows no earlier
# model trained on. It rides in the §6b archive under analysis/.
SUPPLEMENT_NEXT_MANIFEST = Path('models') / 'supplement_manifest_next.csv'
try:
    import csv as _csv
    _nx = globals().get('_SUPPLEMENT_NEXT')
    if not _nx:
        raise RuntimeError('the supplement was not loaded this run')
    _col = 'first_seen_model_version'
    _out_cols = list(_nx['columns']) + ([_col] if _col not in _nx['columns'] else [])
    _rows_out, _n_stamped, _n_added = [], 0, 0
    for _k, _r in _nx['rows']:
        _cls = PurePosixPath(_k).parts[0]
        _trained = (TRAIN_ROOT / _cls / f"{_r['_disk_stem']}.jpg").is_file()
        if _r.get('_unlisted'):
            if not (_nx['has_hint'] or _nx['has_conf']):
                continue        # no routing basis: a listed row would turn eval-eligible
            _o = dict.fromkeys(_out_cols, '')
            _o.update({'filename': _k, 'class': _cls})
            if _nx['has_hint']:
                _o['split_hint'] = 'train_only'
            _n_added += 1
        else:
            _o = {c: (_r.get(c) or '') for c in _out_cols}
        if _trained and (_o.get(_col) or '').strip().lower() in ('', 'never'):
            _o[_col] = MODEL_VERSION
            _n_stamped += 1
        _rows_out.append(_o)
    SUPPLEMENT_NEXT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    with SUPPLEMENT_NEXT_MANIFEST.open('w', newline='', encoding='utf-8') as _fh:
        _w = _csv.DictWriter(_fh, fieldnames=_out_cols)
        _w.writeheader()
        _w.writerows(_rows_out)
    print(f'next supplement manifest: {SUPPLEMENT_NEXT_MANIFEST} — {len(_rows_out)} rows, '
          f'{_n_stamped} stamped first_seen_model_version={MODEL_VERSION} (in this run\'s '
          f'training split), {_n_added} unlisted photos added as train_only rows')
except Exception as _e:  # noqa: BLE001 — a convenience file, never a reason to stop
    print(f'next supplement manifest not written: {type(_e).__name__}: {_e}')


## §3 — Dedup + Stage 1 detector + DINOv2 diagnostic probe

Three sanity passes before we burn GPU on the teacher:

1. **Dedup** the train set (per-class SHA-256 + pHash with Hamming ≤ 6).
   Cross-class similarities are kept on purpose — they are useful as
   hard negatives. Held-out splits are audited (SHA-256 only) but
   never pruned, because pruning a held-out set after the fact biases
   the gate.

2. **Stage 1 YOLOv8n** is re-trained from PlantDoc bboxes **by default**
   (`TRAIN_DETECTOR_FROM_SCRATCH = True`): `find_detector_yaml()` first
   looks for a user-supplied YOLO `data.yaml` under `data_v2/`, and when
   none exists `build_yolo_detection_dataset_from_plantdoc()` generates one
   from the same PlantDoc bboxes used for the classifier crops. Set
   `TRAIN_DETECTOR_FROM_SCRATCH = False` to skip Stage 1 and keep the
   shipped `detector.{pt,onnx}` pair instead.

3. **DINOv2-S linear probe** is the cheap "is the data even
   separable?" check. If linear-on-frozen-DINOv2 cannot get past 30 %
   on `val_realworld` after 8 epochs, the data is the problem, not
   the model.


In [ ]:
# §3 — Dedup + Stage 1 detector + DINOv2 probe
import imagehash
from PIL import Image as PILImage
from ultralytics import YOLO
import yaml

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def dedup_directory(root, phash_threshold=6):
    """Per-class SHA-256 + pHash dedup. Per-file work runs in a thread pool
    (PIL + hashlib release the GIL); the per-class hash-set membership
    check stays serial."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists(): continue
        paths = sorted(iter_images(cls_dir))
        if not paths:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            shas    = list(pool.map(sha256_file, paths))
            phashes = list(pool.map(_phash_file, paths))
        seen_sha, seen_phash = set(), []
        for p, s, ph in zip(paths, shas, phashes):
            if s in seen_sha:
                drop.append((p, 'sha256_dup')); continue
            seen_sha.add(s)
            if ph is None:
                drop.append((p, 'unreadable')); continue
            if any(ph - existing <= phash_threshold for existing in seen_phash):
                drop.append((p, 'phash_dup')); continue
            seen_phash.append(ph)
    for p, _reason in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except Exception:
            pass
    print(f'dedup({root}): dropped={len(drop)}')

def assert_no_exact_split_leakage(train_root, heldout_roots):
    """SHA-256 cross-check between train and every held-out split."""
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_shas = list(pool.map(sha256_file, train_paths))
    train_hash = dict(zip(train_shas, train_paths))
    problems = []
    for root in heldout_roots:
        if not Path(root).exists(): continue
        for p in iter_images(root):
            h = sha256_file(p)
            if h in train_hash:
                problems.append((str(train_hash[h]), str(p)))
                if len(problems) >= 10: break
        if len(problems) >= 10: break
    if problems:
        detail = '\n'.join(f'train={a}  heldout={b}' for a, b in problems)
        raise RuntimeError('Exact train/heldout leakage detected:\n' + detail)
    print('leakage audit (SHA): no exact overlap between train and held-out splits')

def resolve_phash_split_leakage(train_root, heldout_roots, threshold=6,
                                 max_drop_frac=0.25, max_report=10):
    """Perceptual-hash cross-check + auto-resolve. Source datasets (PlantDoc,
    PlantVillage) split near-duplicates across their published train/test
    boundaries; the §2 splitter can't stop those because they live on
    opposite sides of the HF split. This is the last-line defence before
    training reads the splits.

    Train data is never modified — held-out leaks are cheaper to lose
    than diverse training samples. A safety cap (``max_drop_frac``)
    raises if more than the given fraction of any held-out split would be
    removed.
    """
    # Group train pHashes by class. A held-out image is only checked
    # against train images of the **same class** so a leaf-spot photo
    # that pHash-collides with a (visually similar) bacterial photo
    # isn't dropped from val.
    train_hashes_by_class = {}
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_phashes = list(pool.map(_phash_file, train_paths))
    for p, ph in zip(train_paths, train_phashes):
        if ph is not None:
            train_hashes_by_class.setdefault(p.parent.name, []).append(ph)
    if not any(train_hashes_by_class.values()):
        print('pHash leakage resolution: TRAIN_ROOT empty — skipped')
        return
    total_dropped, samples = 0, []
    for root in heldout_roots:
        if not Path(root).exists():
            continue
        held = list(iter_images(root))
        if not held:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            held_phashes = list(pool.map(_phash_file, held))
        to_drop = []
        for p, ph in zip(held, held_phashes):
            if ph is None:
                continue
            same_class_hashes = train_hashes_by_class.get(p.parent.name, [])
            best = min((ph - tph for tph in same_class_hashes), default=None)
            if best is not None and best <= threshold:
                to_drop.append((p, int(best)))
        frac = len(to_drop) / max(len(held), 1)
        if frac > max_drop_frac:
            preview = '\n'.join(f'  heldout={p}  pHash_dist={d}'
                                for p, d in to_drop[:max_report])
            raise RuntimeError(
                f'pHash leakage in {root}: would drop {len(to_drop)}/{len(held)} '
                f'({frac:.0%}) — exceeds max_drop_frac={max_drop_frac:.0%}. '
                f'Investigate the upstream split before re-running.\n' + preview
            )
        for p, _d in to_drop:
            sc = p.with_suffix('.json')
            try:
                p.unlink()
                if sc.exists(): sc.unlink()
            except OSError:
                pass
        if to_drop:
            print(f'  resolved {root}: dropped {len(to_drop)} of {len(held)} '
                  f'(near-duplicate of train, Hamming ≤ {threshold})')
            samples.extend((str(p), d) for p, d in to_drop[:max_report])
        total_dropped += len(to_drop)
    if total_dropped == 0:
        print(f'leakage audit (pHash): no near-duplicates within Hamming {threshold}')
    else:
        preview = '\n'.join(f'  {p}  pHash_dist={d}' for p, d in samples[:max_report])
        print(f'leakage resolution (pHash): dropped {total_dropped} held-out '
              f'near-duplicate(s) within Hamming {threshold}\n' + preview)

def audit_cross_class_phash_leakage(train_root, heldout_roots, threshold=6,
                                    drop=True, max_report=12):
    """Class-AGNOSTIC cross-split pHash audit. resolve_phash_split_leakage
    only compares a held-out image to train images of the SAME class, so a
    PlantDoc near-duplicate that landed under DIFFERENT labels on each side
    of the upstream train/test boundary (e.g. train=leaf_spot,
    val=viral_disease) is never caught and silently inflates the ship-gate
    F1. This pass compares every held-out image to ALL train pHashes
    regardless of class; by default it drops the held-out near-dup (train is
    never touched) and reports both labels so an intentional KD hard-negative
    pair can be audited by eye. Vectorised popcount keeps the full
    held x train comparison cheap."""
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_ph = list(pool.map(_phash_file, train_paths))
    train_cls, train_bits = [], []
    for p, ph in zip(train_paths, train_ph):
        if ph is None:
            continue
        train_cls.append(p.parent.name)
        train_bits.append(ph.hash.flatten())
    if not train_bits:
        print('cross-class pHash audit: TRAIN_ROOT empty — skipped')
        return 0
    train_bits = np.asarray(train_bits, dtype=bool)
    train_cls = np.asarray(train_cls)
    total_cross, samples = 0, []
    for root in heldout_roots:
        if not Path(root).exists():
            continue
        held = list(iter_images(root))
        if not held:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            held_ph = list(pool.map(_phash_file, held))
        to_drop = []
        for p, ph in zip(held, held_ph):
            if ph is None:
                continue
            dists = np.logical_xor(train_bits, ph.hash.flatten()).sum(axis=1)
            j = int(dists.argmin()); d = int(dists[j])
            tcls, hcls = str(train_cls[j]), p.parent.name
            if d <= threshold and tcls != hcls:
                to_drop.append((p, d, tcls, hcls))
        for p, d, tcls, hcls in to_drop:
            samples.append((str(p), d, tcls, hcls))
            if drop:
                sc = p.with_suffix('.json')
                try:
                    p.unlink()
                    if sc.exists(): sc.unlink()
                except OSError:
                    pass
        total_cross += len(to_drop)
    if total_cross == 0:
        print(f'cross-class pHash audit: no cross-label near-dups within Hamming {threshold}')
    else:
        verb = 'dropped' if drop else 'flagged'
        preview = '\n'.join(f'  heldout[{h}]={p}  ~train[{t}]  dist={d}'
                            for p, d, t, h in samples[:max_report])
        print(f'cross-class pHash audit: {verb} {total_cross} cross-label held-out '
              f'near-dup(s) within Hamming {threshold}\n' + preview)
    return total_cross


def dedup_each_split(heldout_roots, threshold=6):
    """Per-split internal dedup, in case the splitters moved a near-dup
    pair into the same held-out slice."""
    for root in heldout_roots:
        if Path(root).exists():
            dedup_directory(root, phash_threshold=threshold)

# §2 already ran _dedup_train_in_place(TRAIN_ROOT) before splitting; only
# dedup the held-out splits here.
# THE HOLDOUT IS NEVER MUTATED HERE. Three of the four guards below are
# read-only or provably no-ops on it, but `audit_cross_class_phash_leakage`
# DROPS held-out rows — and which rows it drops depends on what is in
# TRAIN_ROOT this release. Applied to the holdout that silently destroys the
# one property the sha1 bucket exists to provide: a row's verdict must be a
# function of the row, not of the corpus around it. Two releases would then
# score on different image sets and the deltas between them would mean nothing.
#
# Leakage still has to be RESOLVED, though — a holdout row that near-dups a
# training image measures memorisation. So it is resolved on the mutable side:
# the argument swap below makes HOLDOUT the reference and TRAIN_ROOT the side
# that gets pruned. Same guarantee, opposite victim, and the instrument
# survives intact.
_MUTABLE_SPLITS = tuple(r for r in SPLIT_ROOTS if r != HOLDOUT)
dedup_each_split(_MUTABLE_SPLITS)
assert_no_exact_split_leakage(TRAIN_ROOT, SPLIT_ROOTS)      # read-only; keep HOLDOUT in
resolve_phash_split_leakage(TRAIN_ROOT, _MUTABLE_SPLITS, threshold=6)
audit_cross_class_phash_leakage(TRAIN_ROOT, _MUTABLE_SPLITS, threshold=6, drop=True)
print(f'content-hash memo: {len(_HASH_MEMO)} entries ({sum(1 for k in _HASH_MEMO if k[0] == chr(112)+chr(104))} pHash) — the passes above reuse them instead of re-decoding')
print('holdout leakage: resolving on the TRAIN side so the instrument stays fixed ->')
# Same-class near-duplicates are leakage too. The old cross-class-only pass
# left a holdout photo's same-label augmentation/crop in training. Reverse the
# usual arguments so duplicates are removed from TRAIN_ROOT, never HOLDOUT.
resolve_phash_split_leakage(HOLDOUT, (TRAIN_ROOT,), threshold=6)
audit_cross_class_phash_leakage(HOLDOUT, (TRAIN_ROOT,), threshold=6, drop=True)

# Immutable post-dedup/post-split corpus identity. Counts and source names are
# not enough to prove two runs used the same bytes. Hash each classification
# image plus its provenance sidecar in sorted relative-path order, and bind the
# compact manifest into the final model bundle. No absolute/cache paths enter.
_dataset_hasher = hashlib.sha256(); _dataset_n = 0
_dataset_split_counts = {}
for _dp in sorted(iter_images(CLS_ROOT), key=lambda p: p.relative_to(ROOT).as_posix()):
    _rel = _dp.relative_to(ROOT).as_posix()
    _side = _dp.with_suffix('.json')
    _img_sha = sha256_file(_dp)
    _side_sha = sha256_file(_side) if _side.is_file() else '-'
    _dataset_hasher.update((_rel + '\0' + _img_sha + '\0' + _side_sha + '\n').encode())
    _dataset_n += 1
    _split_key = '/'.join(PurePosixPath(_rel).parts[:3])
    _dataset_split_counts[_split_key] = _dataset_split_counts.get(_split_key, 0) + 1
_source_inventory_path = ROOT / 'source_inventory.json'
if not _source_inventory_path.is_file():
    raise RuntimeError('source_inventory.json missing before data-manifest seal')
DATA_MANIFEST = {
    'schema_version': 1,
    'identity_algorithm': 'sha256(sorted(relative_path\0image_sha256\0sidecar_sha256\n))',
    'dataset_root': 'data_v2',
    'classification_sha256': _dataset_hasher.hexdigest(),
    'n_images': _dataset_n,
    'split_counts': dict(sorted(_dataset_split_counts.items())),
    'source_inventory_sha256': sha256_file(_source_inventory_path),
    'kaggle_revisions': dict(sorted(_KAGGLE_RESOLVED_REVISIONS.items())),
    'huggingface_revisions': {
        k: _HF_RESOLVED_REVISIONS[k] for k in sorted(_HF_RESOLVED_REVISIONS)},
}
Path('models/data_manifest.json').write_text(
    json.dumps(DATA_MANIFEST, indent=2), encoding='utf-8')
print(f'data manifest sealed: n={_dataset_n} '
      f'sha256={DATA_MANIFEST["classification_sha256"][:16]}…')
log_run({'phase': 'data_manifest', **DATA_MANIFEST})

# ── Stage 1 — YOLOv8n single-class plant_leaf ───────────────────
DETECTOR_DATA_ROOT = ROOT / 'detector'
TRAIN_DETECTOR_FROM_SCRATCH = True
DETECTOR_EXPORT_SUCCEEDED = False
YOLO_BASE_WEIGHT_IDENTITY = None
DETECTOR_RUN_DIR = (Path('runs/detect') / RUN_ID).resolve()
if DETECTOR_RUN_DIR.exists():
    raise RuntimeError(f'current detector run directory already exists: {DETECTOR_RUN_DIR}')


def find_detector_yaml():
    """Look for a pre-existing YOLO ``data.yaml`` under ``data_v2/``.

    A user can drop their own labelled detection dataset here; when none
    exists we fall back to ``build_yolo_detection_dataset_from_plantdoc()``
    so Stage 1 can still train end-to-end from the same source we used
    for the classifier crops.
    """
    if not ROOT.exists():
        return None
    cands = [p for p in ROOT.rglob('data.yaml') if 'cache_coco' not in str(p)]
    return cands[0] if cands else None


def _yolo_label_ids(label_dir):
    ids = set()
    for txt in Path(label_dir).rglob('*.txt'):
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                try:
                    ids.add(int(float(parts[0])))
                except ValueError:
                    pass
    return ids


def _link_or_copy(src: Path, dst: Path) -> None:
    """Hard-link ``src`` to ``dst`` (same filesystem: no bytes copied, no
    decode, no re-encode); copy when a link is not possible."""
    try:
        os.link(src, dst)
    except OSError:
        shutil.copyfile(src, dst)


DETECTOR_SUPPLEMENT_VAL_PERMILLE = 150   # boxed supplement groups held for detector val


def _add_supplement_boxes(out_root: Path):
    """Supplement photos that ship YOLO boxes -> Stage 1 train/val.

    §2 wrote SUPPLEMENT_BOXES_INDEX while loading the supplement, so this reads
    ONE json file: no directory walk, no decode, and the photo is hard-linked,
    not re-encoded (§2 already stored it in the frame the boxes were drawn on).
    Only photos still in TRAIN_ROOT are used: anything the splits moved into a
    held-out set — or that dedup removed — never trains the detector, so Stage 1
    never learns a photo the §6 end-to-end eval scores. A capture family goes
    to detector val as a whole (sha1 of its group), so its siblings cannot sit
    on both sides of the recall-aware checkpoint selection."""
    idx = globals().get('SUPPLEMENT_BOXES_INDEX')
    if idx is None or not Path(idx).is_file():
        print('  supplement boxes: no index from §2 — none added')
        return (0, 0)
    entries = json.loads(Path(idx).read_text(encoding='utf-8'))
    added, held_out = {'train': 0, 'val': 0}, 0
    for rel, e in sorted(entries.items()):
        src = TRAIN_ROOT / rel
        if not src.is_file():
            held_out += 1
            continue
        bucket = int.from_bytes(
            hashlib.sha1(f"detector|{e['group']}".encode()).digest()[:4], 'big') % 1000
        target = 'val' if bucket < DETECTOR_SUPPLEMENT_VAL_PERMILLE else 'train'
        img_path = out_root / 'images' / target / src.name
        if img_path.exists():
            continue
        _link_or_copy(src, img_path)
        (out_root / 'labels' / target / f'{src.stem}.txt').write_text(
            ''.join(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n' for cx, cy, w, h in e['boxes']))
        added[target] += 1
    print(f"  supplement boxes added: train={added['train']} val={added['val']} "
          f'({held_out} boxed photos are in a held-out split or were deduped — not used)')
    return added['train'], added['val']


def _add_background_negatives(out_root: Path, n_train_pos: int, n_val_pos: int,
                              bg_frac: float = 0.12, seed: int = 29):
    """Add NAP images as YOLOv8 background negatives (image + EMPTY label
    .txt). Every PlantDoc detection row carries >=1 leaf bbox, so without
    negatives the detector never learns 'no leaf here' and fires at 0.9 on
    pure noise (the actual source of confident-garbage verdicts). An empty
    label file is the explicit YOLO 'background' marker. Pulls from
    not_a_plant — which now includes the §2 hard negatives — so the detector
    also sees leaf-like-but-not-leaf frames as backgrounds."""
    nap_dir = TRAIN_ROOT / 'not_a_plant'
    naps = sorted(iter_images(nap_dir)) if nap_dir.exists() else []
    if not naps:
        print('  no not_a_plant images for detector backgrounds; skipping')
        return (0, 0)
    rng = random.Random(seed); rng.shuffle(naps)
    n_train_bg, n_val_bg = int(n_train_pos * bg_frac), int(n_val_pos * bg_frac)
    take = naps[:n_train_bg + n_val_bg]
    added_tr = added_val = 0
    for k, src in enumerate(take):
        target = 'train' if k < n_train_bg else 'val'
        stem = f'bg_{src.stem}'
        img_path = out_root / 'images' / target / f'{stem}.jpg'
        lbl_path = out_root / 'labels' / target / f'{stem}.txt'
        if img_path.exists():
            continue
        try:
            if src.suffix.lower() in ('.jpg', '.jpeg'):
                _link_or_copy(src, img_path)     # §2 already wrote it as a plain JPEG
            else:
                _save_rgb(PILImage.open(src), img_path)
        except Exception:
            continue
        lbl_path.write_text('')   # empty label = background negative
        if target == 'train':
            added_tr += 1
        else:
            added_val += 1
    print(f'  detector backgrounds added: train={added_tr} val={added_val} (bg_frac={bg_frac})')
    return (added_tr, added_val)


def build_yolo_detection_dataset_from_plantdoc(
    out_root: Path = DETECTOR_DATA_ROOT,
    train_ratio: float = 0.85,
    seed: int = 7,
) -> Optional[Path]:
    """Generate a YOLOv8 single-class detection dataset from PlantDoc.

    PlantDoc HF rows ship the full source image plus one or more
    ``{bbox: [x, y, w, h], category: <leaf-class>}`` objects. This helper
    walks the HF dataset (cached locally after the first §2 run), writes
    the full image to ``out_root/images/{train,val}/`` and a parallel
    YOLO ``.txt`` to ``out_root/labels/{train,val}/`` with one
    ``0 cx cy w h`` row per bbox — every category collapsed to a single
    class ``plant_leaf`` (id 0).

    PlantDoc's own ``test`` split lands entirely in YOLO ``val`` so the
    eval set matches the upstream hold-out. ``train`` rows are sub-split
    deterministically by ``seed`` at ``train_ratio``.

    Returns the generated ``data.yaml`` path, or ``None`` when the HF
    pull failed.

    FAST PATH (2.7.5). Boxes and ids are read from the split without decoding
    a photo, so box-less rows never decode. The rows that do are decoded (by
    the split's own HF decoder), normalised and JPEG-encoded on a bounded
    thread pool — PIL releases the GIL — and written back in the original
    shuffled order, so the seeded train/val draw is the one every earlier
    release made and the files are byte-identical to the old serial loop.
    At most ``4 x workers`` rows are in flight. Then supplement photos that
    ship boxes (§2 index) and not_a_plant backgrounds are hard-linked in.
    """
    try:
        ds = _hf_load_with_retry('agyaatcoder/PlantDoc')
    except Exception as exc:
        print(f'PlantDoc HF pull for YOLO dataset failed: {exc}')
        return None

    if out_root.exists():
        shutil.rmtree(out_root)
    for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
        (out_root / sub).mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    train_count = val_count = skipped_no_bbox = 0
    from collections import deque
    _workers = max(2, min(8, os.cpu_count() or 2))

    def _encode(decode, raw, bboxes):
        """Worker: decode one photo, build its label rows, JPEG-encode it once."""
        img = decode(raw).convert('RGB')
        w, h = img.size
        lines = []
        for bbox in bboxes:
            x, y, bw, bh = (float(v) for v in bbox)
            if bw <= 1 or bh <= 1:
                continue
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            nw = bw / w
            nh = bh / h
            if not (0 < cx < 1 and 0 < cy < 1 and 0 < nw <= 1 and 0 < nh <= 1):
                continue
            lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        if not lines:
            return lines, None
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=92)   # == _save_rgb(img, '<x>.jpg')
        return lines, buf.getvalue()

    with ThreadPoolExecutor(max_workers=_workers) as pool:
        for split_name in [s for s in ('train', 'test') if s in ds]:
            rows, decode = _hf_lazy_images(ds[split_name])
            order = list(range(len(rows)))
            rng.shuffle(order)
            pending, it = deque(), iter(order)

            def _submit(i):
                row = rows[i]
                bboxes = (row.get('objects') or {}).get('bbox') or []
                fut = pool.submit(_encode, decode, row['image'], bboxes) if bboxes else None
                pending.append((i, row.get('image_id', i), fut))

            for _ in range(4 * _workers):
                nxt = next(it, None)
                if nxt is None:
                    break
                _submit(nxt)
            with tqdm(total=len(order), desc=f'YOLO/{split_name}') as bar:
                while pending:
                    i, image_id, fut = pending.popleft()
                    nxt = next(it, None)
                    if nxt is not None:
                        _submit(nxt)
                    bar.update(1)
                    if fut is None:
                        skipped_no_bbox += 1
                        continue
                    lines, jpeg = fut.result()
                    if not lines:
                        skipped_no_bbox += 1
                        continue
                    if split_name == 'test' or rng.random() > train_ratio:
                        target = 'val'
                    else:
                        target = 'train'
                    stem = f'plantdoc_{split_name}_{image_id}'
                    img_path = out_root / 'images' / target / f'{stem}.jpg'
                    lbl_path = out_root / 'labels' / target / f'{stem}.txt'
                    if img_path.exists():
                        continue
                    img_path.write_bytes(jpeg)
                    lbl_path.write_text('\n'.join(lines) + '\n')
                    if target == 'train':
                        train_count += 1
                    else:
                        val_count += 1

    _s_tr, _s_val = _add_supplement_boxes(out_root)
    train_count += _s_tr
    val_count += _s_val
    _add_background_negatives(out_root, train_count, val_count, bg_frac=0.12)

    if train_count == 0 or val_count == 0:
        print(f'PlantDoc → YOLO produced too few rows '
              f'(train={train_count} val={val_count}); skipping.')
        return None

    yaml_path = out_root / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump({
        'path': str(out_root.resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'names': ['plant_leaf'],
        'nc': 1,
    }, sort_keys=False))
    print(f'YOLO dataset built: train={train_count}  val={val_count}  '
          f'skipped_no_bbox={skipped_no_bbox}')
    print(f'  data.yaml = {yaml_path}')
    return yaml_path


yaml_path = find_detector_yaml()
if yaml_path is None and TRAIN_DETECTOR_FROM_SCRATCH:
    print('No YOLO data.yaml under data_v2/ — building one from PlantDoc bboxes.')
    yaml_path = build_yolo_detection_dataset_from_plantdoc()
if yaml_path is None:
    print(f'No YOLO dataset under {ROOT}.  Stage 1 training skipped — '
          f'the shipped detector.{{pt,onnx}} will be used at inference.')
else:
    spec = yaml.safe_load(yaml_path.read_text())
    for key in ('train', 'val'):
        p = spec.get(key)
        if isinstance(p, str):
            labels_dir = Path(yaml_path.parent, p).as_posix().replace('/images/', '/labels/')
            if Path(labels_dir).exists():
                ids = _yolo_label_ids(labels_dir)
                if any(i != 0 for i in ids):
                    raise RuntimeError(
                        f'{key} labels contain class ids {sorted(ids)}; remap to single class 0 first.'
                    )
    spec['names'], spec['nc'] = ['plant_leaf'], 1
    patched = yaml_path.parent / 'plant_leaf.yaml'
    patched.write_text(yaml.safe_dump(spec, sort_keys=False))
    detector = YOLO('yolov8n.pt')
    # ``yolov8n.pt`` is a mutable release alias. Ultralytics may download a
    # different byte stream on a future fresh runtime, so bind the resolved
    # checkpoint bytes into run_manifest.json/bundle_id before training.
    _base_candidates = []
    _reported_ckpt = getattr(detector, 'ckpt_path', None)
    if isinstance(_reported_ckpt, (str, os.PathLike)) and str(_reported_ckpt):
        _base_candidates.append(Path(_reported_ckpt).resolve())
    _base_candidates.append(Path('yolov8n.pt').resolve())
    _base_path = next((p for p in _base_candidates if p.is_file()), None)
    if _base_path is None:
        raise RuntimeError('cannot resolve downloaded yolov8n.pt for identity seal')
    YOLO_BASE_WEIGHT_IDENTITY = {
        'requested_alias': 'yolov8n.pt',
        'resolved_filename': _base_path.name,
        'sha256': sha256_file(_base_path),
        'size_bytes': _base_path.stat().st_size,
    }
    log_run({'phase': 'detector_base_weight', **YOLO_BASE_WEIGHT_IDENTITY})
    detector.train(
        # batch=128 saturates an A100 at imgsz=640 (~12 GB peak). YOLOv8
        # auto-falls back to a smaller batch on OOM so this stays safe on
        # smaller cards too.
        data=str(patched), epochs=60, patience=12, batch=128, imgsz=640,
        optimizer='AdamW', lr0=1e-3, cos_lr=True, amp=True,
        # Indoor leaf bboxes are tight — heavy mosaic = leaks across boxes.
        mosaic=0.1, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4,
        device=0, workers=8,
        project=str(DETECTOR_RUN_DIR.parent), name=DETECTOR_RUN_DIR.name,
        exist_ok=False,
        # Keep every epoch's weights so the recall-aware re-selection below has
        # something to choose from. ~6MB x 60 = ~360MB, deleted with the runtime.
        save_period=1,
    )
    _trainer = getattr(detector, 'trainer', None)
    if _trainer is None or not getattr(_trainer, 'best', None):
        raise RuntimeError('Ultralytics returned no current-run best checkpoint')
    best_pt = Path(_trainer.best).resolve()
    if not best_pt.is_file() or not best_pt.is_relative_to(DETECTOR_RUN_DIR):
        raise RuntimeError(
            f'Ultralytics best checkpoint is not owned by this run: {best_pt}; '
            f'expected under {DETECTOR_RUN_DIR}')
    # ── Recall-aware re-selection ────────────────────────────────
    # Ultralytics ranks checkpoints by fitness = 0.1*mAP50 + 0.9*mAP50-95.
    # Recall is not in it. For a general detector that is fine; for THIS
    # pipeline it is backwards. A missed box becomes a synthetic det_conf=0 in
    # pipeline.py and can trip the NAP gate — the path that cost the deployed
    # 2.6.8 bundle 11.3pp of served leaf_scorch. A spurious box is cheap: it
    # gets classified, mean-voted, and the not_a_plant head veto stops the OOD
    # gate over-firing on it. So Stage 1's loss is asymmetric and fitness
    # does not know that.
    #
    # Measured on the 2026-08-16 run: 13 epochs sat within 1% of best fitness
    # (0.5798-0.5838) with recall spanning 0.718-0.750, and the selected one
    # (ep45) was the BOTTOM of that band at 0.716 — ep43 gave 0.750 for a
    # mAP50-95 cost of 0.001. That is not noise being chased; it is an
    # arbitrary pick among checkpoints Ultralytics itself rates as equivalent.
    #
    # So: among epochs whose fitness is within FITNESS_TOL of the best, take
    # the highest recall. Requires save_period=1 above (weights for every
    # epoch, ~6MB each); without the saved checkpoints this silently keeps
    # best.pt, which is the old behaviour and is stated in the log.
    DETECTOR_FITNESS_TOL = 0.01
    try:
        import csv as _csv
        _res = best_pt.parent.parent / 'results.csv'
        _rows = list(_csv.DictReader(open(_res))) if _res.exists() else []
        def _col(r, *names):
            for n in names:
                for k in r:
                    if k.strip() == n:
                        return float(r[k])
            raise KeyError(names)
        _cand = []
        for _r in _rows:
            _ep = int(_col(_r, 'epoch'))
            _rc = _col(_r, 'metrics/recall(B)')
            _m50 = _col(_r, 'metrics/mAP50(B)')
            _m95 = _col(_r, 'metrics/mAP50-95(B)')
            _w = best_pt.parent / f'epoch{_ep - 1}.pt'
            _cand.append({'ep': _ep, 'recall': _rc, 'fit': 0.1 * _m50 + 0.9 * _m95,
                          'path': _w if _w.exists() else None})
        _top = max((c['fit'] for c in _cand), default=0.0)
        _ok = [c for c in _cand
               if c['path'] and c['fit'] >= _top * (1 - DETECTOR_FITNESS_TOL)]
        if _ok:
            _pick = max(_ok, key=lambda c: c['recall'])
            _cur = min(_cand, key=lambda c: abs(c['fit'] - _top))
            if _pick['recall'] > _cur['recall'] + 1e-9:
                print(f"detector re-selected on recall: ep{_pick['ep']} "
                      f"recall={_pick['recall']:.3f} fitness={_pick['fit']:.4f}  "
                      f"(fitness-best was ep{_cur['ep']} recall={_cur['recall']:.3f}) "
                      f"— {len(_ok)} epochs within {DETECTOR_FITNESS_TOL:.0%} of best fitness")
                # save_period checkpoints keep their optimizer state (~23MB vs
                # the 6MB Ultralytics strips into best.pt/last.pt). Nothing
                # downstream cares — §6 only runs inference and §7 does not
                # bundle detector.pt — but shipping 4x the bytes for state that
                # is never read is untidy, and the stripped file is what every
                # previous release put in models/.
                try:
                    from ultralytics.utils.torch_utils import strip_optimizer
                    strip_optimizer(str(_pick['path']))
                except Exception as _se:  # noqa: BLE001 — cosmetic only
                    print(f"  (could not strip optimizer state: {type(_se).__name__})")
                best_pt = _pick['path']
            else:
                print(f"detector: fitness-best already maximises recall "
                      f"({_cur['recall']:.3f}) among {len(_ok)} equally-fit epochs")
        else:
            print('detector: no per-epoch checkpoints on disk (save_period) — '
                  'keeping the fitness-selected best.pt. Its recall may be several '
                  'points below an equally-fit epoch; see the §6 stage 1 audit.')
    except Exception as _e:  # noqa: BLE001 — selection is an optimisation, not a gate
        print(f'detector recall re-selection skipped: {type(_e).__name__}: {_e}')
    exported = YOLO(str(best_pt)).export(format='onnx', opset=13, dynamic=False)
    Path('models').mkdir(exist_ok=True)
    shutil.copy(best_pt, 'models/detector.pt')
    _exported_path = Path(exported).resolve() if exported else None
    if (_exported_path is None or not _exported_path.is_file() or
            not _exported_path.is_relative_to(DETECTOR_RUN_DIR)):
        raise RuntimeError(
            f'Ultralytics did not return a current-run ONNX artifact: {exported!r}')
    shutil.copy(_exported_path, 'models/detector.onnx')
    DETECTOR_EXPORT_SUCCEEDED = True

# ── DINOv2-S linear probe (diagnostic) ──────────────────────────
# Backbone is frozen — extract features ONCE, then iterate the linear
# head over the cached embeddings. ~30× faster than recomputing the
# ViT-S forward pass per epoch on an A100.
PROBE_SIZE = 224
PROBE_TX = T.Compose([T.Resize(int(PROBE_SIZE * EVAL_RESIZE_RATIO)),
                      T.CenterCrop(PROBE_SIZE), T.ToTensor(), NORM])
train_ds_probe = ImageFolder(str(TRAIN_ROOT), transform=PROBE_TX)
assert train_ds_probe.classes == sorted(CLASSES), (train_ds_probe.classes, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds_probe.class_to_idx)
train_dl_probe = DataLoader(
    train_ds_probe, batch_size=256, shuffle=False,
    **_loader_kwargs(EVAL_WORKERS, drop_last=False),
)
_, val_dl_probe = make_eval_loader(
    VAL_PRIMARY, PROBE_SIZE, batch=256, class_to_idx=TRAIN_CLASS_TO_IDX,
)

backbone = torch.hub.load(DINOV2_REPO_REF, 'dinov2_vits14').eval().to(device)
for p in backbone.parameters(): p.requires_grad_(False)

@torch.inference_mode()
def _extract_features(dl):
    feats, labels = [], []
    for x, y in dl:
        f = backbone(x.to(device, non_blocking=True))
        feats.append(f.detach().cpu())
        labels.append(y.detach().cpu())
    return torch.cat(feats), torch.cat(labels)

print('caching DINOv2 features (train + val)…')
train_feats, train_labels = _extract_features(train_dl_probe)
val_feats,   val_labels   = _extract_features(val_dl_probe)
print(f'  train_feats={tuple(train_feats.shape)}  val_feats={tuple(val_feats.shape)}')

# Move to device for the head loop — small enough (<20 MB even at v2.2 scale).
train_feats_g = train_feats.to(device); train_labels_g = train_labels.to(device)
val_feats_g   = val_feats.to(device);   val_labels_g   = val_labels.to(device)

head = nn.Linear(train_feats.shape[1], len(train_ds_probe.classes)).to(device)
opt  = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn_probe = nn.CrossEntropyLoss(label_smoothing=0.05)
PROBE_BATCH = 1024

probe_curve = []
g = torch.Generator(device='cpu').manual_seed(SEED)
for ep in range(8):
    head.train()
    perm = torch.randperm(train_feats_g.size(0), generator=g)
    for i in range(0, perm.size(0), PROBE_BATCH):
        idx = perm[i:i + PROBE_BATCH].to(device)
        opt.zero_grad(set_to_none=True)
        loss_fn_probe(head(train_feats_g[idx]), train_labels_g[idx]).backward()
        opt.step()
    head.eval()
    with torch.inference_mode():
        pred = head(val_feats_g).argmax(1)
        top1 = float((pred == val_labels_g).float().mean().item())
    probe_curve.append({'epoch': ep + 1, 'val_realworld_top1': top1})
    print(f'ep={ep+1}  val_realworld_top1={top1:.3f}')

Path('models/dinov2_probe.json').write_text(json.dumps(probe_curve, indent=2))
log_run({'phase': 'dinov2_probe', 'curve': probe_curve})
if probe_curve[-1]['val_realworld_top1'] < 0.30:
    print('WARN: DINOv2 frozen+probe < 0.30 — your data is hard. '
          'Audit labels in §2 before spending GPU on the teacher.')

# Per-class recall on val_realworld for the thin classes — surfaces a dead
# thin class here (cheap, on the frozen probe) instead of ~40 min later at
# the §6 ship gate. Warn-only by default; flip the flag to abort early.
PROBE_FAIL_FAST_ON_DEAD_THIN = False
_THIN_CLASSES = ('gray_mold_botrytis', 'leaf_scorch_underwatering', 'nutrient_deficiency')
with torch.inference_mode():
    _probe_pred = head(val_feats_g).argmax(1).cpu().numpy()
_probe_true = val_labels_g.cpu().numpy()
_dead_thin = []
print('thin-class probe recall (val_realworld):')
for _c in _THIN_CLASSES:
    _ci = TRAIN_CLASS_TO_IDX.get(_c)
    if _ci is None:
        continue
    _support = int((_probe_true == _ci).sum())
    if _support == 0:
        print(f'    recall[{_c:25s}] =   n/a  (no val_realworld samples)')
        continue
    _rec = float(((_probe_pred == _ci) & (_probe_true == _ci)).sum() / _support)
    print(f'    recall[{_c:25s}] = {_rec:.3f}  (support={_support})')
    if _rec == 0.0:
        _dead_thin.append(_c)
if _dead_thin:
    _msg = f'DINOv2 probe: zero recall on thin class(es) {_dead_thin} on val_realworld'
    if PROBE_FAIL_FAST_ON_DEAD_THIN:
        raise RuntimeError(_msg + ' — PROBE_FAIL_FAST_ON_DEAD_THIN=True. '
                           'Audit the source for these classes in §2 first.')
    print(f'WARN: {_msg} (warn-only; set PROBE_FAIL_FAST_ON_DEAD_THIN=True to abort).')

# Free memory before §4.
del backbone, head, train_dl_probe, val_dl_probe
del train_feats, train_feats_g, train_labels, train_labels_g
del val_feats,   val_feats_g,   val_labels,   val_labels_g
torch.cuda.empty_cache() if device == 'cuda' else None


## §4 — Loss, augs, frozen DINOv2 ViT-S/14 + trained head

* **Loss:** `LogitAdjustmentLoss(τ=0.5, label_smoothing=0.05)`
  (Menon et al. ICLR 2021). Loss-side rebalancing matched to the
  **training prior** — no `WeightedRandomSampler` by default, because
  stacking the two over-corrects rare classes (Liu et al. 2024,
  *frontiersin*).
* **Sampler off by default.** Set `USE_WEIGHTED_SAMPLER = True` only
  if a single class dominates by > 10×.
* **Model:** frozen DINOv2 ViT-S/14 backbone (`torch.hub`,
  `dinov2_vits14`; flip `USE_REGISTERS` for the `_reg` weights) + a small
  MLP head (`LayerNorm → Linear → GELU → Dropout → Linear`) on the 384-d
  CLS embedding. **Only the head trains**; the backbone stays frozen and
  in `eval()` so its (LayerNorm-only) statistics are identical at train,
  eval and export time. Input is **224 px** (16×14 — DINOv2 patches are
  14 px). Early-stop on `val_indomain` disease macro-F1.
* **No teacher, no KD.** Directly fitting a strong frozen backbone beats
  distilling into a weaker student when there is no size constraint; the
  former EfficientNetV2-S teacher fine-tune was removed.
* **Augs:** conservative indoor-targeted profile at **serving geometry** —
  `Resize(int(IMG * EVAL_RESIZE_RATIO))` then `RandomCrop(IMG)` (the eval /
  calibration / server transform with the centre crop replaced by a random
  window at the identical scale), horizontal flip, one optional
  WB/local-glare/defocus-or-motion/JPEG artifact, and optional mild ColorJitter.
  RandomResizedCrop and RandomErasing are OFF: leaf margins/tips and small
  lesions are label-bearing. This is an explicit unablated default, not a claim
  that augmentation improves accuracy. `TRAIN_AUGMENTATION = 'off'` (default
  `'on'`) is the paired-ablation arm: same geometry, no artifact block and no
  ColorJitter, sealed as profile `serve_parity_v2_geometry_only`. Run it once at
  `SEED=0` and score it against a same-seed default run with
  `tools/eval_bootstrap.py`; neither arm carries an efficacy claim.


In [ ]:
# §4 — Loss, augs, frozen DINOv2 ViT-S/14 + trained head
import math
import torch.nn.functional as F
from collections import Counter
from contextlib import nullcontext
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import WeightedRandomSampler
from PIL import ImageDraw, ImageFilter, Image as PILImage
from io import BytesIO

# §3's detector.train() switched determinism off on return (Ultralytics'
# unset_deterministic). Re-assert it before any §4 training; the state is
# sealed into run_manifest.json as the training-time determinism.
DETERMINISM_AT_TRAINING = assert_determinism('s4_head_training')

# ── Backbone / head config ──────────────────────────────────────
# DINOv2 patches are 14 px, so the input side must be a multiple of 14.
# 224 = 16×14 matches the §3 probe that demonstrated the ~0.91 ceiling.
IMG, BATCH = FAMILY_CONFIG['img_size'], 128
USE_REGISTERS = False   # dinov2_vits14_reg gives slightly cleaner features,
                        # but the §3 probe (and thus the demonstrated 0.91
                        # ceiling) used the non-register weights — default
                        # matches what we validated.
# Augmentation ablation switch. 'on' (default) trains on TRAIN_AUGMENTATION_PROFILE
# below exactly as before (serve_parity_v2_unablated). 'off' is the ablation arm:
# the same serving-parity geometry (Resize -> RandomCrop -> hflip) WITHOUT the
# sensor-artifact block and WITHOUT ColorJitter, sealed under its own profile
# version (run_log, run_manifest.json, model_card.json) so it can never pass for
# the default recipe. Compare an 'off' run only against a same-SEED 'on' run via
# tools/eval_bootstrap.py. Neither arm carries an efficacy claim.
TRAIN_AUGMENTATION = 'on'
TRAIN_AUGMENTATION_MODES = ('on', 'off')
if TRAIN_AUGMENTATION not in TRAIN_AUGMENTATION_MODES:
    raise ValueError(f'TRAIN_AUGMENTATION={TRAIN_AUGMENTATION!r} '
                     f'not in {list(TRAIN_AUGMENTATION_MODES)}')
HEAD_HIDDEN = 384       # ViT-S embedding dim → 1-hidden-layer MLP head.
HEAD_DROPOUT = 0.2
HEAD_EPOCHS = 30
# Stride for the val_studio transfer monitor (see step_eval). Drift is a slow
# signal, so every 5th epoch catches it without paying the eval every epoch.
STUDIO_EVAL_EVERY = 5
HEAD_LR = 1e-3
HEAD_PATIENCE = 5
# Precision-aware checkpoint selection — OPT-IN, OFF BY DEFAULT (v2.5.0).
# The physiological classes are precision-bound (decent recall, low precision —
# they over-trigger). When enabled, the best checkpoint maximises disease
# macro-F1 PLUS a small bonus for the weak classes' F0.5 (precision-weighted,
# recall-aware so a barely-predicting epoch earns ~0) on val_indomain (which
# holds only REAL in-domain sources, never the crop proxies).
#
# DEFAULT 0.0 == pure macro-F1 (the proven selection). A non-zero weight can
# trade up to `weight` of indomain macro-F1 for weak-class precision, and that
# trade is NOT yet A/B-validated to transfer to the val_realworld SHIP metric —
# so it stays OFF until measured on GPU. Raise it (e.g. 0.05) only as a
# deliberate, validated A/B. Precision + CIs are reported/shipped either way
# (§6), so the over-trigger is always visible regardless of this knob.
SELECT_WEAKCLASS_PRECISION_WEIGHT = 0.0
SELECT_WEAKCLASS_PRECISION_CLASSES = ('leaf_scorch_underwatering', 'nutrient_deficiency')
def _weakclass_fbeta(precision_by_cls, recall_by_cls, beta2=0.25):
    """Mean F0.5 (precision-weighted, recall-aware) over the precision-bound
    weak classes — the selection bonus metric. F0.5 -> ~0 as recall -> 0, so a
    degenerate 'predict the class once, precision 1.0' epoch earns no bonus."""
    vals = []
    for c in SELECT_WEAKCLASS_PRECISION_CLASSES:
        if c not in precision_by_cls:
            continue
        p = float(precision_by_cls[c]); r = float(recall_by_cls.get(c, 0.0))
        denom = beta2 * p + r
        vals.append((1.0 + beta2) * p * r / denom if denom > 0 else 0.0)
    return float(sum(vals) / len(vals)) if vals else 0.0

# ── LoRA fine-tune of the backbone (report "last few points") ────
# The frozen probe capped in-domain nutrient_deficiency at ~0.32 — the
# FEATURES, not the head, are the ceiling. LoRA adapts the LATER DINOv2
# blocks to leaf imagery at low rank/LR with minimal forgetting. Validated
# end-to-end on dinov2_vits14 (wrap->train->merge_and_unload->ONNX, parity
# ~1e-6). USE_LORA=False falls back to the pure frozen-head path.
USE_LORA = True
LORA_RANK = 16             # v2.4: ↑ from 8 — the §3 probe showed the FEATURES
LORA_ALPHA = 32            #   (not the head) cap weak-class recall, so give the
LORA_DROPOUT = 0.05        #   adapter more capacity to specialise DINOv2 to leaf
LORA_LR = 1e-4             #   imagery. A100-40GB has the headroom; the merged
                           #   weights still export to a single ONNX graph.
LORA_LAST_K_BLOCKS = 8     # adapt the last K of the ViT's blocks (↑ from 6)

USE_WEIGHTED_SAMPLER = False   # class-balanced sampler — off by default
# Per-image source weighting attacks the agricultural / houseplant domain
# mismatch. PlantDoc is field crops; the deployment target is indoor
# houseplants. Down-weighting PlantDoc rows in the sampler pulls the
# empirical training prior toward the real distribution without dropping
# the hard negatives the NAP head leans on. The LA loss prior is recomputed
# from the sampler weights (``class_priors_from_source_weights``) so the
# correction stays aligned with the empirical per-batch distribution.
# Never enable alongside USE_WEIGHTED_SAMPLER (double rebalance).
USE_SOURCE_WEIGHTED_SAMPLER = True
SOURCE_WEIGHTS = {
    # Rebalance toward the deployment/eval distribution. PlantDoc (field
    # photos) is what val_realworld is seeded from, so favour it;
    # PlantVillage studio images barely transfer; the NAP-only sources
    # (coco / inat_animalia / openimages_indoor) were over-weighted at 1.0.
    'plantdoc':            1.1,
    'plantvillage':        0.35,
    'inat':                1.0,
    # DoctorP real app photos — the deployment distribution itself; full
    # weight, and (with its pepper sibling) eval-eligible.
    'doctorp':             1.0,
    # DoctorP pepper sibling: REAL app photos in the same framing but pepper is
    # a container CROP, so down-weight in TRAINING (a touch below the houseplant-
    # spanning main set) — its real value is the eval/cal supply, which the
    # sampler weight does not touch. Prior decoupled below this (see PRIOR_*).
    'doctorp_pepper':      0.8,
    'inat_animalia':       0.5,
    'inat_physiological':  1.2,
    'coco':                0.5,
    'openimages_indoor':   0.5,
    # MPLD maize phosphorus deficiency: crop-domain down-weight
    # (curated labels, out-of-domain species).
    'mpld':                0.5,
    # Pest classes (mealybug / scale) from iNat research-grade — in-domain
    # houseplant pests, so full weight (not a crop supplement). Inert until
    'inat_pest':           1.0,
    # Kaggle indoor houseplant disease supplement: in-domain (indoor
    # houseplants) but possibly partly studio-framed and unverified, so
    # a moderate weight between in-domain (1.0) and crop supplement (0.5).
    # v2.4: down-weighted further. The set is black-bg-segmented + augmented
    # Aloe (now rejected at fetch) plus a few small real-photo folders.
    'kaggle_houseplant':   0.35,
    # Ornamental disease sets (§2 KAGGLE_ORNAMENTAL_SLUGS): rose + aloe vera
    # leaves with REAL disease morphology on NATURAL backgrounds — much closer
    # to the deployment domain than the field-crop proxies. Re-sourced
    # 2026-07-24 after the original pappu54 slug was deleted upstream; the
    # replacements are all-original captures (no baked-in augmentation), which
    # if anything argues for the full weight the pivot already gave it.
    # NOTE the replacements supply healthy / leaf_spot / powdery_mildew only —
    # the 600 ornamental `viral_disease` images died with pappu54 and are NOT
    # replaced, so treat viral as supply-starved until a new source lands.
    # 2.7.1 ABLATION — RUN, MEASURED, **FALSIFIED**. Reverted to 1.0.
    # Down-weighting this source to 0.35 (draw 0.1107 -> 0.0415) moved the target
    # metric by EXACTLY NOTHING: never-trained ornamental scorch recall stayed
    # 0.065 argmax / 0.000 served, identical to 2.7.0 to three decimals. It also
    # COST healthy: served healthy recall 0.947 -> 0.861 and false-disease-fire
    # 0.046 -> 0.119 (McNemar 2-13 vs 2.6.8, p=0.0074). That is exactly what the
    # source composition predicts in hindsight — it supplies healthy/leaf_spot/
    # powdery ornamentals and ZERO scorch, so cutting its weight removes
    # ornamental HEALTHY signal and cannot touch a class it never contained.
    # Two levers are now falsified BY EXPERIMENT (this one) and two more refuted
    # by the 2026-08-12 investigation (margin-crop/letterbox at conf 0.95,
    # boundary hard-negatives). Stop tuning sampler weights: the surviving
    # mechanism is representational, and roadmap 2.8 item 5 is the actual fix —
    # ornamental leaf_scorch does not exist in the corpus and no weight supplies
    # a missing class. See docs/ROADMAP.md "Model track".
    # ORIGINAL (kept for the record): 1.0 -> 0.35.
    # 2.7.0 collapsed on never-trained ornamental leaf_scorch (served recall
    # 0/200 vs 2.6.8's 71/200) while BEATING 2.6.8 on held-out supplement
    # ornamentals, so the coupling is to training-corpus acquisition style, not
    # to "ornamentals". The one corpus fact that lines up: this source supplies
    # ZERO leaf_scorch (rose/aloe -> healthy, leaf_spot, powdery_mildew only)
    # and the 2.6.12 pivot up-weighted it 0.8 -> 1.0 in exactly the run that
    # collapsed, while its volume nearly halved (2263 -> 1158 rows). A
    # class-incomplete source at full weight teaches "ornamental foliage =
    # one of the three classes I have", and scorch is not one of them.
    # PRIOR_SOURCE_WEIGHTS['kaggle_ornamental'] stays at its decoupled 0.8
    # below, so this run moves exactly ONE knob (2.6.9 and 2.7.0 both moved
    # several and could not be attributed).
    # READ: AUC(z_scorch) on the never-trained benchmark, scored END-TO-END via
    # pipeline.analyze -- not Stage-2 argmax; the OOD AND-gate alone can move
    # ornamental recall ~20pp. FALSIFIED if separability does not move.
    # The real fix is 2.8-era: re-source ornamental leaf_scorch into
    # KAGGLE_ORNAMENTAL_SLUGS. No sampler weight supplies a missing class.
    'kaggle_ornamental':   1.0,
    # Rice N/P/K nutrient-deficiency: crop-domain proxy (gated off by default),
    # down-weighted like the other crop nutrient sources.
    # Real-houseplant healthy species photos: in-domain but whole-plant/
    # species-labelled, so a moderate weight (diversity, not volume).
    'kaggle_species':      0.7,
    # OUR eval-supplement: in-domain houseplant (not a crop proxy), so a
    # strong weight just below the real app-photo sources; prior NOT
    # decoupled (it inherits this via dict(SOURCE_WEIGHTS)).
    'supplement':          0.9,
    # Boundary hard-negatives from the eval-supplement, routed TRAIN-ONLY (see
    # fetch_kaggle_supplement). Same in-domain weight as 'supplement'; prior NOT
    # decoupled (inherited via dict(SOURCE_WEIGHTS)).
    # Supplement rows the offline pipeline marked split_hint=train_only (below
    # its label-confidence floor). They are TRAIN-ONLY but they are ordinary
    # in-domain positives, not hard negatives, so they keep the FULL 'supplement'
    # weight — down-weighting them here would re-create the volume loss that
    # routing them out of eval is explicitly designed to avoid (aqua's KD needs
    # the volume; docs/DATASET_LABELING_PLAN.md v8 item 3). Prior NOT decoupled.
    'supplement_train_only': 0.9,
    'supplement_boundary': 0.35,  # v2.6.10 RCA: 0.9 flooded TRAIN with all 697 boundary-healthy (train-only) and crushed leaf_scorch recall 0.89->0.77; hard-negs nudge, not drive. See docs/DATASET_LABELING_PLAN.md
}

# ── Logit-adjustment prior weights (DECOUPLED from the sampler) ──
# The LA prior should track the DEPLOYMENT class frequency, not the crop-
# inflated training count. MPLD is an out-of-domain crop source that
# teach the deficiency *signal* but say nothing about how often a houseplant
# is nutrient-deficient in production. In v2.3.0 they entered the LA prior at
# their sampler weight (0.5) and, with ~1000 rows, pushed nutrient_deficiency
# to a head-class prior (0.1165) -> LA SUPPRESSED it -> in-domain recall fell
# 0.55->0.26. Here the crop sources get a SMALLER prior weight than their
# sampler weight, so the model still TRAINS on them (sampler 0.5) but LA
# treats nutrient_deficiency as the deployment-rare class it is and applies
# the rare-class boost. The §2 caps keep the sampler/prior gap small enough
# that this does not over-predict the class. (Report Stage 2: decouple
# representation sampling from the rebalancing prior.)
PRIOR_SOURCE_WEIGHTS = dict(SOURCE_WEIGHTS)
PRIOR_SOURCE_WEIGHTS['mpld'] = 0.25
# Kaggle houseplant: decouple the LA prior below the sampler weight too
# (unverified domain mix) so it teaches signal without inflating the prior.
PRIOR_SOURCE_WEIGHTS['kaggle_houseplant'] = 0.25
PRIOR_SOURCE_WEIGHTS['kaggle_ornamental'] = 0.8  # v2.6.12: ornamentals in-domain (was 0.6)
PRIOR_SOURCE_WEIGHTS['kaggle_species'] = 0.5
# DoctorP pepper: crop domain, so the LA prior weight sits below its 0.8 sampler
# weight — teaches the real deficiency/disease signal without pulling the
# deployment (houseplant) prior toward pepper.
PRIOR_SOURCE_WEIGHTS['doctorp_pepper'] = 0.5
LA_TAU = 0.5                   # logit adjustment τ — flattened post-cap
                                #   distribution justifies the lower value.
LABEL_SMOOTHING = 0.05


class LogitAdjustmentLoss(nn.Module):
    """Loss-side Logit Adjustment, Menon et al. ICLR 2021 eq. 4."""
    def __init__(self, class_priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.register_buffer('log_prior', torch.log(class_priors.clamp(min=1e-12)))
        self.tau, self.ls = tau, label_smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.tau * self.log_prior, target,
                               label_smoothing=self.ls)

def class_priors_from_imagefolder(ds):
    counts = Counter([y for _, y in ds.samples])
    pri = torch.tensor([counts.get(i, 0) for i in range(len(ds.classes))], dtype=torch.float32)
    return pri.clamp(min=1.0) / pri.clamp(min=1.0).sum()

def class_priors_from_source_weights(ds, source_weights, default_weight=1.0):
    """Per-class prior under the source-weighted sampler.

    LogitAdjustment (Menon et al. ICLR 2021 eq. 4) assumes the ``log_prior``
    fed to the loss equals the per-batch class distribution the model
    actually sees during training. Under the source-weighted sampler the
    file-count prior is wrong; this helper sums each image's source weight
    into its class bucket. Pass the sampler's SOURCE_WEIGHTS to mirror the
    draw distribution, OR pass PRIOR_SOURCE_WEIGHTS to intentionally DECOUPLE
    the prior from the sampler: out-of-domain representation-only sources
    (mpld) is down-weighted in the prior so the deployment-rare
    classes they pad (nutrient_deficiency) still receive the LA rare-class
    boost while the sampler keeps training on their signal.
    """
    counts = torch.zeros(len(ds.classes), dtype=torch.float32)
    for path, y in ds.samples:
        sc = read_sidecar(Path(path)) or {}
        src_name = sc.get('source')
        if not isinstance(src_name, str):
            src_name = 'unknown'
        w = max(float(source_weights.get(src_name, default_weight)), 0.0)
        counts[y] += w
    return counts.clamp(min=1e-6) / counts.clamp(min=1e-6).sum()

def make_class_balanced_sampler(ds, power=0.5):
    counts = Counter([y for _, y in ds.samples])
    weights = torch.DoubleTensor([(1.0 / counts[y]) ** power for _, y in ds.samples])
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def make_source_weighted_sampler(ds, source_weights, default_weight=1.0):
    """Sample images at a rate proportional to their sidecar `source`. Images
    without a sidecar (or with an unknown source) get ``default_weight``."""
    drawn = Counter()
    sources_per_path = []
    weights = torch.zeros(len(ds.samples), dtype=torch.float64)
    for i, (path, _) in enumerate(ds.samples):
        sc = read_sidecar(Path(path)) or {}
        src_name = sc.get('source')
        if not isinstance(src_name, str):
            src_name = 'unknown'
        weights[i] = max(float(source_weights.get(src_name, default_weight)), 0.0)
        drawn[src_name] += 1
        sources_per_path.append(src_name)
    total = weights.sum().item()
    if total <= 0:
        raise RuntimeError('source-weighted sampler: all weights are zero — '
                           'check sidecar `source` field and SOURCE_WEIGHTS')
    expected = {s: round(weights[[i for i, x in enumerate(sources_per_path) if x == s]].sum().item() / total, 4)
                for s in drawn}
    print(f'source-weighted sampler  raw_counts={dict(drawn)}  '
          f'expected_draw_fraction={expected}  weight_lookup={source_weights}')
    log_run({'phase': 'sampler', 'tag': 'source_weighted',
             'raw_counts': dict(drawn), 'expected_draw_fraction': expected,
             'weight_lookup': source_weights})
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def amp_ctx():
    return autocast(device_type='cuda', dtype=torch.bfloat16) if device == 'cuda' else nullcontext()

class WhiteBalanceShift:
    def __init__(self, p=0.5, strength=(0.85, 1.15)):
        self.p, self.lo, self.hi = p, *strength
    def __call__(self, img):
        if random.random() > self.p: return img
        r_scale = random.uniform(self.lo, self.hi)
        b_scale = random.uniform(self.lo, self.hi)
        r, g, b = img.split()
        r = r.point(lambda v: max(0, min(255, int(v * r_scale))))
        b = b.point(lambda v: max(0, min(255, int(v * b_scale))))
        return PILImage.merge('RGB', (r, g, b))

class LocalSpecularGlare:
    """Localized soft highlight, not a mislabeled global brightness shift."""
    def __init__(self, p=0.12, radius_fraction=(0.06, 0.18), opacity=(35, 105)):
        self.p, self.radius_fraction, self.opacity = p, radius_fraction, opacity
    def __call__(self, img):
        if random.random() > self.p: return img
        w, h = img.size
        radius = max(2, int(min(w, h) * random.uniform(*self.radius_fraction)))
        cx, cy = random.randrange(w), random.randrange(h)
        mask = PILImage.new('L', (w, h), 0)
        ImageDraw.Draw(mask).ellipse(
            (cx - radius, cy - radius, cx + radius, cy + radius),
            fill=random.randint(*self.opacity))
        mask = mask.filter(ImageFilter.GaussianBlur(radius=max(1, radius * 0.55)))
        return PILImage.composite(PILImage.new('RGB', (w, h), 'white'), img, mask)

def _directional_motion_kernel(size=5, angle_degrees=0.0):
    """Sparse normalized line kernel used by DefocusOrMotionBlur."""
    if size != 5:
        raise ValueError('Pillow supports the audited motion kernel at size=5')
    theta = math.radians(angle_degrees)
    centre = size // 2
    weights = [0.0] * (size * size)
    for offset in range(-centre, centre + 1):
        x = max(0, min(size - 1, round(centre + offset * math.cos(theta))))
        y = max(0, min(size - 1, round(centre + offset * math.sin(theta))))
        weights[y * size + x] += 1.0
    return weights

class DefocusOrMotionBlur:
    def __init__(self, p=0.15): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        if random.random() < 0.5:
            return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.2)))
        kernel = _directional_motion_kernel(angle_degrees=random.uniform(0, 180))
        return img.filter(ImageFilter.Kernel((5, 5), kernel, scale=sum(kernel)))

class JpegRecompress:
    def __init__(self, p=0.35, q_range=(50, 92)): self.p, self.q_range = p, q_range
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.q_range))
        buf.seek(0)
        return PILImage.open(buf).convert('RGB').copy()

# GEOMETRY PARITY WITH SERVING. Every eval slice, both calibration splits, the
# server (classifier.py, from classes.json -> training.eval_resize_ratio) and
# the Aqua client all see Resize(int(IMG * EVAL_RESIZE_RATIO)) -> CenterCrop(IMG).
# Training uses the SAME resize with RandomCrop(IMG) in place of the centre
# crop: a positional jitter (<= int(IMG * (EVAL_RESIZE_RATIO - 1)) px along the
# short axis, anywhere along the long axis) at exactly the scale that serves.
#
# The first 2.8.0 draft letterboxed the full frame here (PadToSquare -> Resize
# to IMG x IMG) while serving centre-crops. A 4:3 photo then reached the head
# ~1.4x larger at serve time than at train time, with padding bars the server
# never produces -- a train/serve distribution shift, not margin preservation:
# the margins the letterbox kept are cut by the serving centre crop anyway, and
# temperature / conformal / the OOD bank are all fit on the centre-crop view.
# If full-frame inference is ever wanted, change the SERVING transform and
# calibrate on it; never change only the training side.
#
# Explicit, shared Gamma/Aqua profile. No augmentation benefit is claimed: this
# is a conservative label-preserving default pending a locked, paired ablation.
TRAIN_AUGMENTATION_PROFILE = {
    'version': 'serve_parity_v2_unablated',
    'geometry': 'resize_shorter_side_eval_ratio_then_random_crop+hflip',
    'eval_resize_ratio': EVAL_RESIZE_RATIO,
    'sensor_artifact': 'one_of(wb,localized_glare,defocus_or_motion,jpeg)',
    'sensor_artifact_p': 0.35, 'color_jitter_p': 0.25,
    'random_resized_crop': False, 'random_erasing': False,
    'label_preservation_rationale': ('no scale/aspect jitter; crop window at the '
                                     'serving scale keeps margins/tips and small lesions'),
    'efficacy_status': 'not established; requires locked ablation',
}
if TRAIN_AUGMENTATION == 'off':
    # Ablation arm: same keys in the same order, photometric pieces zeroed and
    # relabelled, so no record of this run carries the default version string.
    TRAIN_AUGMENTATION_PROFILE = {
        **TRAIN_AUGMENTATION_PROFILE,
        'version': 'serve_parity_v2_geometry_only',
        'sensor_artifact': 'none',
        'sensor_artifact_p': 0, 'color_jitter_p': 0,
        'efficacy_status': ('ablation arm (TRAIN_AUGMENTATION=off) of '
                            'serve_parity_v2_unablated; compare only against a '
                            'same-seed default-profile run via tools/eval_bootstrap.py; '
                            'no efficacy claim either way'),
    }
# The Compose is BUILT FROM the profile, so the sealed record and the transform
# cannot drift apart. A zero-probability stage is omitted outright (never wrapped
# at p=0): the default path keeps exactly the pre-switch stage list and RNG draws.
_photometric_tx = []
if TRAIN_AUGMENTATION_PROFILE['sensor_artifact_p'] > 0:
    _photometric_tx.append(T.RandomApply([T.RandomChoice([
        WhiteBalanceShift(p=1.0), LocalSpecularGlare(p=1.0),
        DefocusOrMotionBlur(p=1.0), JpegRecompress(p=1.0, q_range=(65, 95)),
    ])], p=TRAIN_AUGMENTATION_PROFILE['sensor_artifact_p']))
if TRAIN_AUGMENTATION_PROFILE['color_jitter_p'] > 0:
    _photometric_tx.append(T.RandomApply([T.ColorJitter(
        brightness=0.10, contrast=0.10, saturation=0.08, hue=0.0)],
        p=TRAIN_AUGMENTATION_PROFILE['color_jitter_p']))
assert len(_photometric_tx) == (0 if TRAIN_AUGMENTATION == 'off' else 2), (
    TRAIN_AUGMENTATION, TRAIN_AUGMENTATION_PROFILE)
indoor_train_pil_tx = T.Compose([
    T.Resize(int(IMG * EVAL_RESIZE_RATIO)), T.RandomCrop(IMG), T.RandomHorizontalFlip(),
    *_photometric_tx,
])
indoor_train_tx = T.Compose([indoor_train_pil_tx, T.ToTensor(), NORM])
print(f'train augmentation profile: {TRAIN_AUGMENTATION_PROFILE}')
log_run({'phase': 'augmentation_profile', **TRAIN_AUGMENTATION_PROFILE})

# ── Datasets / loaders ──────────────────────────────────────────
train_ds = ImageFolder(str(TRAIN_ROOT), transform=indoor_train_tx)
if len(train_ds) == 0:
    raise RuntimeError('TRAIN_ROOT empty — re-run §2.')
CLASS_NAMES = list(train_ds.classes)
assert CLASS_NAMES == sorted(CLASSES), (CLASS_NAMES, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds.class_to_idx)
N_CLASSES = len(CLASS_NAMES)

if USE_SOURCE_WEIGHTED_SAMPLER and not USE_WEIGHTED_SAMPLER:
    priors = class_priors_from_source_weights(train_ds, PRIOR_SOURCE_WEIGHTS).to(device)
    print(f'LA prior (PRIOR_SOURCE_WEIGHTS; crop decoupled from sampler): '
          f'{ {n: round(float(p), 4) for n, p in zip(train_ds.classes, priors.cpu().tolist())} }')
else:
    priors = class_priors_from_imagefolder(train_ds).to(device)
loss_fn = LogitAdjustmentLoss(priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING)

if USE_SOURCE_WEIGHTED_SAMPLER and USE_WEIGHTED_SAMPLER:
    raise RuntimeError('USE_SOURCE_WEIGHTED_SAMPLER and USE_WEIGHTED_SAMPLER are '
                       'mutually exclusive — they would compose into a double '
                       'rebalance on top of the logit-adjustment loss prior.')
LOADER_WORKERS = min(12, EVAL_WORKERS)
if USE_SOURCE_WEIGHTED_SAMPLER:
    sampler = make_source_weighted_sampler(train_ds, SOURCE_WEIGHTS)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))
elif USE_WEIGHTED_SAMPLER:
    sampler = make_class_balanced_sampler(train_ds, power=0.5)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))
else:
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED),
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))

val_indomain_ds, val_indomain_dl = make_eval_loader(
    VAL_INDOMAIN, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
val_realworld_ds, val_realworld_dl = make_eval_loader(
    VAL_PRIMARY, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
temp_cal_ds, temp_cal_dl = make_eval_loader(
    TEMP_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
# The permanent never-trained holdout (§2 make_holdout). Fail-soft: a run whose
# supplement did not load still trains and still ships — it just cannot report
# the improvement metric, and §6 says so out loud rather than silently omitting
# a line nobody would notice was missing.
try:
    holdout_ds, holdout_dl = make_eval_loader(
        HOLDOUT, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
    )
except (RuntimeError, FileNotFoundError):
    print('holdout not available (no class subdirs) — the 2.7.2 improvement metric '
          'will be MISSING from this run. Check §2 make_holdout.')
    holdout_ds = holdout_dl = None
try:
    val_studio_ds, val_studio_dl = make_eval_loader(
        VAL_STUDIO, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
    )
except (RuntimeError, FileNotFoundError):
    print('val_studio not available (no class subdirs); skipping studio monitor.')
    val_studio_ds = val_studio_dl = None

# Studio is a non-blocking transfer monitor, but a sudden drift is worth
# a loud line. Warn when bias falls outside this range.
STUDIO_BIAS_BAND = (0.7, 1.5)

def _maybe_warn_studio_bias(metrics, tag):
    if metrics is None:
        return
    bias = float(metrics.get('healthy_bias', 1.0))
    lo, hi = STUDIO_BIAS_BAND
    if not lo <= bias <= hi:
        print(f'WARN: [{tag}] studio healthy_bias={bias:.2f} outside '
              f'[{lo:.2f}, {hi:.2f}]  (transfer monitor; non-blocking)')

def _eval_under_amp(model, dl, tag):
    with amp_ctx():
        return evaluate(model, dl, CLASS_NAMES, device, tag)

def step_eval(model, tag, studio=True):
    """Per-epoch eval used for early-stopping decisions only.

    ``studio`` gates the val_studio pass. Studio is a non-blocking transfer
    monitor that never feeds selection (only val_indomain does) but is a
    comparably sized split, so running it every epoch roughly doubles
    per-epoch eval cost for no selection value — the caller runs it on a
    stride instead.

    val_realworld is intentionally NOT computed per epoch: it is the ship
    gate, and looking at it every epoch leaks via human inspection of
    run_log.jsonl. The single post-training measurement happens in §6
    against the deployed INT8 ORT session.
    """
    out = {'indomain': _eval_under_amp(model, val_indomain_dl, f'{tag} val_indomain')}
    if studio and val_studio_dl is not None:
        out['studio'] = _eval_under_amp(model, val_studio_dl, f'{tag} val_studio')
        _maybe_warn_studio_bias(out['studio'], tag)
    row = {'phase': 'head', 'tag': tag,
           'indomain_disease_f1': out['indomain']['disease_macro_f1'],
           'indomain_healthy_bias': out['indomain']['healthy_bias']}
    if 'studio' in out:
        row['studio_disease_f1'] = out['studio']['disease_macro_f1']
        row['studio_healthy_bias'] = out['studio']['healthy_bias']
    log_run(row)
    return out

def _make_sched(opt, n_epochs, n_steps_per_epoch, cosine=True):
    """Linear-warmup + cosine schedule. Set ``cosine=False`` for a flat
    post-warmup LR."""
    if n_steps_per_epoch <= 0:
        raise RuntimeError('Training loader has zero batches; lower BATCH or add data.')
    warmup = LinearLR(opt, start_factor=0.1, total_iters=n_steps_per_epoch)
    if not cosine:
        return warmup
    return SequentialLR(opt, schedulers=[
        warmup,
        CosineAnnealingLR(opt, T_max=max(1, (n_epochs - 1) * n_steps_per_epoch)),
    ], milestones=[n_steps_per_epoch])

# ── Frozen DINOv2 ViT-S/14 + trained MLP head ───────────────────
# The §3 frozen DINOv2-S linear probe already reached ~0.91 top-1 on
# val_realworld, proving the features separate the 8 classes. We promote
# that result to the *production* model: freeze the ViT-S/14 backbone and
# train a small MLP head on the CLS embedding under the same logit-
# adjustment loss. No teacher, no knowledge distillation — with no model-
# size constraint and ~10× CPU latency headroom, directly fitting a strong
# frozen backbone is the higher-ceiling, lower-risk path (vanilla KD into a
# weaker, INT8-hostile MobileNetV3 student is exactly the regime where KD
# underperforms; see DIST/DKD). The backbone is exported into the same ONNX
# graph as the head in §6, so serving stays a single self-contained file.
_HUB_ID = FAMILY_CONFIG['hub_id']
if USE_REGISTERS and _HUB_ID == 'dinov2_vits14':
    _HUB_ID = 'dinov2_vits14_reg'
ARCHITECTURE = f'{_HUB_ID}+mlp_head' + ('+lora' if USE_LORA else '')

dino_backbone = torch.hub.load(DINOV2_REPO_REF, _HUB_ID).eval().to(device)
for p in dino_backbone.parameters():
    p.requires_grad_(False)
EMBED_DIM = int(getattr(dino_backbone, 'embed_dim', 384))
print(f'DINOv2 backbone={_HUB_ID}  embed_dim={EMBED_DIM}  (frozen)')

# Wrap the LATER blocks with LoRA adapters (base stays frozen + eval). peft
# matches target_modules as a REGEX on module names; we encode both the
# block range and the attn/mlp Linear suffixes. merge_and_unload() in §6
# folds the adapters into the base Linears so the ONNX stays one graph.
if USE_LORA:
    from peft import LoraConfig, get_peft_model
    _n_blocks = len(dino_backbone.blocks)
    _lo = max(0, _n_blocks - LORA_LAST_K_BLOCKS)
    _rng = '|'.join(str(i) for i in range(_lo, _n_blocks))
    LORA_TARGET_REGEX = rf'blocks\.({_rng})\.(attn\.(qkv|proj)|mlp\.(fc1|fc2))'
    dino_backbone = get_peft_model(dino_backbone, LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_REGEX, bias='none'))
    _n_lora = sum(p.numel() for p in dino_backbone.parameters() if p.requires_grad)
    print(f'LoRA enabled: rank={LORA_RANK} alpha={LORA_ALPHA} '
          f'blocks=[{_lo},{_n_blocks}) trainable_backbone_params={_n_lora:,}')

def _set_backbone_train_mode(backbone):
    """Keep frozen DINO layers in eval while training LoRA dropout.

    Calling ``backbone.eval()`` after ``model.train()`` used to disable every
    PEFT ``lora_dropout`` module, making LORA_DROPOUT=0.05 an inert knob. Only
    those adapter-owned Dropout children are restored to train mode; frozen
    backbone normalization/stochastic layers remain deterministic."""
    backbone.eval()
    if not USE_LORA:
        return 0
    enabled = 0
    for name, module in backbone.named_modules():
        if 'lora_dropout' in name and isinstance(module, nn.Dropout):
            module.train()
            enabled += 1
    if LORA_DROPOUT > 0 and enabled == 0:
        raise RuntimeError('USE_LORA=True but no PEFT lora_dropout modules were found')
    return enabled

class DINOv2Classifier(nn.Module):
    """Frozen DINOv2 ViT-S/14 backbone + trainable MLP head on the CLS
    embedding.

    The backbone runs under ``torch.no_grad`` so only the head learns and
    the backbone never accumulates an autograd graph. ``torch.onnx.export``
    traces the executed ops regardless of grad mode, so the exported ONNX
    is the full, self-contained backbone+head graph.
    """
    def __init__(self, backbone, embed_dim, num_classes, hidden, p_drop,
                 lora_active=False):
        super().__init__()
        self.backbone = backbone
        self.lora_active = lora_active
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        if self.lora_active:
            feats = self.backbone(x)          # LoRA adapters need a graph
        else:
            with torch.no_grad():
                feats = self.backbone(x)      # (B, embed_dim) CLS embedding
        return self.head(feats)

model = DINOv2Classifier(
    dino_backbone, EMBED_DIM, N_CLASSES, HEAD_HIDDEN, HEAD_DROPOUT,
    lora_active=USE_LORA,
).to(device)

# The head always trains. With LoRA the adapter params on the later blocks
# train too (lower LR); the frozen base stays in eval() so its LayerNorm
# stats + base dropout are identical at train, eval and export time.
if USE_LORA:
    _lora_params = [p for p in model.backbone.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{'params': model.head.parameters(), 'lr': HEAD_LR},
         {'params': _lora_params, 'lr': LORA_LR}], weight_decay=1e-4)
    print(f'optimiser: head@{HEAD_LR} + {len(_lora_params)} LoRA tensors@{LORA_LR}')
else:
    opt = torch.optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=1e-4)
sched = _make_sched(opt, HEAD_EPOCHS, len(train_dl))

best_f1, best_ep, best_state, plateau = -1.0, 0, None, 0
best_sel = -1.0   # best precision-aware selection score (>= best_f1 by design)
for ep in range(HEAD_EPOCHS):
    model.train()
    _set_backbone_train_mode(model.backbone)
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with amp_ctx():
            logits = model(x)
            loss = loss_fn(logits, y)
        if not torch.isfinite(loss):
            print(f'WARN: non-finite loss at head ep{ep+1}; skipping batch'); continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=5.0)
        opt.step(); sched.step()
    m = step_eval(model, f'head ep{ep+1}',
                  studio=((ep + 1) % STUDIO_EVAL_EVERY == 0 or ep + 1 == HEAD_EPOCHS))
    ind = m['indomain']
    f1 = ind['disease_macro_f1']
    # Precision-aware selection score (see SELECT_WEAKCLASS_PRECISION_* above).
    _mean_weak_f05 = _weakclass_fbeta(ind.get('per_class_precision', {}),
                                      ind.get('per_class_recall', {}))
    sel = f1 + SELECT_WEAKCLASS_PRECISION_WEIGHT * _mean_weak_f05
    if sel > best_sel:
        best_sel, best_f1, best_ep, plateau = sel, f1, ep + 1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        log_run({'phase': 'head_select', 'tag': f'ep{ep+1}', 'sel_score': sel,
                 'indomain_disease_f1': f1, 'mean_weak_f05': _mean_weak_f05})
    else:
        plateau += 1
        if plateau >= HEAD_PATIENCE:
            print(f'head early stop @ ep{ep+1}  (best ep={best_ep} f1={best_f1:.3f} '
                  f'sel={best_sel:.3f})')
            break

if best_state is None:
    raise RuntimeError('head training produced no checkpoint — check the loop.')
model.load_state_dict(best_state)
print(f'restored best head  best_indomain_disease_f1={best_f1:.3f} (ep{best_ep})')

Path('models').mkdir(exist_ok=True)
torch.save(model.state_dict(), 'models/classifier.pth')
log_run({'phase': 'head_done', 'architecture': ARCHITECTURE,
         'best_indomain_f1': best_f1, 'best_ep': best_ep})

# ── Free the §4 training loader (its persistent workers) + optimiser/scheduler
# before §5/§6. Left resident they sit through the aqua KD + ONNX/TF export tail
# (a contributor to the aqua-combo OOM). The val/cal loaders are KEPT — §6 needs
# them; only the train-side objects (unused after here) are released.
import gc as _gc4
for _nm in ('train_dl', 'sampler', 'opt', 'sched'):
    globals().pop(_nm, None)
_gc4.collect()
if device == 'cuda':
    torch.cuda.empty_cache()


## §5 — Finalize teacher; Aqua: distill → MobileNetV3-Small TFLite student

For **gamma/frontier** this just freezes the §4 frozen-DINOv2 + MLP head for the §6 ONNX export. In **combined** mode (`TRAIN_MODE='combined'`) the gamma §4 model is also the *teacher* and we knowledge-distill it into an Aqua MobileNetV3-Small student exported as strict FLOAT32 TFLite for fully offline on-device inference. KD is warranted here because the target is a hard on-device size/latency budget — a tiny student is the deliverable, not a regression.

**Current export contract.** Aqua no longer routes through `tf.lite.TFLiteConverter.from_saved_model`: current onnx2tf writes `.tflite` files directly and may not emit `saved_model.pb`. §5 requires exactly one emitted FLOAT32 graph, verifies FLOAT32 input/output and numeric parity with the torch student, and fails the combined run if that exact artifact is unavailable. The conversion runs in a separate CPython 3.12 env (TensorFlow 2.19 + onnx2tf 1.28.8, provisioned with `uv` and preflighted at the end of §1), so the training kernel's Python version does not matter and TensorFlow never loads into it. INT8 is known to collapse this MobileNetV3 student and FLOAT16 requires a delegate the Android runtime contract does not promise; neither may be substituted.

In [ ]:
# §5 — Finalize teacher; for Aqua: distill into a MobileNetV3-Small TFLite student
#
# Gamma / Frontier: the production model IS the §4 frozen-DINOv2 + MLP head, so
# this cell only freezes it for the §6 ONNX export.
#
# Aqua (FAMILY_CONFIG['distill']): the §4 model becomes the TEACHER and we
# knowledge-distill it into a small MobileNetV3-Small student that exports to
# TFLite for fully offline on-device inference. KD is justified HERE (unlike the
# removed DINOv2 KD): the target is a hard on-device latency/size budget, so a
# weaker-but-tiny student is the deliverable, not a regression.

assert_determinism('s5_finalize_distill')

model.eval()
for p in model.parameters():
    p.requires_grad_(False)
if device == 'cuda':
    torch.cuda.empty_cache()
print(f'Teacher (DINOv2 + head) ready  best_indomain_disease_f1={best_f1:.3f} (ep{best_ep})')

if FAMILY_CONFIG.get('distill'):
    # A combined run promises Gamma + Aqua. Export is fail-closed: silently
    # swallowing an on-device failure relabels a partial Gamma-only run as a
    # successful combined bundle and can pick up stale models/aqua siblings.
    try:
        import torchvision
        from PIL import Image as _PILImage
        AQUA_DIR = Path('models/aqua'); AQUA_DIR.mkdir(parents=True, exist_ok=True)

        # ── Hyperparameters ────────────────────────────────────────────
        STUDENT_IMG = 224        # MobileNetV3 native res (teacher still sees IMG)
        KD_EPOCHS   = 30         # SAME budget as v2.6.6 -> no added runtime. Feature
                                 # distillation gives a richer gradient PER epoch
                                 # (a tiny extra Linear), not more epochs; the
                                 # teacher backbone forward was already computed.
        KD_T        = 3.0        # softmax temperature for the soft-target KL term
        KD_ALPHA    = 0.7        # weight on the KD (soft) term vs hard CE
        KD_BETA     = 0.5        # weight on FEATURE distillation (cosine to the
                                 # teacher's DINOv2 CLS embedding). 0.0 reverts to
                                 # the logits-only v2.6.6 recipe. Added because aqua
                                 # collapsed toward healthy/not_a_plant on hard real
                                 # photos -- a representation gap logits-only KD
                                 # cannot close (gap teacher 0.886 -> student 0.780).
        KD_LR       = 1e-3
        KD_WD       = 1e-4
        IMAGENET_MEAN = [0.485, 0.456, 0.406]
        IMAGENET_STD  = [0.229, 0.224, 0.225]

        # ── Student: ImageNet-pretrained MobileNetV3-Small, head -> N_CLASSES ──
        student = torchvision.models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        _in = student.classifier[-1].in_features
        student.classifier[-1] = nn.Linear(_in, N_CLASSES)
        student = student.to(device)
        _n_params = sum(p.numel() for p in student.parameters())
        print(f'Aqua student = MobileNetV3-Small  params={_n_params/1e6:.2f}M  classes={N_CLASSES}')
        # Feature-distillation projection (TRAINING-ONLY; never exported): maps the
        # student's pooled features to the teacher's CLS-embedding dim so the loss
        # can pull the student's REPRESENTATION onto DINOv2's, not just its logits.
        # Dims derived from a dummy forward so this is robust to arch changes.
        with torch.no_grad():
            _sf_dim = student.avgpool(student.features(torch.zeros(
                1, 3, STUDENT_IMG, STUDENT_IMG, device=device))).flatten(1).shape[1]
            _tf_dim = model.backbone(torch.zeros(
                1, 3, IMG, IMG, device=device)).shape[1]
        feat_proj = nn.Linear(_sf_dim, _tf_dim).to(device)

        # ── Paired dataset: ONE augmented PIL view -> (student tensor, teacher
        #    tensor, label). Both views share the same geometry so the teacher's
        #    soft target corresponds to exactly what the student sees; only the
        #    resolution differs (student @224, teacher @IMG). ──────────────────
        # Reuse the exact audited PIL augmentation distribution from §4. The
        # former Aqua-only RRC(scale=.60)+stronger jitter trained the student on
        # a different, more destructive distribution than its teacher.
        _geo_aug = indoor_train_pil_tx
        def _to_norm(img, size):
            img = T.functional.center_crop(T.functional.resize(img, size), size)
            return T.functional.normalize(T.functional.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)

        class _KDPairDataset(torch.utils.data.Dataset):
            def __init__(self, samples, train=True):
                self.samples = samples; self.train = train
            def __len__(self):
                return len(self.samples)
            def __getitem__(self, i):
                path, y = self.samples[i]
                img = _PILImage.open(path).convert('RGB')
                img = _geo_aug(img) if self.train else \
                    T.functional.center_crop(T.functional.resize(img, 256), 256)
                s = _to_norm(img, STUDENT_IMG)
                # When student and teacher input sizes match (the gamma/aqua
                # default, both 224) the two normalised tensors are byte-identical
                # — compute it ONCE and share it instead of resizing+normalising
                # the same crop twice. Halves the per-sample CPU transform (the
                # dataloader bottleneck that starves the A100); no behaviour change.
                t = s if STUDENT_IMG == IMG else _to_norm(img, IMG)
                return s, t, y

        _kd_src = ImageFolder(str(TRAIN_ROOT))  # samples + class order only
        assert _kd_src.classes == CLASS_NAMES, (_kd_src.classes, CLASS_NAMES)
        # Mirror §4's class-imbalance handling for the student. §4's DINOv2 teacher
        # is trained with a class-balanced/source-weighted sampler + logit-adjusted
        # loss; a plain-shuffle student would see the thin classes
        # (gray_mold_botrytis, the physiological pair) in only ~a few % of batches
        # and REGRESS their recall vs the teacher. The class-balanced sampler
        # indexes _kd_src.samples — the same list (same order) _KDPairDataset is
        # built from — so its indices line up. With balanced batches the hard term
        # stays plain CE: a single rebalance (logit-adjustment on top would
        # double-correct). The KD soft term transfers the teacher's (source-weighted,
        # houseplant-domain) distribution on top of that.
        kd_sampler = make_class_balanced_sampler(_kd_src, power=0.5)
        # More workers + warm (persistent) workers: the paired-view transform is
        # CPU-heavy and 2 workers starve the A100. Same data, just fed faster.
        _kd_workers = min(8, (os.cpu_count() or 2))
        kd_loader = DataLoader(_KDPairDataset(_kd_src.samples, train=True),
                               batch_size=BATCH, sampler=kd_sampler,
                               num_workers=_kd_workers, pin_memory=(device == 'cuda'),
                               persistent_workers=(_kd_workers > 0), prefetch_factor=2,
                               drop_last=True)
        # Remap VAL_INDOMAIN's ImageFolder label ids into the student's class order
        # (CLASS_NAMES). VAL_INDOMAIN may not hold every class as a subdir, which
        # would otherwise shift the ImageFolder ids and silently corrupt the macro-F1
        # early-stop signal (the §4/§6 eval loaders already guard this via class_to_idx).
        _kd_val_if = ImageFolder(str(VAL_INDOMAIN))
        _kd_val_remap = {i: CLASS_NAMES.index(c) for i, c in enumerate(_kd_val_if.classes)}
        _kd_val_samples = [(p, _kd_val_remap[y]) for p, y in _kd_val_if.samples]
        kd_val = DataLoader(_KDPairDataset(_kd_val_samples, train=False),
                            batch_size=BATCH, shuffle=False,
                            num_workers=min(4, (os.cpu_count() or 2)))

        opt = torch.optim.AdamW(
            list(student.parameters()) + list(feat_proj.parameters()),
            lr=KD_LR, weight_decay=KD_WD)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=KD_EPOCHS)
        ce = nn.CrossEntropyLoss()

        @torch.no_grad()
        def _student_macro_f1(loader):
            # Macro-F1 (not top-1) so early stopping tracks the THIN classes the
            # student is most at risk of dropping — top-1 on an imbalanced val set
            # is dominated by the majority classes. Mirrors §4's disease-F1 gate.
            student.eval(); ys, ps = [], []
            for sx, _, y in loader:
                with amp_ctx():
                    out = student(sx.to(device, non_blocking=True))
                ps.append(out.argmax(1).cpu()); ys.append(y)
            y_true, y_pred = torch.cat(ys).numpy(), torch.cat(ps).numpy()
            return float(f1_score(y_true, y_pred, average='macro', zero_division=0))

        best_student_f1, best_student_state, _kd_plateau = -1.0, None, 0
        KD_PATIENCE = 8   # generous: stops only on a real plateau (8 epochs of no
                          # macro-F1 gain), so the cosine LR-anneal tail isn't cut.
                          # Best-macro-F1 checkpoint is always restored -> the
                          # shipped student is never worse than running all KD_EPOCHS.
        model.eval()  # frozen teacher: clean soft targets + features (no head dropout)
        for ep in range(KD_EPOCHS):
            student.train(); run = 0.0
            for sx, tx, y in kd_loader:
                sx = sx.to(device, non_blocking=True); tx = tx.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                # bf16 autocast — the teacher DINOv2 forward dominates and §4 already
                # trains/evals it in bf16, so this is the SAME numerics, ~2-3x faster
                # on the A100. softmax/log_softmax stay fp32 under autocast (KD
                # targets unaffected); backward runs in fp32 (bf16 needs no scaler).
                with amp_ctx():
                    with torch.no_grad():
                        t_feats = model.backbone(tx)          # (B, embed_dim) CLS
                        t_logits = model.head(t_feats)        # == model(tx)
                    s_feats = student.avgpool(student.features(sx)).flatten(1)
                    s_logits = student.classifier(s_feats)    # == student(sx)
                    soft = F.kl_div(F.log_softmax(s_logits / KD_T, dim=1),
                                    F.softmax(t_logits / KD_T, dim=1),
                                    reduction='batchmean') * (KD_T * KD_T)
                    hard = ce(s_logits, y)
                    # Feature distillation: pull the student representation onto the
                    # teacher's DINOv2 CLS embedding (cosine; fp32 to avoid bf16
                    # underflow on the dot product). feat_proj is training-only.
                    feat = (1.0 - F.cosine_similarity(
                        feat_proj(s_feats).float(), t_feats.float(), dim=1)).mean()
                    loss = (KD_ALPHA * soft + (1.0 - KD_ALPHA) * hard
                            + KD_BETA * feat)
                opt.zero_grad(); loss.backward(); opt.step()
                run += loss.item()
            sched.step()
            f1 = _student_macro_f1(kd_val)
            if f1 > best_student_f1:
                best_student_f1 = f1
                best_student_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
                _kd_plateau = 0
            else:
                _kd_plateau += 1
            print(f'  KD ep{ep+1:02d}/{KD_EPOCHS}  loss={run/len(kd_loader):.3f}  val_macroF1={f1:.3f}')
            if _kd_plateau >= KD_PATIENCE:
                print(f'  KD early stop @ ep{ep+1} (best val_macroF1={best_student_f1:.3f})')
                break
        if best_student_state is not None:
            student.load_state_dict(best_student_state)
        student.eval()
        print(f'Aqua student trained  best_val_macroF1={best_student_f1:.3f}')

        # ── Export → TFLite (onnx2tf direct, no SavedModel round-trip) ──
        # Path: torch -> ONNX -> onnx2tf -> strict FLOAT32 TFLite. The Android
        # serving contract requires FLOAT32 NHWC input/output. Dynamic-range
        # INT8 collapsed this MobileNetV3 student; FLOAT16 needs an unpromised
        # delegate. Neither is an eligible shipping fallback.
        #
        # v2.5.1 FIX — the previous build crashed here on every run. Current
        # onnx2tf writes the .tflite variants DIRECTLY via its LiteRT flatbuffer
        # backend and no longer always emits a TF SavedModel (`saved_model.pb`),
        # so `tf.lite.TFLiteConverter.from_saved_model(_saved)` raised
        # `OSError: SavedModel file does not exist` and dropped the aqua bundle
        # (the gamma §6 bundle was unaffected, which masked it). We now read the
        # .tflite onnx2tf already produced and require its FLOAT32 build, which
        # runs on the plain CPU interpreter without a delegate. The INT8 sibling
        # COLLAPSES MobileNetV3-Small to near-constant output (it scored ~0.01 macro-F1
        # on the §6 exact-metrics eval vs ~0.82 for the float student, v2.6.8), so the
        # faithful float32 build (~5.8 MB, downloaded not APK-bundled) ships; a float16
        # build needs a float16 delegate (XNNPACK/GPU) the export box + app LiteRT
        # lack (CONV_2D fails to prepare, v2.6.9); int8 collapses MobileNetV3.
        #
        # The conversion runs in the CPython 3.12 export env that §1 provisioned
        # and preflighted (provision_aqua_export_env / aqua_onnx_to_tflite): TF
        # 2.19 has no 3.13+ wheel, and installing TF into THIS kernel could not
        # swap the protobuf §3's ONNX export already imported. This kernel only
        # exports the ONNX and verifies the result.
        _onnx_path = str(AQUA_DIR / 'aqua_student.onnx')
        _dummy = torch.randn(1, 3, STUDENT_IMG, STUDENT_IMG, device=device)
        torch.onnx.export(student, _dummy, _onnx_path, opset_version=18,
                          input_names=['input'], output_names=['logits'],
                          dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}})

        _saved = str(AQUA_DIR / 'aqua_tf_export')
        _chosen = aqua_onnx_to_tflite(_onnx_path, _saved)
        _tflite_path = AQUA_DIR / 'aqua_classifier.tflite'
        shutil.copyfile(_chosen, _tflite_path)
        print(f'Aqua TFLite: selected {_chosen.name}  (FLOAT32 contract, '
              f'{_tflite_path.stat().st_size/1e6:.2f} MB)')

        # Verify the shipped student loads + runs and tracks the torch student —
        # catches a broken NHWC transpose/dtype before it ships. FLOAT32 dtype,
        # the app's tensor shapes and numeric parity are mandatory.
        try:
            aqua_verify_tflite(_tflite_path, student, _dummy, N_CLASSES)
        except Exception as _verr:
            # Fail CLOSED: combined mode promises both artifacts, so a model that
            # cannot satisfy the exact FLOAT32 contract aborts the current run.
            _tflite_path.unlink(missing_ok=True)
            print(f'Aqua TFLite NOT shipped -- verification failed: '
                  f'{type(_verr).__name__}: {_verr}')
            raise

        # ── Classifier-side OOD floor for Aqua (MaxLogit, plant-only) ──
        # Aqua is single-stage: no detector and no embedding bank, so until now it
        # abstained only on blur or a natural `not_a_plant` argmax. The cheapest
        # honest signal is §6's single-image MaxLogit floor: the PLANT-ONLY max
        # logit (not_a_plant column excluded, same scope as §6) at the 5th
        # percentile of real plant photos, so ~5 % of plants are refused in
        # exchange for catching inputs that look like no condition we know.
        # Fit on temperature_cal with the APP's preprocessing (resize the short
        # side to img_size * eval_resize_ratio, centre-crop img_size: exactly
        # AquaClassifier.resizeAndCenterCrop) and the FP32 torch student, whose
        # FLOAT32 TFLite export is parity-verified below. Like §6, a floor that
        # does not separate plants from not_a_plant ships as null (gate off),
        # and so does one with too few rows to tell. val_realworld reports the
        # out-of-sample cost. The app reads `classes.json -> ood`; a sidecar
        # without it (or with a null floor) serves exactly as before.
        AQUA_OOD_TARGET_PLANT_FPR = 0.05
        _aq_nap = CLASS_NAMES.index('not_a_plant') if 'not_a_plant' in CLASS_NAMES else None
        _aq_scope = 'plant_only' if _aq_nap is not None else 'all_classes'
        _aq_cols = [i for i in range(N_CLASSES) if _aq_scope == 'all_classes' or i != _aq_nap]
        _aq_tx = T.Compose([T.Resize(int(STUDENT_IMG * EVAL_RESIZE_RATIO)), T.CenterCrop(STUDENT_IMG),
                            T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

        class _AquaServedDS(torch.utils.data.Dataset):
            def __init__(self, root):
                _if = ImageFolder(str(root))
                _rm = {i: CLASS_NAMES.index(c) for i, c in enumerate(_if.classes)}
                self.samples = [(p, _rm[y]) for p, y in _if.samples]
            def __len__(self):
                return len(self.samples)
            def __getitem__(self, i):
                path, y = self.samples[i]
                return _aq_tx(_PILImage.open(path).convert('RGB')), y

        @torch.no_grad()
        def _aqua_served_logits(root):
            """(plant-only max logit, argmax, label) per photo; FP32, eval mode."""
            dl = DataLoader(_AquaServedDS(root), batch_size=BATCH, shuffle=False,
                            num_workers=min(4, (os.cpu_count() or 2)))
            ml, am, ys = [], [], []
            for x, y in dl:
                logits = student(x.to(device, non_blocking=True)).float()
                ml.append(logits[:, _aq_cols].max(1).values.cpu())
                am.append(logits.argmax(1).cpu()); ys.append(y)
            return (torch.cat(ml).numpy().astype(np.float64), torch.cat(am).numpy(),
                    torch.cat(ys).numpy())

        student.eval()
        _aq_ml, _aq_am, _aq_y = _aqua_served_logits(TEMP_CAL)
        _aq_plant = _aq_ml[_aq_y != _aq_nap] if _aq_nap is not None else _aq_ml
        _aq_napml = _aq_ml[_aq_y == _aq_nap] if _aq_nap is not None else np.array([])
        AQUA_ML_FLOOR = (float(np.percentile(_aq_plant, AQUA_OOD_TARGET_PLANT_FPR * 100))
                         if len(_aq_plant) >= 30 else None)
        _aq_sep = (float(np.percentile(_aq_plant, 50) - np.percentile(_aq_napml, 50))
                   if len(_aq_plant) and len(_aq_napml) else None)
        if AQUA_ML_FLOOR is None:
            print(f'WARN: Aqua OOD floor not fitted -- only {len(_aq_plant)} plant rows in temperature_cal')
        elif _aq_sep is None or len(_aq_napml) < 10:
            print(f'WARN: Aqua OOD floor disabled -- {len(_aq_napml)} not_a_plant rows cannot show separation')
            AQUA_ML_FLOOR = None
        elif _aq_sep < 0:
            print(f'WARN: Aqua OOD floor disabled -- not_a_plant max-logit median above plants '
                  f'(sep={_aq_sep:+.3f})')
            AQUA_ML_FLOOR = None

        # Out-of-sample cost and catch on val_realworld (never used to fit).
        _rw_ml, _rw_am, _rw_y = _aqua_served_logits(VAL_PRIMARY)
        _rw_plant = _rw_y != _aq_nap if _aq_nap is not None else np.ones_like(_rw_y, dtype=bool)
        _rw_missed_nap = (~_rw_plant) & (_rw_am != _aq_nap) if _aq_nap is not None else ~_rw_plant
        if AQUA_ML_FLOOR is not None:
            _rw_reject = _rw_ml < AQUA_ML_FLOOR
            _rw_plant_fr = float(_rw_reject[_rw_plant].mean()) if _rw_plant.any() else None
            _rw_nap_catch = (float(_rw_reject[_rw_missed_nap].mean())
                             if _rw_missed_nap.any() else None)
        else:
            _rw_plant_fr = _rw_nap_catch = None
        AQUA_OOD = {
            'score': 'max_logit',
            'max_logit_floor': AQUA_ML_FLOOR,
            'max_logit_scope': _aq_scope,
            'target_plant_fpr': AQUA_OOD_TARGET_PLANT_FPR,
            'fit_source': 'torch_fp32_student',
            'fit_split': 'temperature_cal',
            'preprocess': 'resize_short_side_eval_ratio_center_crop',
            'calibration': {
                'plant_n': int(len(_aq_plant)), 'nap_n': int(len(_aq_napml)),
                'plant_p05': float(np.percentile(_aq_plant, 5)) if len(_aq_plant) else None,
                'plant_p50': float(np.percentile(_aq_plant, 50)) if len(_aq_plant) else None,
                'nap_p50': float(np.percentile(_aq_napml, 50)) if len(_aq_napml) else None,
                'separation_p50': _aq_sep,
                'val_realworld_plant_false_reject': _rw_plant_fr,
                'val_realworld_missed_nap_caught': _rw_nap_catch,
                'val_realworld_missed_nap_n': int(_rw_missed_nap.sum()),
            },
        }
        print('Aqua OOD floor:', json.dumps(AQUA_OOD, indent=2))

        # classes.json mirrors the server bundle so the app reads the same metadata
        # (class order, input size, normalisation, family tag).
        (AQUA_DIR / 'classes.json').write_text(json.dumps({
            'model_version': MODEL_VERSION,
            'model_family': 'aqua',
            'architecture': 'mobilenetv3_small',
            'class_names': CLASS_NAMES,
            'training': {'img_size': STUDENT_IMG,
                         'eval_resize_ratio': EVAL_RESIZE_RATIO,
                         'distilled_from': FAMILY_CONFIG['hub_id']},
            'inference': {'layout': 'NHWC', 'normalize_mean': IMAGENET_MEAN,
                          'normalize_std': IMAGENET_STD, 'tta': 'none',
                          'output_semantics': 'logits'},
            # Classifier-side OOD floor (above); null = gate off.
            'ood': AQUA_OOD,
        }, indent=2), encoding='utf-8')
        print(f'Aqua TFLite written: {_tflite_path}  ({_tflite_path.stat().st_size/1e6:.2f} MB)  '
              f'-> drop {_tflite_path.name} + classes.json into server static/aqua/; '
              f'the app downloads + caches it from GET /api/aqua/model (no APK bundling).')
        AQUA_EXPORT_SUCCEEDED = True
    except Exception as _aqua_err:
        import traceback
        print(f'Aqua distillation/TFLite export FAILED ({type(_aqua_err).__name__}: {_aqua_err})')
        traceback.print_exc()
        raise RuntimeError('combined run aborted: current-run Aqua artifact was not produced') from _aqua_err
    finally:
        # Always release the aqua KD + export memory (loader workers, student,
        # ONNX/TF intermediates) before §6 — left resident they stack into the
        # aqua-combo §6 OOM. The teacher (`model`) is KEPT: §6 exports it. Runs
        # on both the success and the failure path (a partial export still
        # leaves the student + warm KD workers behind otherwise).
        import gc as _gc5, shutil as _sh5
        try:
            _sh5.rmtree(_saved, ignore_errors=True)   # large onnx2tf TF export dir
        except Exception:
            pass
        for _nm in ('kd_loader', 'kd_val', 'kd_sampler', '_kd_src', '_kd_val_if',
                    'student', 'opt', 'sched', 'tflite_bytes', 'conv', '_dummy',
                    '_geo_aug', '_aq_ml', '_aq_am', '_aq_y', '_rw_ml', '_rw_am', '_rw_y'):
            globals().pop(_nm, None)
        _gc5.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()
        print('Aqua: released KD loader / student / TF intermediates before §6.')


## §6 — Production export: ONNX → dynamic INT8 + temperature + conformal

### Export & quantisation order of operations

1. Export FP32 ONNX from the trained **frozen-DINOv2 + head** model
   (backbone+head in one graph). CPU export keeps any CUDA-only ops out
   of the graph; `quant_pre_process` (symbolic-shape inference) runs with
   a copy-through fallback if it can't resolve the ViT graph. An
   **ONNX↔torch parity assert** (max abs logit diff < 1e-2) fails loudly
   if the position-embedding interpolation / opset broke the export
   before it can poison calibration.
2. **Dynamic INT8** quantisation (`quantize_dynamic`, `QInt8` weights).
   ORT's official guidance is dynamic quant for transformer/ViT models:
   activation ranges are computed at run time (no calibration set), and
   the weight-heavy MatMul/Gemm ops — most of a ViT — are quantised for
   ~2–3× CPU speedup at ~0 accuracy loss. This sidesteps the
   HardSwish/SE static-PTQ traps that crippled MobileNetV3.
3. Drift gate on val_realworld disease macro-F1:
   * `drift ≤ 0.02` → ship INT8.
   * `0.02 < drift ≤ 0.05` → ship INT8 with WARN.
   * `drift > 0.05` → **honest FP32 fallback**: write `classifier_fp32.onnx`,
     delete `classifier_int8.onnx`, record `int8.shipped = False`.
   The int8-vs-fp32 ordering is additionally reported with a **paired
   bootstrap** CI (report Stage 0) — a 0.03–0.04 gap at n≈450 is within
   sampling noise, so `paired_bootstrap.significant` is the honest test,
   not the raw F1 difference.
4. Fit temperature on the SHIPPED artefact's logits (INT8 if it passed
   the drift gate, FP32 otherwise — never fit T on a collapsed graph).
5. Fit conformal RAPS on `conformal_cal` softmaxed through the fitted T.
   Use ~80/20 fit/audit when `n_cal ≥ 200`; otherwise fit on the full
   split and report `audit_n = 0`.
6. Calibrate the MaxLogit OOD floor on `val_realworld` (plant vs NAP);
   the gate is disabled (`max_logit_floor=null`) when separation is the
   wrong sign.
7. Final eval on every val slice via the shipped ORT session.
8. **Ship gate** runs BEFORE any declarative artefact is persisted. Every
   calibration step above (F1, T, conformal, MaxLogit) is computed
   through the same `inference.tta` mode, so enabling `USE_TTA_HFLIP`
   never decalibrates the shipped model.


In [ ]:
# §6 — DINOv2 export: ONNX FP32 → dynamic INT8 + temperature + conformal RAPS + ship gate
assert_determinism('s6_export')  # labels any warning from here on as §6
import datetime
import shutil as _sh
import onnx  # noqa: F401  (import validates the onnx runtime dep is present)
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
except ImportError as e:
    raise RuntimeError(
        'onnxruntime.quantization.shape_inference.quant_pre_process is required. '
        'Upgrade onnxruntime to >=1.17 and re-run §1 install.'
    ) from e

Path('models').mkdir(exist_ok=True)

# Which family this §6 ONNX bundle (models/) is labelled as. In a combined run
# the §4 DINOv2 model IS gamma (Aqua's teacher = dinov2_vits14 @224), so ONE
# TRAIN_MODE='combined' run yields BOTH the gamma server bundle (models/, below)
# AND the aqua TFLite student (models/aqua/, from §5). A 'gamma' run is gamma
# only; a 'frontier' run is the ViT-B/14 backbone, exported as 'frontier'.
# Decided in §1: FAMILY_CONFIG['export_family']. A `combined` run's PRIMARY
# bundle is the gamma server model (§5 writes the aqua TFLite separately).
_EXPORT_FAMILY = MODEL_FAMILY
if FAMILY_CONFIG.get('distill'):
    print(f"combined run: §6 exports the DINOv2 teacher as the '{_EXPORT_FAMILY}' bundle "
          "(models/); the shipped aqua model is models/aqua/aqua_classifier.tflite (§5). "
          "-> gamma + aqua from one run.")

# ── Test-time augmentation (report Stage 3) ─────────────────────
# Horizontal-flip TTA (2×) is "nearly free accuracy" given the CPU latency
# headroom. It is OFF by default: when ON, EVERYTHING below — the ship-gate
# F1, temperature, conformal q_hat and the MaxLogit floor — is computed
# through the SAME flip-averaging, and ``inference.tta='hflip'`` is written
# to classes.json so the server (`classifier.py`) reproduces the identical
# averaging. We average LOGITS (not softmax probs): that keeps temperature
# scaling, MaxLogit and conformal all operating on the same quantity they
# were fit on, so enabling TTA never silently decalibrates the model.
USE_TTA_HFLIP = True
TTA_MODE = 'hflip' if USE_TTA_HFLIP else 'none'

# INT8-vs-FP32 ship decision (report P1: ship INT8 for Gamma). Gamma has
# CPU latency headroom, but INT8 is ~4x smaller (=22.5 vs 87 MB) and faster
# on CPU, and it is the natural base for the Aqua tier -- so we now DEFAULT
# TO INT8 and only fall back to FP32 when INT8 is actually worse. "Actually
# worse" is decided below by the hard drift gate (drift > FAIL_DRIFT) AND the
# paired bootstrap (INT8 significantly below FP32, CI excludes 0) -- never by
# a point-estimate inversion at this sample size. Set PREFER_FP32=True to
# force FP32 regardless (a tier that wants max accuracy and has the budget).
PREFER_FP32 = False


def _ort_session(path):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=['CPUExecutionProvider'])


def _ort_logits(sess, x_np):
    in_name = sess.get_inputs()[0].name
    logits = sess.run(None, {in_name: x_np})[0]
    if USE_TTA_HFLIP:
        flipped = np.ascontiguousarray(x_np[..., ::-1])
        logits = (logits + sess.run(None, {in_name: flipped})[0]) * 0.5
    return logits


def _ort_forward(sess, x_np):
    """Like _ort_logits but also returns the CLS embedding (2nd output)
    TTA-averaged the same way; embedding is None on a legacy 1-output graph."""
    in_name = sess.get_inputs()[0].name
    has_emb = len(sess.get_outputs()) >= 2
    outs = sess.run(None, {in_name: x_np})
    logits, emb = outs[0], (outs[1] if has_emb else None)
    if USE_TTA_HFLIP:
        fouts = sess.run(None, {in_name: np.ascontiguousarray(x_np[..., ::-1])})
        logits = (logits + fouts[0]) * 0.5
        if has_emb:
            emb = (emb + fouts[1]) * 0.5
    return logits, emb


disease_idx = [i for i, n in enumerate(CLASS_NAMES) if n not in ('healthy', 'not_a_plant')]


def _disease_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))



def _predict_onnx(path, dl):
    """Return (y_true, y_pred) integer arrays — needed for the paired
    bootstrap so the int8-vs-fp32 comparison is on identical indices."""
    sess = _ort_session(path)
    y_true, y_pred = [], []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        y_pred.extend(logits.argmax(1).tolist())
        y_true.extend(y.tolist())
    return np.asarray(y_true), np.asarray(y_pred)


def _paired_bootstrap_f1(y_true, y_pred_a, y_pred_b, *, n_boot=5000, ci=0.95, seed=SEED):
    """Paired bootstrap for (A − B) disease macro-F1 on a shared test set
    (report Stage 0). Resample one index vector per iteration, apply to
    BOTH models, summarise the difference distribution. ``significant`` is
    True only when the CI of the difference excludes zero — the honest test
    the historical ``f1_int8=0.73 > f1_fp32=0.69`` inversion never passed.
    """
    y_true = np.asarray(y_true); a = np.asarray(y_pred_a); b = np.asarray(y_pred_b)
    n = len(y_true)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        diffs[i] = _disease_f1(yt, a[idx]) - _disease_f1(yt, b[idx])
    lo = float(np.percentile(diffs, (1 - ci) / 2 * 100))
    hi = float(np.percentile(diffs, (1 + ci) / 2 * 100))
    return {
        'metric': 'disease_macro_f1', 'compared': 'int8_minus_fp32',
        'diff_point': float(_disease_f1(y_true, a) - _disease_f1(y_true, b)),
        'diff_ci_low': lo, 'diff_ci_high': hi, 'ci_level': ci,
        'p_int8_gt_fp32': float(np.mean(diffs > 0.0)),
        'significant': bool(lo > 0.0 or hi < 0.0),
        'n': int(n), 'n_boot': int(n_boot),
    }


def _per_class_recall(y_true, y_pred):
    """Recall per class index; nan when a class has no support (so an
    absent class is never silently scored 0/1)."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    out = np.full(len(CLASS_NAMES), np.nan, dtype=np.float64)
    for i in range(len(CLASS_NAMES)):
        sup = int(np.sum(yt == i))
        if sup:
            out[i] = float(np.sum((yt == i) & (yp == i)) / sup)
    return out


def _per_class_recall_ci(y_true, y_pred, *, n_boot=2000, ci=0.95, seed=SEED):
    """Percentile-bootstrap recall CI per class on the shipped model
    (report Stage 0: report per-class recall CIs, not just point
    estimates — a ~30-50 sample class has a recall CI of +/-0.10+, so the
    weak-class numbers the report targets must ship with their CI)."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    n = len(yt)
    rng = np.random.default_rng(seed)
    boot = np.full((n_boot, len(CLASS_NAMES)), np.nan, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = _per_class_recall(yt[idx], yp[idx])
    point = _per_class_recall(yt, yp)
    out = {}
    for i, c in enumerate(CLASS_NAMES):
        col = boot[:, i]; valid = col[~np.isnan(col)]
        if valid.size == 0:
            out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                      'ci_low': None, 'ci_high': None,
                      'support': int(np.sum(yt == i)), 'n_boot_valid': 0}
            continue
        lo = float(np.percentile(valid, (1 - ci) / 2 * 100))
        hi = float(np.percentile(valid, (1 + ci) / 2 * 100))
        out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                  'ci_low': lo, 'ci_high': hi, 'ci_half_width': (hi - lo) / 2,
                  'support': int(np.sum(yt == i)), 'n_boot_valid': int(valid.size)}
    return {'metric': 'per_class_recall', 'ci_level': ci, 'n': int(n),
            'n_boot': int(n_boot), 'classes': out}


def _per_class_precision(y_true, y_pred):
    """Precision per class index; nan when a class is never PREDICTED (so an
    unpredicted class is never silently scored 0/1). The physiological classes
    are precision-bound (they over-trigger), so precision ships beside recall."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    out = np.full(len(CLASS_NAMES), np.nan, dtype=np.float64)
    for i in range(len(CLASS_NAMES)):
        pred = int(np.sum(yp == i))
        if pred:
            out[i] = float(np.sum((yt == i) & (yp == i)) / pred)
    return out


def _per_class_precision_ci(y_true, y_pred, *, n_boot=2000, ci=0.95, seed=SEED):
    """Percentile-bootstrap precision CI per class on the shipped model — the
    precision companion to _per_class_recall_ci. 'support' is true-class support
    (comparable to the recall CI); 'n_predicted' is precision's denominator. A
    replicate that never predicts a class yields nan and is dropped from its
    percentile computation."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    n = len(yt)
    rng = np.random.default_rng(seed)
    boot = np.full((n_boot, len(CLASS_NAMES)), np.nan, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = _per_class_precision(yt[idx], yp[idx])
    point = _per_class_precision(yt, yp)
    out = {}
    for i, c in enumerate(CLASS_NAMES):
        col = boot[:, i]; valid = col[~np.isnan(col)]
        base = {'support': int(np.sum(yt == i)), 'n_predicted': int(np.sum(yp == i))}
        if valid.size == 0:
            out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                      'ci_low': None, 'ci_high': None, 'n_boot_valid': 0, **base}
            continue
        lo = float(np.percentile(valid, (1 - ci) / 2 * 100))
        hi = float(np.percentile(valid, (1 + ci) / 2 * 100))
        out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                  'ci_low': lo, 'ci_high': hi, 'ci_half_width': (hi - lo) / 2,
                  'n_boot_valid': int(valid.size), **base}
    return {'metric': 'per_class_precision', 'ci_level': ci, 'n': int(n),
            'n_boot': int(n_boot), 'classes': out}


# ── Export the frozen-DINOv2 + head model to ONNX ───────────────
# Move to CPU for a deployment-faithful FP32 export (no CUDA-only ops in
# the graph). DINOv2 interpolates its position embedding to the 224-px
# input; opset 17 covers the LayerNorm / SDPA / grid-sample ops the ViT
# emits. The parity assert below fails loudly if the export diverged.
# Merge LoRA adapters into the base Linears so the export is a single,
# self-contained graph (validated: ONNX<->torch parity ~1e-6 after merge).
# `hasattr` guard, not just `USE_LORA`: §6 gets re-run by hand whenever
# something late in it fails (ship gate raises, an OOM in the aqua eval), and on
# the second pass the backbone is already a plain module — `merge_and_unload`
# would raise AttributeError and kill the cell before it reached the export,
# the calibration or the Stage-1 audit. Merging is idempotent by intent, so make
# it idempotent in fact.
if USE_LORA and hasattr(model.backbone, 'merge_and_unload'):
    model.backbone = model.backbone.merge_and_unload()
    model.lora_active = False
    print('LoRA adapters merged into the backbone for export.')
elif USE_LORA:
    print('LoRA adapters already merged (§6 re-run) — skipping.')
# Export with a SECOND output -- the DINOv2 CLS embedding -- so the server
# computes an embedding-space k-NN OOD score (ood.py) without a second model.
# output[0] stays the logits, so every existing run(...)[0] call site (eval,
# INT8 drift, temperature, conformal) is unaffected; a legacy bundle without
# this output simply has no embedding to score and OOD falls back to MaxLogit.
class _ExportWithEmbedding(nn.Module):
    def __init__(self, clf):
        super().__init__()
        self.clf = clf
    def forward(self, x):
        feats = self.clf.backbone(x)          # (N, embed_dim) CLS embedding
        return self.clf.head(feats), feats

# Fold CLASS_LOGIT_BIAS (§1) into the head's final bias vector. Everything
# downstream — the FP32/INT8 export, the parity assert, temperature, conformal,
# the OOD calibration and every §6 metric — then sees the biased logits, which
# is the point: the bias becomes part of the model rather than a serving-time
# adjustment that the calibration knows nothing about.
CLASS_LOGIT_BIAS_APPLIED = {}
if CLASS_LOGIT_BIAS:
    _final = model.head[-1]
    if not isinstance(_final, nn.Linear) or _final.out_features != len(CLASS_NAMES):
        raise RuntimeError('CLASS_LOGIT_BIAS expects the head to end in '
                           f'nn.Linear(_, {len(CLASS_NAMES)}); got {_final!r}. '
                           'Fold it manually or clear the dict.')
    if _final.bias is None:
        raise RuntimeError('CLASS_LOGIT_BIAS needs a head whose last Linear HAS a bias.')
    _unknown = set(CLASS_LOGIT_BIAS) - set(CLASS_NAMES)
    if _unknown:
        raise RuntimeError(f'CLASS_LOGIT_BIAS names classes that do not exist: {sorted(_unknown)}')
    # RESTORE-THEN-APPLY, never accumulate. `model` is a live object and §6 gets
    # re-run by hand all the time (ship gate raises, an OOM in the aqua eval, a
    # typo further down) — a bare `+=` would silently double the bias on the
    # second pass, and every downstream fit (temperature, INT8, conformal) would
    # be consistent with the doubled model while classes.json recorded the
    # single value. With USE_LORA=True the merge_and_unload above happens to
    # raise on a re-run and hides this; with USE_LORA=False — one of the three
    # documented experiments — §6 is cleanly re-runnable and it would not.
    with torch.no_grad():
        if not hasattr(model, '_head_bias_pristine'):
            model._head_bias_pristine = _final.bias.detach().clone()
        _final.bias.copy_(model._head_bias_pristine)
        for _cname, _b in CLASS_LOGIT_BIAS.items():
            _b = float(_b)
            if _b == 0.0:
                continue
            _final.bias[CLASS_NAMES.index(_cname)] += _b
            CLASS_LOGIT_BIAS_APPLIED[_cname] = _b
    print(f'CLASS_LOGIT_BIAS folded into the head: {CLASS_LOGIT_BIAS_APPLIED} '
          '(temperature / conformal / INT8 are fit AFTER this, on the biased logits)')
    log_run({'phase': 'class_logit_bias', 'applied': CLASS_LOGIT_BIAS_APPLIED})

export_model = model.to('cpu').eval()
export_wrapped = _ExportWithEmbedding(export_model).eval()
raw_path  = 'models/classifier_raw.onnx'
fp32_onnx = 'models/classifier.onnx'
int8_onnx = 'models/classifier_int8.onnx'
dummy = torch.randn(1, 3, IMG, IMG)
torch.onnx.export(
    export_wrapped, dummy, raw_path,
    opset_version=17, input_names=['input'],
    output_names=['logits', 'embedding'],
    dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}, 'embedding': {0: 'N'}},
    dynamo=False,
)
try:
    quant_pre_process(raw_path, fp32_onnx, skip_symbolic_shape=False)
except Exception as exc:  # noqa: BLE001 — fall back to the un-preprocessed graph
    print(f'WARN: quant_pre_process failed ({type(exc).__name__}: {exc}); '
          f'using the raw export as the FP32 graph.')
    _sh.copyfile(raw_path, fp32_onnx)

# Export parity check — catches a broken ViT export (bad interpolation /
# opset) before it silently poisons every downstream calibration step.
with torch.no_grad():
    torch_logits = export_wrapped(dummy)[0].numpy()
_p_sess = _ort_session(fp32_onnx)
onnx_logits = _p_sess.run(None, {_p_sess.get_inputs()[0].name: dummy.numpy().astype(np.float32)})[0]
_parity = float(np.abs(torch_logits - onnx_logits).max())
print(f'ONNX↔torch parity: max_abs_logit_diff={_parity:.3e}')
if _parity > 1e-2:
    raise RuntimeError(
        f'ONNX export diverged from torch (max_abs_logit_diff={_parity:.3e}). '
        f'Investigate the DINOv2 position-embedding interpolation / opset before shipping.'
    )

# ── Dynamic INT8 quantisation ───────────────────────────────────
# ONNX Runtime's official guidance: dynamic quantization for transformer /
# ViT models, static for CNNs. Dynamic quant computes activation ranges at
# run time (no calibration set) and quantises the weight-heavy MatMul/Gemm
# ops — which is most of a ViT — for ~2–3× CPU speedup at ~0 accuracy loss,
# avoiding the HardSwish/SE static-PTQ traps that crippled MobileNetV3.
quantize_dynamic(fp32_onnx, int8_onnx, weight_type=QuantType.QInt8)
print(f'dynamic INT8 quantised: {Path(int8_onnx).stat().st_size / 1e6:.1f} MB '
      f'(fp32 {Path(fp32_onnx).stat().st_size / 1e6:.1f} MB)')
Path(raw_path).unlink(missing_ok=True)

# ── Drift gate (val_realworld) + paired bootstrap ───────────────
yt_rw, yp_fp32 = _predict_onnx(fp32_onnx, val_realworld_dl)
_,     yp_int8 = _predict_onnx(int8_onnx, val_realworld_dl)
f1_fp32 = _disease_f1(yt_rw, yp_fp32)
f1_int8 = _disease_f1(yt_rw, yp_int8)
drift = f1_fp32 - f1_int8
paired_int8_vs_fp32 = _paired_bootstrap_f1(yt_rw, yp_int8, yp_fp32)
print(f'val_realworld disease F1  fp32={f1_fp32:.4f}  int8={f1_int8:.4f}  drift={drift:+.4f}')
print(f'paired bootstrap (int8−fp32): diff={paired_int8_vs_fp32["diff_point"]:+.4f}  '
      f'95% CI=[{paired_int8_vs_fp32["diff_ci_low"]:+.4f}, {paired_int8_vs_fp32["diff_ci_high"]:+.4f}]  '
      f'significant={paired_int8_vs_fp32["significant"]}')

WARN_DRIFT, FAIL_DRIFT = 0.02, 0.05
canonical_fp32_fallback = Path('models/classifier_fp32.onnx')
# Ship FP32 when explicitly preferred, OR when INT8 fails the hard drift
# gate, OR when the paired bootstrap says INT8 is SIGNIFICANTLY worse than
# FP32 (CI excludes 0 and the point difference is negative). The bootstrap
# term is the honest backstop: a -0.01 drift the bootstrap cannot separate
# from noise is NOT a reason to forfeit INT8's 4x size + latency win.
_int8_significantly_worse = bool(
    paired_int8_vs_fp32['significant'] and paired_int8_vs_fp32['diff_point'] < 0.0
)
_ship_fp32 = PREFER_FP32 or (drift > FAIL_DRIFT) or _int8_significantly_worse
if _ship_fp32:
    why = (f'INT8 drift {drift:+.4f} > FAIL_DRIFT={FAIL_DRIFT}' if drift > FAIL_DRIFT
           else 'INT8 significantly worse than FP32 (paired bootstrap CI excludes 0)'
           if _int8_significantly_worse
           else f'PREFER_FP32=True (INT8 drift {drift:+.4f}; CPU latency headroom)')
    print(f'Shipping FP32 as classifier.onnx. Reason: {why}. '
          f'classifier_int8.onnx will NOT be emitted.')
    Path(int8_onnx).unlink(missing_ok=True)
    _sh.copyfile(fp32_onnx, canonical_fp32_fallback)
    int8_shipped, shipped_artifact = False, f'classifier.onnx (FP32; {why})'
    eval_sess_path = fp32_onnx
else:
    if drift > WARN_DRIFT:
        print(f'WARN: INT8 drift {drift:+.4f} > WARN_DRIFT={WARN_DRIFT}; shipping anyway.')
    canonical_fp32_fallback.unlink(missing_ok=True)
    int8_shipped, shipped_artifact = True, 'classifier_int8.onnx'
    eval_sess_path = int8_onnx

eval_sess = _ort_session(eval_sess_path)
QUANT_FIT_SOURCE = 'int8' if int8_shipped else 'fp32_fallback'

# Per-class recall bootstrap CI on the SHIPPED artefact's val_realworld
# predictions (report Stage 0). This is where the weak classes the report
# targets (nutrient_deficiency, leaf_scorch_*, gray_mold_*) live; their
# point recall is only trustworthy alongside a CI.
shipped_rw_pred = yp_int8 if int8_shipped else yp_fp32
recall_ci_realworld = _per_class_recall_ci(yt_rw, shipped_rw_pred)
print('per-class recall (val_realworld, shipped):')
for _c, _d in recall_ci_realworld['classes'].items():
    if _d.get('ci_low') is None:
        print(f'  {_c:28s} n={_d["support"]:>3}  (no bootstrap support)')
    else:
        print(f'  {_c:28s} n={_d["support"]:>3}  recall={_d["point"]:.3f} '
              f'95% CI=[{_d["ci_low"]:.3f}, {_d["ci_high"]:.3f}]')

# Per-class PRECISION bootstrap CI on the same shipped val_realworld
# predictions. The weak physiological classes are precision-bound, so their
# precision (and its CI) is the metric that actually tracks the over-trigger.
precision_ci_realworld = _per_class_precision_ci(yt_rw, shipped_rw_pred)
print('per-class precision (val_realworld, shipped):')
for _c, _d in precision_ci_realworld['classes'].items():
    if _d.get('ci_low') is None:
        print(f'  {_c:28s} pred={_d["n_predicted"]:>3}  (never predicted / no support)')
    else:
        print(f'  {_c:28s} pred={_d["n_predicted"]:>3}  precision={_d["point"]:.3f} '
              f'95% CI=[{_d["ci_low"]:.3f}, {_d["ci_high"]:.3f}]')

# Conformal split — used only for RAPS quantile fitting downstream.
cal_dataset, cal_dl = make_eval_loader(
    CONFORMAL_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)


def _collect_logits(sess, dl):
    all_logits, all_y = [], []
    for x, y in dl:
        all_logits.append(_ort_logits(sess, x.numpy().astype(np.float32)))
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)


# ── Temperature scaling on the shipped artefact's logits ────────
logits_temp_np, y_temp_np = _collect_logits(eval_sess, temp_cal_dl)
logits_temp = torch.from_numpy(logits_temp_np)
y_temp = torch.from_numpy(y_temp_np).long()

log_T = torch.tensor(0.0, requires_grad=True)
opt_T = torch.optim.LBFGS([log_T], lr=0.05, max_iter=50)
def closure():
    opt_T.zero_grad()
    T_now = log_T.exp().clamp(0.05, 10.0)
    loss = F.cross_entropy(logits_temp / T_now, y_temp)
    loss.backward(); return loss
opt_T.step(closure)
T_val = float(log_T.detach().exp().clamp(0.05, 10.0))
temperature_payload = {
    'temperature': T_val,
    'source': QUANT_FIT_SOURCE,
    'tta': TTA_MODE,
    '_note': f'Fitted with LBFGS on temperature_cal NLL using {QUANT_FIT_SOURCE} ONNX '
             f'logits (tta={TTA_MODE}).',
    'fit_sources': ['plantdoc', 'inat', 'coco', 'inat_animalia', 'openimages_indoor',
                    'coco_hard_partial_plant', 'inat_hard_greentexture'],
}
print(f'fitted T={T_val:.3f}  source={QUANT_FIT_SOURCE}  '
      f'temp_cal MaxLogit mean={logits_temp.max(1).values.numpy().mean():.2f}')

# ── Conformal RAPS on conformal_cal probabilities (after temperature) ──
# RAPS non-conformity score — INLINE copy so the notebook is fully self-contained
# on Colab (only this notebook ships there, not the repo). Keep in lockstep with
# server/conformal.py:raps_score — its signature (probs_sorted_desc, lambda_reg,
# k_reg) matches so the served prediction sets never drift from this calibration.
def _raps_score(p_sorted, lam, k_reg):
    cs = np.cumsum(p_sorted)
    return cs + lam * np.maximum(0.0, np.arange(1, len(p_sorted)+1, dtype=np.float64) - k_reg)

def _eval_raps_set(probs, labels, q_hat, lambda_reg, k_reg):
    n = probs.shape[0]
    if n == 0:
        return float('nan'), float('nan')
    covered, set_sizes = 0, []
    for i in range(n):
        order = np.argsort(-probs[i])
        s = _raps_score(probs[i, order], lambda_reg, k_reg)
        above = np.where(s >= q_hat)[0]
        size = max(int(above[0]) + 1 if above.size else len(s), 1)
        set_sizes.append(size)
        covered += int(labels[i] in order[:size])
    return covered / n, float(np.mean(set_sizes))

def calibrate_q_hat(cal_probs, cal_labels, *, alpha=0.1, lambda_reg=0.01, k_reg=1,
                    audit_probs=None, audit_labels=None):
    n = cal_probs.shape[0]
    if n == 0:
        raise RuntimeError('Empty conformal calibration split.')
    nonconf = np.empty(n)
    for i in range(n):
        order = np.argsort(-cal_probs[i])
        s = _raps_score(cal_probs[i, order], lambda_reg, k_reg)
        nonconf[i] = s[int(np.where(order == cal_labels[i])[0][0])]
    q = float(np.quantile(nonconf, min(1.0, np.ceil((n+1) * (1-alpha)) / n), method='higher'))
    in_cov, in_size = _eval_raps_set(cal_probs, cal_labels, q, lambda_reg, k_reg)
    out_cov = out_size = float('nan'); audit_n = 0
    if audit_probs is not None and audit_probs.shape[0] > 0:
        audit_n = int(audit_probs.shape[0])
        out_cov, out_size = _eval_raps_set(audit_probs, audit_labels, q, lambda_reg, k_reg)
    primary_cov  = out_cov  if audit_n > 0 else in_cov
    primary_size = out_size if audit_n > 0 else in_size
    return {'method': 'raps', 'alpha': alpha, 'q_hat': q,
            'lambda_reg': lambda_reg, 'k_reg': k_reg,
            'calibration_n': int(n),
            'empirical_coverage': primary_cov,
            'empirical_coverage_kind': 'out_of_sample' if audit_n > 0 else 'in_sample',
            'mean_set_size': primary_size,
            'audit_n': audit_n,
            'in_sample_coverage': in_cov,
            'in_sample_mean_set_size': in_size}

logits_cal_np, y_cal_np = _collect_logits(eval_sess, cal_dl)
probs_cal = torch.softmax(torch.from_numpy(logits_cal_np) / T_val, dim=1).numpy()
n_cal = int(y_cal_np.shape[0])
order_cal = np.random.RandomState(SEED + 7).permutation(n_cal)
# ── Conformal hyper-parameter search (report P1: fix over-coverage) ──
# The old call hardcoded alpha=0.10, lambda_reg=0.01, k_reg=1 and shipped
# whatever coverage fell out -- on this very-confident classifier that was
# 99.4% (vs the 90% target) with a 2.56/9 mean set size, i.e. sets too big
# to be the honest "either of these two" hint they exist to be. RAPS only
# *guarantees* coverage >= 1 - alpha; on a sharp model the realised q_hat
# over-shoots, so we grid-search (alpha, lambda_reg, k_reg) and keep the
# config with the SMALLEST out-of-sample (audit) mean set size whose audit
# coverage still clears the product target (CONF_TARGET_COVERAGE) -- exactly
# the smallest set that still covers the truth ~90% of the time. The
# advertised coverage_target = 1 - chosen alpha then honestly reflects what
# was calibrated instead of a nominal 90% the sets silently overshoot.
CONF_TARGET_COVERAGE = 0.90
# Widened THREE times, each time because the search pinned to the boundary.
# 06-29 / 07-01 / 07-03 / 07-20 all chose alpha=0.20, lambda=0.50 (the rail of
# the grid of the day); the 2026-08-07 2.7.0 run then chose **alpha=0.35, the
# last value of the widened grid** -- coverage 0.967 and mean set size 1.62
# against a 0.90 target, i.e. still over-covering, so the optimum is STILL
# outside. lambda converged this time (0.75, not the 1.0 rail), so only alpha
# needed extending. It now runs to 0.60, far enough that a fourth boundary hit
# would mean something is wrong with the scores rather than the grid.
# The coverage >= CONF_TARGET_COVERAGE guard below is what keeps this honest,
# so a wider grid can only shrink sets it is allowed to shrink. If a run again
# picks the final value of either list, widen again -- a boundary selection
# means the search never converged.
#
# 2026-08-08 UPDATE — STOP WIDENING. The widened grid was hit again (alpha=0.60,
# the new rail) and the diagnosis is that alpha is a REDUNDANT PARAMETERISATION,
# not a too-small grid: what is really being searched is q_hat, and for a model
# this confident (q_hat=0.972 even at alpha=0.60) coverage stays at 0.980, so the
# search will always run to whatever rail exists. Reaching 0.90 coverage would
# need alpha ~0.85+, and chasing it is actively wrong -- the guard is measured on
# a 256-row audit split, so selecting AT the target fits its noise. The
# _coverage_floor margin below is the real fix: the rail is now harmless because
# nothing can be selected without clearing coverage with room to spare. Judge
# this search by mean_set_size vs baseline (2.7.0: 1.54 vs 2.87 at 0.98 coverage),
# not by where alpha landed.
# NOTE alpha is a SEARCH KNOB, not a promise. What the server advertises as
# `coverage` (server/conformal.py:coverage_target) is `target_coverage` (0.90)
# whenever the selected config cleared it on the OUT-OF-SAMPLE audit split with
# the 2-SE margin (`coverage_basis` below); a sidecar without that evidence
# falls back to the nominal 1 - alpha. Never read 1 - alpha as the guarantee:
# 2.7.0-2.7.2 selected alpha=0.60 and would have advertised 40 %.
CONF_ALPHA_GRID  = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30, 0.35,
                    0.40, 0.45, 0.50, 0.55, 0.60]
CONF_LAMBDA_GRID = [0.0, 0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.0]
CONF_KREG_GRID   = [1, 2]

def _search_conformal(fit_p, fit_y, audit_p, audit_y):
    trials = []
    for a in CONF_ALPHA_GRID:
        for lam in CONF_LAMBDA_GRID:
            for kr in CONF_KREG_GRID:
                trials.append(calibrate_q_hat(
                    fit_p, fit_y, alpha=a, lambda_reg=lam, k_reg=kr,
                    audit_probs=audit_p, audit_labels=audit_y))
    cov  = lambda c: c['empirical_coverage']
    size = lambda c: c['mean_set_size']
    # Smallest set whose (audit, else in-sample) coverage still clears the
    # product target; if the grid over-shrank everything, keep the most
    # covering trial rather than ship an under-covering set.
    # Selection margin. The guard is measured on a ~250-row audit split, where a
    # coverage estimate carries SE = sqrt(p(1-p)/n) ~= 0.019 at p=0.90. Picking the
    # config that sits EXACTLY on the target therefore ships sets whose TRUE
    # coverage is below target roughly half the time -- selecting on audit noise.
    # Requiring target + 2 SE makes the choice robust to the size of the split it
    # was chosen on, and it is why a boundary alpha is no longer a problem: the
    # search may run to the end of the grid, but it can only get there through
    # configs that clear coverage with room to spare. Inert when the search is far
    # from the boundary (the 2.7.0 run selected at 0.980 against a 0.938 floor).
    def _coverage_floor(c):
        n = int(c.get('audit_n') or c.get('calibration_n') or 0)
        if n <= 0:
            return CONF_TARGET_COVERAGE
        se = (CONF_TARGET_COVERAGE * (1.0 - CONF_TARGET_COVERAGE) / n) ** 0.5
        return CONF_TARGET_COVERAGE + 2.0 * se
    covering = [c for c in trials if cov(c) is not None and cov(c) >= _coverage_floor(c)]
    best = (min(covering, key=lambda c: (size(c), abs(cov(c) - CONF_TARGET_COVERAGE)))
            if covering else max(trials, key=cov))
    best['target_coverage'] = CONF_TARGET_COVERAGE
    best['grid'] = {'alpha': CONF_ALPHA_GRID, 'lambda_reg': CONF_LAMBDA_GRID,
                    'k_reg': CONF_KREG_GRID, 'n_trials': len(trials)}
    # Carry the pre-tune baseline (the old fixed config) so the over-coverage
    # fix is auditable before/after in the model card.
    base = next((c for c in trials if c['alpha'] == 0.10
                 and c['lambda_reg'] == 0.01 and c['k_reg'] == 1), None)
    if base is not None:
        best['baseline_alpha010_lambda001_k1'] = {
            'empirical_coverage': base['empirical_coverage'],
            'mean_set_size': base['mean_set_size']}
    return best

if n_cal >= 200:
    fit_n = int(n_cal * 0.80)
    fit_idx, audit_idx = order_cal[:fit_n], order_cal[fit_n:]
    cfg = _search_conformal(probs_cal[fit_idx], y_cal_np[fit_idx],
                            probs_cal[audit_idx], y_cal_np[audit_idx])
else:
    print(f'conformal: n_cal={n_cal} < 200 — grid-searching on the full split '
          f'with in-sample coverage (audit_n=0; coverage is optimistic)')
    cfg = _search_conformal(probs_cal, y_cal_np, None, None)
print(f"conformal: selected alpha={cfg['alpha']} lambda_reg={cfg['lambda_reg']} "
      f"k_reg={cfg['k_reg']} -> coverage={cfg['empirical_coverage']:.3f} "
      f"set_size={cfg['mean_set_size']:.2f} (kind={cfg['empirical_coverage_kind']})")
cfg['coverage_scope'] = (f'all {len(CLASS_NAMES)} classes (plantdoc, inat, coco, '
                          'inat_animalia, openimages_indoor); '
                          'not_a_plant is additionally guarded by the '
                          'Stage 1 detector + embedding k-NN OOD gate.')
cfg['cal_sources'] = ['plantdoc', 'inat', 'coco', 'inat_animalia',
                       'openimages_indoor', 'coco_hard_partial_plant',
                       'inat_hard_greentexture']
cfg['fit_source'] = QUANT_FIT_SOURCE
cfg['tta'] = TTA_MODE
# The server advertises `target_coverage` only when the audit backs it.
cfg['coverage_basis'] = (
    'audit_target' if (cfg.get('empirical_coverage_kind') == 'out_of_sample'
                       and int(cfg.get('audit_n') or 0) >= 50
                       and float(cfg['empirical_coverage']) >= CONF_TARGET_COVERAGE)
    else 'nominal_one_minus_alpha')
print('conformal:', cfg)

# ── MaxLogit OOD floor on the shipped session ───────────────────
TARGET_PLANT_FPR = 0.05
NAP_CLASS_IDX = (CLASS_NAMES.index('not_a_plant')
                 if 'not_a_plant' in CLASS_NAMES else None)

# v2.6.13: score the floor over the PLANT logits only (excluding not_a_plant).
#
# The all-class max was a DEAD signal on this head, and structurally so:
# `not_a_plant` is a TRAINED class, so a NAP photo produces a confident NAP
# logit and its all-class max lands ABOVE the plant median. Every run since
# v2.3 therefore took the `separation < 0` branch below and shipped
# `max_logit_floor: null` (measured separation_p50 -0.939 / -0.403 / -0.199 /
# -0.104 across the 06-29 → 07-20 runs), leaving the AND-gate on the embedding
# bank alone. The §2 hard-negative fetchers were added to fix this and could
# not: the ordering is an artefact of the taxonomy, not of the data.
#
# Excluding the NAP column makes the score mean "does this look like any PLANT
# condition we know?" — which is the question the gate is actually asking, and
# which separates by construction. The scope is persisted to
# `classes.json -> ood.max_logit_scope`; the server reads the matching per-crop
# key and defaults to 'all_classes' for bundles predating the key, so old
# bundles keep their exact historical numerics.
MAXLOGIT_SCOPE = 'plant_only' if NAP_CLASS_IDX is not None else 'all_classes'
_PLANT_LOGIT_COLS = [i for i in range(len(CLASS_NAMES))
                     if not (MAXLOGIT_SCOPE == 'plant_only' and i == NAP_CLASS_IDX)]

def _maxlogit_distribution(sess, dl, *, only_class=None, exclude_class=None):
    """MaxLogit per sample over the SCOPED logit columns (``MAXLOGIT_SCOPE``).

    ``only_class`` / ``exclude_class`` filter which SAMPLES are scored (by
    label); ``_PLANT_LOGIT_COLS`` controls which LOGIT COLUMNS the max runs
    over. The two are independent — don't conflate them."""
    out = []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        for i in range(logits.shape[0]):
            yi = int(y[i])
            if only_class is not None and yi != only_class:
                continue
            if exclude_class is not None and yi == exclude_class:
                continue
            out.append(float(logits[i][_PLANT_LOGIT_COLS].max()))
    return np.array(out, dtype=np.float64)

plant_ml = _maxlogit_distribution(eval_sess, val_realworld_dl, exclude_class=NAP_CLASS_IDX)
nap_ml = (_maxlogit_distribution(eval_sess, val_realworld_dl, only_class=NAP_CLASS_IDX)
          if NAP_CLASS_IDX is not None else np.array([]))

if len(plant_ml) >= 30:
    ML_FLOOR = float(np.percentile(plant_ml, TARGET_PLANT_FPR * 100))
else:
    print(f'WARN: only {len(plant_ml)} plant MaxLogit samples — falling back to 4.0 default')
    ML_FLOOR = 4.0

ML_SEPARATION = (float(np.percentile(plant_ml, 50) - np.percentile(nap_ml, 50))
                  if len(plant_ml) and len(nap_ml) else None)
if ML_SEPARATION is not None and ML_SEPARATION < 0:
    print(f'WARN: NAP MaxLogit median above plant median (sep={ML_SEPARATION:+.3f}) '
          f'— disabling the MaxLogit gate (max_logit_floor=null).')
    ML_FLOOR_PERSISTED = None
else:
    ML_FLOOR_PERSISTED = ML_FLOOR

ML_CAL_META = {
    'target_plant_fpr': TARGET_PLANT_FPR,
    'plant_n': int(len(plant_ml)),
    'plant_p05': float(np.percentile(plant_ml, 5)) if len(plant_ml) else None,
    'plant_p50': float(np.percentile(plant_ml, 50)) if len(plant_ml) else None,
    'plant_p95': float(np.percentile(plant_ml, 95)) if len(plant_ml) else None,
    'nap_n': int(len(nap_ml)),
    'nap_p05': float(np.percentile(nap_ml, 5)) if len(nap_ml) else None,
    'nap_p50': float(np.percentile(nap_ml, 50)) if len(nap_ml) else None,
    'nap_p95': float(np.percentile(nap_ml, 95)) if len(nap_ml) else None,
    'separation_p50': ML_SEPARATION,
    'scope': MAXLOGIT_SCOPE,
    'fit_source': QUANT_FIT_SOURCE,
}
print(f'MaxLogit OOD floor calibrated [{MAXLOGIT_SCOPE}]: '
      f'{"off" if ML_FLOOR_PERSISTED is None else f"{ML_FLOOR_PERSISTED:.3f}"}  '
      f'(plant n={len(plant_ml)} p50={ML_CAL_META["plant_p50"]}, '
      f'nap n={len(nap_ml)} p50={ML_CAL_META["nap_p50"]}, '
      f'sep={ML_SEPARATION if ML_SEPARATION is None else round(ML_SEPARATION, 3)})')

# ── Embedding-space OOD (k-NN) calibration (report P1) ──────────────
# MaxLogit went dead once the NAP head grew over-confident. Calibrate a
# live, logit-independent OOD signal: a deep k-NN cosine distance in the
# DINOv2 CLS embedding space (Sun et al. 2022). Build an in-distribution
# memory bank of PLANT embeddings from conformal_cal, then set the OOD
# threshold on a DISJOINT split (temperature_cal) at TARGET_PLANT_FPR, so
# the bank and the threshold are never fit on the same samples. Both use the
# SHIPPED (possibly INT8) session so the bank/threshold match what serves.
OOD_KNN_K = 5
OOD_BANK_CAP = 4000

def _collect_embeddings(sess, dl):
    embs, ys = [], []
    for x, y in dl:
        _lg, e = _ort_forward(sess, x.numpy().astype(np.float32))
        if e is None:
            return None, None
        embs.append(e); ys.append(y.numpy())
    return np.concatenate(embs), np.concatenate(ys)

def _l2n(a):
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)

def _knn_scores(bank_norm, q, k):
    k = max(1, min(k, bank_norm.shape[0]))
    sims = _l2n(q) @ bank_norm.T
    topk = np.partition(sims, -k, axis=1)[:, -k:] if k < sims.shape[1] else sims
    return 1.0 - topk.mean(axis=1)

EMBEDDING_OOD, ood_bank_ship = None, None
_emb_bank, _y_bank = _collect_embeddings(eval_sess, cal_dl)        # conformal_cal
_emb_thr,  _y_thr  = _collect_embeddings(eval_sess, temp_cal_dl)   # temperature_cal
if (_emb_bank is not None and _emb_thr is not None
        and NAP_CLASS_IDX is not None):
    _bank_plant = _emb_bank[_y_bank != NAP_CLASS_IDX]
    _thr_plant  = _emb_thr[_y_thr != NAP_CLASS_IDX]
    _thr_nap    = _emb_thr[_y_thr == NAP_CLASS_IDX]
    if _bank_plant.shape[0] >= 10 and _thr_plant.shape[0] >= 10:
        if _bank_plant.shape[0] > OOD_BANK_CAP:
            _sel = np.random.RandomState(SEED + 11).choice(
                _bank_plant.shape[0], OOD_BANK_CAP, replace=False)
            _bank_plant = _bank_plant[_sel]
        _bn = _l2n(_bank_plant)
        _ps = _knn_scores(_bn, _thr_plant, OOD_KNN_K)
        # Threshold at the (1 - FPR) quantile of in-dist plant scores: only
        # TARGET_PLANT_FPR of real plants exceed it (false-OOD rate).
        _thr = float(np.quantile(_ps, 1.0 - TARGET_PLANT_FPR))
        _ns = _knn_scores(_bn, _thr_nap, OOD_KNN_K) if len(_thr_nap) else np.array([])
        _nap_recall = float(np.mean(_ns > _thr)) if len(_ns) else None
        ood_bank_ship = _bank_plant
        EMBEDDING_OOD = {
            'method': 'knn', 'metric': 'cosine', 'k': OOD_KNN_K,
            'threshold': _thr, 'embed_dim': int(_bank_plant.shape[1]),
            'bank_file': 'ood_embedding.npz',
            '_calibration': {
                'target_plant_fpr': TARGET_PLANT_FPR,
                'bank_n': int(_bank_plant.shape[0]),
                'thr_plant_n': int(len(_ps)), 'thr_nap_n': int(len(_ns)),
                'nap_recall_at_threshold': _nap_recall,
                'plant_score_p50': float(np.percentile(_ps, 50)),
                'plant_score_p95': float(np.percentile(_ps, 95)),
                'nap_score_p50': float(np.percentile(_ns, 50)) if len(_ns) else None,
                'fit_source': QUANT_FIT_SOURCE,
            },
        }
        print(f'Embedding OOD calibrated: threshold={_thr:.4f} '
              f'bank_n={_bank_plant.shape[0]} '
              f'nap_recall@thr={_nap_recall} (plant FPR={TARGET_PLANT_FPR})')
if EMBEDDING_OOD is None:
    print('Embedding OOD: graph has no embedding output or too few '
          'plant/NAP samples — not shipping a bank (OOD falls back to MaxLogit).')

# ── Final metrics across all val slices via the shipped ORT session ──
class _OnnxModelAdapter:
    """Thin wrapper so evaluate() can call model(x) on an ORT session."""
    def __init__(self, sess): self.sess = sess
    def eval(self): return self
    def __call__(self, x):
        out = _ort_logits(self.sess, x.detach().cpu().numpy().astype(np.float32))
        return torch.from_numpy(out).to(x.device)

eval_adapter = _OnnxModelAdapter(eval_sess)
ONNX_EVAL_DEVICE = 'cpu'
final_metrics = {
    'indomain':  evaluate(eval_adapter, val_indomain_dl,  CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_indomain'),
    # `_rw_logits` is filled here and reused by the Z1 sweep below instead of
    # re-running the same ~2k-image forward pass.
    'realworld': evaluate(eval_adapter, val_realworld_dl, CLASS_NAMES, ONNX_EVAL_DEVICE,
                          'final val_realworld', logit_sink=(_rw_logits := [])),
}
if val_studio_dl is not None:
    final_metrics['studio'] = evaluate(eval_adapter, val_studio_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_studio')
if holdout_dl is not None:
    # THE 2.7.2 IMPROVEMENT METRIC. Everything else here is a regression metric:
    # val_realworld and val_indomain contain rows this model trained near, and
    # the supplement as a whole is a training-set-membership detector (2.6.8:
    # 0.964 on rows it trained on vs 0.807 on rows it did not). This split is
    # the only one that is clean by construction. It does NOT gate yet — there
    # is no baseline for it until the first 2.7.2 run produces one.
    final_metrics['holdout'] = evaluate(
        eval_adapter, holdout_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final holdout')
# These blocks are CLASSIFIER-ONLY (Stage 2 on pre-cropped val images, no
# detector). The end-to-end detector->crop->classify numbers are appended
# below as final_metrics['end_to_end'] by the detector-crop eval, so the two
# eval modes are never conflated in training_metrics.json / model_card.json.
for _m in final_metrics.values():
    _m.setdefault('eval_mode', 'classifier_only')
# val_studio's plain disease_macro_f1 averages over all disease classes, but
# PlantVillage supplies only a few -> the supported number is the honest head.
if 'studio' in final_metrics:
    _st = final_metrics['studio']
    print(f"val_studio: supported_disease_macro_f1="
          f"{_st['supported_disease_macro_f1']:.3f} over "
          f"{_st['n_supported_disease_classes']} supported disease classes "
          f"(all-class={_st['disease_macro_f1']:.3f} drags in absent-class zeros)")

# ── Aqua (on-device) EXACT metrics ─ eval the SHIPPED int8 TFLite on the same
# val slices as gamma so the on-device numbers are MEASURED (not guessed) and
# ship in model_card.json. Fail-soft: a TFLite hiccup never blocks the gamma
# bundle (already gated above). Reuses evaluate(), so per-class recall/precision
# + disease/supported macro-F1 + the split/eval_mode tags match gamma exactly.
aqua_metrics = None
_aqua_tflite = Path('models/aqua/aqua_classifier.tflite')
if _aqua_tflite.exists():
    try:
        try:
            from ai_edge_litert.interpreter import Interpreter as _AQInterp
        except Exception:
            from tensorflow.lite import Interpreter as _AQInterp  # type: ignore
        _aqi = _AQInterp(model_path=str(_aqua_tflite)); _aqi.allocate_tensors()
        _ain = _aqi.get_input_details()[0]; _aout = _aqi.get_output_details()[0]

        class _TFLiteEvalAdapter:
            """Let evaluate() drive the int8 TFLite: NCHW float torch batch ->
            NHWC per-sample invoke -> logits torch tensor (the graph has float I/O)."""
            def eval(self): return self
            def __call__(self, x):
                xn = x.detach().cpu().numpy().transpose(0, 2, 3, 1).astype(_ain['dtype'])
                out = np.empty((xn.shape[0], len(CLASS_NAMES)), dtype=np.float32)
                for _i in range(xn.shape[0]):
                    _aqi.set_tensor(_ain['index'], xn[_i:_i+1]); _aqi.invoke()
                    out[_i] = _aqi.get_tensor(_aout['index'])[0]
                return torch.from_numpy(out)

        _aq = _TFLiteEvalAdapter()
        aqua_metrics = {
            'indomain':  evaluate(_aq, val_indomain_dl,  CLASS_NAMES, 'cpu', 'aqua val_indomain'),
            'realworld': evaluate(_aq, val_realworld_dl, CLASS_NAMES, 'cpu', 'aqua val_realworld'),
        }
        if val_studio_dl is not None:
            aqua_metrics['studio'] = evaluate(_aq, val_studio_dl, CLASS_NAMES, 'cpu', 'aqua val_studio')
        for _m in aqua_metrics.values():
            _m['eval_mode'] = 'classifier_only'; _m['tier'] = 'aqua_tflite'
        # Teacher->student drift on the SAME slices (gamma - aqua) so the KD gap is
        # recorded, not inferred from the KD-val number alone.
        aqua_metrics['drift_vs_gamma_disease_macro_f1'] = {
            k: round(final_metrics[k]['disease_macro_f1'] - aqua_metrics[k]['disease_macro_f1'], 4)
            for k in ('indomain', 'realworld', 'studio')
            if k in final_metrics and k in aqua_metrics}
        print('Aqua EXACT metrics (shipped TFLite):', {k: round(v['disease_macro_f1'], 4)
              for k, v in aqua_metrics.items() if isinstance(v, dict) and 'disease_macro_f1' in v})
        print('Aqua teacher->student drift:', aqua_metrics['drift_vs_gamma_disease_macro_f1'])
        _aq_rw = (aqua_metrics.get('realworld') or {}).get('disease_macro_f1', 1.0)
        if _aq_rw < 0.30:
            raise RuntimeError(
                f'Aqua FLOAT32 TFLite is degenerate on realworld '
                f'(disease_macro_f1={_aq_rw:.3f}); artifact substitution is forbidden')
    except Exception as _ae:  # noqa: BLE001 — combined contract is fail-closed
        print(f'Aqua exact-metrics validation failed: {type(_ae).__name__}: {_ae}')
        if FAMILY_CONFIG.get('distill'):
            raise RuntimeError(
                'combined run aborted: Aqua exact served-artifact validation failed') from _ae

# NAP head veto (2.7.2). pipeline.py refuses to run the OOD gate when the
# classifier's own not_a_plant head is confident the frame IS a plant. Measured
# on the deployed 2.6.8 bundle: all 13 never-trained ornamental leaf_scorch
# clusters the gate wrongly rejected had P(nap) <= 0.03 (median 0.0018), while
# 372 of 373 real not_a_plant images sit above 0.05.
#
# ⚠ INHERITED, NOT FITTED. Unlike the MaxLogit floor beside it, nothing in §6
# measures P(not_a_plant) separation for THIS bundle — 0.05 was measured on
# 2.6.8 and every later bundle inherits it. pipeline.py evaluates it against
# POST-temperature, mean-voted probabilities, so a bundle whose fitted
# temperature moves far from 2.6.8's 0.849 could sit at different percentiles.
# The failure direction is safe — a flatter distribution lifts P(nap) above the
# floor, so the veto fires LESS and the gate reverts toward its historical
# two-of-two behaviour — but re-measure before leaning on it. Set it to None to
# restore that historical gate explicitly.
NAP_HEAD_MASS_FLOOR = 0.05


# ── SERVED-PATH OOD calibration (2.7.5): the overlay that actually runs ──
# pipeline.py does not score one image at a time. It runs Stage 1, crops up to
# `max_crops` leaf boxes (padding_ratio / min_size_px / min_area_fraction),
# folds the whole frame in as one more member, and takes the MIN k-NN distance
# and the MAX plant-scope logit ACROSS THOSE MEMBERS. The single-image
# thresholds fitted above are a different statistic (min-over-members <= the
# whole-frame distance), so pipeline.py refuses to serve them and the overlay
# stayed OFF on every 2.8.0-draft bundle. This block fits the served statistic,
# here, on the SHIPPED graphs: Stage 1 ONNX -> the crop rules below (copied
# verbatim from serving_contract.crop / .detector; server/tests/
# test_data_invariants.py pins them to pipeline.py) -> whole-frame fold ->
# per-image min distance / max logit / mean-vote P(not_a_plant) / best det
# conf. Thresholds sit at TARGET_PLANT_FPR on temperature_cal plants (the same
# disjoint split the single-image threshold uses), and the resulting
# three-of-three gate is REPORTED on temperature_cal NAP rows next to the
# head-only rejection rate, so a reviewer can see what the overlay buys.
SERVED_GATE_CROP = {'max_crops': 4, 'min_size_px': 32, 'min_area_fraction': 0.005,
                    'padding_ratio': 0.10, 'fold_whole_frame_vote': True}
SERVED_GATE_DETECTOR = {'img_size': 640, 'min_confidence': 0.15, 'nms_iou': 0.45,
                        'max_detections': 10}
SERVED_GATE_DET_CONF_FLOOR = 0.40      # pipeline.NAP_DETECTOR_CONF_FLOOR
SERVED_GATE_PLANT_CAP = 800            # temperature_cal plants scored (CPU budget)
SERVED_GATE = None

def _served_letterbox(pil, size):
    """Mirror of server/detector.py:letterbox."""
    w, h = pil.size
    s = size / max(w, h)
    nw, nh = int(round(w * s)), int(round(h * s))
    canvas = PILImage.new('RGB', (size, size), (114, 114, 114))
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas.paste(pil.resize((nw, nh), PILImage.BILINEAR), (px, py))
    arr = np.asarray(canvas, dtype=np.float32) / 255.0
    return np.ascontiguousarray(arr.transpose(2, 0, 1)[None]), s, (px, py)

def _served_nms(xyxy, conf, iou_thr):
    """Mirror of server/detector.py:nms (greedy, IoU <= thr survives)."""
    order = conf.argsort()[::-1]; keep = []
    x1, y1, x2, y2 = xyxy.T; areas = (x2 - x1) * (y2 - y1)
    while order.size:
        i = order[0]; keep.append(int(i))
        xx1 = np.maximum(x1[i], x1[order[1:]]); yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]]); yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0.0, xx2 - xx1) * np.maximum(0.0, yy2 - yy1)
        union = areas[i] + areas[order[1:]] - inter
        ovr = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        order = order[np.where(ovr <= iou_thr)[0] + 1]
    return keep

def _served_detect(det_sess, pil):
    """Mirror of server/detector.py:LeafDetector.detect -> (detections, best_conf)."""
    cfg = SERVED_GATE_DETECTOR
    x, s, (px, py) = _served_letterbox(pil, cfg['img_size'])
    out = np.asarray(det_sess.run(None, {det_sess.get_inputs()[0].name: x})[0], dtype=np.float32)
    out = out[0] if out.ndim == 3 else out
    out = out if out.shape[0] == 5 else out.T
    conf = out[4]; boxes = out[:4].T
    ok = (np.isfinite(conf) & (conf >= cfg['min_confidence']) & (conf <= 1.0)
          & np.isfinite(boxes).all(1) & (boxes[:, 2] > 0) & (boxes[:, 3] > 0))
    idx = np.where(ok)[0]
    if not len(idx):
        return [], 0.0
    cx, cy, bw, bh = boxes[idx].T; conf = conf[idx]
    xyxy = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1)
    keep = _served_nms(xyxy, conf, cfg['nms_iou'])
    W, H = pil.size; dets = []
    for i in keep:
        bx1 = min(max((xyxy[i, 0] - px) / s, 0.0), float(W)); by1 = min(max((xyxy[i, 1] - py) / s, 0.0), float(H))
        bx2 = min(max((xyxy[i, 2] - px) / s, 0.0), float(W)); by2 = min(max((xyxy[i, 3] - py) / s, 0.0), float(H))
        w_, h_ = bx2 - bx1, by2 - by1
        if w_ >= 1 and h_ >= 1:
            dets.append({'x': int(round(bx1)), 'y': int(round(by1)),
                         'width': int(round(w_)), 'height': int(round(h_)),
                         'confidence': float(conf[i])})
    dets.sort(key=lambda d: -d['confidence'])
    dets = dets[:cfg['max_detections']]
    return dets, (max(d['confidence'] for d in dets) if dets else 0.0)

def _served_members(pil, dets):
    """Mirror of pipeline._size_filter / _extract_crop / _pick_primary members."""
    c = SERVED_GATE_CROP
    if not dets:
        return [pil]                      # _analyze_no_detection: whole frame only
    W, H = pil.size; area = max(W * H, 1)
    sized = [d for d in dets if d['width'] >= c['min_size_px'] and d['height'] >= c['min_size_px']
             and d['width'] * d['height'] / area >= c['min_area_fraction']]
    if not sized:
        sized = [max(dets, key=lambda d: d['width'] * d['height'])]
    ranked = sorted(sized, key=lambda d: -d['confidence'])[:c['max_crops']]
    members = []
    for d in ranked:
        if d['width'] < c['min_size_px'] or d['height'] < c['min_size_px']:
            members.append(pil); continue
        padx, pady = int(d['width'] * c['padding_ratio']), int(d['height'] * c['padding_ratio'])
        x1, y1 = max(0, d['x'] - padx), max(0, d['y'] - pady)
        x2, y2 = min(W, d['x'] + d['width'] + padx), min(H, d['y'] + d['height'] + pady)
        members.append(pil.crop((x1, y1, x2, y2)) if (x2 > x1 and y2 > y1) else pil)
    if c['fold_whole_frame_vote']:
        members.append(pil)
    return members

# ── Served-path streaming (2.7.5) — shared by the calibration below and the
# Stage-1 audit / end-to-end eval at the end of §6. One photo at a time, at the
# size the app uploads, through the shipped graphs, nothing retained.
SERVED_UPLOAD_MAX_SIDE = 1024   # android util/ImageUtils.prepareImageForUpload(maxSize = 1024)
SERVED_PASS_PREFETCH = 8        # photos decoded ahead of the detector (~2.4 MB each at 1024 px)

def _served_load(path, max_side=SERVED_UPLOAD_MAX_SIDE):
    """Decode a photo at the size it reaches the server.

    The app sends every upload at a ``max_side`` long edge: libjpeg power-of-two
    subsampling at decode, then an exact bilinear scale, floor-rounded
    (ImageUtils.prepareImageForUpload). JPEG ``draft`` IS that libjpeg
    subsampling, so a 12 MP phone photo costs a ~3 MP decode instead of 36 MB of
    RGB, and every crop is cut from the frame the server actually sees."""
    im = PILImage.open(path)
    w, h = im.size
    if not max_side or max(w, h) <= max_side:
        return im.convert('RGB')
    r = max_side / max(w, h)
    tw, th = max(1, int(w * r)), max(1, int(h * r))
    if im.format == 'JPEG':
        im.draft('RGB', (tw, th))
    im = im.convert('RGB')
    return im if im.size == (tw, th) else im.resize((tw, th), PILImage.BILINEAR)

def _served_classify(sess, members, tx, tta=None):
    """Mirror of classifier.py _preprocess_batch + _onnx_forward: all members of
    ONE photo in one batch, their hflip mirrors concatenated into the SAME run.
    The INT8 graph quantises activations per tensor, so what shares a batch is
    part of the served statistic — batching photos together would not be.
    -> (logits [n, C], CLS embeddings [n, D] or None), TTA-averaged."""
    tta = TTA_MODE if tta is None else tta
    x = np.stack([tx(m).numpy() for m in members]).astype(np.float32)
    in_name = sess.get_inputs()[0].name
    has_emb = len(sess.get_outputs()) == 2    # classifier.py: _has_embedding
    n = x.shape[0]
    if tta == 'hflip':
        outs = sess.run(None, {in_name: np.concatenate(
            [x, np.ascontiguousarray(x[..., ::-1])], axis=0)})
        logits = (outs[0][:n] + outs[0][n:]) * 0.5
        emb = ((outs[1][:n] + outs[1][n:]) * 0.5) if has_emb else None
    else:
        outs = sess.run(None, {in_name: x})
        logits, emb = outs[0], (outs[1] if has_emb else None)
    return (np.asarray(logits, dtype=np.float32),
            None if emb is None else np.asarray(emb, dtype=np.float32))

def _served_pass(samples, det_sess, cls_sess, tx, temperature, *, prefetch=SERVED_PASS_PREFETCH):
    """Stream ``(path, label)`` samples through the served path, in input order.

    Decoding runs ``prefetch`` photos ahead on two threads (PIL releases the
    GIL), so the detector and classifier never wait on JPEG work. Yields one
    small dict per photo — label, best detector confidence, whether Stage 1 found
    a box, member count, member logits / embeddings, temperature-scaled mean
    vote — and keeps nothing once it is yielded: peak host RAM is ``prefetch``
    decoded photos plus one photo's members."""
    from collections import deque
    from concurrent.futures import ThreadPoolExecutor
    src = iter(samples)
    window = deque()
    with ThreadPoolExecutor(max_workers=2) as pool:
        def _feed():
            nxt = next(src, None)
            if nxt is not None:
                window.append((nxt[1], pool.submit(_served_load, nxt[0])))
        for _ in range(max(1, prefetch)):
            _feed()
        while window:
            y, fut = window.popleft()
            _feed()
            pil = fut.result()
            dets, det_conf = _served_detect(det_sess, pil)
            logits, emb = _served_classify(cls_sess, _served_members(pil, dets), tx)
            del pil
            z = logits / temperature
            p = np.exp(z - z.max(axis=1, keepdims=True))
            p /= p.sum(axis=1, keepdims=True)
            yield {'y': y, 'det_conf': float(det_conf), 'has_det': bool(dets),
                   'n': int(logits.shape[0]), 'logits': logits, 'emb': emb,
                   'vote': p.mean(axis=0)}

def _served_gate_fires(det_conf, d_min, ml_max, p_nap, *, emb_thr, ml_floor,
                       head_mass_floor, det_floor):
    """Vectorised mirror of pipeline._decide_nap_route's three-of-three overlay,
    WITHOUT its natural-not_a_plant short-circuit (OR the head's own argmax in
    separately). Per photo the classifier signal is the k-NN distance when a
    threshold and a finite distance exist, else the MaxLogit floor when it is
    positive, else nothing — and with no signal the overlay never fires."""
    det_conf = np.asarray(det_conf, dtype=float)
    sig = np.zeros(det_conf.shape, dtype=bool)
    if ml_floor is not None and ml_floor > 0:
        sig = np.asarray(ml_max, dtype=float) < ml_floor
    if emb_thr is not None and d_min is not None:
        d = np.asarray(d_min, dtype=float)
        sig = np.where(np.isfinite(d), d > emb_thr, sig)
    if head_mass_floor and head_mass_floor > 0:
        veto = np.asarray(p_nap, dtype=float) < head_mass_floor
    else:
        veto = np.zeros(det_conf.shape, dtype=bool)
    return (det_conf < det_floor) & sig & ~veto

if EMBEDDING_OOD is not None or ML_FLOOR_PERSISTED is not None:
    try:
        _det_onnx = Path('models/detector.onnx')
        if not _det_onnx.is_file():
            raise FileNotFoundError('models/detector.onnx missing — §3 did not export the detector')
        _det_sess = _ort_session(_det_onnx)
        _srv_tx = T.Compose([T.Resize(int(IMG * EVAL_RESIZE_RATIO)), T.CenterCrop(IMG),
                             T.ToTensor(), NORM])
        _srv_ds = temp_cal_dl.dataset
        _srv_tt = getattr(_srv_ds, 'target_transform', None)
        _srv_samples = [(p, int(_srv_tt(l) if _srv_tt is not None else l)) for p, l in _srv_ds.samples]
        _plant_s = [s for s in _srv_samples if s[1] != NAP_CLASS_IDX]
        _nap_s = [s for s in _srv_samples if s[1] == NAP_CLASS_IDX]
        if len(_plant_s) > SERVED_GATE_PLANT_CAP:
            _sel = np.random.RandomState(SEED + 13).choice(len(_plant_s), SERVED_GATE_PLANT_CAP, replace=False)
            _plant_s = [_plant_s[i] for i in sorted(_sel)]
        _bn_srv = _l2n(ood_bank_ship) if ood_bank_ship is not None else None

        def _served_stats(samples):
            rows = []
            for r in _served_pass(samples, _det_sess, eval_sess, _srv_tx, T_val):
                d_min = (float(_knn_scores(_bn_srv, r['emb'], OOD_KNN_K).min())
                         if (_bn_srv is not None and r['emb'] is not None) else None)
                rows.append((r['det_conf'], d_min, float(r['logits'][:, _PLANT_LOGIT_COLS].max()),
                             float(r['vote'][NAP_CLASS_IDX]), int(r['vote'].argmax()), r['n']))
            return rows

        _sp = _served_stats(_plant_s); _sn = _served_stats(_nap_s)
        if len(_sp) < 30 or len(_sn) < 10:
            raise RuntimeError(f'too few served samples: plants={len(_sp)} nap={len(_sn)}')
        _det_p = np.array([r[0] for r in _sp]); _det_n = np.array([r[0] for r in _sn])
        _has_emb = _bn_srv is not None and all(r[1] is not None for r in _sp + _sn)
        _emb_p = np.array([r[1] for r in _sp], dtype=float) if _has_emb else None
        _emb_n = np.array([r[1] for r in _sn], dtype=float) if _has_emb else None
        _ml_p = np.array([r[2] for r in _sp]); _ml_n = np.array([r[2] for r in _sn])
        _hn_p = np.array([r[3] for r in _sp]); _hn_n = np.array([r[3] for r in _sn])
        _am_p = np.array([r[4] for r in _sp]); _am_n = np.array([r[4] for r in _sn])
        # Threshold at the (1 - FPR) quantile of served plant distances: only
        # TARGET_PLANT_FPR of real plants exceed it on the statistic that serves.
        _srv_emb_thr = float(np.quantile(_emb_p, 1.0 - TARGET_PLANT_FPR)) if _has_emb else None
        _srv_ml_floor = float(np.percentile(_ml_p, TARGET_PLANT_FPR * 100))
        _srv_ml_sep = float(np.percentile(_ml_p, 50) - np.percentile(_ml_n, 50))
        if _srv_ml_sep < 0:
            _srv_ml_floor = None      # same auto-disable rule as the single-image floor
        if _srv_emb_thr is None and _srv_ml_floor is None:
            raise RuntimeError('neither served signal separates plants from not_a_plant '
                               f'(max-logit separation {_srv_ml_sep:+.3f}, no embedding bank)')

        _gate_kw = dict(emb_thr=_srv_emb_thr, ml_floor=_srv_ml_floor,
                        head_mass_floor=NAP_HEAD_MASS_FLOOR or 0.0,
                        det_floor=SERVED_GATE_DET_CONF_FLOOR)
        _g_p = _served_gate_fires(_det_p, _emb_p, _ml_p, _hn_p, **_gate_kw)
        _g_n = _served_gate_fires(_det_n, _emb_n, _ml_n, _hn_n, **_gate_kw)
        _nat_p = _am_p == NAP_CLASS_IDX
        _nat_n = _am_n == NAP_CLASS_IDX
        SERVED_GATE = {
            'schema_version': 1,
            'calibration_mode': 'full_served_pipeline',
            'embedding_aggregation': 'min_member_distance',
            'max_logit_aggregation': 'max_member_logit',
            'detector_confidence_floor': SERVED_GATE_DET_CONF_FLOOR,
            'fold_whole_frame_vote': SERVED_GATE_CROP['fold_whole_frame_vote'],
            'embedding_threshold': _srv_emb_thr,
            'max_logit_floor': _srv_ml_floor,
            'max_logit_scope': MAXLOGIT_SCOPE,
            'nap_head_mass_floor': NAP_HEAD_MASS_FLOOR,
            'target_plant_fpr': TARGET_PLANT_FPR,
            'fit_source': QUANT_FIT_SOURCE,
            'fit_split': 'temperature_cal',
            'crop': dict(SERVED_GATE_CROP),
            'detector': dict(SERVED_GATE_DETECTOR),
            'calibration': {
                'plant_n': int(len(_sp)), 'nap_n': int(len(_sn)),
                'members_mean': float(np.mean([r[5] for r in _sp + _sn])),
                'det_conf_below_floor_plant': float((_det_p < SERVED_GATE_DET_CONF_FLOOR).mean()),
                'det_conf_below_floor_nap': float((_det_n < SERVED_GATE_DET_CONF_FLOOR).mean()),
                'served_min_distance_plant_p50': float(np.percentile(_emb_p, 50)) if _has_emb else None,
                'served_min_distance_plant_p95': float(np.percentile(_emb_p, 95)) if _has_emb else None,
                'served_min_distance_nap_p50': float(np.percentile(_emb_n, 50)) if _has_emb else None,
                'served_max_logit_plant_p05': float(np.percentile(_ml_p, 5)),
                'served_max_logit_nap_p50': float(np.percentile(_ml_n, 50)),
                'served_max_logit_separation_p50': _srv_ml_sep,
                'gate_nap_recall': float(_g_n.mean()),
                'gate_plant_false_reject': float(_g_p.mean()),
                'head_nap_recall': float(_nat_n.mean()),
                'head_plant_false_nap': float(_nat_p.mean()),
                'combined_nap_recall': float((_g_n | _nat_n).mean()),
                'combined_plant_false_reject': float((_g_p | _nat_p).mean()),
            },
        }
        _c = SERVED_GATE['calibration']
        print(f"Served OOD gate calibrated on temperature_cal (plants={_c['plant_n']} nap={_c['nap_n']}): "
              f"emb_thr={_srv_emb_thr} ml_floor={_srv_ml_floor}")
        print(f"  gate alone     NAP recall={_c['gate_nap_recall']:.3f}  plant false-reject={_c['gate_plant_false_reject']:.3f}")
        print(f"  head alone     NAP recall={_c['head_nap_recall']:.3f}  plant false-NAP={_c['head_plant_false_nap']:.3f}")
        print(f"  head OR gate   NAP recall={_c['combined_nap_recall']:.3f}  plant false-reject={_c['combined_plant_false_reject']:.3f}")
        log_run({'phase': 'served_gate',
                 **{k: v for k, v in SERVED_GATE.items() if k not in ('crop', 'detector')}})
        del _det_sess
    except Exception as _e:  # noqa: BLE001 — the overlay stays OFF, the bundle still ships
        SERVED_GATE = None
        print(f'WARN: served OOD gate NOT calibrated ({type(_e).__name__}: {_e}); '
              'classes.json -> ood.served_gate = null and pipeline.py keeps the overlay OFF')
else:
    print('Served OOD gate: no classifier-side signal (no embedding bank, MaxLogit disabled) — overlay OFF')

# ── Compose payloads (NOT yet written — the ship gate runs first) ──
classes_payload = {
    'model_version': MODEL_VERSION,
    'run_id': RUN_ID,
    'data_manifest_sha256': DATA_MANIFEST['classification_sha256'],
    'model_family': _EXPORT_FAMILY,
    'architecture': ARCHITECTURE,
    'class_names': CLASS_NAMES,
    'detector_classes': ['plant_leaf'],
    'training': {
        'img_size': IMG,
        'architecture': ARCHITECTURE,
        # Eval/calibration transform: Resize(int(img_size*eval_resize_ratio))
        # -> CenterCrop(img_size). Serving reads this so train/serve crop
        # parity holds (conformal q_hat + temperature were fit on it).
        'eval_resize_ratio': EVAL_RESIZE_RATIO,
        'batch_size': BATCH,
        'loss': 'logit_adjustment_ce',
        'logit_adjustment_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'use_source_weighted_sampler': bool(USE_SOURCE_WEIGHTED_SAMPLER),
        'source_weights': dict(SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'prior_source_weights': dict(PRIOR_SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'lr_schedule': 'cosine + warmup',
        'precision': 'bf16-mixed' if device == 'cuda' else 'fp32',
        'primary_metric': 'val_realworld_disease_macro_f1',
        'primary_val_set': 'data_v2/classification/val/realworld',
        'isolated_val_slice': 'data_v2/classification/val/plantvillage_studio',
    },
    'backbone': {
        'name': _HUB_ID,
        'source': 'facebookresearch/dinov2 (torch.hub)',
        'frozen': True,
        'embed_dim': EMBED_DIM,
        'head': 'LayerNorm -> Linear -> GELU -> Dropout -> Linear (on CLS embedding)',
        'head_hidden': HEAD_HIDDEN,
        'head_dropout': HEAD_DROPOUT,
        '_note': 'Frozen DINOv2 ViT-S/14 feature extractor + trained MLP head. '
                 'No knowledge distillation (see §5).',
    },
    'inference': {
        'layout': 'NCHW',
        'normalize_mean': [0.485, 0.456, 0.406],
        'normalize_std': [0.229, 0.224, 0.225],
        'tta': TTA_MODE,
        'output_semantics': 'logits',
        # ALREADY FOLDED INTO THE SHIPPED WEIGHTS — provenance, not an
        # instruction. The server must NOT re-apply it; doing so would double
        # the shift and break the calibration fit on top of it.
        'class_logit_bias': CLASS_LOGIT_BIAS_APPLIED,
        '_class_logit_bias_note': 'Folded into the head bias BEFORE export, so '
                                  'temperature/conformal/INT8 were all fit on '
                                  'these logits. Recorded for traceability only.',
        '_tta_note': 'hflip averages original+mirror LOGITS; temperature / '
                     'conformal / MaxLogit above are all fit with the same mode.',
    },
    'ood': {
        'score': 'embedding_knn' if EMBEDDING_OOD is not None else 'max_logit',
        'max_logit_floor': ML_FLOOR_PERSISTED,
        'max_logit_scope': MAXLOGIT_SCOPE,
        'nap_head_mass_floor': NAP_HEAD_MASS_FLOOR,
        'embedding': EMBEDDING_OOD,
        # Fitted on the served statistic (see the SERVED-PATH block); the ONLY
        # thresholds pipeline.py will apply. None -> overlay OFF.
        'served_gate': SERVED_GATE,
        '_score_note': ('Primary: embedding-space k-NN distance on the DINOv2 '
                        'CLS vector (Sun 2022). Fallback: the MaxLogit floor, '
                        'scored over max_logit_scope (v2.6.13+ exports use '
                        "'plant_only' -- the max EXCLUDING the not_a_plant "
                        'logit; the all-class max cannot separate because NAP '
                        'is a trained class). The server prefers the embedding '
                        'gate when a bank is shipped, else MaxLogit. Both are '
                        'VETOED by nap_head_mass_floor: neither consults the '
                        'not_a_plant head, and on out-of-domain damaged foliage '
                        'that head is the better separator -- P(nap) p05=0.815 '
                        'on real not_a_plant vs p95<=0.04 on plants (measured on '
                        '2.6.8). The gate refuses to fire below the floor.'),
        '_calibration': ML_CAL_META,
    },
}

# ── Inference benchmark (latency + size) for the shipped artefacts ──
# Measured at export time and embedded in model_card.json so EVERY tier
# (gamma / frontier server ONNX, aqua on-device TFLite) carries a comparable
# latency + size record for deploy capacity-planning and offline research.
# cpu_1thread is the canonical cross-tier number: it matches the conservative
# single-core server budget and is the only setting comparable across tiers and
# hardware; a cuda timing is added when a GPU provider is present (the frontier
# tier serves on GPU). FULLY GUARDED -- a benchmark failure never blocks the
# ship (the model is graded by final_metrics above, not by latency).
def _file_mb(p):
    try:
        return round(Path(p).stat().st_size / 1e6, 2)
    except Exception:
        return None

def _lat_stats(ts_ms):
    ts = sorted(ts_ms)
    return {'latency_ms_p50': round(float(np.percentile(ts, 50)), 2),
            'latency_ms_p95': round(float(np.percentile(ts, 95)), 2),
            'latency_ms_mean': round(float(np.mean(ts)), 2),
            'n_iter': len(ts)}

def _benchmark_onnx(onnx_path, img_size, n_warmup=5, n_iter=50):
    import time as _time
    avail = set(ort.get_available_providers())
    # hflip-TTA makes the runtime do 2 forward passes per image; bench the
    # SAME work the server spends so the number is deploy-faithful.
    _calls = 2 if USE_TTA_HFLIP else 1
    x = np.random.rand(1, 3, img_size, img_size).astype(np.float32)
    out = {}
    for prov, tag, threads in (('CPUExecutionProvider', 'cpu_1thread', 1),
                               ('CUDAExecutionProvider', 'cuda', 0)):
        if prov not in avail:
            continue
        try:
            so = ort.SessionOptions()
            so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
            if threads:
                so.intra_op_num_threads = threads
                so.inter_op_num_threads = 1
            sess = ort.InferenceSession(str(onnx_path), sess_options=so, providers=[prov])
            in_name = sess.get_inputs()[0].name
            for _ in range(n_warmup):
                sess.run(None, {in_name: x})
            ts = []
            for _ in range(n_iter):
                _t0 = _time.perf_counter()
                for _ in range(_calls):
                    sess.run(None, {in_name: x})
                ts.append((_time.perf_counter() - _t0) * 1e3)
            out[tag] = _lat_stats(ts)
        except Exception as _be:  # noqa: BLE001 — diagnostic only
            out[tag] = {'error': f'{type(_be).__name__}: {_be}'}
    return out

def _benchmark_tflite(tflite_path, n_warmup=5, n_iter=50):
    import time as _time
    try:
        try:
            from ai_edge_litert.interpreter import Interpreter as _Interp
        except Exception:
            from tensorflow.lite import Interpreter as _Interp
        interp = _Interp(model_path=str(tflite_path)); interp.allocate_tensors()
        ind = interp.get_input_details()[0]
        shp = [int(d) if int(d) > 0 else 1 for d in ind['shape']]
        if np.issubdtype(ind['dtype'], np.floating):
            x = np.random.rand(*shp).astype(ind['dtype'])
        else:
            x = np.random.randint(0, 255, size=shp).astype(ind['dtype'])
        for _ in range(n_warmup):
            interp.set_tensor(ind['index'], x); interp.invoke()
        ts = []
        for _ in range(n_iter):
            _t0 = _time.perf_counter()
            interp.set_tensor(ind['index'], x); interp.invoke()
            ts.append((_time.perf_counter() - _t0) * 1e3)
        return {'cpu_1thread': _lat_stats(ts)}
    except Exception as _be:  # noqa: BLE001 — diagnostic only
        return {'error': f'{type(_be).__name__}: {_be}'}

benchmark = {
    '_note': ('Inference latency (ms/image, batch=1) + artefact size on disk. '
              'cpu_1thread is the canonical cross-tier number (single-core server '
              'budget) and INCLUDES hflip-TTA (2 forward passes) when '
              'inference.tta=hflip -- i.e. the work the runtime actually spends. '
              f'Measured on the export box (device={device}); RELATIVE, not an SLA.'),
    'measured_on': device,
    'tta': TTA_MODE,
    'classifier_onnx': {
        'artifact': shipped_artifact,
        'size_mb': _file_mb(eval_sess_path),
        'img_size': IMG,
        **_benchmark_onnx(eval_sess_path, IMG),
    },
}
# Aqua on-device student (§5): benchmark its TFLite when this run produced it.
_aqua_tflite = Path('models/aqua/aqua_classifier.tflite')
if _aqua_tflite.exists():
    benchmark['aqua_tflite'] = {
        'artifact': 'aqua/aqua_classifier.tflite',
        'size_mb': _file_mb(_aqua_tflite),
        **_benchmark_tflite(_aqua_tflite),
    }
print('Benchmark:\n' + json.dumps(benchmark, indent=2))

def _load_source_inventory():
    """Read §2's persisted per-source inventory. Returns None when absent (a
    kernel that resumed after §2) rather than raising -- the model card is
    analysis metadata and must never block an otherwise-good export."""
    try:
        return json.loads((ROOT / 'source_inventory.json').read_text())
    except Exception:
        return None

model_card = {
    'version': MODEL_VERSION,
    'run_id': RUN_ID,
    'data_manifest': DATA_MANIFEST,
    'model_family': _EXPORT_FAMILY,
    'architecture': ARCHITECTURE,
    'trained_at': datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat(),
    'classes': CLASS_NAMES,
    'img_size': IMG,
    'temperature': T_val,
    'conformal': cfg,
    'train_counts':           {c: count_images(TRAIN_ROOT  / c) for c in CLASS_NAMES},
    'val_realworld_counts':   {c: count_images(VAL_PRIMARY / c) for c in CLASS_NAMES},
    'val_indomain_counts':    {c: count_images(VAL_INDOMAIN/ c) for c in CLASS_NAMES},
    'temperature_cal_counts': {c: count_images(TEMP_CAL    / c) for c in CLASS_NAMES},
    'conformal_cal_counts':   {c: count_images(CONFORMAL_CAL/ c) for c in CLASS_NAMES},
    'val_studio_counts':      {c: count_images(VAL_STUDIO  / c) for c in CLASS_NAMES} if val_studio_dl is not None else None,
    'holdout_counts':         {c: count_images(HOLDOUT     / c) for c in CLASS_NAMES},
    # 2.6.9 and 2.7.0 are permanently uninterpretable partly because nothing
    # recorded the seed. `hyperparams` below carries the rest.
    'seed': SEED,
    # Per-source image inventory from §2 (the REQUIRED_SOURCE_FLOORS gate).
    # Class counts alone cannot tell you WHICH upstream corpus a run was
    # trained on -- on 2026-07-20 a 2,264-image source vanished and the class
    # counts merely looked "a bit smaller". A paired A/B between two versions
    # is only meaningful when their inventories match, so ship the inventory
    # with the model card and DIFF IT before believing any version-over-version
    # delta. Falls back to None on a resumed session that skipped §2.
    'source_inventory': (SOURCE_INVENTORY if 'SOURCE_INVENTORY' in globals()
                         else _load_source_inventory()),
    'hyperparams': {
        'backbone': _HUB_ID,
        'backbone_frozen': True,
        'embed_dim': EMBED_DIM,
        'head_hidden': HEAD_HIDDEN,
        'head_dropout': HEAD_DROPOUT,
        'head_epochs': HEAD_EPOCHS,
        'head_lr': HEAD_LR,
        'la_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'select_weakclass_precision_weight': SELECT_WEAKCLASS_PRECISION_WEIGHT,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'use_source_weighted_sampler': bool(USE_SOURCE_WEIGHTED_SAMPLER),
        'source_weights': dict(SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'prior_source_weights': dict(PRIOR_SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'batch': BATCH, 'img_size': IMG,
        # Profile version labels the arm (default vs TRAIN_AUGMENTATION='off').
        'augmentation_profile': TRAIN_AUGMENTATION_PROFILE,
        'lora': ({'enabled': True, 'rank': LORA_RANK, 'alpha': LORA_ALPHA,
                  'last_k_blocks': LORA_LAST_K_BLOCKS, 'lr': LORA_LR}
                 if USE_LORA else {'enabled': False}),
    },
    'int8': {
        '_note': ('f1_int8 / f1_fp32 are measured by raw argmax(logits) on '
                  'val_realworld — no temperature scaling, no weak-healthy '
                  'reroute, no very-low-confidence demotion. The runtime '
                  'pipeline applies all three on top; production disease '
                  'macro-F1 is therefore at least as good as f1_int8, never '
                  'less. The ship gate uses f1_int8 as a conservative floor. '
                  'The paired_bootstrap entry is the honest int8-vs-fp32 test: '
                  'treat the ordering as real only when significant=true.'),
        'drift': drift, 'f1_fp32': f1_fp32, 'f1_int8': f1_int8,
        'warn_drift': WARN_DRIFT, 'fail_drift': FAIL_DRIFT,
        'quant_mode': 'dynamic',
        'weight_type': 'QInt8',
        'paired_bootstrap': paired_int8_vs_fp32,
        'tta': TTA_MODE,
        'shipped': int8_shipped,
        'shipped_artifact': shipped_artifact,
    },
    'per_class_recall_ci_realworld': recall_ci_realworld,
    'per_class_precision_ci_realworld': precision_ci_realworld,
    'head': {'best_indomain_disease_f1': best_f1, 'best_epoch': best_ep},
    'reporting_backend': ('onnxruntime_int8_cpu' if int8_shipped
                          else 'onnxruntime_fp32_cpu'),
    'benchmark': benchmark,
    'aqua': aqua_metrics,
    'reproducibility': {
        'seed': SEED,
        'torch': torch.__version__,
        'onnxruntime': ort.__version__,
        'cudnn_deterministic': bool(getattr(torch.backends.cudnn, 'deterministic', False)),
        'cudnn_benchmark':     bool(getattr(torch.backends.cudnn, 'benchmark', True)),
        # The state §4 trained under (the keys above are read at export time).
        'training_determinism': DETERMINISM_AT_TRAINING,
        'nondeterministic_ops': {op: list(ph) for op, ph in sorted(DETERMINISM_WARNINGS.items())},
    },
    # Headline disease macro-F1 per split WITH split name + eval mode, so the
    # card separates classifier-only (Stage 2 on pre-cropped val) from
    # end-to-end (detector->crop->classify, appended by the detector-crop eval).
    # metrics_by_split['studio']['supported_disease_macro_f1'] is the
    # val_studio_supported_macro_f1 (PlantVillage covers only a few classes).
    'metrics_by_split': {
        _k: {'split': _v.get('split', _k),
             'eval_mode': _v.get('eval_mode', 'classifier_only'),
             'n': _v.get('n'),
             'disease_macro_f1': _v['disease_macro_f1'],
             'supported_disease_macro_f1': _v.get('supported_disease_macro_f1',
                                                  _v['disease_macro_f1']),
             'n_supported_disease_classes': _v.get('n_supported_disease_classes'),
             'healthy_bias': _v['healthy_bias']}
        for _k, _v in final_metrics.items()
    },
}

# ── Ship gate — runs BEFORE we persist anything declarative ─────
MIN_DISEASE_F1, MAX_HEALTHY_BIAS, MIN_HEALTHY_RECALL = 0.70, 1.50, 0.70
rw = final_metrics['realworld']
disease_classes = [c for c in CLASS_NAMES if c not in ('healthy', 'not_a_plant')]
zero_recall = [c for c in disease_classes if rw['per_class_recall'].get(c, 0.0) <= 0.0]
healthy_recall = float(rw['per_class_recall'].get('healthy', 0.0))
failures = []
if rw['disease_macro_f1'] < MIN_DISEASE_F1:
    failures.append(f'disease_macro_f1={rw["disease_macro_f1"]:.3f} < {MIN_DISEASE_F1}')
if rw['healthy_bias'] >= MAX_HEALTHY_BIAS:
    failures.append(f'healthy_bias={rw["healthy_bias"]:.2f} >= {MAX_HEALTHY_BIAS}')
if healthy_recall < MIN_HEALTHY_RECALL:
    failures.append(f'recall[healthy]={healthy_recall:.3f} < {MIN_HEALTHY_RECALL} (false-positive disease risk)')
if zero_recall:
    failures.append(f'zero recall classes: {zero_recall}')
if failures:
    raise RuntimeError('SHIP GATE FAILED: ' + '; '.join(failures))
print('SHIP GATE PASSED')

# ── Unseen-domain regression diagnostic — NEVER a ship gate ─────
# `val_realworld` is still domain-limited, so a separate unseen-domain sentinel
# is useful for qualitative regression alerts. It does not grade or bless the
# shipped artefact. Historical release comparisons, pooled-arm values and claims
# that this set decisively located a model failure were invalidated by the
# 2026-08-23 audit: denominators/stages were mixed, evaluation was classifier-only
# rather than served, and nominal rows were treated as independent captures.
# See docs/ROADMAP.md; archive-derived values below are not
# calibration targets or evidence of improvement.
#
# KNOWN LIMITATION, do not lose it: this leaves the scorch arm at ONE species
# from one studio/white-background domain. The nominal 31 rows are not 31
# independent captures: SIFT+RANSAC found at least 11 rotated/noised
# pseudo-replicates that pHash<=6 missed, and several images show wilt/torn/dark
# leaves rather than clean marginal necrosis. Treat v2 only as a regression
# sentinel. A deployment-representative, robust-family-deduped v3 with multiple
# species/capture sessions is required before any release floor is defensible.
#
# Report the healthy false-fire arm beside scorch recall so a diagnostic alert
# cannot be improved merely by firing `leaf_scorch` more often. The numeric
# references remain non-blocking heuristics until a family-clean v3 is built.
OOD_BENCHMARK_SLUG = 'maxayyt5/plantvision-ood-benchmark'
REQUIRE_OOD_BENCHMARK = False         # v2 is diagnostic-only; v3 must replace it
# The diagnostic being non-BLOCKING must not let it quietly stop EXISTING: a
# production Run-all already needs the same Kaggle credentials for the §2
# supplement, so the sentinel must be downloaded, scored and written into
# model_card.json / training_metrics.json on every run. Only its floors are
# advisory. Set False only for a deliberately non-production diagnostic run.
REQUIRE_OOD_DIAGNOSTIC_RUN = True
OOD_ARM_FLOORS = {'hibiscus': 0.70}   # advisory heuristic, never a v2 ship gate
OOD_MAX_HEALTHY_FALSE_FIRE = 0.12     # advisory heuristic pending family-clean v3

ood_benchmark = None
_ood_root = _kaggle_download(OOD_BENCHMARK_SLUG) if OOD_BENCHMARK_SLUG else None
if _ood_root is None:
    _msg = (f'unseen-domain diagnostic {OOD_BENCHMARK_SLUG!r} unavailable (no Kaggle '
            f'credentials, no network, or the slug is wrong). v2 is advisory and '
            f'cannot establish release fitness; build a family-clean v3 before setting '
            f'REQUIRE_OOD_BENCHMARK=True.')
    if REQUIRE_OOD_BENCHMARK:
        raise RuntimeError('SHIP GATE FAILED: ' + _msg)
    if REQUIRE_OOD_DIAGNOSTIC_RUN:
        raise RuntimeError(
            'OOD DIAGNOSTIC NOT RUN: ' + _msg + ' The sentinel is advisory but '
            'mandatory to record; set REQUIRE_OOD_DIAGNOSTIC_RUN=False only for a '
            'non-production run.')
    print('WARN: ' + _msg)
else:
    import csv as _csv
    _ood_root = Path(_ood_root)
    # server/tools/build_ood_benchmark.py writes manifest.csv (with an `arm`
    # column); the published v2 also carries the older manifest_arms.csv.
    # Accept either — a freshly built v3 must not fail here on a filename.
    _arm_csv = (next(_ood_root.rglob('manifest_arms.csv'), None)
                or next(_ood_root.rglob('manifest.csv'), None))
    if _arm_csv is None:
        raise RuntimeError(
            f'unseen-domain diagnostic at {_ood_root} has no manifest_arms.csv or '
            f'manifest.csv; class/path/family-keyed reporting cannot be verified. '
            f'Re-publish it with a complete manifest.')
    _img_root = _arm_csv.parent
    with open(_arm_csv, newline='') as _f:
        _arm_rows = list(_csv.DictReader(_f))
    _arm_of = {}
    for _r in _arm_rows:
        _raw_key = str(_r.get('filename', '')).strip().replace('\\', '/')
        _path_key = PurePosixPath(_raw_key)
        _key = _path_key.as_posix()
        _declared_class = str(_r.get('class', '')).strip()
        if (not _raw_key or _path_key.is_absolute() or '..' in _path_key.parts or
                len(_path_key.parts) < 2 or _path_key.parts[0] != _declared_class or
                _key in _arm_of):
            raise RuntimeError(f'OOD manifest has missing/duplicate class-path key: {_key!r}')
        _arm_of[_key] = _r.get('species') or '?'
    _family_ids = [str(r.get('capture_family_id', '')).strip() for r in _arm_rows]
    _family_verified = bool(_family_ids) and all(_family_ids) and len(set(_family_ids)) == len(_family_ids)
    if not _family_verified:
        _msg = ('benchmark manifest has no unique capture_family_id per row; nominal image '
                'counts are not independent-family counts (known defect in private v2)')
        if REQUIRE_OOD_BENCHMARK:
            raise RuntimeError('OOD benchmark cannot gate: ' + _msg)
        print('WARN: ' + _msg)

    _ood_ds, _ood_dl = make_eval_loader(_img_root, IMG, class_to_idx=TRAIN_CLASS_TO_IDX)
    _ood_logits, _ood_y = _collect_logits(eval_sess, _ood_dl)
    _ood_pred = _ood_logits.argmax(1)
    _ood_rel = [Path(p).relative_to(_img_root).as_posix() for p, _ in _ood_ds.samples]
    _ood_sp = np.array([_arm_of.get(r, '?') for r in _ood_rel])

    _sc_i = CLASS_NAMES.index('leaf_scorch_underwatering')
    _he_i = CLASS_NAMES.index('healthy')
    _dis_i = [i for i, c in enumerate(CLASS_NAMES) if c not in ('healthy', 'not_a_plant')]
    _m_sc, _m_he = _ood_y == _sc_i, _ood_y == _he_i

    _arms = {}
    for _sp in sorted(set(_ood_sp[_m_sc])):
        _m = _m_sc & (_ood_sp == _sp)
        _arms[_sp] = {'n': int(_m.sum()), 'recall': float((_ood_pred[_m] == _sc_i).mean())}
    _ff = float(np.isin(_ood_pred[_m_he], _dis_i).mean()) if _m_he.any() else 0.0

    ood_benchmark = {
        'slug': OOD_BENCHMARK_SLUG, 'nominal_n': int(len(_ood_y)),
        'artefact': shipped_artifact, 'eval_mode': 'classifier_only',
        'diagnostic_only': True, 'release_blocking': False,
        'independent_family_count_verified': _family_verified,
        'scorch_recall_pooled': float((_ood_pred[_m_sc] == _sc_i).mean()),
        'scorch_recall_by_arm': _arms,
        'healthy_n': int(_m_he.sum()), 'healthy_false_fire': _ff,
        'arm_floors': OOD_ARM_FLOORS, 'max_healthy_false_fire': OOD_MAX_HEALTHY_FALSE_FIRE,
    }
    print(f'\nUnseen-domain diagnostic ({OOD_BENCHMARK_SLUG}, nominal n={len(_ood_y)}):')
    for _sp, _d in _arms.items():
        _fl = OOD_ARM_FLOORS.get(_sp)
        print(f'  leaf_scorch[{_sp:16s}] recall={_d["recall"]:.3f} (n={_d["n"]})'
              + (f'  reference={_fl:.2f}' if _fl is not None else '  (reported)'))
    print(f'  healthy false-fire      {_ff:.3f} (n={int(_m_he.sum())})  '
          f'ceiling={OOD_MAX_HEALTHY_FALSE_FIRE:.2f}')

    _ood_fail = []
    for _sp, _fl in OOD_ARM_FLOORS.items():
        if _sp not in _arms:
            _ood_fail.append(f'arm {_sp!r} missing from the benchmark (floors are stale)')
        elif _arms[_sp]['recall'] < _fl:
            _ood_fail.append(f'leaf_scorch[{_sp}]={_arms[_sp]["recall"]:.3f} < {_fl}')
    if _ff > OOD_MAX_HEALTHY_FALSE_FIRE:
        _ood_fail.append(f'healthy false-fire={_ff:.3f} > {OOD_MAX_HEALTHY_FALSE_FIRE}')
    # Alerts ride in the artefact, not only in the Colab log: a reviewer of
    # model_card.json must see the same lines the run printed.
    ood_benchmark['alerts'] = list(_ood_fail)
    log_run({'phase': 'ood_diagnostic', 'slug': OOD_BENCHMARK_SLUG,
             'nominal_n': ood_benchmark['nominal_n'],
             'scorch_recall_by_arm': _arms, 'healthy_false_fire': _ff,
             'alerts': list(_ood_fail), 'release_blocking': bool(REQUIRE_OOD_BENCHMARK)})
    if _ood_fail and REQUIRE_OOD_BENCHMARK:
        raise RuntimeError('OOD benchmark gate failed: ' + '; '.join(_ood_fail))
    if _ood_fail:
        print('WARN: unseen-domain diagnostic alerts (non-blocking): ' + '; '.join(_ood_fail))
    else:
        print('Unseen-domain diagnostic references met (this is not a ship gate)')

model_card['ood_benchmark'] = ood_benchmark
final_metrics['ood_benchmark'] = ood_benchmark


# ── The four populations, and the band a claim has to clear ─────
# Read these together or not at all. Three of the four are REGRESSION metrics
# with floors; exactly one is the improvement metric. Quoting a single pooled
# number is how 2.6.9, 2.7.0 and 2.7.1 each got shipped-then-unshipped.
#
# THE REPLICATE BAND. Two archived 2.6.0 bundles share a byte-identical
# hyperparams dict and corpora differing by SEVEN images; on the never-trained
# ornamental benchmark they disagree on EIGHT healthy clusters (149/151 vs
# 141/151, McNemar 8-0, p=0.0078) and one scorch cluster. A second pair (2.4.0,
# 794-image corpus delta) swings 13 scorch clusters. So ~8 healthy clusters and
# up to ~13 scorch clusters are inside pure run-to-run noise, at p-values this
# gate would otherwise call significant. Until a true seed replicate measures the
# band properly (re-run with SEED changed and nothing else), treat any benchmark
# delta below that as NOT MEASURED.
REPLICATE_BAND = {'benchmark_healthy_clusters': 8, 'benchmark_scorch_clusters': 13,
                  'source': '2.6.0 x5-vs-x6 (7-image corpus delta) and 2.4.0 x8-vs-x9',
                  'note': 'upper bound: both pairs differ slightly in corpus too, so this is '
                          'not pure seed noise. No true replicate has ever been run.'}
print('\n── the four populations ──────────────────────────────────────')
if 'holdout' in final_metrics:
    _hd = final_metrics['holdout']
    _hcm = np.asarray(_hd['confusion_matrix'])
    _hacc = float(np.trace(_hcm) / max(_hcm.sum(), 1))
    _hd['accuracy'] = _hacc
    _hd['per_class_support'] = {c: int(n) for c, n in zip(CLASS_NAMES, _hcm.sum(1))}
    # SUPPORTED macro-F1, not the all-class one. The holdout is drawn from the
    # supplement, which is supply-starved in viral_disease (2 eval-eligible rows
    # in all of v8), so that class lands at n=0 here and the plain macro-F1
    # averages in a hard zero — 0.730 where the honest number is 0.852. This is
    # the same trap val_studio already documents; the improvement metric must
    # not inherit it.
    _hsup = _hd.get('supported_disease_macro_f1', _hd['disease_macro_f1'])
    _hnc = _hd.get('n_supported_disease_classes', '?')
    print(f"  [improvement] holdout            acc={_hacc:.4f} "
          f"disease_macro_f1={_hsup:.4f} over {_hnc} supported classes "
          f"(n={_hd['n']}; all-class={_hd['disease_macro_f1']:.4f} drags in "
          f"absent-class zeros — do NOT quote it)")
else:
    print('  [improvement] holdout            MISSING — this run cannot report an improvement')
print(f"  [regression ] val_realworld      disease_macro_f1={rw['disease_macro_f1']:.4f} "
      f"healthy_recall={healthy_recall:.4f}  (floor {MIN_DISEASE_F1} / {MIN_HEALTHY_RECALL})")
print(f"  [regression ] val_indomain       disease_macro_f1="
      f"{final_metrics['indomain']['disease_macro_f1']:.4f}")
print(f"  [regression ] not_a_plant recall="
      f"{rw['per_class_recall'].get('not_a_plant', float('nan')):.4f}  "
      f"(val_realworld NAP = COCO/OpenImages/iNat hard negatives. NOT comparable "
      f"to the 0.9894 measured on supplement eval_ok NAP — different population; "
      f"score that one with tools/paired_eval.py)")
print(f"  the never-trained OOD benchmark is scored OUTSIDE this notebook — "
      f"server/tools/build_ood_benchmark.py + paired_eval.py --served, "
      f"baseline = 2.6.8 + the NAP head veto, NOT bare 2.6.8")
print(f"  replicate band: a benchmark delta under "
      f"{REPLICATE_BAND['benchmark_healthy_clusters']} healthy / "
      f"{REPLICATE_BAND['benchmark_scorch_clusters']} scorch clusters is INSIDE RUN NOISE")
if 'holdout' in final_metrics:
    _hd = final_metrics['holdout']
    for _c in CLASS_NAMES:
        _n = int((_hd.get('per_class_support') or {}).get(_c, 0))
        if _n:
            print(f"      holdout recall[{_c:25s}] = "
                  f"{_hd['per_class_recall'].get(_c, float('nan')):.3f}  (n={_n})")

# Non-blocking weak-class precision audit (v2.5.1). The physiological classes
# over-trigger; a sharp argmax-precision drop is worth a loud line. NOT a hard
# gate — precision on a ~25-45 sample slice has a wide CI, so failing the run on
# it would be noise-driven; the runtime (temperature + low-confidence demotion +
# conformal sets) absorbs much of the raw-argmax over-trigger.
# 0.50 was unreachable: the project's weak-class precision concern is 0.80 (the
# whole point of the v8 data round), and nutrient shipped at 0.722 in 2.7.0
# without this ever printing. A monitor calibrated below every value it will
# ever see is not a monitor. Non-blocking either way -- it informs, the ship
# gate decides.
WEAK_PRECISION_WARN = 0.80
for _c in ('leaf_scorch_underwatering', 'nutrient_deficiency'):
    _pd = precision_ci_realworld['classes'].get(_c) or {}
    _pt = _pd.get('point')
    if _pt is not None and _pt < WEAK_PRECISION_WARN:
        print(f'WARN: weak-class precision[{_c}]={_pt:.3f} < {WEAK_PRECISION_WARN} '
              f'(over-triggering; n_pred={_pd.get("n_predicted")}, '
              f'CI=[{_pd.get("ci_low")}, {_pd.get("ci_high")}]) — non-blocking; '
              f'runtime demotion/conformal mitigate.')


# ── Persist the declarative artefacts now that the gate is green ──
Path('models/classes.json').write_text(json.dumps(classes_payload, indent=2))
Path('models/temperature.json').write_text(json.dumps(temperature_payload, indent=2))
Path('models/conformal.json').write_text(json.dumps(cfg, indent=2))
Path('models/training_metrics.json').write_text(json.dumps(final_metrics, indent=2))
Path('models/model_card.json').write_text(json.dumps(model_card, indent=2))
if EMBEDDING_OOD is not None and ood_bank_ship is not None:
    np.savez_compressed('models/ood_embedding.npz',
                        bank=ood_bank_ship.astype(np.float16))
    print(f'ood_embedding.npz: bank shape={ood_bank_ship.shape}')
print(f'classes.json: {len(CLASS_NAMES)} classes  architecture={ARCHITECTURE}')
print('Persisted: classes.json, temperature.json, conformal.json, '
      'training_metrics.json, model_card.json')

# Fail-closed, checksummed serving manifest. Runtime must load the ONE selected
# graph and every bound sidecar by this manifest; no filename fallback.
import hashlib as _bundle_hashlib
import importlib.metadata as _importlib_metadata
import platform as _platform
_selected_classifier = ('classifier_int8.onnx' if int8_shipped else 'classifier.onnx')
_expected_fit_source = 'int8' if int8_shipped else 'fp32_fallback'
_temperature_fit_source = temperature_payload.get('source')
_conformal_fit_source = cfg.get('fit_source')
if (_temperature_fit_source != _expected_fit_source or
        _conformal_fit_source != _expected_fit_source):
    raise RuntimeError(
        'calibration/classifier mismatch: selected graph is '
        f'{_selected_classifier}, temperature={_temperature_fit_source}, '
        f'conformal={_conformal_fit_source}; calibration may not fall back')
if not DETECTOR_EXPORT_SUCCEEDED:
    raise RuntimeError('current run did not produce detector.onnx')
if FAMILY_CONFIG.get('distill') and not AQUA_EXPORT_SUCCEEDED:
    raise RuntimeError('combined run did not produce a verified current-run Aqua artifact')
if 'DATA_MANIFEST' not in globals():
    raise RuntimeError('current run has no sealed DATA_MANIFEST')
if YOLO_BASE_WEIGHT_IDENTITY is None:
    raise RuntimeError('current detector run has no sealed base-weight identity')
# Bind the complete resolved environment, not a hand-picked package subset.
# This is the explicit audit trail for the Colab CUDA-runtime exception in §1.
_resolved_distributions = []
for _dist in _importlib_metadata.distributions():
    _name = str(_dist.metadata.get('Name') or '').strip().lower()
    if _name:
        _resolved_distributions.append({
            'name': re.sub(r'[-_.]+', '-', _name),
            'version': str(_dist.version),
        })
_resolved_distributions.sort(key=lambda x: (x['name'], x['version']))
_runtime_environment = {
    'resolution_policy': PACKAGE_RESOLUTION_POLICY,
    'python': sys.version, 'python_cache_tag': sys.implementation.cache_tag,
    'platform': _platform.platform(),
    'cuda_runtime': str(torch.version.cuda),
    'cudnn': str(torch.backends.cudnn.version()),
    'cuda_device': torch.cuda.get_device_name(0),
    'distributions': _resolved_distributions,
}
# The Aqua converter ran in its own interpreter (§1); bind it too, so a
# toolchain change is a different recipe identity, not a silent one.
if AQUA_EXPORT_ENVIRONMENT is not None:
    _runtime_environment['aqua_export_environment'] = AQUA_EXPORT_ENVIRONMENT
_recipe_environment_sha256 = _bundle_hashlib.sha256(
    json.dumps(_runtime_environment, sort_keys=True, separators=(',', ':')).encode()
).hexdigest()
_run_manifest_payload = {
    'schema_version': 1, 'run_id': RUN_ID, 'started_at': RUN_STARTED_AT,
    'model_version': MODEL_VERSION, 'train_mode': TRAIN_MODE,
    'model_family': _EXPORT_FAMILY, 'seed': SEED,
    'python': sys.version, 'cuda_device': torch.cuda.get_device_name(0),
    'package_resolution_policy': PACKAGE_RESOLUTION_POLICY,
    'recipe_environment_sha256': _recipe_environment_sha256,
    'runtime_environment': _runtime_environment,
    'dinov2_repo_ref': DINOV2_REPO_REF,
    'detector_base_weights': YOLO_BASE_WEIGHT_IDENTITY,
    'source_revisions': {
        'huggingface': {k: _HF_RESOLVED_REVISIONS[k]
                        for k in sorted(_HF_RESOLVED_REVISIONS)},
        'kaggle': dict(sorted(_KAGGLE_RESOLVED_REVISIONS.items())),
    },
    'data_identity': DATA_MANIFEST['classification_sha256'],
    'augmentation_profile': TRAIN_AUGMENTATION_PROFILE,
    # Training-time determinism, re-asserted at the top of §4 after
    # Ultralytics switched it off, plus every op that had no deterministic
    # kernel (op -> phases). An empty map means the run is bitwise-repeatable
    # on the same GPU type and stack.
    'determinism': {
        'at_training': DETERMINISM_AT_TRAINING,
        'nondeterministic_ops': {op: list(ph) for op, ph in sorted(DETERMINISM_WARNINGS.items())},
    },
}
Path('models/run_manifest.json').write_text(
    json.dumps(_run_manifest_payload, indent=2), encoding='utf-8')
_bound = {
    'classifier': _selected_classifier, 'detector': 'detector.onnx',
    'classes': 'classes.json', 'temperature': 'temperature.json',
    'conformal': 'conformal.json', 'data_manifest': 'data_manifest.json',
    'run_manifest': 'run_manifest.json',
}
if EMBEDDING_OOD is not None:
    _bound['ood_embedding'] = 'ood_embedding.npz'
if FAMILY_CONFIG.get('distill'):
    _bound.update({
        'aqua_classifier': 'aqua/aqua_classifier.tflite',
        'aqua_classes': 'aqua/classes.json',
        'aqua_onnx': 'aqua/aqua_student.onnx',
    })
_missing_bound = [name for name in _bound.values() if not (Path('models') / name).is_file()]
if _missing_bound:
    raise RuntimeError(f'cannot emit deploy bundle: required artifacts missing {_missing_bound}')
def _artifact_digest(_name):
    _p = Path('models') / _name
    _h = _bundle_hashlib.sha256()
    with _p.open('rb') as _f:
        for _chunk in iter(lambda: _f.read(1 << 20), b''):
            _h.update(_chunk)
    return {'path': _name, 'sha256': _h.hexdigest(), 'size_bytes': _p.stat().st_size}
_data_manifest_artifact = _artifact_digest('data_manifest.json')
_served_gate = ((classes_payload.get('ood') or {}).get('served_gate') or {})
_bundle_manifest = {
    'schema_version': 1,
    'bundle_id_algorithm': 'sha256_canonical_json_without_bundle_id',
    'manifest_file': 'bundle_manifest.json',
    'run_id': RUN_ID, 'model_version': MODEL_VERSION, 'train_mode': TRAIN_MODE,
    'model_family': _EXPORT_FAMILY,
    'data_manifest': {
        'path': 'data_manifest.json',
        'manifest_sha256': _data_manifest_artifact['sha256'],
        'dataset_sha256': DATA_MANIFEST['classification_sha256'],
        'n_images': DATA_MANIFEST['n_images'],
    },
    'classifier_selection': {
        'path': _selected_classifier, 'fallback_allowed': False,
        'calibration_source': _expected_fit_source,
    },
    'artifacts': {role: _artifact_digest(name) for role, name in _bound.items()},
    'serving_contract': {
        'schema_version': 1,
        'detector': {
            'path': 'detector.onnx', 'class_names': ['plant_leaf'],
            **SERVED_GATE_DETECTOR,
        },
        'crop': {
            **SERVED_GATE_CROP,
            'member_selection': 'detector_confidence_descending',
            'vote_aggregation': 'equal_mean_probabilities',
            'whole_frame_vote_weight': 1.0,
        },
        'classifier': {
            'path': _selected_classifier, 'img_size': IMG,
            'eval_resize_ratio': EVAL_RESIZE_RATIO,
            'resize': 'shorter_side', 'crop': 'center',
            'normalize_mean': [0.485, 0.456, 0.406],
            'normalize_std': [0.229, 0.224, 0.225],
            'tta': TTA_MODE, 'tta_aggregation': 'mean_logits',
            'output_semantics': 'logits',
        },
        'blur': {
            'enabled': True, 'policy': 'abstain',
            'variance_laplacian_floor': 50.0, 'downscale_max_side': 320,
        },
        'ood': {
            'policy': 'detector_and_classifier_signal_with_nap_head_veto',
            'detector_confidence_floor': SERVED_GATE_DET_CONF_FLOOR,
            'classifier_signal_priority': 'embedding_knn_else_max_logit',
            'embedding_aggregation': 'min_member_distance',
            'max_logit_aggregation': 'max_member_logit',
            'nap_head_veto': 'veto_when_probability_below_configured_floor',
            'served_gate_enabled': bool(_served_gate),
            'served_gate': _served_gate or None,
            'config_source': 'classes.json#ood',
        },
    },
}
_bundle_identity_bytes = json.dumps(
    _bundle_manifest, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()
_bundle_manifest['bundle_id'] = _bundle_hashlib.sha256(_bundle_identity_bytes).hexdigest()
Path('models/bundle_manifest.json').write_text(
    json.dumps(_bundle_manifest, indent=2), encoding='utf-8')
print(f'bundle_manifest.json: bundle_id={_bundle_manifest["bundle_id"][:16]}…; '
      f'selected {_selected_classifier}; {len(_bound)} checksummed artifacts; '
      'fallback_allowed=False')

log_run({'phase': 'export_done', 'architecture': ARCHITECTURE,
         'f1_fp32': f1_fp32, 'f1_int8': f1_int8, 'T': T_val,
         'q_hat': cfg['q_hat'], 'empirical_coverage': cfg['empirical_coverage'],
         'paired_significant': paired_int8_vs_fp32['significant'],
         'int8_shipped': int8_shipped, 'shipped_artifact': shipped_artifact})

# ── Z1: per-class decision-bias sweep (diagnostic, gates nothing) ────────
# A constant added to one class's LOGIT before argmax. This is NOT the refuted
# lever: that one swept a bias on 2.7.1 chasing 2.6.8's out-of-domain recall and
# cost 14x the false-fire. This sweeps on THIS run's own val_realworld and
# reports the whole curve, so the trade is visible instead of asserted.
#
# The motivating measurement, on the deployed 2.6.8 over the 585 supplement rows
# it never trained on: leaf_scorch recall 0.739 -> 0.841 at bias +1.0 with
# healthy recall UNCHANGED at 0.934, other-disease accuracy 0.810 -> 0.795,
# overall 0.807 -> 0.821. Most of that class's errors are near the boundary.
#
# READ IT AS A DIAGNOSTIC, NOT A VALUE TO SHIP. Picking the argmax of a sweep on
# the same split you then quote is selection on the test set. If a bias is to
# ship, choose it on temperature_cal, then set CLASS_LOGIT_BIAS in §1 and re-run
# §4-§6 so the bias is folded into the exported weights and every calibration is
# fitted on top of it. Do NOT hand-edit classes.json ->
# inference.class_logit_bias: the server does not read that key (it is written
# for provenance), so it would be a silent no-op that the model card
# nonetheless reports as applied.
Z1_BIAS_CLASSES = ['leaf_scorch_underwatering', 'leaf_spot']
Z1_BIAS_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
try:
    # Reuse the logits `evaluate` already produced for final_metrics['realworld']
    # rather than paying for a second full pass. Fall back to recomputing them if
    # that block was skipped or the sink is empty.
    _z_true = np.concatenate([_yb.numpy() for _, _yb in val_realworld_dl])
    if _rw_logits:
        _z_logits = np.concatenate(_rw_logits)
        print(f'[Z1] reusing the val_realworld logits from final_metrics '
              f'({_z_logits.shape[0]} rows) — no second forward pass')
    else:
        _z_logits = np.concatenate([_ort_logits(eval_sess, _xb.numpy().astype(np.float32))
                                    for _xb, _ in val_realworld_dl])
    if _z_logits.shape[0] != _z_true.shape[0]:
        raise RuntimeError(f'Z1: {_z_logits.shape[0]} logits vs {_z_true.shape[0]} labels')
    _z_report = {}
    for _cname in Z1_BIAS_CLASSES:
        if _cname not in CLASS_NAMES:
            continue
        _ci = CLASS_NAMES.index(_cname)
        _hi = CLASS_NAMES.index('healthy') if 'healthy' in CLASS_NAMES else None
        _rows = []
        print(f'\n[Z1 bias sweep] {_cname} on val_realworld (n={len(_z_true)})')
        print(f"  {'bias':>6s} {'that class':>11s} {'healthy':>9s} {'other dis':>10s} "
              f"{'disease_f1':>11s}")
        for _b in Z1_BIAS_GRID:
            _Z = _z_logits.copy(); _Z[:, _ci] += _b
            _yp = _Z.argmax(1)
            _rec = float((( _yp == _ci) & (_z_true == _ci)).sum() / max((_z_true == _ci).sum(), 1))
            _hrec = (float(((_yp == _hi) & (_z_true == _hi)).sum() / max((_z_true == _hi).sum(), 1))
                     if _hi is not None else float('nan'))
            _om = np.isin(_z_true, [k for k, c in enumerate(CLASS_NAMES)
                                    if c not in ('healthy', 'not_a_plant', _cname)])
            _oacc = float((_yp[_om] == _z_true[_om]).mean()) if _om.any() else float('nan')
            _f1 = _disease_f1(_z_true, _yp)
            _rows.append({'bias': _b, 'class_recall': _rec, 'healthy_recall': _hrec,
                          'other_disease_acc': _oacc, 'disease_macro_f1': _f1})
            print(f'  {_b:6.2f} {_rec:11.3f} {_hrec:9.3f} {_oacc:10.3f} {_f1:11.3f}')
        _z_report[_cname] = _rows
        _base, _best = _rows[0], max(_rows, key=lambda r: r['disease_macro_f1'])
        if _best['bias'] > 0 and _best['disease_macro_f1'] - _base['disease_macro_f1'] > 0.005:
            print(f"  -> bias {_best['bias']:.2f} would move disease_macro_f1 "
                  f"{_base['disease_macro_f1']:.3f} -> {_best['disease_macro_f1']:.3f} at a "
                  f"healthy cost of {_base['healthy_recall'] - _best['healthy_recall']:+.3f}. "
                  f"Selected ON this split — re-choose on temperature_cal before shipping it.")
        else:
            print('  -> no bias beats 0.0 by more than noise on this split; leave it at 0.')
    final_metrics['z1_bias_sweep'] = _z_report
    log_run({'phase': 'z1_bias_sweep',
             'best': {k: max(v, key=lambda r: r['disease_macro_f1'])['bias']
                      for k, v in _z_report.items()}})
except Exception as _e:  # noqa: BLE001 — diagnostic only, model already shipped
    print(f'Z1 bias sweep skipped: {type(_e).__name__}: {_e}')

# ── Stage 1 audit + production-faithful eval: ONE streamed served pass ──
# WHY THE STAGE-1 AUDIT EXISTS (2.7.2). Scoring every archived bundle on a
# never-trained ornamental benchmark turned up a defect that had been invisible
# for fifteen versions: every run re-trains and re-exports its own detector, and
# NOTHING had ever measured it. Detector quality was a per-run lottery, and the
# deployed 2.6.8 bundle drew badly: Stage 1 returned ZERO boxes on 22.6% of
# out-of-domain damaged foliage vs 3.3% of healthy (Fisher p=3.2e-05), and every
# no-box image is then forced down pipeline.py's synthetic det_conf=0 path.
# Swapping in 2.6.9's detector — same classifier, same bank, same calibration —
# recovered +11.3pp served scorch recall, 95% CI [+0.032, +0.210]. A miss on
# damaged foliage is not symmetric with a miss on healthy foliage: it is exactly
# the input the user photographed BECAUSE it looked wrong.
#
# WHY IT IS ONE PASS NOW (2.7.5). The audit used to be an Ultralytics
# `YOLO.predict` over detector.pt and the end-to-end eval a second pass that
# decoded every photo AGAIN at full camera resolution and buffered 256 crop +
# 256 whole-frame tensors per chunk. On a `combined` run that sat on top of §5's
# TensorFlow/TFLite objects and every idle persistent DataLoader worker pool
# (`_loader_kwargs` keeps EVAL_WORKERS processes alive per loader, prefetched
# batches included), and the kernel OOMed — so the end-to-end eval was OFF for
# the mode that ships. Now:
#   - the pools and the §4/§5 training objects are released first;
#   - every photo goes through the SHIPPED graphs (models/detector.onnx +
#     eval_sess) with the server's own detector, crop and vote rules — the
#     `_served_*` mirrors above, pinned to detector.py / pipeline.py /
#     classifier.py by server/tests/test_served_gate_mirror.py;
#   - photos are decoded at the size the app uploads (SERVED_UPLOAD_MAX_SIDE,
#     JPEG draft = the same libjpeg subsampling Android uses), two threads ahead;
#   - each photo is reduced to a few scalars before the next one is touched.
# Peak extra host RAM: SERVED_PASS_PREFETCH decoded 1024 px photos plus one
# photo's (<= max_crops + 1) members and their mirrors — tens of MB, in every
# mode. One detector run per photo feeds all three readouts below.
#
# Both readouts are FULLY GUARDED: the model + artefacts are already shipped
# above, so a failure here only skips a diagnostic.
#
# STAGE1_MISS_RATIO_WARN is warn-only on purpose. It is a brand-new metric with
# no baseline: two bundles have been measured off-corpus (2.6.8 6.8x, 2.7.0
# 1.4x) and zero on val_realworld. Collect three runs of it, then decide the
# hard-gate value — the same discipline the weak-class precision audit follows.
STAGE1_MISS_RATIO_WARN = 2.0     # damaged no-box rate / healthy no-box rate
STAGE1_MISS_ABS_WARN = 0.15      # any single class losing >15% of its images
STAGE1_AUDIT_CONF = 0.25         # the no-box threshold every earlier audit used
RUN_DETECTOR_CROP_EVAL = True    # every mode, `combined` included: the pass is bounded


def _release_host_memory():
    """Give back what the rest of §6 no longer needs. Returns how many idle
    DataLoader worker pools were shut down."""
    import gc as _gc
    import sys as _sys
    _pools = 0
    for _v in list(globals().values()):
        if isinstance(_v, DataLoader):
            _it = getattr(_v, '_iterator', None)
            if _it is not None and hasattr(_it, '_shutdown_workers'):
                try:
                    _it._shutdown_workers()
                    _pools += 1
                except Exception:
                    pass
                _v._iterator = None
    # §4/§5 training state. Nothing after this point reads any of it (§6b
    # bundles from disk), and a re-run of §5 rebuilds all of it.
    for _name in ('interp', '_interp', '_p_sess', 'student', 'best_student_state',
                  'feat_proj', 'kd_loader', 'kd_val', 'kd_sampler', 'opt', 'sched',
                  'teacher_logits', 's_logits', 't_logits', 's_feats', 't_feats',
                  '_z_logits', 'train_dl', 'train_dl_probe'):
        globals().pop(_name, None)
    if 'tensorflow' in _sys.modules:
        try:
            _sys.modules['tensorflow'].keras.backend.clear_session()
        except Exception:
            pass
    # Classification runs through the shipped ONNX session from here on, so
    # park the torch graph on CPU and hand its VRAM back.
    try:
        model.to('cpu')
        dino_backbone.to('cpu')
    except Exception:
        pass
    _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return _pools


_served_ok = False
try:
    import time as _time
    _pools = _release_host_memory()
    try:
        import psutil as _ps
        _avail_gb = _ps.virtual_memory().available / 2**30
    except Exception:
        _avail_gb = None
    _det_onnx = Path('models/detector.onnx')
    if not _det_onnx.is_file():
        raise FileNotFoundError('models/detector.onnx missing — §3 did not export the detector')
    _det_sess_e2e = _ort_session(_det_onnx)
    _srv_tx = T.Compose([T.Resize(int(IMG * EVAL_RESIZE_RATIO)), T.CenterCrop(IMG),
                         T.ToTensor(), NORM])
    _tt = getattr(val_realworld_ds, 'target_transform', None)
    _samples = [(p, int(_tt(l) if _tt is not None else l)) for p, l in val_realworld_ds.samples]
    _N = len(_samples)
    _yt = np.array([y for _, y in _samples], dtype=np.int64)
    _best_conf = np.zeros(_N, dtype=np.float32)
    _n_mem = np.zeros(_N, dtype=np.int16)
    _yp_crop = np.zeros(_N, dtype=np.int64)      # top-confidence crop alone
    _yp_fold = np.zeros(_N, dtype=np.int64)      # that crop + the whole frame
    _yp_served = np.zeros(_N, dtype=np.int64)    # the served mean vote (all members)
    _p_nap = np.full(_N, np.nan, dtype=np.float32)
    _ml_max = np.zeros(_N, dtype=np.float32)
    _d_min = np.full(_N, np.nan, dtype=np.float32)
    _bn_eval = (_l2n(ood_bank_ship) if (SERVED_GATE is not None and ood_bank_ship is not None
                                        and SERVED_GATE.get('embedding_threshold') is not None)
                else None)
    print(f'served pass on val_realworld: {_N} photos, upload side {SERVED_UPLOAD_MAX_SIDE}px, '
          f'{_pools} idle DataLoader worker pool(s) shut down'
          + (f', host RAM available {_avail_gb:.1f} GB' if _avail_gb is not None else ''))

    def _sm(z):
        e = np.exp(z - z.max())
        return e / e.sum()

    _t0, _i = _time.perf_counter(), -1
    for _i, _r in enumerate(_served_pass(_samples, _det_sess_e2e, eval_sess, _srv_tx, T_val)):
        _lg = _r['logits']
        _best_conf[_i] = _r['det_conf']
        _n_mem[_i] = _r['n']
        # Member 0 is the top-confidence crop (the whole frame when Stage 1 found
        # nothing); the last member is the folded whole frame.
        _yp_crop[_i] = int(_lg[0].argmax())
        _yp_fold[_i] = (int((_sm(_lg[0]) + _sm(_lg[-1])).argmax()) if _r['has_det']
                        else _yp_crop[_i])
        _yp_served[_i] = int(_r['vote'].argmax())
        if NAP_CLASS_IDX is not None:
            _p_nap[_i] = _r['vote'][NAP_CLASS_IDX]
        _ml_max[_i] = _lg[:, _PLANT_LOGIT_COLS].max()
        if _bn_eval is not None and _r['emb'] is not None:
            _d_min[_i] = _knn_scores(_bn_eval, _r['emb'], OOD_KNN_K).min()
        if (_i + 1) % 500 == 0:
            print(f'  {_i + 1}/{_N} photos  ({(_i + 1) / (_time.perf_counter() - _t0):.1f}/s)')
    _served_secs = _time.perf_counter() - _t0
    if _i + 1 != _N:
        raise RuntimeError(f'served pass scored {_i + 1} of {_N} photos — truncated')
    del _det_sess_e2e
    _served_ok = True
    print(f'served pass done: {_N} photos in {_served_secs:.0f}s '
          f'({_N / max(_served_secs, 1e-9):.1f}/s), {float(_n_mem.mean()):.2f} members/photo')
except Exception as _e:  # noqa: BLE001 — diagnostic only, model already shipped
    print(f'served pass skipped: {type(_e).__name__}: {_e}')

# ── Readout 1: Stage 1 audit (every mode) ────────────────────────────────
if _served_ok:
    try:
        _nobox = _best_conf < STAGE1_AUDIT_CONF
        _below_floor = _best_conf < SERVED_GATE_DET_CONF_FLOOR
        _nap_i = CLASS_NAMES.index('not_a_plant') if 'not_a_plant' in CLASS_NAMES else -1
        _healthy_i = CLASS_NAMES.index('healthy') if 'healthy' in CLASS_NAMES else -1
        _damaged = np.isin(_yt, [i for i, c in enumerate(CLASS_NAMES)
                                 if c not in ('healthy', 'not_a_plant')])
        _h_mask = _yt == _healthy_i
        _r_dam = float(_nobox[_damaged].mean()) if _damaged.any() else 0.0
        _r_hea = float(_nobox[_h_mask].mean()) if _h_mask.any() else 0.0
        # not_a_plant is reported but excluded from the ratio: a no-box on a photo
        # of a keyboard is the detector being RIGHT.
        _r_nap = float(_nobox[_yt == _nap_i].mean()) if _nap_i >= 0 and (_yt == _nap_i).any() else None
        _ratio = (_r_dam / _r_hea) if _r_hea > 0 else float('inf') if _r_dam > 0 else 1.0
        print(f'[stage 1 audit] no-box rate  damaged={_r_dam:.3f} (n={int(_damaged.sum())})  '
              f'healthy={_r_hea:.3f} (n={int(_h_mask.sum())})  '
              f'ratio={_ratio:.2f}' + (f'  not_a_plant={_r_nap:.3f}' if _r_nap is not None else ''))
        _stage1_per_class = {}
        for _ci, _c in enumerate(CLASS_NAMES):
            _sup = int((_yt == _ci).sum())
            if not _sup:
                continue
            _rate = float(_nobox[_yt == _ci].mean())
            # below_gate_floor: the share that reaches the NAP overlay's first
            # condition (det_conf < SERVED_GATE_DET_CONF_FLOOR) — the photos the
            # classifier-side signal alone then decides on.
            _stage1_per_class[_c] = {'no_box_rate': _rate, 'n': _sup,
                                     'below_gate_floor_rate': float(_below_floor[_yt == _ci].mean())}
            print(f'    no_box[{_c:25s}] = {_rate:.3f}  below_gate_floor = '
                  f"{_stage1_per_class[_c]['below_gate_floor_rate']:.3f}  (n={_sup})")
        # WORST-CLASS ratio, not just the pooled one. The 2026-08-17 run showed why:
        # pooled damaged/healthy was 1.42 — under the threshold, apparently fine —
        # while gray_mold sat at 0.435 and pests at 0.449 against healthy's 0.133.
        # Four well-detected classes (leaf_spot 0.026, nutrient 0.046, powdery 0.043,
        # viral 0.052) diluted the average until the two worst classes vanished
        # inside it. A pooled mean is the wrong summary for a per-class failure.
        _per_ratio = {c: v['no_box_rate'] / _r_hea for c, v in _stage1_per_class.items()
                      if c not in ('healthy', 'not_a_plant') and _r_hea > 0}
        _worst = max(_per_ratio.items(), key=lambda kv: kv[1], default=(None, 0.0))
        if _worst[0]:
            print(f'    worst class: {_worst[0]} at {_worst[1]:.2f}x the healthy no-box rate')
        if _ratio > STAGE1_MISS_RATIO_WARN or _worst[1] > STAGE1_MISS_RATIO_WARN:
            _lead = (f'pooled {_ratio:.2f}x' if _ratio > STAGE1_MISS_RATIO_WARN
                     else f'{_worst[0]} {_worst[1]:.2f}x')
            print(f'WARN [stage 1]: the detector misses damaged foliage more often than healthy '
                  f'({_lead} > {STAGE1_MISS_RATIO_WARN}x). Every miss becomes a synthetic '
                  f'det_conf=0 in pipeline.py and is a candidate for the NAP gate. This is a '
                  f'STAGE-1 problem — do not chase it with classifier weights. Stage 1 trains on '
                  f'PlantDoc LEAF boxes plus whatever supplement rows ship boxes, so classes '
                  f'whose imagery is not leaf-shaped (iNat botrytis on flowers/fruit, macro pest '
                  f'close-ups on stems) are the ones it cannot box — box those in the supplement.')
        _abs_bad = {c: v['no_box_rate'] for c, v in _stage1_per_class.items()
                    if c != 'not_a_plant' and v['no_box_rate'] > STAGE1_MISS_ABS_WARN}
        if _abs_bad:
            print(f'WARN [stage 1]: classes losing >{STAGE1_MISS_ABS_WARN:.0%} of their images '
                  f'before Stage 2 ever sees them: '
                  + ', '.join(f'{c}={r:.3f}' for c, r in sorted(_abs_bad.items())))
        _s1 = {'split': 'val_realworld', 'no_box_rate_damaged': _r_dam,
               'worst_class': _worst[0], 'worst_class_ratio': _worst[1],
               'no_box_rate_healthy': _r_hea, 'no_box_rate_not_a_plant': _r_nap,
               'damaged_healthy_miss_ratio': None if _ratio == float('inf') else _ratio,
               'per_class': _stage1_per_class, 'detector_conf': STAGE1_AUDIT_CONF,
               # 2.7.5: shipped detector.onnx through the served letterbox/NMS at the
               # app's upload size. <= 2.7.2 ran Ultralytics on detector.pt at full
               # resolution — read cross-release deltas of a few points as runtime.
               'detector_runtime': 'served_onnx',
               'upload_max_side': SERVED_UPLOAD_MAX_SIDE,
               'below_gate_floor_rate': float(_below_floor[_yt != _nap_i].mean())
                                        if (_yt != _nap_i).any() else None}
        log_run({'phase': 'stage1_audit', **{k: v for k, v in _s1.items() if k != 'per_class'}})
        final_metrics['stage1_detection'] = _s1
        if 'metrics_by_split' in model_card:
            model_card['metrics_by_split']['stage1_detection'] = _s1
    except Exception as _e:  # noqa: BLE001 — diagnostic only, model already shipped
        print(f'stage 1 audit skipped: {type(_e).__name__}: {_e}')

# ── Readout 2: end-to-end eval — detector -> crops -> classify -> vote ──
# The headline metrics above score the WHOLE val image; production classifies
# DETECTOR CROPS and mean-votes them with the whole frame (pipeline.py). Three
# numbers from the same forward passes:
#   served_vote — what a user gets: up to max_crops padded crops + the whole
#                 frame, temperature-scaled mean vote (the headline here)
#   crop_only   — the top-confidence crop alone (the pre-2.7.5 headline)
#   fold        — that crop + the whole frame (FOLD_WHOLE_FRAME_VOTE's effect)
# Abstentions are not applied: the served gate's cost is reported separately.
if _served_ok and RUN_DETECTOR_CROP_EVAL:
    try:
        _f1s = _disease_f1(_yt, _yp_served)
        _f1c = _disease_f1(_yt, _yp_crop)
        _f1f = _disease_f1(_yt, _yp_fold)
        _whole = final_metrics['realworld']['disease_macro_f1']
        print(f'[detector-crop val_realworld] served_vote={_f1s:.3f} crop_only={_f1c:.3f} '
              f'crop+whole_fold={_f1f:.3f} whole_image={_whole:.3f}  '
              f'(served vs whole {_f1s-_whole:+.3f}; fold recovers {_f1f-_f1c:+.3f} vs crop-only)')
        for _ci, _c in enumerate(CLASS_NAMES):
            _sup = int((_yt == _ci).sum())
            if _sup:
                _rec_s = float(((_yp_served == _ci) & (_yt == _ci)).sum() / _sup)
                _rec_c = float(((_yp_crop == _ci) & (_yt == _ci)).sum() / _sup)
                print(f'    recall[{_c:25s}] served={_rec_s:.3f}  crop_only={_rec_c:.3f}  (n={_sup})')
        log_run({'phase': 'detector_crop_eval', 'disease_macro_f1': _f1s,
                 'crop_only_disease_macro_f1': _f1c, 'fold_disease_macro_f1': _f1f,
                 'whole_image_f1': _whole, 'seconds': _served_secs})
        # Persist the END-TO-END numbers next to the classifier-only ones so both
        # files carry the two eval modes, clearly separated.
        _e2e = {'split': 'val_realworld', 'eval_mode': 'end_to_end',
                'definition': 'served_vote',
                'disease_macro_f1': float(_f1s),
                'served_vote_disease_macro_f1': float(_f1s),
                'crop_only_disease_macro_f1': float(_f1c),
                'fold_disease_macro_f1': float(_f1f),
                'classifier_only_disease_macro_f1': float(_whole),
                'members_mean': float(_n_mem.mean()),
                'upload_max_side': SERVED_UPLOAD_MAX_SIDE,
                'served_recall': {c: float(((_yp_served == i) & (_yt == i)).sum() / (_yt == i).sum())
                                  for i, c in enumerate(CLASS_NAMES) if (_yt == i).any()}}
        final_metrics['end_to_end'] = _e2e
        if 'metrics_by_split' in model_card:
            model_card['metrics_by_split']['end_to_end'] = _e2e
    except Exception as _e:  # noqa: BLE001 — diagnostic only, model already shipped
        print(f'detector-crop eval skipped: {type(_e).__name__}: {_e}')

# ── Readout 3: the served OOD gate, out of sample ────────────────────────
# SERVED_GATE was fitted on temperature_cal; val_realworld never saw that fit.
# Same rule as pipeline._decide_nap_route, on the statistics the pass above
# already computed — no extra forward pass.
if _served_ok and SERVED_GATE is not None and NAP_CLASS_IDX is not None:
    try:
        _g = _served_gate_fires(
            _best_conf, _d_min if _bn_eval is not None else None, _ml_max, _p_nap,
            emb_thr=SERVED_GATE.get('embedding_threshold'),
            ml_floor=SERVED_GATE.get('max_logit_floor'),
            head_mass_floor=SERVED_GATE.get('nap_head_mass_floor') or 0.0,
            det_floor=SERVED_GATE['detector_confidence_floor'])
        _nat = _yp_served == NAP_CLASS_IDX
        _is_nap = _yt == NAP_CLASS_IDX
        _oos = {'split': 'val_realworld', 'fit_split': SERVED_GATE.get('fit_split'),
                'plant_n': int((~_is_nap).sum()), 'nap_n': int(_is_nap.sum())}
        if _is_nap.any() and (~_is_nap).any():
            _oos.update({
                'gate_nap_recall': float(_g[_is_nap].mean()),
                'gate_plant_false_reject': float(_g[~_is_nap].mean()),
                'head_nap_recall': float(_nat[_is_nap].mean()),
                'head_plant_false_nap': float(_nat[~_is_nap].mean()),
                'combined_nap_recall': float((_g | _nat)[_is_nap].mean()),
                'combined_plant_false_reject': float((_g | _nat)[~_is_nap].mean()),
            })
            print(f"[served gate, out of sample] plants={_oos['plant_n']} nap={_oos['nap_n']}")
            print(f"  gate alone     NAP recall={_oos['gate_nap_recall']:.3f}  "
                  f"plant false-reject={_oos['gate_plant_false_reject']:.3f}")
            print(f"  head alone     NAP recall={_oos['head_nap_recall']:.3f}  "
                  f"plant false-NAP={_oos['head_plant_false_nap']:.3f}")
            print(f"  head OR gate   NAP recall={_oos['combined_nap_recall']:.3f}  "
                  f"plant false-reject={_oos['combined_plant_false_reject']:.3f}")
            _fit_fr = (SERVED_GATE.get('calibration') or {}).get('gate_plant_false_reject')
            if _fit_fr is not None and _oos['gate_plant_false_reject'] > 2 * max(_fit_fr, TARGET_PLANT_FPR):
                print(f"WARN [served gate]: plant false-reject {_oos['gate_plant_false_reject']:.3f} "
                      f'on val_realworld vs {_fit_fr:.3f} where it was fitted — the thresholds do '
                      f'not transfer; consider shipping served_gate=null for this bundle.')
        log_run({'phase': 'served_gate_oos', **_oos})
        final_metrics['served_gate_oos'] = _oos
        if 'metrics_by_split' in model_card:
            model_card['metrics_by_split']['served_gate_oos'] = _oos
    except Exception as _e:  # noqa: BLE001 — diagnostic only, model already shipped
        print(f'served gate out-of-sample check skipped: {type(_e).__name__}: {_e}')

if _served_ok:
    Path('models/training_metrics.json').write_text(json.dumps(final_metrics, indent=2))
    Path('models/model_card.json').write_text(json.dumps(model_card, indent=2))
    print('stage 1 + end-to-end metrics persisted to training_metrics.json + model_card.json')


In [ ]:
# §6b — Bundle the run's artefacts into ONE per-family deploy+analysis archive.
# Named for the TRAINING MODE (MODEL_FAMILY) so aqua / gamma / frontier runs
# never clobber each other and the bundle is self-describing. Contents:
#   * Deploy artefacts at the archive ROOT (classifier*.onnx, detector.onnx,
#     classes.json, temperature.json, conformal.json, ood_embedding.npz, ...)
#     so the server's flatten step drops them straight into its models/ dir.
#   * Analysis/research artefacts ride along: training_metrics.json, the
#     model_card.json (per-class CIs, int8 paired-bootstrap, NEW latency/size
#     benchmark) and run_log.jsonl (per-epoch/phase post-mortem) -- everything
#     needed to analyse results offline without the training box.
#   * The on-device aqua/ subdir (aqua_classifier.tflite + its classes.json +
#     aqua_student.onnx) is included UNDER aqua/ when an aqua run produced it, so
#     the on-device deliverable ships in the SAME bundle as its gamma teacher
#     (the old top-level-files-only tar silently dropped it).
# Heavy torch checkpoints (classifier.pth ~86 MB, detector.pt) are NOT bundled:
# the runtime is ONNX/TFLite-only and they'd only bloat it. ``.npz`` is REQUIRED
# -- it carries the embedding-OOD bank classes.json -> ood.embedding references;
# omitting it silently disables the embedding OOD gate at serve time (the bug
# that shipped in v2.3.1).
_bundle_inventory = (SOURCE_INVENTORY if 'SOURCE_INVENTORY' in globals() else
                     json.loads((ROOT / 'source_inventory.json').read_text()))
_lic_policy = _bundle_inventory.get('license_policy') or {}
print(f"licence policy: {_lic_policy.get('policy', 'unrecorded')}; non-commercial sources "
      f"in the mix: {_lic_policy.get('noncommercial_sources_used') or {}} (recorded in "
      'model_card.json -> source_inventory.license_policy; never a gate)')
import tarfile as _tarfile
import hashlib as _hashlib
BUNDLE_PATH = Path('models') / f'plantvision_{RUN_ID}.tar'
_manifest_path = Path('models/bundle_manifest.json')
if not _manifest_path.is_file():
    raise RuntimeError('bundle_manifest.json missing; refusing heuristic bundling')
_ship_manifest = json.loads(_manifest_path.read_text(encoding='utf-8'))
if _ship_manifest.get('schema_version') != 1:
    raise RuntimeError('unsupported bundle manifest schema')
_ship_artifacts = _ship_manifest.get('artifacts')
if not isinstance(_ship_artifacts, dict) or not _ship_artifacts:
    raise RuntimeError('bundle manifest has no artifact allow-list')
_selection = _ship_manifest.get('classifier_selection') or {}
if _selection.get('fallback_allowed') is not False:
    raise RuntimeError('classifier fallback must be explicitly false')
_selected = _selection.get('path')
if ((_ship_artifacts.get('classifier') or {}).get('path') != _selected):
    raise RuntimeError('manifest classifier role disagrees with exact selection')
if FAMILY_CONFIG.get('distill') and not AQUA_EXPORT_SUCCEEDED:
    raise RuntimeError('combined bundle requires current-run Aqua success')
# Calibration is complete; remove current-run alternate graphs so neither an
# operator nor a legacy filename resolver can fall back to a sibling.
for _alt in ('classifier.onnx', 'classifier_int8.onnx',
             'classifier_fp32.onnx', 'classifier_raw.onnx'):
    if _alt != _selected:
        (Path('models') / _alt).unlink(missing_ok=True)
_analysis_allow = {'training_metrics.json', 'model_card.json',
                   'run_log.jsonl', 'dinov2_probe.json',
                   # §2's re-stamped supplement manifest: publish it as the next
                   # revision's manifest.csv (no per-row first-seen by hand).
                   'supplement_manifest_next.csv'}
_root_allow = {'bundle_manifest.json', *_analysis_allow}
_root_allow.update(
    spec['path'] for spec in _ship_artifacts.values()
    if isinstance(spec, dict) and isinstance(spec.get('path'), str)
    and '/' not in spec['path'])
_recognized_suffixes = {'.onnx', '.json', '.jsonl', '.npz', '.tflite'}
_extra_root = sorted(
    p.name for p in Path('models').iterdir()
    if p.is_file() and p.suffix.lower() in _recognized_suffixes
    and p.name not in _root_allow)
if _extra_root:
    raise RuntimeError(
        f'unowned recognized artifacts in deploy staging root: {_extra_root}')
_added = []

def _sha256(_p):
    _h = _hashlib.sha256()
    with open(_p, 'rb') as _f:
        for _chunk in iter(lambda: _f.read(1 << 20), b''):
            _h.update(_chunk)
    return _h.hexdigest()

def _verified_manifest_artifact(_role, _spec):
    if not isinstance(_spec, dict):
        raise RuntimeError(f'artifact {_role!r} spec is not an object')
    _raw = _spec.get('path')
    _rel = PurePosixPath(_raw) if isinstance(_raw, str) else None
    if (_rel is None or _rel.is_absolute() or '..' in _rel.parts or
            not _rel.parts or '\\' in _raw):
        raise RuntimeError(f'artifact {_role!r} has unsafe path {_raw!r}')
    _p = Path('models').joinpath(*_rel.parts)
    if not _p.is_file():
        raise RuntimeError(f'manifest artifact missing: {_role} -> {_raw}')
    _actual_sha = _sha256(_p); _actual_size = _p.stat().st_size
    if _actual_sha != _spec.get('sha256') or _actual_size != _spec.get('size_bytes'):
        raise RuntimeError(f'manifest artifact changed after sealing: {_role} -> {_raw}')
    return _p, _rel.as_posix()

_ANALYSIS_FILES = tuple(sorted(_analysis_allow))
with _tarfile.open(BUNDLE_PATH, 'w') as _tar:
    _tar.add(_manifest_path, arcname='bundle_manifest.json')
    _added.append('bundle_manifest.json')
    for _role, _spec in sorted(_ship_artifacts.items()):
        _p, _arc = _verified_manifest_artifact(_role, _spec)
        _tar.add(_p, arcname=_arc)
        _added.append(_arc)
    for _name in _ANALYSIS_FILES:
        _p = Path('models') / _name
        if _p.is_file():
            _arc = f'analysis/{_name}'
            _tar.add(_p, arcname=_arc)
            _added.append(_arc)

_root_members = [m for m in _added if '/' not in m]
_root_classifiers = sorted(m for m in _root_members
                           if m.startswith('classifier') and m.endswith('.onnx'))
if _root_classifiers != [_selected]:
    raise RuntimeError(f'deploy root must contain exactly {_selected!r}; got {_root_classifiers}')
_cj = json.loads((Path('models') / 'classes.json').read_text())
_bank = (((_cj.get('ood') or {}).get('embedding') or {}) or {}).get('bank_file')
if _bank and _bank not in _added:
    raise RuntimeError(f'classes.json references OOD bank {_bank!r} not bound by manifest')
print(f'wrote {BUNDLE_PATH}  ({BUNDLE_PATH.stat().st_size / 1e6:.1f} MB)')
print(f'  root (manifest-only server artifacts): {_root_members}')
_aqua_members = [m for m in _added if m.startswith('aqua/')]
if _aqua_members:
    print(f'  on-device (aqua/): {_aqua_members}')
print(f'  analysis/: {[m for m in _added if m.startswith("analysis/")]}')
print('\nShip checklist (manual deploy — the server does NOT auto-download):')
print(f'  1. Extract the COMPLETE {BUNDLE_PATH.name} tree into a fresh server')
print( '     models/ directory. Keep aqua/ nested: strict boot verifies every')
print( '     artifact bound by bundle_manifest.json before serving.')
if _aqua_members:
    print( '  2. Additionally publish aqua/aqua_classifier.tflite + aqua/classes.json to')
    print( '     the server static/aqua/ dir (served via GET /api/aqua/model).')
    print( '  3. Restart the server — boot is fail-fast if an artefact is missing.')
else:
    print( '  2. Restart the server — boot is fail-fast if an artefact is missing.')


## §7 — Evaluation contract

| Slice | Role | Pass criterion |
|---|---|---|
| `val_indomain` | training-loop metric, drives early stopping | tracked; best head checkpoint restored |
| `val_realworld` | **primary — ship/no-ship gate** | disease macro-F1 ≥ **0.70** (must not regress below the live model); no disease class with recall = 0; healthy-bias < 1.5; healthy recall ≥ 0.70 |
| `val_studio` | PlantVillage transfer / background-bias monitor | tracked, never blocking |
| `temperature_cal` | temperature scaling | held out |
| `conformal_cal` | RAPS calibration | held out; 80/20 fit/audit when n ≥ 200, else fit-only |
| private v2 unseen-domain set | contaminated studio-domain regression sentinel (151 healthy + 31 scorch rows with geometric pseudoreplicates) | **diagnostic only**; `REQUIRE_OOD_BENCHMARK=False`; never a release gate and never interpreted as 31 independent scorch captures |

The deploy archive is manifest-owned. `bundle_manifest.json` selects **one exact classifier graph**, sets `fallback_allowed=false`, binds every deploy byte by SHA-256/size, and binds the data/run lineage plus the exact serving contract. The archive root contains only its allow-listed server artefacts; stale sibling graphs and heuristic filename fallback are forbidden. In combined mode, the manifest also binds the immutable `aqua/` subtree. Analysis files live under `analysis/` in the archive.

Required manifest-bound deploy roles are:

* `classifier_int8.onnx` **or** `classifier.onnx` — exactly the graph that passed drift/calibration; never both and never a runtime fallback.
* `detector.onnx`, `classes.json`, `temperature.json`, `conformal.json`. Temperature and conformal fit-source fields must match the selected classifier.
* `data_manifest.json` and `run_manifest.json` — corpus digest/source revisions and current-run recipe/environment/base-weight identity.
* `ood_embedding.npz` when `classes.json` configures the embedding bank; a referenced bank must be present and bound.
* Combined mode additionally binds `aqua/aqua_classifier.tflite`, `aqua/classes.json`, and `aqua/aqua_student.onnx`; the TFLite graph is strict FLOAT32.

`classifier.pth`, `detector.pt`, `classifier_fp32.onnx`, raw/intermediate graphs, and prior-run siblings are **not deploy artefacts**. Named diagnostics (`dinov2_probe.json`, `training_metrics.json`, `model_card.json`, `run_log.jsonl`) are archived under `analysis/` and are not runtime filename candidates.

OOD status: 2.7.5 calibrates `classes.json -> ood.served_gate` in §6 through the served path — Stage 1 ONNX → the same crop rules as `serving_contract.crop` → whole-frame fold → min member k-NN distance / max member plant logit — on `temperature_cal` at `TARGET_PLANT_FPR`, and records the resulting three-of-three gate's NAP recall and plant false-reject beside the head-only numbers. `served_gate_enabled=true` exactly when that calibration produced a threshold; a run where it fails ships `served_gate=null` (`served_gate_enabled=false`) and the overlay stays off. Single-image thresholds are never served. The private v2 sentinel cannot enable or tune that gate.

For an out-of-notebook A/B, `tools/paired_eval.py` requires verified bound manifests by default, executes served semantics (including blur/NAP-OOD abstentions), reports coverage/abstention, and carries capture-family IDs into `tools/eval_bootstrap.py`. Bootstrap confidence intervals resample independent capture families with paired indices; row-level results without groups are advisory, not release evidence.
